# Tiled matrix transpose

A transpose does no arithmetic, yet a naive one is slow: if each program reads a row, it writes a column, and consecutive threads store to addresses `M` elements apart. Working on 2-D tiles lets every load *and* every store touch contiguous memory.

Write `transpose(x)` returning a **contiguous** copy of `x.T`.

## Inputs
- `x`: CUDA tensor `(M, N)`, $M, N \ge 1$, of any dtype (tests use `float32` and `int32`). Use `x.stride(0)` and `x.stride(1)`; do not assume contiguity.

## Output
- A new contiguous tensor of shape `(N, M)` and the same dtype, with `out[j, i] = x[i, j]`. Values are copied exactly.

## Requirements
- A 2-D grid of `(cdiv(M, BM), cdiv(N, BN))` programs. Program `(pid_m, pid_n)` loads the tile `x[pid_m*BM : +BM, pid_n*BN : +BN]` with a 2-D mask, transposes it in registers with `tl.trans`, and stores it as the tile `out[pid_n*BN : +BN, pid_m*BM : +BM]`.
- `BM`, `BN` are `tl.constexpr` powers of two (e.g. 32).

This problem runs in **Colab with a GPU runtime**. Expected outputs come from an equivalent PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def transpose(x):
    """x: (M, N) CUDA tensor, any dtype and strides. Return x.T as a new contiguous (N, M) tensor."""
    raise NotImplementedError


@triton.jit
def _transpose_kernel(x_ptr, out_ptr, M, N, stride_xm, stride_xn, stride_on, stride_om,
                      BM: tl.constexpr, BN: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], "dtype": "float32"}], "e": {"__nd__": [[1.0, 4.0], [2.0, 5.0], [3.0, 6.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]], "dtype": "int32"}], "e": {"__nd__": [[0, 4, 8], [1, 5, 9], [2, 6, 10], [3, 7, 11]], "dtype": "int32"}}, {"a": [{"__nd__": [[9.0]], "dtype": "float32"}], "e": {"__nd__": [[9.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.0, 1.0, 2.0, 3.0, 4.0]], "dtype": "float32"}], "e": {"__nd__": [[0.0], [1.0], [2.0], [3.0], [4.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.8999031186103821, -0.4583752751350403, -1.3140853643417358, 0.6586976647377014, 0.9554258584976196, 0.5246310234069824, -2.558269500732422, -1.5353412628173828, 0.6136307120323181, 1.494365930557251, -1.4945822954177856, 0.9703254699707031, 1.1190851926803589, 0.017410578206181526, -1.339133381843567, -0.6403135657310486, -0.36200037598609924, 0.6909022927284241, 0.8993356227874756, -0.986848771572113, -0.10213501006364822, -0.03492661938071251, -0.1748647689819336, 0.9459331631660461, -0.2716423571109772, -0.0700131207704544, -1.4837714433670044, 0.14615555107593536, -0.9387549757957458, 1.8856043815612793, 0.14862462878227234, 0.1838734894990921, 0.0986730232834816, -1.1710097789764404, -0.4353877604007721, 0.11089904606342316, 0.9117034077644348, 0.21966463327407837, -1.3220744132995605, -0.21426066756248474, -0.13816185295581818, -1.6847881078720093, 0.5395525693893433, 0.7637683749198914, -0.9124293923377991, 2.1853322982788086, 0.324837327003479, -0.5884791612625122, 1.9458446502685547, -0.5886527895927429, 0.14952750504016876, -1.812504529953003, -1.097822904586792, 1.750251054763794, -1.0016318559646606, -0.8418654203414917, -0.26036927103996277, -0.8247355222702026, -1.2520310878753662, -1.0220783948898315, -0.17003101110458374, 1.5131309032440186, 0.7696735262870789, 2.263970136642456], [-0.8698101043701172, 0.17317888140678406, 0.25686517357826233, 0.5076766014099121, 0.7218186855316162, -1.7291629314422607, 1.4759763479232788, -0.013240644708275795, -0.4972066581249237, 0.25351643562316895, -0.5871797204017639, -1.2591904401779175, -0.37412479519844055, -1.9415563344955444, -1.127770185470581, 0.7236483693122864, 0.4756447672843933, -1.151097297668457, -2.2454750537872314, 0.17502057552337646, 0.36982715129852295, -0.05980798602104187, -1.0016217231750488, 0.05884239077568054, -0.36620163917541504, 0.1824212670326233, 0.5660898685455322, -0.04666045308113098, -0.2123861461877823, 0.5310394167900085, 0.26019051671028137, -0.8776059150695801, -0.10265259444713593, 0.6619710326194763, 0.566676676273346, -0.32494738698005676, -0.16475775837898254, -1.0546441078186035, 0.22098210453987122, 1.0684579610824585, 0.4071141481399536, -0.46759316325187683, 0.6242401003837585, -0.004906778689473867, 0.6311420798301697, -1.2471394538879395, -0.4167402982711792, 0.7652712464332581, 1.1058533191680908, -0.7801048755645752, -0.5019945502281189, 1.8491978645324707, 0.3650195896625519, -0.153121218085289, 0.48895737528800964, -0.5500261783599854, -1.6647868156433105, 0.026143142953515053, -0.34950289130210876, -0.6110222935676575, 0.007791154086589813, -0.8798443078994751, -0.4935612678527832, -1.8657894134521484], [1.1923428773880005, 1.4240317344665527, -1.1406865119934082, -1.5961689949035645, 0.3878454864025116, 0.7894242405891418, -0.4591422379016876, -1.1156907081604004, 1.8323572874069214, -0.6859242916107178, 0.36959096789360046, 0.2881930470466614, 0.3097412884235382, 1.2666064500808716, 1.3789180517196655, -1.4739190340042114, -0.12562070786952972, 1.0913655757904053, 0.17375719547271729, -0.6511379480361938, 0.7755193114280701, -0.5232162475585938, -0.18898096680641174, 0.8250610828399658, 0.5654955506324768, 1.880777359008789, 0.6875735521316528, 0.003970490302890539, 1.0332063436508179, -0.04454539716243744, 0.06852097064256668, 0.13934198021888733, -0.41248008608818054, 0.26471850275993347, 0.1666335165500641, 0.17127445340156555, -0.9556545615196228, 0.5732484459877014, 0.10141777247190475, -0.15393570065498352, -0.1486535668373108, -0.6692723631858826, -0.6993617415428162, 1.8623509407043457, 0.17674580216407776, 0.7380908131599426, 0.6799888610839844, -0.7537494897842407, -0.14892086386680603, -0.5416708588600159, 0.517824649810791, 0.35509219765663147, -0.9670684933662415, 0.9322242140769958, 0.9030817151069641, -1.3903950452804565, 0.3607134521007538, 0.2591469883918762, -1.7361066341400146, -1.160036325454712, 0.5330931544303894, -1.2110098600387573, -1.9539152383804321, 0.03853512182831764], [-1.9655067920684814, -0.20679813623428345, 0.09442973881959915, -0.10538601130247116, 0.5674481987953186, -0.0816906988620758, -0.2834911048412323, -1.1433733701705933, 1.1357825994491577, -1.1710913181304932, 0.3445030450820923, -0.09653632342815399, -0.4260581433773041, -1.315321683883667, -1.6966569423675537, -0.09251436591148376, 0.6657434105873108, 0.05454370751976967, -1.1645617485046387, 0.9531631469726562, -2.605194568634033, -0.13056007027626038, 0.9154669642448425, 1.0347535610198975, -0.14629866182804108, 1.2095682621002197, 1.728361964225769, -0.27340182662010193, 1.104597806930542, 0.06376875191926956, 1.0828369855880737, -1.1844961643218994, 0.7466009259223938, 1.2563724517822266, 1.1375545263290405, 0.09971880912780762, -1.9823657274246216, 0.2861904799938202, -2.318258762359619, 0.14128269255161285, 0.20135077834129333, 0.6472941637039185, -0.5949511528015137, 0.6877020597457886, -0.3357521891593933, -1.1458567380905151, 0.17348985373973846, 2.262974739074707, 0.35219985246658325, 0.47316688299179077, 1.6408905982971191, 1.1046124696731567, 1.102184772491455, -1.4408499002456665, -0.7911708354949951, 2.235787868499756, 0.2224586308002472, -0.8280426263809204, 1.261252999305725, 0.3924678862094879, -1.9293192625045776, -0.6627941131591797, -1.1360650062561035, -1.2008289098739624], [-0.7337416410446167, -1.3539620637893677, -0.45091667771339417, 0.23689255118370056, -0.20710228383541107, -0.2102249711751938, 0.9449697732925415, -0.2974686622619629, 0.1296825259923935, -0.8770400285720825, -1.1862388849258423, 0.51100093126297, -0.972798228263855, 0.13335035741329193, 0.8983229994773865, -0.4918181598186493, 2.117896318435669, 1.154932975769043, -0.774941623210907, 2.13230562210083, 0.5126007199287415, 1.0842838287353516, -0.4711441099643707, 0.3590772747993469, -0.23570898175239563, 0.3278779685497284, -0.6708052158355713, -0.4871984124183655, -0.7413039207458496, -2.210026502609253, -0.8632765412330627, 0.270134299993515, -1.2500736713409424, -1.0433697700500488, 1.3556078672409058, -1.7961822748184204, -0.24617260694503784, 0.4725686311721802, 0.6061264872550964, 0.3723907768726349, 0.2877083122730255, -0.07896576821804047, -1.1026699542999268, -2.212771415710449, 1.6808733940124512, -0.41940876841545105, 0.10577099025249481, -1.0259913206100464, 0.02224990352988243, -1.428452730178833, -0.8158507943153381, 0.19532310962677002, 1.770680546760559, 0.49967917799949646, 2.0179903507232666, 0.27029886841773987, 0.20965175330638885, 1.4535316228866577, 0.012672136537730694, -1.4806346893310547, 0.5149244070053101, 0.3988657593727112, 0.18802908062934875, 0.45602163672447205], [-0.340660959482193, -0.08864977955818176, -1.288060188293457, 0.7480973601341248, -0.684158980846405, 1.2231075763702393, 0.10027065128087997, -0.8516147136688232, 0.6974550485610962, 0.3599795401096344, 0.43590420484542847, 0.05157991871237755, -0.5600208044052124, 1.4960922002792358, -1.552392840385437, -1.041038990020752, 1.2759385108947754, 1.7062774896621704, -1.9293212890625, 1.7610541582107544, 0.6014499068260193, -2.00620174407959, -0.6632086634635925, -1.1056885719299316, 0.07043773680925369, -0.2917822599411011, 1.2332061529159546, 0.18745777010917664, 0.8555373549461365, 0.997447669506073, -1.104976773262024, -0.14799921214580536, 0.47859784960746765, 0.2995857894420624, -0.657784104347229, 0.4879797101020813, -1.1764971017837524, -0.6216272711753845, 0.5223340392112732, 0.6654309630393982, -1.0732537508010864, 0.2159944325685501, 0.5570297241210938, 0.9360799789428711, 0.6457073092460632, 0.16403627395629883, -1.0509146451950073, -1.6835135221481323, -0.8572655320167542, -0.6963451504707336, -0.4555371403694153, 0.25345197319984436, 0.5250223278999329, 1.9980356693267822, -1.6953502893447876, -0.9134610295295715, -1.7069451808929443, -1.1620583534240723, -1.446288824081421, -0.9352262616157532, -0.9528447389602661, -1.2973151206970215, -0.749412477016449, 0.6836749911308289], [0.6057178974151611, -0.15176734328269958, 0.7778471112251282, -2.188178300857544, -1.0612519979476929, 0.7438154816627502, -0.44453996419906616, -0.6817123889923096, -0.45675763487815857, -0.4625297784805298, 0.704009473323822, 1.447244644165039, -1.3964546918869019, -1.1432619094848633, -0.6047202944755554, 0.23877355456352234, 0.6882261633872986, -0.08775756508111954, 0.3833174705505371, -0.30538854002952576, 0.37029939889907837, -2.07387638092041, 0.7120949625968933, 0.045047517865896225, -1.1478885412216187, -0.06779567152261734, -1.2625044584274292, 1.1089059114456177, 1.073300838470459, 0.0848100483417511, -1.7510160207748413, -2.063983678817749, 0.0727948397397995, -1.1467972993850708, 2.0426647663116455, -1.507251262664795, -1.5780147314071655, 0.9184601306915283, 2.0231704711914062, -1.669063687324524, 0.803791880607605, 0.38384389877319336, -0.4328260123729706, -1.3933830261230469, -0.7584786415100098, -0.4709814190864563, 0.5679681301116943, 0.5609595775604248, -0.37571558356285095, -2.0546727180480957, -0.6836041212081909, -0.5735498070716858, -1.2105238437652588, 0.29197072982788086, 1.272192120552063, 0.665062665939331, -0.41499340534210205, -1.666552186012268, 0.4583747088909149, 1.1336168050765991, 0.20790615677833557, 0.163664311170578, 0.3044619560241699, -0.13469940423965454], [-1.089009165763855, 1.8226481676101685, 1.1215150356292725, 1.3628349304199219, 0.009091629646718502, 0.7252418994903564, -2.0238122940063477, -0.1765352487564087, -0.1620151400566101, -0.8795292973518372, -0.922248363494873, -1.7394911050796509, 0.3160697817802429, 0.14133892953395844, 0.45134201645851135, -0.6857211589813232, 0.8102511763572693, 2.148977518081665, -0.5179665088653564, 0.6146741509437561, -2.6837375164031982, 1.754376769065857, 1.4886592626571655, 0.06065455451607704, 0.7225856184959412, -1.5005122423171997, 1.131286859512329, -1.0014513731002808, -0.3416120111942291, -0.5470161437988281, 0.9327576160430908, 0.22310537099838257, 0.24521410465240479, 1.0738639831542969, -1.2151941061019897, 0.4829792380332947, 0.9331173896789551, 2.1121885776519775, 1.2370816469192505, -1.0399430990219116, 0.9409905672073364, -1.5432658195495605, -0.42526742815971375, -0.36796969175338745, 0.6471886038780212, -0.48888903856277466, -0.1507345587015152, 0.34696587920188904, 0.015107192099094391, 0.31262239813804626, -0.5860698819160461, 0.03495031222701073, 1.0742604732513428, 0.7831284403800964, 0.41479969024658203, 0.3535030484199524, -1.6817735433578491, 1.1966789960861206, 1.6724965572357178, 1.3022451400756836, -1.2942326068878174, -0.6232863664627075, -0.6558471322059631, 0.23779083788394928], [-0.5155795812606812, 0.18819868564605713, 0.6934308409690857, 0.06432077288627625, 1.6394416093826294, 1.1536107063293457, 0.26563167572021484, -0.6244698762893677, 0.08083276450634003, -0.5325934290885925, -1.2438243627548218, 0.15908955037593842, 1.307270884513855, -0.40861353278160095, -0.9304059147834778, 0.08489415794610977, -0.6191263198852539, 0.9043327569961548, -0.24656984210014343, 0.07291946560144424, -0.187511146068573, 0.09801522642374039, -0.8479596972465515, 0.14680016040802002, -0.6895408630371094, -0.7100067138671875, 0.29836875200271606, -1.6988108158111572, -0.7515841722488403, -0.2632159888744354, -0.23131847381591797, 0.47007840871810913, 1.118277668952942, 0.05540956184267998, 0.024942995980381966, -0.12039241939783096, 0.04569206386804581, -0.08779995143413544, 2.937380313873291, -0.7223324775695801, 0.9342005848884583, 0.9290662407875061, 0.6426178812980652, -0.10218202322721481, 0.5072893500328064, 1.085374116897583, -0.8222247958183289, 1.181255578994751, 0.27646568417549133, 1.5324430465698242, 0.19275467097759247, 0.225307896733284, 0.2398947924375534, 1.2878522872924805, -0.29720813035964966, 1.4777028560638428, 0.2831805348396301, -1.604684829711914, -0.2553133964538574, 1.4359402656555176, -0.03268507122993469, -1.3366377353668213, 0.47181636095046997, -0.1753959208726883], [0.6676793098449707, -0.4114547371864319, 1.2054307460784912, 0.5911117196083069, 0.12867727875709534, -1.2888044118881226, -0.3759533762931824, 1.6365784406661987, 2.461146593093872, 0.8043172955513, -0.04201611131429672, 0.8933999538421631, -0.18157339096069336, -1.2520462274551392, -0.2233319878578186, -1.5462781190872192, -0.11772065609693527, 0.21148231625556946, 0.8776394724845886, 2.142606735229492, -0.23554906249046326, 1.0518031120300293, -0.08785352855920792, 0.8981180191040039, 0.17291632294654846, 0.061453528702259064, -0.12160354107618332, -0.8075142502784729, -0.8923971652984619, 2.1207849979400635, -0.4519121050834656, -0.1141229122877121, -0.699027955532074, -1.239908218383789, 0.6202710866928101, -0.8065798282623291, -1.3927572965621948, -1.1554834842681885, -0.1550513058900833, 0.8607157468795776, -0.5249208211898804, 2.4977715015411377, -1.2541661262512207, 1.0243734121322632, -1.7396317720413208, -0.16639791429042816, 0.15475785732269287, -1.0757285356521606, 0.035045016556978226, 0.11347280442714691, -0.22673399746418, 1.1841027736663818, -0.18521659076213837, 1.355492115020752, -0.9944339990615845, 0.23443466424942017, -0.09827530384063721, -0.6036983132362366, 0.30331557989120483, -0.6957701444625854, 0.5019524097442627, -1.223759651184082, 0.004288239870220423, -0.4606814682483673], [1.5303953886032104, -0.01700490713119507, -1.1904680728912354, -2.1827242374420166, -0.5535383224487305, -0.3185899257659912, 1.7100979089736938, 2.3178536891937256, 0.013037207536399364, -0.3308594524860382, 1.3968191146850586, -1.1285423040390015, -0.4913652241230011, 1.455488920211792, -0.04435897618532181, 0.03992754966020584, 0.10061267018318176, -0.8511046767234802, -0.6384986042976379, -0.06595558673143387, -1.65879225730896, -0.3902221918106079, 0.703824520111084, -0.19587136805057526, -2.3835530281066895, 0.12895779311656952, 0.7925776243209839, -0.2523709535598755, 1.3760493993759155, 0.3249971270561218, 0.14536908268928528, -0.38748422265052795, -1.5404058694839478, -1.0179729461669922, -0.575279176235199, -1.827839732170105, 0.600877583026886, -1.0819159746170044, -1.6162034273147583, 0.0171206071972847, -0.780884325504303, -0.8399583697319031, -0.5029340982437134, 1.6197550296783447, -0.6615617871284485, 0.4144236445426941, 0.2150653451681137, -0.13213010132312775, -0.21591006219387054, -0.5756670236587524, 0.8215908408164978, 0.47491052746772766, -1.1652508974075317, 1.6954865455627441, 1.3738999366760254, 1.534897804260254, 1.7814605236053467, -0.7402556538581848, 0.13519427180290222, 0.4030706584453583, -0.7339342832565308, 0.2864648103713989, -0.5419684648513794, 1.4307308197021484], [-0.13700881600379944, -1.7071185111999512, -1.4293266534805298, -0.8566039800643921, -0.7225134968757629, 1.216440200805664, -0.8717086315155029, -1.494749903678894, 0.4862147867679596, 0.6418257355690002, 1.2611799240112305, -0.9460441470146179, -0.3007318675518036, -0.5822508931159973, 0.36769694089889526, 1.2012457847595215, -0.45043936371803284, -1.3434103727340698, 0.08650703728199005, -1.3707393407821655, -2.541722536087036, 0.28454267978668213, 0.9492698311805725, -1.7157375812530518, -0.006869330070912838, -1.038689136505127, -0.6974465250968933, 0.058501001447439194, -0.37954795360565186, -1.1146408319473267, 1.9081367254257202, -0.15841223299503326, 1.0266318321228027, -0.6831233501434326, 0.8704742789268494, 0.5657032132148743, -0.8921152353286743, -1.1389124393463135, 0.5938212275505066, -0.031848639249801636, -1.4403527975082397, -1.1494170427322388, -1.4370230436325073, 0.8036302924156189, 0.5587061643600464, -0.7483652234077454, 1.8650603294372559, -0.2549581527709961, 0.7426502704620361, 1.556613564491272, -0.8623237609863281, -0.6465769410133362, -1.0332574844360352, -0.14799988269805908, -1.8309932947158813, 1.3821425437927246, 0.7643731236457825, -0.0799730122089386, -0.4333830177783966, 0.6291009187698364, -1.0556741952896118, -0.8429441452026367, -0.1958383321762085, 0.5132637619972229], [-1.3493880033493042, -2.1075692176818848, -1.2414186000823975, 0.3184157907962799, 0.6318437457084656, 1.1161473989486694, 1.088407039642334, -2.3566079139709473, 1.9803273677825928, -1.195037603378296, 1.3977832794189453, 0.1043664887547493, -0.3818780481815338, 0.5468370914459229, -0.7387341260910034, -1.9393256902694702, -0.2706186771392822, -1.1248379945755005, 0.534800112247467, -0.03687746077775955, -0.23443321883678436, -0.8537653088569641, -1.1263427734375, 1.00685453414917, -0.1887790411710739, 0.7323111295700073, 0.6601270437240601, -0.7505186796188354, -0.27323904633522034, 1.2058132886886597, -0.8728693127632141, -0.08649203181266785, 0.8504368662834167, 1.2942299842834473, -0.7061967849731445, -1.2057045698165894, -0.03805288299918175, -0.8950231075286865, 0.7048787474632263, 0.6489371061325073, -1.7708303928375244, -0.24275313317775726, -0.8262097835540771, -0.033612336963415146, 1.1155413389205933, 0.43447861075401306, 0.8978976011276245, 0.0027777496725320816, -2.6910126209259033, -2.9569592475891113, -0.31769460439682007, 0.744182825088501, -0.3085881173610687, 0.17100951075553894, -1.5078818798065186, 0.8876871466636658, -2.4846179485321045, -1.3491706848144531, -1.546278476715088, 1.0564754009246826, -1.4004573822021484, 1.209554672241211, -0.1911104917526245, -1.5221372842788696], [-1.0479254722595215, 0.5509051084518433, -0.9176278710365295, 1.0885967016220093, -0.7233878970146179, 1.569963812828064, -0.751186728477478, 0.9699144959449768, 0.04679206758737564, -0.10635776817798615, 0.2956945598125458, 0.9242297410964966, 0.2530613839626312, -0.6040695309638977, -0.002265071962028742, 0.2286248803138733, -0.5424455404281616, 0.8305144906044006, -0.96439129114151, -1.3908461332321167, -0.06986145675182343, -1.0823297500610352, 0.9213810563087463, 1.228393316268921, -1.919921875, 0.04241737723350525, -0.3029714822769165, 0.6972513794898987, -0.19374139606952667, 0.8257158994674683, -0.05092581734061241, 0.7098743915557861, -0.9867958426475525, 1.4491692781448364, -1.5946506261825562, -1.4568476676940918, 0.3270825743675232, 0.964439868927002, 0.2709067463874817, -2.6652159690856934, -1.3865383863449097, 0.24557794630527496, 0.06523028761148453, 0.12412471324205399, -0.2938699424266815, -0.2669796049594879, 0.13816572725772858, 1.2594645023345947, 0.6779926419258118, -0.5075498819351196, 0.6025964021682739, 0.2775305509567261, -0.9836982488632202, 0.8962817788124084, 0.11050381511449814, -0.34924063086509705, 0.37133851647377014, -2.9414453506469727, 1.601850152015686, -0.31375640630722046, -0.44220975041389465, -0.9816311597824097, 0.30627667903900146, 0.7132523059844971], [1.1218878030776978, -0.6482441425323486, 1.9496804475784302, -0.4865599572658539, 0.010171583853662014, 2.970968008041382, 0.7807389497756958, -1.9831628799438477, 0.1812298446893692, 0.6112141609191895, -0.4037332832813263, 0.2940548360347748, 1.3554388284683228, -0.6453508734703064, -0.49437010288238525, 0.7139834761619568, -0.38205093145370483, -0.22394582629203796, 0.44720879197120667, 1.068036675453186, 1.0667312145233154, 2.111229181289673, 0.8928000926971436, 0.4409791827201843, -2.1375954151153564, -0.001538573415018618, -0.65567547082901, -1.6248083114624023, 0.02314361184835434, 0.45823681354522705, -0.6269470453262329, 0.7082387208938599, -1.1235982179641724, -0.7026557922363281, 1.1051366329193115, 0.5595276951789856, 0.4839227795600891, -0.34300899505615234, 0.7244430184364319, -1.2352527379989624, -0.05574558302760124, 0.08569443225860596, -0.26980888843536377, 1.271958589553833, 1.0944594144821167, 0.18767233192920685, 0.004210630431771278, 0.2315317541360855, 0.853624165058136, 0.34284618496894836, 0.24331901967525482, 0.4315861761569977, 1.1104165315628052, 0.5473667979240417, -0.4876408576965332, 0.5577390193939209, -0.9523491263389587, 1.203368902206421, 2.287093162536621, 0.15321217477321625, 0.9807662963867188, 1.265201210975647, -0.5547460913658142, 0.854436457157135], [1.3333178758621216, -0.26146236062049866, 0.8939447402954102, 0.5393257141113281, -0.9474541544914246, 0.8511086702346802, 1.8847929239273071, 0.24231652915477753, 0.48922330141067505, -0.8519030213356018, -0.9525532126426697, -0.4582779109477997, 0.3089849054813385, 0.4178001880645752, 0.43185341358184814, 0.9222221970558167, 0.5821750164031982, -0.13951098918914795, 0.8311170339584351, -0.7039746046066284, 1.8731147050857544, -0.7294390797615051, 0.04164654389023781, -0.9582239389419556, 1.4264137744903564, 0.5558668971061707, -0.7786097526550293, -0.794025719165802, 0.9583896994590759, -1.3961421251296997, 0.2524065375328064, -0.9657363891601562, 0.37706106901168823, 1.1542261838912964, 0.02713397517800331, 0.5405804514884949, 0.3851805031299591, -0.8401992917060852, -0.4251210391521454, -0.7389296293258667, -1.179717779159546, -2.665703773498535, 1.4356073141098022, -0.491498202085495, 0.46894386410713196, 0.4630524814128876, -0.6732829809188843, 0.44945430755615234, 0.6006954908370972, -0.36795246601104736, -0.6248364448547363, 0.41569629311561584, 0.13297678530216217, -0.5198304653167725, 0.898234486579895, -1.3164219856262207, 1.0312918424606323, 0.10628420859575272, -1.1473209857940674, 0.41652318835258484, -0.7588562369346619, 0.03826719522476196, -0.15688541531562805, -0.6990015506744385], [1.4016379117965698, -0.08112846314907074, -0.20107074081897736, 0.8807601928710938, 0.4938201904296875, -0.04028857499361038, 0.22751930356025696, -1.2422022819519043, 1.3364770412445068, -1.5608032941818237, 1.522441029548645, 0.5861697793006897, -0.4113505184650421, -0.715015709400177, -1.796675682067871, -1.5641100406646729, 1.4067883491516113, 0.7288343906402588, 0.7648867964744568, 0.5806168913841248, -1.5306833982467651, 0.6830989122390747, -0.8719853758811951, -1.8474117517471313, -0.663332462310791, -0.5502601861953735, 1.2292487621307373, -1.1766968965530396, -1.116884469985962, -0.22315442562103271, 0.9783003926277161, -2.2504019737243652, -0.026549089699983597, 0.9600598812103271, 1.7288378477096558, 0.4192456305027008, 0.7869210243225098, -0.2834038734436035, -0.2535204291343689, 0.006409502122551203, -0.9464609622955322, 0.11732052266597748, 0.23467959463596344, 0.9794695973396301, 1.7882970571517944, 0.33955255150794983, 0.4407752454280853, 0.38020560145378113, -1.8028666973114014, 0.478577196598053, 0.034354954957962036, -0.8846963047981262, 0.4098007082939148, -1.2382882833480835, 0.8156172633171082, -0.6123212575912476, -1.0886752605438232, -0.35756537318229675, -0.10970030725002289, -1.2354501485824585, -2.0281107425689697, 1.2981054782867432, -1.284261703491211, -0.659424364566803], [-0.35577183961868286, -1.3108446598052979, -0.4365514814853668, 0.3914957344532013, 2.1902589797973633, 1.455220103263855, 0.9997928738594055, -1.157912015914917, 0.960088849067688, -0.4760870337486267, 1.9024279117584229, -0.8372438549995422, -1.1885597705841064, 0.2792993485927582, 1.240312099456787, -0.8726211190223694, -1.1538082361221313, 0.29970884323120117, -1.222969651222229, -0.34200412034988403, -1.1053526401519775, -0.7955606579780579, 0.33235424757003784, 0.5915045738220215, 0.462802916765213, -1.1798439025878906, 0.05416422709822655, 0.24310925602912903, -1.1842577457427979, -0.4464147686958313, 0.6097335815429688, 1.0225083827972412, 1.207926630973816, -0.9957178831100464, -0.8028660416603088, -0.08736041188240051, -0.5841209292411804, -1.1596986055374146, -0.19880369305610657, 0.039243679493665695, 1.016602635383606, 0.0021594504360109568, -1.1832267045974731, 1.138392448425293, -0.7801547646522522, 0.9778569340705872, -0.05609746649861336, -0.6341299414634705, 1.9264453649520874, -0.4241088330745697, -2.276127815246582, -0.9432900547981262, -0.6659168601036072, 0.8874530792236328, -0.33722952008247375, -0.6951429843902588, -0.7948446273803711, -1.3862978219985962, -0.1582811176776886, 0.3939249813556671, 0.4637182354927063, -0.09862518310546875, -1.7092335224151611, -0.5752314925193787], [0.33637985587120056, 0.2500740587711334, 1.3591388463974, 1.2648379802703857, 1.0045390129089355, 1.777077078819275, 0.8857879638671875, 0.24294352531433105, 1.485949993133545, -0.5768786668777466, 0.8962158560752869, -1.8457896709442139, -0.09838689118623734, 0.9826818108558655, 1.8390215635299683, -1.9730743169784546, -1.2194558382034302, -0.22949755191802979, -1.3033612966537476, -0.468979150056839, 0.03609301894903183, 1.4470247030258179, -0.892238974571228, 0.5959594249725342, -1.5505070686340332, 1.156083583831787, 0.45588386058807373, 0.5596024394035339, -0.34148871898651123, 1.210231065750122, 0.7127923965454102, -1.7253141403198242, 1.3332270383834839, 2.0510940551757812, 0.5522165298461914, -0.1679176539182663, 0.2850044369697571, 2.28482723236084, -0.2683456540107727, -1.4254059791564941, 1.4899109601974487, 0.6456810235977173, -0.0008382191881537437, 0.5424531102180481, -0.5704626441001892, 1.505360722541809, 0.4919479489326477, -0.5281388163566589, -1.7758952379226685, -1.4390945434570312, -1.0993258953094482, 0.5244581699371338, 0.12036565691232681, -0.5530344843864441, 0.21207955479621887, 0.985340416431427, 0.0844932273030281, 1.2224406003952026, -0.9929238557815552, -0.16473734378814697, 0.8481938242912292, -0.3639611303806305, -0.14604413509368896, -2.2999157905578613], [0.49820196628570557, 0.5129743218421936, -0.6230924725532532, -0.49698272347450256, 0.1296909749507904, 0.9923047423362732, -2.2392148971557617, -2.616438388824463, 0.6562344431877136, 1.5428861379623413, 0.062160421162843704, -0.40183451771736145, -0.1819300800561905, -0.01330346055328846, 0.11034388840198517, 1.6083965301513672, -1.5473889112472534, 0.7326810956001282, 0.2484724223613739, 0.5079293847084045, -2.182305097579956, 0.8615733981132507, -2.32364821434021, -1.1983821392059326, -0.8048321008682251, -0.4721190929412842, 0.09090957790613174, -0.9109734892845154, 0.6957644820213318, 0.40681755542755127, 0.7599347233772278, 0.3173638880252838, -0.3694393038749695, 1.2629344463348389, -0.25047004222869873, -0.330195814371109, 0.4560188353061676, 1.0703057050704956, 0.8246799111366272, -0.31268805265426636, 0.8544406294822693, 0.32552021741867065, -1.805362582206726, -0.005000210367143154, -0.2121913731098175, -0.5081697702407837, -0.4858408570289612, -0.6500990986824036, -0.5463030934333801, 0.00952125247567892, 1.1016736030578613, -0.7539150714874268, 2.0666449069976807, -1.006325364112854, 1.6236298084259033, -0.6658770442008972, -0.0837690532207489, -0.4862793982028961, 1.8097050189971924, -0.3744311034679413, 0.36108261346817017, -1.3231927156448364, -0.30176061391830444, -1.089979887008667], [-0.05505608394742012, -2.0198400020599365, -1.8138787746429443, 1.4165327548980713, -0.2320563644170761, -0.9649589657783508, 1.0070873498916626, -0.7442213892936707, 0.710289478302002, -0.2076931893825531, -1.116620421409607, 0.03305329382419586, -0.9776239991188049, -1.3432561159133911, -2.6633312702178955, 2.1998233795166016, -1.7679386138916016, -1.0564184188842773, -2.084948778152466, -1.9199737310409546, 0.8787534236907959, 1.5946288108825684, -1.158696174621582, -0.12082263082265854, 0.4983322024345398, 0.27485665678977966, 0.45786553621292114, 1.1695462465286255, -1.6934912204742432, 0.5562804937362671, 1.2203669548034668, 0.9696736335754395, 0.059353966265916824, 0.48526284098625183, 0.48054903745651245, 1.360451579093933, -0.1990768164396286, -0.1440541297197342, -1.8188444375991821, -0.0011682405602186918, -0.4306592345237732, 1.2280662059783936, -0.7029686570167542, 0.44393065571784973, 0.25271859765052795, -0.3569774925708771, 0.9649531245231628, 0.04803653806447983, 0.5708579421043396, -0.1743210107088089, -0.3089892566204071, -1.7813185453414917, 0.34780484437942505, -0.19395321607589722, -1.9013041257858276, -0.3555911183357239, -0.545335590839386, 0.8513015508651733, -0.4159545302391052, 0.22912654280662537, -0.3120248019695282, -1.2175706624984741, 0.024887073785066605, -0.41345661878585815], [-1.2927535772323608, -0.9435593485832214, -1.2080312967300415, -0.3977668583393097, 0.487494558095932, 2.5930190086364746, 0.1864939033985138, 0.2801334857940674, 0.562183141708374, -1.180060625076294, -1.2601020336151123, 0.8675949573516846, 0.5094172954559326, 0.938113272190094, 0.14628809690475464, -1.6217007637023926, 0.07194868475198746, -2.0524168014526367, 0.5233405232429504, 0.2769964933395386, -0.8159721493721008, -0.8666795492172241, -0.9094231724739075, -0.8983151912689209, 0.3509356677532196, 0.5026132464408875, -0.7427237033843994, 0.13776817917823792, 1.419980764389038, -0.4051627516746521, 0.258024662733078, -0.9395707249641418, -2.0066866874694824, -0.3979945182800293, -0.7039204835891724, 1.2575139999389648, -0.8063749074935913, -0.11140891164541245, -0.12228339910507202, -0.274550199508667, 2.056575059890747, 0.0007191899348981678, -1.0769264698028564, -1.0662426948547363, -0.7396999597549438, -1.038934588432312, 0.08246397227048874, -0.6776779294013977, -0.7065669298171997, -1.1987769603729248, 0.6441663503646851, -0.10321833193302155, -0.6512730121612549, 0.7108118534088135, 0.6076623797416687, 1.2074064016342163, -0.9303717017173767, 1.4292367696762085, 0.7266244888305664, 0.1793835312128067, 0.8136462569236755, 0.7081722617149353, -0.9408215880393982, -1.330755352973938], [-1.789583444595337, -0.920840859413147, -0.9281929731369019, 0.6046157479286194, -0.49416065216064453, -0.8374435901641846, 0.5765594244003296, 0.37432289123535156, 0.3760901391506195, 1.4074956178665161, -0.3568952679634094, -0.2807236611843109, 1.914574384689331, 1.3255313634872437, -1.792946696281433, -0.09823252260684967, 0.32115840911865234, 2.112454414367676, -0.01061192061752081, 0.5304145216941833, -2.3519225120544434, 0.04478298872709274, -0.860512375831604, 1.9200011491775513, -1.3756797313690186, 0.008779442869126797, -2.1717987060546875, 0.7402613162994385, -0.9781771898269653, 0.5742835402488708, 0.8983065485954285, -0.9302676916122437, 1.3348839282989502, 1.5149152278900146, 0.594433069229126, -1.1432583332061768, 0.09890557825565338, -0.5026213526725769, -0.23091302812099457, 0.03197353705763817, -0.26873883605003357, 0.25820693373680115, -0.5280998945236206, 1.431689977645874, -2.526808261871338, -0.6436015963554382, 0.1617695391178131, -1.284970998764038, -0.412663072347641, -0.2437162697315216, -1.4966198205947876, -0.6292728185653687, -0.04020804166793823, -1.8384084701538086, -0.06322100013494492, 1.3774735927581787, -0.0024771252647042274, 1.507949948310852, 1.1397091150283813, -0.21432091295719147, 0.515099823474884, 1.0319112539291382, -0.6079912185668945, 2.072019100189209], [0.6979082822799683, 0.255918949842453, -0.021805427968502045, -0.19099211692810059, -0.45064279437065125, -0.1313873827457428, 0.41438692808151245, -0.634353518486023, -1.1304303407669067, 1.5309072732925415, 1.590489387512207, -0.10195150971412659, -0.21800914406776428, -1.0979174375534058, -0.023040322586894035, -2.2529568672180176, 0.8131577968597412, 0.9166340231895447, -1.255744218826294, 0.1477547436952591, -1.6343077421188354, -0.27313435077667236, -0.60235196352005, 0.1792914867401123, 1.3425698280334473, 1.0641388893127441, 0.1138862892985344, 0.5967855453491211, -1.7611325979232788, -0.44235482811927795, -0.14458729326725006, -0.1922607570886612, 0.21417218446731567, 0.5368334054946899, -0.021568486467003822, 0.4227941632270813, -0.5910213589668274, -1.8227453231811523, -0.5585258603096008, 0.5273871421813965, 0.8045322299003601, -1.885867714881897, -1.4262117147445679, -0.617684543132782, -0.2348073571920395, 1.9817605018615723, -0.45795929431915283, 1.5475995540618896, 1.3176418542861938, -0.8948923349380493, 0.8774781823158264, 0.6532900333404541, -0.860852837562561, -0.2434271275997162, -0.5453047752380371, -0.31026092171669006, -0.6048600077629089, -1.192702054977417, -0.20955362915992737, -0.6233299970626831, 0.29834872484207153, -0.6292323470115662, 0.7884424924850464, -0.5981305241584778], [0.20838885009288788, -0.3676600456237793, 1.6633070707321167, -0.018315935507416725, 0.31944897770881653, -0.6153111457824707, 1.487412929534912, -2.3068196773529053, -2.049015760421753, -1.137660264968872, -1.6164379119873047, -0.8252266645431519, -0.6303850412368774, -0.028483035042881966, -1.4813190698623657, 0.6765457391738892, 0.7064183950424194, 0.6991832852363586, 1.1151520013809204, -0.670414388179779, -3.7679977416992188, 0.5549207329750061, 1.4494202136993408, -0.27025195956230164, 0.739774227142334, 0.03226511552929878, 1.3589082956314087, -0.12282413244247437, 0.38222771883010864, 2.450981855392456, -2.2131237983703613, -0.2488706111907959, 1.4983537197113037, -1.6479604244232178, 0.09928475320339203, 0.8983843326568604, -1.7217707633972168, -0.3000392019748688, 0.912118673324585, 0.6295017600059509, 1.4861860275268555, -1.713024616241455, 0.6913756728172302, -0.4565344452857971, -0.6052491068840027, 1.447505235671997, -1.737714409828186, 0.3383203148841858, 1.1490925550460815, -1.3965381383895874, 0.7470375299453735, -0.4334375262260437, -1.8461668491363525, 0.980688750743866, -1.8626978397369385, 1.383813738822937, -0.711910605430603, -0.2723172605037689, 0.39477482438087463, 0.7907900214195251, 0.9467736482620239, -0.8245117664337158, -1.489797830581665, -0.34852996468544006], [-0.6639066338539124, 2.2239181995391846, 0.21247488260269165, 0.2475304901599884, 0.6061961054801941, -1.4521735906600952, -0.5143530964851379, -1.4138424396514893, 1.9953510761260986, -0.6346685886383057, 0.2459847331047058, 0.7155243754386902, 0.3113636076450348, 0.3626692593097687, 0.7222578525543213, 0.76004958152771, -0.4648844599723816, 0.5418962240219116, 1.5209969282150269, -0.4043193459510803, 2.213106870651245, -1.053531289100647, -0.2759469151496887, 0.8472574949264526, 0.1256166249513626, 1.580712914466858, 1.057334303855896, -0.02894216775894165, 0.4695548713207245, -0.11845706403255463, -1.211267352104187, 1.9116370677947998, 0.05762818083167076, 0.5327285528182983, -2.755430221557617, -0.9666861295700073, -0.509304404258728, -0.4227880835533142, 0.14325475692749023, -0.44659146666526794, -1.4629899263381958, 0.46694308519363403, 0.6687488555908203, -0.3058018386363983, -0.2474314272403717, 0.20040200650691986, 0.878034770488739, -0.9595992565155029, 1.4995898008346558, -0.7835356593132019, -0.08539485931396484, 0.3021692931652069, 0.8105026483535767, -0.8883257508277893, 0.16308173537254333, -0.8294118642807007, 1.4996343851089478, -0.0625474750995636, 1.5083374977111816, 1.039313793182373, 0.3828728497028351, -0.23036427795886993, 0.3595101237297058, -0.34171175956726074], [0.352510541677475, 1.0944455862045288, -1.0603115558624268, 1.1162198781967163, 0.14490365982055664, 0.22394491732120514, 0.6653387546539307, -0.14154288172721863, -0.14901958405971527, 0.19632019102573395, -1.8851860761642456, -0.835516631603241, -0.20279796421527863, 0.9377351999282837, -0.8374294638633728, 0.6385728120803833, 0.12492170184850693, 0.4615686535835266, 0.04279348999261856, 0.30709534883499146, 0.2576027512550354, -0.8271039724349976, 3.2502903938293457, -0.19153210520744324, 0.21993862092494965, 0.18383772671222687, 0.16326306760311127, -1.5419727563858032, -2.391817808151245, -1.1703927516937256, -0.36263731122016907, 1.0515587329864502, 1.8410060405731201, 1.0539880990982056, 2.069675922393799, 1.0463762283325195, -0.6795795559883118, -0.7563943266868591, 0.34195154905319214, -0.46421507000923157, -0.21407008171081543, -1.2966785430908203, 0.5747382044792175, 1.3843786716461182, -2.5175297260284424, 0.5084304213523865, -0.2783489227294922, -1.240092396736145, 0.1603882610797882, -0.8913378119468689, -0.08792746812105179, 0.02707221917808056, 0.7074143886566162, 0.9234904050827026, 1.2984203100204468, -0.22021719813346863, -0.45979249477386475, -1.5067737102508545, -0.505081057548523, -0.5878749489784241, 1.0580514669418335, 0.08533702045679092, 1.4125248193740845, 0.29294946789741516], [-0.5791202187538147, 1.099886178970337, -0.035009145736694336, -0.6068607568740845, 0.24506793916225433, -0.2784549593925476, 0.12435968965291977, 0.9012701511383057, 1.127774715423584, -0.07393535226583481, -1.0988141298294067, -1.4582359790802002, 1.1060227155685425, -0.44124269485473633, -0.5298418402671814, -0.9505932927131653, -0.639820396900177, -0.7663716673851013, 1.2306078672409058, 1.4097703695297241, 0.7234137654304504, -0.719792366027832, -0.5610424876213074, -0.35913681983947754, -0.9751121401786804, 0.5336808562278748, -1.0024164915084839, -0.16312481462955475, 0.0009374450310133398, 0.031466878950595856, -1.7540545463562012, 0.6035351157188416, -2.1358070373535156, -0.12762746214866638, 0.11102542281150818, 0.15819939970970154, 0.5185714960098267, 1.39644193649292, -1.8108388185501099, -0.990541398525238, -2.3956949710845947, -0.8946470618247986, 0.647798478603363, -0.41940346360206604, 0.3882126212120056, 0.35430118441581726, 2.3599650859832764, -0.23136983811855316, -0.4936714768409729, -0.36463090777397156, 1.022636890411377, -1.0653650760650635, -0.2703433930873871, 0.9643120169639587, -0.2583121657371521, -0.526416003704071, 0.09918398410081863, -0.35272854566574097, 0.3935942053794861, -0.6569238305091858, 1.5015417337417603, -1.5121046304702759, -2.145592451095581, -1.065731167793274], [0.7704298496246338, -0.37895503640174866, 1.637758731842041, -1.077773928642273, 0.43484577536582947, -0.05734919384121895, 1.444725751876831, 0.0979139655828476, -0.06382862478494644, -0.3386208117008209, 0.9276825785636902, -0.37600061297416687, 1.1527037620544434, 0.08764442056417465, -0.7881914973258972, 0.3407512903213501, -0.2344415783882141, -0.4014061987400055, 1.1215636730194092, 0.33226141333580017, -0.8036428689956665, 1.5453014373779297, -0.4230853319168091, -1.2981587648391724, 0.4916646182537079, 0.02014278434216976, -1.212384581565857, -1.024172306060791, -1.1282484531402588, 1.3797791004180908, 0.9831665754318237, -0.7361054420471191, 0.0031751771457493305, 0.6935985088348389, 0.6855688095092773, 1.1869772672653198, 2.457927942276001, -0.897288978099823, -0.2037564069032669, -0.743502140045166, 1.3265676498413086, 0.11827308684587479, 1.0294450521469116, -0.1681685447692871, 0.037038106471300125, 1.4155890941619873, -0.024147426709532738, 0.12740236520767212, -1.256752848625183, 1.732814908027649, -0.658704400062561, 0.33684584498405457, -0.21729344129562378, -0.16970239579677582, -1.0838935375213623, -0.7002350687980652, -1.8287628889083862, 0.43022847175598145, 1.3552838563919067, -1.6214100122451782, -0.4875483512878418, 0.4143708050251007, 0.5146179795265198, -0.3156066834926605], [-1.1812398433685303, -1.8035709857940674, -1.6437913179397583, 2.114628553390503, -0.6022742390632629, -2.1312084197998047, -0.6494404077529907, 0.6762714385986328, 0.3991154730319977, -1.4838532209396362, 0.1026792898774147, -1.2792972326278687, -1.2131582498550415, 0.6953432559967041, 1.0922316312789917, -2.2884328365325928, 0.6799390912055969, 0.18415388464927673, 0.40973392128944397, -0.12326642125844955, 0.3602060079574585, -0.18213045597076416, 0.3457046449184418, -0.3708389401435852, 0.8817909955978394, -1.5168603658676147, 0.7180557250976562, -0.8634173274040222, -1.9698121547698975, -1.1364762783050537, -0.6883548498153687, 0.9740656018257141, 1.4669350385665894, 0.17532455921173096, -1.044979214668274, 0.7143205404281616, 2.689962387084961, -2.578665018081665, 1.376423954963684, 0.37085410952568054, -1.2945469617843628, -0.20865505933761597, -0.8598510026931763, -0.11789493262767792, -0.24957609176635742, 0.28290244936943054, -0.784021258354187, -1.3176707029342651, 0.09295748174190521, 0.030073143541812897, 0.5392067432403564, 1.8996367454528809, -0.545200526714325, 0.9804509878158569, -0.39754199981689453, -1.0288605690002441, 1.2147682905197144, -1.8796896934509277, 0.8849246501922607, -0.7431023120880127, 0.02508961781859398, 0.021814610809087753, 0.12624585628509521, -1.771342396736145], [-1.3708492517471313, -1.378656029701233, -2.095831871032715, -0.0691559687256813, -0.41842585802078247, -0.3496246635913849, 0.08096820116043091, -0.14532925188541412, 1.437666654586792, 0.5193088054656982, -1.4708075523376465, 0.4889136254787445, -1.2152122259140015, 0.9030336141586304, -1.2275418043136597, -0.8061491847038269, -0.3852047622203827, 0.4848991632461548, -0.41770437359809875, 0.4540065824985504, 2.6646270751953125, -0.6805050373077393, 1.0918060541152954, -1.6112881898880005, -0.2455417662858963, -0.6864802837371826, -1.2117727994918823, 1.2038301229476929, -0.2591373324394226, -0.28814172744750977, -1.6071385145187378, -1.43028724193573, -1.4036364555358887, -0.5248169898986816, 0.34404894709587097, -0.3734819293022156, -0.2872104346752167, 0.43879565596580505, -0.8108862638473511, 0.005336306523531675, -0.3300354480743408, -0.12913525104522705, 0.7253822684288025, 0.7893970012664795, -0.237039715051651, 1.173269271850586, 1.6224768161773682, -0.9495989680290222, 0.5134251117706299, 0.5040676593780518, -1.1526801586151123, 0.4949263334274292, 0.49130168557167053, -2.070441246032715, 0.8422613739967346, -1.4758999347686768, -0.8901575207710266, -1.911316990852356, -0.33437785506248474, -0.47482815384864807, 0.8722385764122009, -0.4562698304653168, -1.37643563747406, 1.4179246425628662], [-0.17544561624526978, -0.3007773458957672, 0.9156194925308228, -0.9888858795166016, -1.8128583431243896, 0.2562413513660431, -0.2672420144081116, -1.227473497390747, -0.402447372674942, 0.6951013803482056, 1.3002431392669678, -2.2080814838409424, 0.10351710021495819, 0.20042270421981812, -1.2821789979934692, -0.729926586151123, 0.40107157826423645, 0.11248844861984253, 1.5171936750411987, -0.05625048279762268, 1.162052869796753, -1.5830533504486084, 0.13167443871498108, -0.47474050521850586, -1.6676690578460693, 0.7157323360443115, 1.5018305778503418, 0.26352810859680176, -1.7888318300247192, 0.08597050607204437, -0.31659382581710815, -1.4124858379364014, 0.8543019890785217, 1.2984063625335693, 0.13837644457817078, 0.8985907435417175, 1.06361985206604, 0.7507938742637634, 1.4183677434921265, 1.023157000541687, 0.6626275181770325, 1.6546343564987183, -0.8233789205551147, 2.005817413330078, -0.6204504370689392, 0.06315172463655472, -0.23119528591632843, 0.6624128818511963, 1.2157474756240845, 0.7406305074691772, -0.21093013882637024, -1.0813395977020264, 2.304866075515747, -1.3469252586364746, -0.08770667761564255, 0.7094663381576538, -0.5715884566307068, 1.368421196937561, -1.3708343505859375, 0.525062620639801, -2.228431463241577, 1.952713966369629, 2.638108253479004, -0.9800573587417603], [-0.27161484956741333, 0.7141802906990051, -0.062568299472332, 1.5313307046890259, 0.8119798302650452, -1.1985061168670654, -0.8411993980407715, -0.731806218624115, -0.46005794405937195, 0.019985433667898178, -0.10329987853765488, 0.8273008465766907, 0.9678579568862915, 0.9882715940475464, -1.7561088800430298, -0.029074518010020256, 0.2534950077533722, -0.5133402347564697, 1.834403157234192, 1.673601746559143, 0.49277228116989136, -0.5848450660705566, -0.5493894815444946, 0.47072291374206543, -0.44195806980133057, 0.0907803401350975, 0.26928067207336426, 0.4881858825683594, 2.25488018989563, -1.3602641820907593, -1.5005815029144287, -0.09589898586273193, -0.5117193460464478, -1.2149391174316406, 0.24449960887432098, -0.23208093643188477, 0.1697874367237091, -0.6839438676834106, -0.663449227809906, -2.0876529216766357, -0.697304904460907, -0.6779983043670654, 0.030023908242583275, -0.8261248469352722, 0.9288634657859802, -0.021958010271191597, -0.2878284156322479, 1.7334332466125488, 1.1394572257995605, 0.4025708734989166, -0.8080047965049744, 0.6093148589134216, 2.4937400817871094, 0.7076113820075989, -0.393223375082016, -0.9662273526191711, -1.1197113990783691, -1.2541718482971191, -0.29387161135673523, -2.061673641204834, -1.3096446990966797, 0.196504145860672, 0.8062432408332825, -1.014363169670105], [1.085580825805664, 1.144005298614502, 0.4837956130504608, -0.5743128657341003, -0.5039026141166687, 0.8465922474861145, -0.9432703852653503, -0.6749463677406311, 0.346348375082016, 1.0394617319107056, 0.5261312127113342, 1.860133171081543, 0.5382035374641418, 0.7095589637756348, -0.4374868869781494, 1.7808033227920532, -1.1994032859802246, 0.2578565776348114, 0.003710675286129117, 1.5860525369644165, -1.7977465391159058, 0.6933899521827698, 0.07299965620040894, 0.7408111691474915, -0.06671296805143356, -2.286343812942505, -0.6706362962722778, -0.4070431590080261, -0.18183347582817078, 1.3550323247909546, -0.14667661488056183, -0.1749134659767151, -0.5882591605186462, -0.3224141299724579, -0.980094850063324, 1.8164191246032715, 1.2524516582489014, -0.1938246488571167, -1.971239686012268, 0.605668306350708, -0.6217864155769348, -0.4065493941307068, 0.3026559054851532, -0.5929405689239502, 0.21755218505859375, -0.82145094871521, 0.1995057612657547, 0.9547842144966125, 0.22981497645378113, -0.6596660614013672, -0.5865942239761353, -1.6826391220092773, -0.4429577589035034, -1.0463193655014038, -0.03405516967177391, -1.2811641693115234, 0.644148051738739, 1.1679718494415283, 2.295997142791748, 0.49946463108062744, 1.1549549102783203, 0.6597863435745239, -0.4566473662853241, -0.27927330136299133], [0.6340168714523315, -0.5666196942329407, -0.4418620467185974, -0.8041290044784546, -0.8639131188392639, 0.9051325917243958, -1.2173289060592651, -1.7691930532455444, -1.3990808725357056, 1.7209707498550415, -0.7921659350395203, 2.2339189052581787, -0.6516728401184082, -0.7325198650360107, 2.583237648010254, 0.7035468816757202, 0.0368322879076004, 0.6401119232177734, -1.3064908981323242, -0.6730167269706726, 0.06121606379747391, -1.7836315631866455, 1.8410627841949463, -0.7834659218788147, 0.21497222781181335, -0.2512907087802887, 2.9512500762939453, 0.8258538842201233, 0.8299887776374817, -0.8573771715164185, -1.9652012586593628, 1.6658973693847656, -0.9953504800796509, -0.12151817977428436, -0.5166004300117493, 1.321007251739502, -1.113547682762146, -0.9945765137672424, -0.15878714621067047, -1.2645045518875122, 0.705176055431366, -0.6546556949615479, -2.190046787261963, -1.2402119636535645, 0.9216005206108093, -0.22152233123779297, -0.2380335032939911, -2.119309425354004, -0.05773083493113518, -0.02946457266807556, -1.432288646697998, 1.2479020357131958, -0.7588814496994019, 0.7944391965866089, 1.2127660512924194, 1.0010355710983276, 0.2142678052186966, 1.2218976020812988, -0.44252803921699524, -0.8713173270225525, -0.7309157848358154, 0.6863226890563965, 0.22883771359920502, -0.5854735374450684], [-1.8622945547103882, 0.13047955930233002, -0.7632626295089722, 1.2823892831802368, -0.5077354311943054, 0.7208672761917114, 1.323121428489685, -2.210054397583008, -0.46826601028442383, 1.2202192544937134, -0.15905001759529114, 0.09857494384050369, -0.7462381720542908, 0.3020547926425934, -0.28252890706062317, -0.048897720873355865, 1.230759859085083, -1.9943397045135498, 0.8300361633300781, 3.7575411796569824, -0.46699193120002747, -0.8483969569206238, -1.5228526592254639, -0.9330482482910156, 0.9225545525550842, -0.4808100163936615, 0.10179721564054489, -2.2259693145751953, 0.05433781072497368, -0.26724621653556824, 0.09602277725934982, 0.20396873354911804, 0.20479841530323029, -1.0090302228927612, -0.013549759984016418, -0.9637280106544495, -0.08468464016914368, 1.1313281059265137, -0.6971340775489807, -1.0856798887252808, 1.4179487228393555, 1.6853034496307373, -0.3449119031429291, 0.7436985373497009, 1.4514081478118896, 0.09272009134292603, 1.586503267288208, -0.17085234820842743, 0.26527172327041626, -1.000540018081665, -0.9464293718338013, -0.14191733300685883, -1.1170077323913574, -0.6841689348220825, -0.8874569535255432, 1.2730120420455933, -1.107324481010437, 0.6400001645088196, -0.628736138343811, -0.2108902782201767, -0.49769070744514465, -1.1219767332077026, 2.0365638732910156, 0.4585055410861969], [-0.8124166131019592, -1.4190102815628052, -1.0102483034133911, 0.07199376821517944, -0.34911245107650757, 0.03120686672627926, -1.2435638904571533, -0.5446205735206604, -0.21722254157066345, -0.934386134147644, -0.1185380145907402, -0.5104355216026306, 0.6091113686561584, -0.04846624284982681, 1.269591212272644, 1.8234503269195557, -0.12896761298179626, -0.996695339679718, -0.2065039873123169, 1.108787178993225, 0.6029279828071594, -0.5478240847587585, -0.5212636590003967, -0.23756884038448334, -0.9419698715209961, -1.0419690608978271, -1.16730535030365, -0.29786044359207153, -1.102216362953186, 0.43141257762908936, 0.9966529011726379, 0.7729189991950989, -0.6717371344566345, 1.8373103141784668, -0.37766000628471375, 1.1194566488265991, -1.077149510383606, -0.11799504607915878, 1.8300775289535522, 0.022798944264650345, -0.06551290303468704, 0.6136327385902405, -1.609268307685852, 1.218510389328003, -0.6036399006843567, -0.3058144748210907, 1.687220573425293, 0.6324081420898438, -0.7054689526557922, -0.6047933101654053, -0.3377457559108734, -0.37432658672332764, 0.31795868277549744, 1.0969336032867432, -0.917602002620697, -0.03657042235136032, 0.5233070850372314, 0.9182479381561279, -0.8860645294189453, -1.6849993467330933, 0.3372024595737457, 1.5494998693466187, 0.5091332793235779, -0.3024965524673462], [-1.1873646974563599, 1.4508296251296997, -1.3742297887802124, 0.34418976306915283, 1.310017704963684, 0.20195774734020233, 0.1554163098335266, 1.0745983123779297, -1.5146942138671875, -0.6049176454544067, -0.7351235151290894, 1.084187388420105, 0.8891127705574036, -0.35399699211120605, 1.950961709022522, 0.9877452850341797, -0.13326673209667206, -0.2075616717338562, 0.9928106069564819, 0.831153392791748, 0.08828262239694595, 0.16622860729694366, 0.6299760341644287, 0.2295009344816208, 1.2608602046966553, 0.591141402721405, -0.43176138401031494, -1.450405478477478, -1.0189845561981201, -0.739018440246582, -0.30841341614723206, 0.9905819296836853, -0.6460029482841492, -0.14006198942661285, -0.21028642356395721, 1.0718525648117065, -0.19503161311149597, -0.5414747595787048, -0.37144431471824646, -0.3351280987262726, 0.5114478468894958, 0.474945992231369, 0.37564346194267273, 0.5163120627403259, 0.4124102294445038, -0.27548131346702576, 0.5331630110740662, 0.41746777296066284, -0.5865136384963989, -1.3342688083648682, -0.5657451152801514, -0.4520595967769623, -1.2742491960525513, -2.032310724258423, -0.16981783509254456, 0.6690875291824341, -0.008969045244157314, 0.10773415863513947, -1.0828536748886108, 0.0085885189473629, 0.701532781124115, -0.0630626529455185, -0.73561030626297, 0.7569798231124878], [-0.6334829926490784, -0.15557312965393066, -1.120652675628662, 0.1579175442457199, 0.36152297258377075, 0.012591136619448662, -0.8080368041992188, -0.46157991886138916, 0.21639692783355713, -0.09482179582118988, -0.22171662747859955, 1.8458296060562134, 2.478273391723633, 0.1350417584180832, 0.9387320876121521, -0.10986661911010742, 0.8777616024017334, 1.5539265871047974, 0.34063130617141724, -0.35331013798713684, 0.20970883965492249, 0.37985625863075256, -0.3012240529060364, 0.20893658697605133, -0.7905332446098328, 0.7114876508712769, 0.7298285961151123, -0.2809576988220215, 0.27354779839515686, 1.4109400510787964, -0.5502831935882568, 0.779044508934021, -0.04133983701467514, 0.4400462508201599, -0.5696076154708862, 2.109607219696045, -0.6122036576271057, 1.0780926942825317, -0.7593865990638733, -1.1546369791030884, 0.01904722861945629, -0.20769155025482178, -0.6423287391662598, -0.7379605770111084, -0.5655580759048462, 0.984368622303009, 0.16269102692604065, 0.2727546691894531, 0.5281720161437988, 0.5240306854248047, 1.1009448766708374, 0.5318252444267273, -1.5854336023330688, -0.352181613445282, 0.7004758715629578, 1.3182944059371948, 0.05420364439487457, 0.18721118569374084, -0.22967472672462463, 0.7112058401107788, 1.1726067066192627, 0.04680094122886658, 0.3065390884876251, -0.7644605040550232], [-0.9042505621910095, 0.42656898498535156, 1.1022566556930542, -0.3991999626159668, -0.2379390150308609, 0.08623664081096649, -0.360666424036026, -0.2719908356666565, 0.9519915580749512, 1.3257395029067993, 0.6603543162345886, -0.2601564824581146, 1.022500991821289, 0.7164359092712402, -1.0427571535110474, 0.48402711749076843, 0.8427267670631409, 0.21236921846866608, 2.0899717807769775, 0.9459696412086487, 0.9011909365653992, 0.3307592272758484, 0.9071002006530762, -0.9317687749862671, -1.2943469285964966, 0.7911126613616943, -1.8438796997070312, 0.9963704347610474, -0.644330620765686, 0.819264829158783, -1.5665993690490723, -0.5940564274787903, 1.2286591529846191, 1.493628740310669, 1.3109318017959595, 0.5686963200569153, -1.4720616340637207, 0.6178896427154541, -0.18978016078472137, -0.18092814087867737, -0.3438883423805237, 1.7818210124969482, 1.6354472637176514, -0.14776656031608582, 0.25841113924980164, -0.04556020721793175, -0.9596010446548462, -0.03289930522441864, 1.037282109260559, -0.8466551899909973, -0.47087839245796204, 1.8749223947525024, 0.4546196162700653, -1.516983151435852, 0.754109799861908, 1.6816319227218628, -0.14924365282058716, -0.6934044361114502, 0.39501163363456726, 1.0393571853637695, -0.21611429750919342, -0.2467990219593048, -2.6200971603393555, -1.9792057275772095], [-1.309715747833252, 0.14208142459392548, 1.006695032119751, 0.70172518491745, 0.9722415804862976, -0.8592473268508911, -0.4835117757320404, -1.098766803741455, -0.7759919166564941, 1.0016374588012695, 0.16032874584197998, -0.2587509751319885, -0.5948218703269958, 0.5651354193687439, 1.0164674520492554, 1.1092535257339478, -0.4605601727962494, 1.7539442777633667, 0.6694806218147278, -0.676331102848053, -0.07376173138618469, 1.2833938598632812, 1.4656051397323608, -0.450529545545578, -1.0701922178268433, -0.6440037488937378, 0.34168320894241333, -0.6756145358085632, 0.29671505093574524, 1.5783675909042358, 0.8688259124755859, 0.4775187373161316, -0.6283763647079468, 1.4007364511489868, -1.2984215021133423, -0.0011815647594630718, 0.5408336520195007, 1.2541484832763672, -1.8811681270599365, -1.575905442237854, -0.5601462125778198, 2.255281686782837, -0.04707985743880272, 0.5321341753005981, 1.566293716430664, -1.4594324827194214, -0.47837722301483154, 0.10580122470855713, 0.6754146218299866, 0.2063751518726349, -0.665755569934845, -0.14942653477191925, 0.33964693546295166, -0.5366119742393494, 0.055525023490190506, -0.44056177139282227, -0.5111770033836365, -1.7284820079803467, 0.2210964858531952, -0.8771241307258606, -0.04530470445752144, -0.7823072671890259, -1.2890735864639282, -0.862220048904419], [-0.6011978387832642, -0.020903071388602257, -0.10028854012489319, -0.5430030226707458, 0.01554049551486969, 0.10107069462537766, 0.08623089641332626, 1.115495204925537, -0.056531719863414764, 0.2520125210285187, 0.08887916803359985, -0.03696179389953613, -0.42896780371665955, -0.7431235313415527, -1.3119257688522339, 0.33971694111824036, -0.29489365220069885, -0.22762563824653625, 0.7245655655860901, 0.7632038593292236, 0.6924785375595093, 1.2822744846343994, 0.33860987424850464, -0.7426949143409729, 0.7297540903091431, 0.6514515280723572, 1.1964656114578247, -0.17519479990005493, 0.6640900373458862, -1.2036164999008179, 0.5920833945274353, 0.6605721712112427, 1.4935624599456787, 1.4716242551803589, 0.3876299262046814, 1.2174279689788818, 0.552365243434906, 0.2945398688316345, 1.5597779750823975, 0.4132893681526184, 0.7978946566581726, -0.16306520998477936, -0.3453797996044159, 0.6241661906242371, -0.8171179294586182, 1.9362022876739502, 1.5046889781951904, -1.3194129467010498, 1.2019692659378052, -1.2878085374832153, -0.14486850798130035, 0.6767037510871887, -0.31237009167671204, 0.9981086254119873, -0.19530817866325378, -1.8152176141738892, 0.9754126071929932, -0.1876583695411682, 1.437214970588684, -0.3460627794265747, -1.0855785608291626, -0.2759855091571808, 0.49442726373672485, -2.136962652206421], [1.8169363737106323, -0.3671073615550995, 0.4259355962276459, 0.21883925795555115, 1.5929914712905884, 0.15204942226409912, -2.1361606121063232, 0.7942264080047607, 0.2651371359825134, 0.14422249794006348, -1.2239391803741455, 0.8267810940742493, -0.9230107069015503, 1.123164415359497, 1.4918309450149536, 0.3459322452545166, 0.3403257131576538, 0.1072925478219986, 2.093668222427368, -0.47250956296920776, 1.3616018295288086, -1.2418612241744995, -0.8002734184265137, -1.8122563362121582, 0.03704371303319931, -1.5730395317077637, 0.574286937713623, 0.4278097450733185, -1.0498722791671753, -0.5534213781356812, -1.0351301431655884, -0.4824362099170685, 0.6159270405769348, -0.3318521976470947, -0.7905750870704651, 0.013681449927389622, -1.450927734375, -0.21188265085220337, -0.1774272620677948, -1.429725170135498, 0.21854358911514282, -0.12850825488567352, 0.948599636554718, -0.5846222639083862, -0.9605821371078491, 0.8891125321388245, 0.7549342513084412, 0.1392548829317093, -0.902070164680481, -0.7359170913696289, 0.5266828536987305, -0.6730678677558899, 1.1058584451675415, 0.020072191953659058, -1.0594167709350586, 1.3594410419464111, 1.607681155204773, 0.4355831444263458, -0.22123484313488007, -0.18188869953155518, -0.3594989478588104, 0.037390079349279404, 0.5025942325592041, 0.2918994724750519], [-0.4856294095516205, 1.3691530227661133, -0.21753084659576416, 2.0714495182037354, -0.9945359826087952, 0.9617483615875244, 0.014530928805470467, 0.947394609451294, -0.5744219422340393, -1.0477172136306763, -1.7359687089920044, -0.5857177972793579, 0.13518157601356506, -0.8992565870285034, 0.44859060645103455, 1.0830693244934082, 0.8345542550086975, -0.06412634253501892, -1.2746319770812988, 0.6428699493408203, -0.5165280103683472, -0.7929400205612183, -0.2462347447872162, 0.3125182092189789, 0.3748677670955658, -0.11998848617076874, -0.9997764229774475, -1.0564124584197998, 0.5960147380828857, 0.26356688141822815, -0.41538000106811523, -1.13937246799469, -1.6378142833709717, 0.1507745087146759, 1.005835771560669, 0.1357908993959427, -1.0524604320526123, 1.5208725929260254, -1.9510256052017212, -0.7427549958229065, -0.7636708617210388, 0.19022785127162933, -0.7258796095848083, -2.1682193279266357, 0.9184992909431458, -0.9415436387062073, 0.2763286530971527, -0.780536413192749, 1.4110639095306396, -1.5974417924880981, 1.974958062171936, -1.0856789350509644, 2.157703399658203, 1.1058357954025269, 2.17228627204895, 1.866078495979309, -0.24190212786197662, -0.9003875851631165, 1.5390464067459106, 0.47608843445777893, -1.2149338722229004, -1.7279694080352783, -0.4448949098587036, 0.5574769973754883], [-1.2485790252685547, -0.7256300449371338, 0.6946454644203186, -0.9892962574958801, -0.2594333589076996, 1.338673710823059, -1.4280219078063965, 1.1843323707580566, 0.31568512320518494, 1.02118980884552, -3.09879469871521, 1.5253710746765137, -0.7671810984611511, -0.9793148636817932, 0.2448863536119461, 1.2907251119613647, 0.06015590578317642, -1.6980342864990234, 1.4137508869171143, 0.15094105899333954, 1.046862244606018, 1.0392628908157349, -0.32319456338882446, -1.2166028022766113, 0.29989761114120483, 0.5807323455810547, 0.5006057024002075, -0.3911826014518738, 1.3174676895141602, 0.3667182922363281, 0.17916399240493774, 1.5232495069503784, 0.6737160682678223, 0.8948745131492615, -0.33569005131721497, -0.41660773754119873, -1.5355099439620972, 1.6363356113433838, -1.6590888500213623, 1.0900300741195679, -1.398013949394226, 0.7475332021713257, 0.7482397556304932, 0.7281690835952759, -0.6074103116989136, -0.2798879146575928, 0.9788053035736084, -0.5724311470985413, 0.8041585683822632, -1.0203309059143066, -0.9856090545654297, 1.569303035736084, 2.149243116378784, 1.5257282257080078, -0.3930453658103943, -0.5309750437736511, 0.9312905669212341, 0.5511982440948486, 0.11713720858097076, 0.0029402594082057476, 1.3937196731567383, 0.49271219968795776, -0.02808060124516487, -0.20819970965385437], [-0.2503089904785156, -1.241460919380188, 1.2265700101852417, 0.007894786074757576, -1.6700396537780762, -0.7983174920082092, -0.4583938717842102, 0.05360901355743408, 1.570939064025879, -1.1148244142532349, -0.8898813724517822, -0.9318233132362366, 0.04167100787162781, -0.1704513281583786, -3.1053059101104736, -0.10652477294206619, -0.6032898426055908, -0.399217814207077, -0.7045793533325195, 0.6333852410316467, -1.212041974067688, 0.29239121079444885, -0.18179109692573547, 1.3543646335601807, -2.4340851306915283, -0.20719781517982483, 0.9852440357208252, 0.45120495557785034, -0.08513346314430237, -0.28626638650894165, -0.6989208459854126, 0.02041679620742798, -1.4590871334075928, -0.18408291041851044, -0.34965381026268005, 0.7216296195983887, -0.14063112437725067, -1.7219278812408447, 0.797718346118927, -0.3235187232494354, 0.8392775654792786, 0.9606361985206604, 1.2364064455032349, 1.118633508682251, -1.210412859916687, -0.6867432594299316, 0.33344703912734985, -0.0530235581099987, 0.5973663926124573, -1.2463345527648926, 0.2534125745296478, -0.7202317118644714, 1.14622962474823, -0.24326181411743164, 0.5368741154670715, -1.0494822263717651, 1.4175360202789307, 0.043993156403303146, -0.9838204383850098, 0.758858323097229, 1.5793057680130005, -0.11655687540769577, 1.9637420177459717, -0.6975983381271362], [0.1137014850974083, -0.4138297140598297, 0.43363022804260254, 0.7406242489814758, -1.0598641633987427, 0.229350283741951, -0.9108262658119202, 0.45690053701400757, -0.8669142723083496, -1.0755616426467896, 0.13132734596729279, -0.408875048160553, -1.1348172426223755, -1.1636073589324951, -1.4616849422454834, -1.4716285467147827, 0.7339748740196228, -0.6436705589294434, 0.07269831001758575, -2.344944715499878, -0.14216892421245575, 0.0062080384232103825, 0.05107325315475464, -0.24638426303863525, 0.44922059774398804, -2.09222412109375, -0.670089602470398, 1.245070219039917, -0.2361678183078766, -0.1035064160823822, 3.6640148162841797, 1.7282503843307495, 0.6571133732795715, -0.1402338296175003, -0.6737102270126343, 0.07283665984869003, -0.7320036888122559, 0.3420828878879547, -0.5979408621788025, 0.013463586568832397, -0.7292788028717041, -0.1334851235151291, -0.1413317620754242, -1.2083656787872314, -0.1230081245303154, -0.8350988030433655, -0.42145299911499023, -0.2895812690258026, -1.3793940544128418, 0.35791879892349243, 0.24459724128246307, -0.8562648892402649, -1.0982460975646973, 0.6396048069000244, 1.3979591131210327, -1.053088903427124, 0.10936755686998367, 1.4218318462371826, 0.9678647518157959, 2.4593923091888428, -0.4806312620639801, -0.6072132587432861, -1.1947648525238037, 0.06972446292638779], [1.2912312746047974, -1.0667694807052612, 0.10151305794715881, -0.3949592113494873, -0.8640904426574707, -0.6457409858703613, -0.5671964287757874, -1.7500213384628296, -0.5317729115486145, -0.41117021441459656, -0.6758841872215271, -0.4630023241043091, 0.9152907133102417, -0.9752108454704285, 0.13157658278942108, 0.9673910737037659, -1.231221079826355, 1.7662363052368164, -1.823956847190857, -0.3850262761116028, 0.5621654987335205, -0.16995283961296082, -0.26935964822769165, 1.415971040725708, 1.1519259214401245, -0.933907151222229, -0.5798577070236206, 1.9327162504196167, 0.6194162368774414, 0.38547512888908386, 1.1952109336853027, -0.3759394586086273, -0.4196530282497406, -0.07429429143667221, 0.9007154703140259, 0.6696605086326599, -0.7281674742698669, 1.30608332157135, -0.20198164880275726, 2.9270763397216797, -0.41142794489860535, 0.8790708184242249, -1.9834808111190796, -0.4311593174934387, -1.5148568153381348, -1.0679436922073364, 0.7250745892524719, -1.6210922002792358, 0.9958165287971497, 0.7441652417182922, -0.7882323265075684, 0.5373652577400208, -0.6139528155326843, 0.6588292121887207, 0.23751311004161835, 0.7906549572944641, -2.3296992778778076, -2.031126022338867, 0.589658260345459, -0.17000220715999603, -1.1054574251174927, 1.569218397140503, -1.9500060081481934, 2.072145462036133], [1.0490366220474243, -0.43739596009254456, 1.663769006729126, 0.4400269091129303, -0.4579814374446869, 0.21640408039093018, -1.8854526281356812, 0.4417535364627838, -0.6903690695762634, 0.47022807598114014, -0.3895149230957031, -0.4373016059398651, 0.2726907730102539, 1.0872281789779663, -0.01909545063972473, 0.607519268989563, -2.464831829071045, -1.7101333141326904, 1.657670021057129, -0.8716546893119812, -1.6895188093185425, -0.7895626425743103, -0.9290140271186829, 1.508525013923645, 1.4036284685134888, -0.23755072057247162, -0.5631338357925415, -0.5158030986785889, 1.3873507976531982, -1.198851227760315, -1.6062389612197876, -1.487395167350769, -0.3538319766521454, 1.117097020149231, -1.6440178155899048, 1.0789886713027954, -0.5433033108711243, -0.3179700970649719, 0.12425066530704498, -0.05559340864419937, -2.264462471008301, 2.429957389831543, -0.2926662266254425, -0.7138704061508179, -0.918107807636261, 0.22878092527389526, 0.11786015331745148, -0.5116540789604187, -0.8582608699798584, 1.7566680908203125, -0.5813036561012268, 0.4918835759162903, -0.2022688090801239, -0.8058778047561646, -0.63057941198349, 1.4461710453033447, -1.6421838998794556, -0.1111140251159668, -1.1836652755737305, 1.154997706413269, 0.9427019357681274, 0.07050982862710953, 0.578978955745697, -1.533183217048645], [1.0563734769821167, 1.025160312652588, 0.7964121103286743, -1.3578532934188843, 1.417523741722107, -0.16682256758213043, -2.263535737991333, 0.9802421927452087, 0.6105175018310547, -1.5634444952011108, 2.191577911376953, 0.7925490140914917, 0.8642694354057312, 0.22017601132392883, 1.0302220582962036, -2.156918525695801, -0.2931259274482727, -0.144461527466774, 0.41816744208335876, 1.680561900138855, -0.46954694390296936, 0.4291965663433075, -1.074721336364746, -0.4903208911418915, 0.8734135627746582, 0.1788935512304306, -1.1942930221557617, -0.9327466487884521, -0.25638964772224426, -1.0826427936553955, -0.38776034116744995, -1.165462851524353, -1.1282427310943604, -0.4040403664112091, -0.0850614458322525, 1.4715311527252197, -0.7339051961898804, 1.182477593421936, -2.1911840438842773, 0.12064320594072342, -1.871583342552185, -0.5847625136375427, 0.5013070106506348, 0.0446026585996151, -0.4075547456741333, 0.949201762676239, -1.700259804725647, -0.13947871327400208, 1.0736174583435059, -0.7368273735046387, -0.28714582324028015, 1.184221625328064, -0.7876341342926025, 1.2922176122665405, 0.058138009160757065, -0.423669695854187, -0.8614294528961182, 0.6368367075920105, 1.6361660957336426, -2.243959665298462, 2.631810426712036, -0.3393356502056122, -0.09503275156021118, -0.8179579973220825], [-1.2120676040649414, 0.6494922637939453, -1.7295782566070557, 0.2986627221107483, -0.7859243750572205, -0.10322856158018112, -0.6283469200134277, 0.7088385820388794, 1.322974681854248, 0.29603174328804016, -1.0060392618179321, -0.08539615571498871, -1.3594313859939575, -2.4176902770996094, 0.12756936252117157, 0.38068047165870667, -0.91008460521698, 0.002672482980415225, -0.04392004758119583, -0.8426980972290039, -0.9130293726921082, -0.025842798873782158, 0.5816428065299988, 1.0986531972885132, 0.39145514369010925, 0.7380170226097107, -1.4038331508636475, -1.3520005941390991, 0.6645359992980957, -1.2096714973449707, -0.6510354280471802, 0.2867007851600647, 0.8583133816719055, 0.8945786952972412, -0.3836454749107361, 0.468523234128952, 2.0307819843292236, 0.37245655059814453, 1.060547113418579, -1.0211269855499268, -1.100699782371521, 1.4606682062149048, 0.5242165327072144, 0.8691370487213135, -1.6984246969223022, -2.638676881790161, 0.03080044686794281, -0.4149455726146698, -0.2504291832447052, -0.5746155381202698, 1.0549793243408203, 0.47744378447532654, 1.7973991632461548, -0.9096495509147644, -0.3227140009403229, -0.13451696932315826, -0.42088234424591064, 2.856851577758789, 1.0292963981628418, 0.5245506167411804, -0.5371926426887512, -0.9079551696777344, -2.0850069522857666, -1.6557585000991821], [1.48731529712677, -0.4345776438713074, 0.3613467514514923, 0.8643251061439514, -2.197402000427246, 0.3389036953449249, 1.1320533752441406, -1.2814130783081055, -0.6208911538124084, -1.5836894512176514, -1.9882844686508179, 0.0022190995514392853, -0.38683080673217773, 1.0104882717132568, 1.0024493932724, -0.16466820240020752, 1.130163311958313, 0.6545385718345642, 1.7192281484603882, 1.4566140174865723, -0.6041373014450073, 1.9719364643096924, 0.8945356607437134, -0.7342209219932556, -1.778987169265747, 0.3683503568172455, 0.2274673730134964, -1.1464083194732666, 0.31683114171028137, 2.335076093673706, 0.9457080960273743, -1.8271437883377075, -0.22689183056354523, -0.2992511987686157, -0.45797455310821533, -0.3293705880641937, -0.3317406475543976, 0.8416281342506409, -0.7243439555168152, -0.15906254947185516, -0.8317772150039673, -1.1626076698303223, -1.3342163562774658, -0.44414588809013367, 0.037974096834659576, -0.18865545094013214, 2.063314437866211, 0.6440221667289734, -0.8066678643226624, 2.288508415222168, 1.3564568758010864, -1.171634554862976, 1.4285507202148438, -1.152898907661438, -0.026333967223763466, 0.33161717653274536, 0.6162164211273193, -0.5590810179710388, 0.5057956576347351, 0.30211424827575684, -0.30958321690559387, 0.27419760823249817, 0.30397137999534607, 0.18316225707530975], [-0.7922418117523193, -1.181965708732605, -0.13090930879116058, 0.9758108854293823, 0.7478390336036682, 2.1813831329345703, 2.349515914916992, -1.1602091789245605, -1.218732476234436, -1.103071689605713, 0.47750595211982727, -0.9114797711372375, 0.8208231925964355, -1.3531502485275269, -0.36717158555984497, -0.6692163348197937, -0.8183422088623047, 0.9733860492706299, 0.454222708940506, -1.137001872062683, -1.0501372814178467, 1.0033860206604004, 0.04216624051332474, 0.03060969151556492, -0.23344601690769196, -0.7699293494224548, 2.883819580078125, 0.508497953414917, -0.6910054087638855, 0.7385743856430054, 0.2378004491329193, 0.0003857812553178519, -0.17077453434467316, -0.5855147838592529, -0.4461878538131714, -1.7479370832443237, 1.7832345962524414, 2.2117466926574707, 1.2401021718978882, -1.005082607269287, -0.6470393538475037, 1.494524359703064, 0.8221380710601807, 0.4370344877243042, -1.1000499725341797, -1.0477491617202759, 1.255324125289917, 0.11983102560043335, -0.935347855091095, -2.0659537315368652, 1.5308603048324585, -0.41651588678359985, -1.2884187698364258, -0.05240415036678314, -1.3184007406234741, 0.5026088356971741, -1.2561591863632202, -0.2778390049934387, -2.145725727081299, -0.5164003968238831, -1.1357921361923218, -0.3676931858062744, 0.9510999917984009, 0.861781895160675], [0.3359469175338745, 1.244585394859314, -0.7295312285423279, -0.9887392520904541, 1.148601770401001, 0.0012989359674975276, -1.7451603412628174, -1.815014123916626, -0.28310659527778625, 1.8824976682662964, -0.06262164562940598, -0.8988449573516846, 0.6487563848495483, 0.6088430285453796, 1.8563889265060425, -1.7182183265686035, 2.000383138656616, 0.009843943640589714, 0.5807315707206726, 2.2699813842773438, 0.7363042831420898, 0.08203806728124619, 0.43421801924705505, -0.44385215640068054, -1.0568619966506958, -1.516505479812622, -1.4657427072525024, -0.7875732779502869, -1.8867067098617554, -0.7557993531227112, 2.044092893600464, 0.359360009431839, -2.387854814529419, -0.345485657453537, 0.15020103752613068, 0.34058114886283875, 0.625632643699646, -1.5233278274536133, 0.3228168189525604, 0.5766950845718384, 0.1084413230419159, -0.06159384548664093, -0.36275380849838257, -0.5115762948989868, 0.2390671968460083, -0.1434325873851776, -0.8470759391784668, -0.9222463965415955, -2.0242156982421875, 0.7371552586555481, 1.1880767345428467, -0.581394374370575, 1.6014389991760254, 1.100539207458496, -0.05674423649907112, -1.0440618991851807, -0.37988579273223877, -1.9199150800704956, 0.06974037736654282, 0.2909139096736908, -0.1565067023038864, 1.2828341722488403, -1.1116100549697876, -1.8374346494674683], [0.1655832976102829, -1.763810157775879, 1.458625078201294, -0.32280653715133667, 0.5343409776687622, 0.07247921079397202, 0.00240227859467268, -0.9826515316963196, -1.4164223670959473, -0.18427687883377075, -0.33124032616615295, 0.7507110834121704, 0.0648527592420578, 3.563298225402832, 0.949830949306488, -1.6927876472473145, 0.3487931787967682, -0.05120568722486496, 2.6270344257354736, -0.1337510347366333, -0.3148338496685028, -1.1405142545700073, 0.2304186224937439, 0.3848130404949188, 0.6951789259910583, -0.4257126748561859, -0.8432270288467407, 1.296546220779419, -0.21474522352218628, -0.9033794403076172, 0.6301528811454773, 1.6115275621414185, -0.6332997679710388, 0.30823302268981934, 0.4004955291748047, -2.2941722869873047, 0.7246740460395813, -1.342728614807129, -1.0246773958206177, 0.48517483472824097, 0.04159156605601311, 1.73343825340271, -0.7444857954978943, 0.9491186141967773, -1.7668023109436035, -0.984320342540741, -0.82405024766922, 1.4308844804763794, 0.49168938398361206, 0.06905114650726318, -0.40329599380493164, -0.4204418957233429, 0.3097444176673889, -0.2689228057861328, 1.4489576816558838, 0.7856079936027527, 0.9471412301063538, 0.8131369948387146, -0.792937695980072, -0.48119235038757324, -0.580651044845581, 0.9371213316917419, 2.05790114402771, -1.8183590173721313], [0.43523627519607544, 0.5625103712081909, -0.21339797973632812, -0.45415738224983215, -0.9195059537887573, 0.17652449011802673, 1.2584058046340942, 0.7772361636161804, -2.9645304679870605, -2.466975688934326, -0.2907254695892334, 1.4538545608520508, 0.2231423556804657, 0.8517555594444275, 0.3945610225200653, -0.04022981598973274, -1.5782973766326904, -0.5692254900932312, -0.6765251159667969, 1.2131667137145996, 0.40207353234291077, 0.10512244701385498, 0.2715143859386444, 0.38285601139068604, -0.07153027504682541, -0.6734410524368286, -2.2633697986602783, -1.562956690788269, -0.6956251859664917, -1.6425645351409912, -0.4042893350124359, 0.6252009272575378, 0.31482192873954773, 0.14000007510185242, 0.4763847589492798, -0.15480248630046844, 0.6526604890823364, -1.1937259435653687, 0.5352431535720825, -0.4138876497745514, -1.4445794820785522, 1.2144170999526978, 0.4054468870162964, -1.911102533340454, -1.024153709411621, 1.3246814012527466, 0.4732552170753479, -2.2607622146606445, -0.23188894987106323, 1.0528916120529175, -0.2823069095611572, -0.3435870409011841, -0.09265919029712677, 0.8146039247512817, 0.04438291862607002, 0.511604905128479, 1.0389039516448975, -0.05748613923788071, -1.5582164525985718, 0.6185095906257629, 1.544312834739685, -0.3276195824146271, 0.17975029349327087, 0.21151147782802582], [-1.0759713649749756, 0.38520580530166626, -0.4685728847980499, -0.15503093600273132, -0.4414218068122864, -2.4265310764312744, -0.9510751366615295, -0.5035111308097839, 0.6238594055175781, -0.8673586249351501, -1.6202940940856934, 0.062088873237371445, -1.186166524887085, -0.4850134253501892, 0.5505208969116211, 0.9274881482124329, -0.1481567919254303, -0.8234906792640686, -0.3723835051059723, -1.845357894897461, 1.100740909576416, 0.8810153603553772, -0.002623653970658779, 0.6124134063720703, -1.4372907876968384, -0.7699332237243652, 1.3551331758499146, 0.8439428806304932, -1.1965917348861694, 0.6047784686088562, -1.3623746633529663, -0.6742733716964722, -0.2198168784379959, 0.5312091708183289, 0.4965271055698395, 1.0624805688858032, -0.5517519116401672, 1.724762201309204, 2.162310838699341, -0.11051683127880096, -1.5263540744781494, -1.0760961771011353, 0.3452169895172119, -0.3347579538822174, 0.7947940230369568, -0.7388173341751099, -0.14604303240776062, -0.8924309611320496, -0.702032208442688, -1.77492094039917, 1.6046111583709717, 0.7742415070533752, 0.32492026686668396, -1.5936734676361084, -1.4467658996582031, -0.2679276466369629, 0.08887839317321777, 0.486747682094574, -0.9904703497886658, -0.08090400695800781, -0.6087045669555664, 0.5146785974502563, 1.2874619960784912, 1.0321069955825806], [1.1789354085922241, -0.44243523478507996, 1.0979571342468262, -0.6152939796447754, -0.5412255525588989, -0.5066548585891724, 0.3310129940509796, 0.450759619474411, 0.4042391777038574, 0.777944028377533, -1.6679843664169312, -0.7540982961654663, 0.07207559049129486, 0.23821303248405457, -1.5235519409179688, -2.834602117538452, 0.8113420009613037, -0.371684193611145, -0.14082327485084534, 0.15598678588867188, -0.7522332072257996, -1.3620445728302002, -0.37569186091423035, -0.7001377940177917, 1.2189351320266724, -1.3860772848129272, 1.0451960563659668, -1.5135915279388428, -1.0728355646133423, -1.2743911743164062, 0.33978497982025146, 1.122406244277954, 0.6892522573471069, -0.6707316040992737, -0.6173789501190186, 0.3471987843513489, -0.306484580039978, -1.5812734365463257, -0.4740085303783417, -0.45739102363586426, -0.36191651225090027, -0.9897106289863586, 0.03456631675362587, 1.0068877935409546, 0.9901683330535889, -1.7632108926773071, -2.043210029602051, -0.7302117943763733, 2.3583998680114746, -0.12064532935619354, 1.317313313484192, 0.5067875385284424, -0.8583232760429382, -1.231028437614441, -0.6065053343772888, 0.18404312431812286, -0.5671141147613525, -0.17289474606513977, 1.0806645154953003, 0.6885601282119751, -0.010809788480401039, 0.06210758909583092, 0.30858495831489563, -1.0192981958389282], [-0.291686475276947, 2.3683764934539795, 1.0145875215530396, 0.8986137509346008, -0.8877911567687988, -0.13251850008964539, -0.7545868158340454, 2.034396171569824, 1.5814673900604248, 1.0819729566574097, -0.32791149616241455, 0.34058713912963867, -0.693020761013031, -0.41850805282592773, -0.4396449029445648, -0.15532584488391876, 0.07502425462007523, -1.075295090675354, -1.1419671773910522, 1.3774865865707397, -0.6555546522140503, 0.12731966376304626, 0.14919936656951904, 0.41699886322021484, 0.801291823387146, 3.087733030319214, -1.967136263847351, -0.9586904048919678, 1.6990491151809692, -1.6908495426177979, -2.1583290100097656, 1.2315696477890015, 0.6144797205924988, 0.6308344602584839, -1.1030566692352295, -2.0396950244903564, -1.2315764427185059, 3.1737704277038574, 1.0972310304641724, -0.651327908039093, 2.589353084564209, -0.12645192444324493, -0.733984112739563, -0.4447003901004791, -1.6352204084396362, 0.8739778399467468, 0.16411912441253662, 0.8902482390403748, -0.5627276301383972, 1.3324308395385742, -1.6777002811431885, 1.0164928436279297, 0.8641127943992615, -0.6487624049186707, -0.7091285586357117, 0.565910816192627, -1.2900338172912598, -0.3599904179573059, -0.7019692063331604, 1.5109970569610596, -0.8248934149742126, -0.3169247806072235, -0.24441945552825928, -0.03825821354985237], [0.7952659726142883, -0.1652614176273346, -0.23448796570301056, -0.41918742656707764, 1.243301272392273, -0.5376797318458557, 1.579474925994873, -1.3879152536392212, -0.02550435997545719, 0.43106499314308167, -0.11495061963796616, 1.3409146070480347, 0.28697487711906433, -1.325626254081726, 0.8257302641868591, 0.015898263081908226, -1.0439766645431519, -0.3167405128479004, 0.9022573232650757, 1.350619912147522, 1.3031965494155884, 0.5110377669334412, 0.03950054571032524, 0.8826654553413391, 0.6325737237930298, -0.5793702006340027, -0.7923598885536194, 0.2497047483921051, -0.09346086531877518, -0.39633703231811523, -1.2053470611572266, 0.43532949686050415, -0.5452427268028259, 0.4893653690814972, -0.3822953999042511, -0.3119625151157379, 0.8756542205810547, 1.624958872795105, -0.2284456342458725, 0.007414609659463167, -1.369726538658142, 0.40484797954559326, -0.05271105840802193, 1.1864213943481445, -1.0409959554672241, 0.6146268844604492, 1.8834606409072876, 0.6883020401000977, 0.7080949544906616, -0.23436854779720306, 0.7710458040237427, -1.3887121677398682, 1.5176633596420288, 2.326855182647705, 0.04452645033597946, 0.3372933566570282, 0.28869467973709106, -0.8523188829421997, -1.365804672241211, -1.8711243867874146, 0.7854153513908386, 0.2953707277774811, 1.9025086164474487, -1.9221235513687134], [-0.38497933745384216, 0.31259357929229736, -0.6277384757995605, 0.39448922872543335, -0.38517990708351135, 0.40253040194511414, -2.051766872406006, -1.1210989952087402, -1.6314711570739746, -0.26929548382759094, 0.4677882492542267, -0.7009813785552979, 0.5358653664588928, 0.7267922759056091, -0.6825883388519287, 0.17337901890277863, 0.6321101188659668, -0.30569660663604736, -1.1334599256515503, 0.7635881900787354, 2.633904218673706, -1.082214117050171, -0.5668436884880066, 0.2964678704738617, 1.0823205709457397, -0.6341980695724487, -1.0165845155715942, 1.6083532571792603, -0.047822579741477966, 0.24267315864562988, 2.807063102722168, -1.290223240852356, -1.3550776243209839, 0.6651382446289062, -1.4036744832992554, 0.42101186513900757, 0.8094975352287292, 0.39794817566871643, 1.660935401916504, 0.7019261717796326, -0.013973316177725792, 1.2127724885940552, -0.23943153023719788, -0.2233610898256302, 0.5575894713401794, -1.5181576013565063, -0.28715214133262634, -0.45258164405822754, 0.380522221326828, -0.6722116470336914, -0.46946296095848083, -0.11020923405885696, 0.49035611748695374, -1.3400346040725708, -0.5221797823905945, 0.10261697322130203, -1.3498848676681519, 1.2300807237625122, 0.6810116171836853, 1.7957606315612793, -0.05276034027338028, 0.23544643819332123, -1.2966738939285278, 2.13960337638855], [0.9595680236816406, -0.08622090518474579, 0.019455045461654663, 0.8094500303268433, 0.8753104209899902, 1.906605839729309, 0.45513179898262024, -0.6490171551704407, -1.4912357330322266, -0.4829266667366028, 0.251412570476532, -0.43936917185783386, 0.7207179069519043, 1.1647554636001587, 0.03174485266208649, -0.8073902130126953, -0.29842087626457214, -0.15358257293701172, 0.3084877133369446, -0.15620721876621246, -1.8959981203079224, -0.39409494400024414, 0.6452283263206482, -0.627346932888031, -0.3807252049446106, 2.6497926712036133, -0.31007444858551025, -2.7115371227264404, -0.5048575401306152, -0.8735126852989197, -0.9590675830841064, 0.5231752395629883, -0.3524523973464966, 0.10181336104869843, -1.3948580026626587, -0.5918375849723816, -2.262822151184082, 2.156470537185669, -1.2465171813964844, -0.5837334394454956, 0.38781803846359253, 0.5623561143875122, -1.9372867345809937, -0.35438302159309387, 0.42990803718566895, -0.3002009987831116, -0.39242637157440186, -0.8695212602615356, 0.3480885922908783, -0.5723089575767517, -0.4769570827484131, 1.754491925239563, -1.9854826927185059, -0.41845500469207764, -0.0695427879691124, 0.7170159816741943, -0.5426803231239319, -0.15873974561691284, 0.1625450849533081, -1.6427712440490723, -0.35976698994636536, 1.1381419897079468, 1.0746616125106812, 0.8566325902938843], [1.3095371723175049, 0.2988162040710449, -0.19490614533424377, -0.6808376908302307, 0.3308844566345215, 1.6061105728149414, 1.236817479133606, -0.19533294439315796, 0.1076713353395462, 0.9446232318878174, 1.06084406375885, 1.330500602722168, -0.3338034749031067, -0.6151017546653748, 0.23462006449699402, 0.5346651077270508, -0.6818456649780273, -1.1438515186309814, 0.28193625807762146, -1.5950977802276611, -2.4353907108306885, -0.29843002557754517, 1.6595933437347412, -0.4622820019721985, -0.47850871086120605, -0.4227825105190277, 0.2984481751918793, -1.1699106693267822, 0.10567401349544525, -1.0472159385681152, -1.6841188669204712, -0.37310791015625, -0.11294993758201599, 1.5016872882843018, -1.5569384098052979, -0.06332091242074966, 0.0554133802652359, 1.7240574359893799, 1.817244291305542, 1.4335006475448608, -1.1260862350463867, -0.15105916559696198, 3.2155938148498535, 1.6254605054855347, -0.3911444842815399, -0.781146228313446, 0.43410637974739075, -0.9875398874282837, 1.0202765464782715, -0.6201590895652771, 0.8611266016960144, -1.8285696506500244, 0.9328784942626953, 0.2900281250476837, 1.707410216331482, -1.014646053314209, -0.049045808613300323, -0.7862211465835571, 0.8087397217750549, -0.13845643401145935, 1.3661569356918335, 1.684255838394165, 0.3427085876464844, 0.7393924593925476], [0.7639765739440918, 1.074670433998108, -0.8179081082344055, -1.2729685306549072, 0.7478671073913574, -0.2758277654647827, -1.001766562461853, -0.03670923411846161, -0.7025889158248901, -0.5753227472305298, 0.29576170444488525, -2.3653311729431152, 0.0991709977388382, 0.7668958306312561, -0.13582603633403778, 0.10684617608785629, -0.9430207014083862, 0.1773575395345688, 0.8785650134086609, 0.06626737117767334, 0.7994179725646973, 0.933002233505249, -0.7612165808677673, -0.06740135699510574, 0.4980207085609436, -0.1485130339860916, 0.1385861039161682, -0.20327775180339813, -0.16777074337005615, 0.8190354704856873, -0.521034300327301, -1.1536626815795898, 2.5530474185943604, 1.7250263690948486, 0.9327877163887024, -1.0466697216033936, -2.0998568534851074, -0.685957133769989, 0.5229571461677551, 0.7834556102752686, -0.6687882542610168, 0.0832347422838211, 1.0543309450149536, 0.41502445936203003, 0.0663740485906601, 1.68538236618042, -1.211496353149414, -1.3490861654281616, 0.16668039560317993, 1.3427386283874512, 1.1687078475952148, 0.35855376720428467, -0.20234571397304535, -1.1258273124694824, -0.6164645552635193, -0.3361782431602478, 0.3565721809864044, -0.6747019290924072, -0.6791474223136902, 0.557927131652832, -0.3058668076992035, -1.5166490077972412, -0.06123563274741173, -0.41388365626335144]], "dtype": "float32"}], "e": {"__nd__": [[0.8999031186103821, -0.8698101043701172, 1.1923428773880005, -1.9655067920684814, -0.7337416410446167, -0.340660959482193, 0.6057178974151611, -1.089009165763855, -0.5155795812606812, 0.6676793098449707, 1.5303953886032104, -0.13700881600379944, -1.3493880033493042, -1.0479254722595215, 1.1218878030776978, 1.3333178758621216, 1.4016379117965698, -0.35577183961868286, 0.33637985587120056, 0.49820196628570557, -0.05505608394742012, -1.2927535772323608, -1.789583444595337, 0.6979082822799683, 0.20838885009288788, -0.6639066338539124, 0.352510541677475, -0.5791202187538147, 0.7704298496246338, -1.1812398433685303, -1.3708492517471313, -0.17544561624526978, -0.27161484956741333, 1.085580825805664, 0.6340168714523315, -1.8622945547103882, -0.8124166131019592, -1.1873646974563599, -0.6334829926490784, -0.9042505621910095, -1.309715747833252, -0.6011978387832642, 1.8169363737106323, -0.4856294095516205, -1.2485790252685547, -0.2503089904785156, 0.1137014850974083, 1.2912312746047974, 1.0490366220474243, 1.0563734769821167, -1.2120676040649414, 1.48731529712677, -0.7922418117523193, 0.3359469175338745, 0.1655832976102829, 0.43523627519607544, -1.0759713649749756, 1.1789354085922241, -0.291686475276947, 0.7952659726142883, -0.38497933745384216, 0.9595680236816406, 1.3095371723175049, 0.7639765739440918], [-0.4583752751350403, 0.17317888140678406, 1.4240317344665527, -0.20679813623428345, -1.3539620637893677, -0.08864977955818176, -0.15176734328269958, 1.8226481676101685, 0.18819868564605713, -0.4114547371864319, -0.01700490713119507, -1.7071185111999512, -2.1075692176818848, 0.5509051084518433, -0.6482441425323486, -0.26146236062049866, -0.08112846314907074, -1.3108446598052979, 0.2500740587711334, 0.5129743218421936, -2.0198400020599365, -0.9435593485832214, -0.920840859413147, 0.255918949842453, -0.3676600456237793, 2.2239181995391846, 1.0944455862045288, 1.099886178970337, -0.37895503640174866, -1.8035709857940674, -1.378656029701233, -0.3007773458957672, 0.7141802906990051, 1.144005298614502, -0.5666196942329407, 0.13047955930233002, -1.4190102815628052, 1.4508296251296997, -0.15557312965393066, 0.42656898498535156, 0.14208142459392548, -0.020903071388602257, -0.3671073615550995, 1.3691530227661133, -0.7256300449371338, -1.241460919380188, -0.4138297140598297, -1.0667694807052612, -0.43739596009254456, 1.025160312652588, 0.6494922637939453, -0.4345776438713074, -1.181965708732605, 1.244585394859314, -1.763810157775879, 0.5625103712081909, 0.38520580530166626, -0.44243523478507996, 2.3683764934539795, -0.1652614176273346, 0.31259357929229736, -0.08622090518474579, 0.2988162040710449, 1.074670433998108], [-1.3140853643417358, 0.25686517357826233, -1.1406865119934082, 0.09442973881959915, -0.45091667771339417, -1.288060188293457, 0.7778471112251282, 1.1215150356292725, 0.6934308409690857, 1.2054307460784912, -1.1904680728912354, -1.4293266534805298, -1.2414186000823975, -0.9176278710365295, 1.9496804475784302, 0.8939447402954102, -0.20107074081897736, -0.4365514814853668, 1.3591388463974, -0.6230924725532532, -1.8138787746429443, -1.2080312967300415, -0.9281929731369019, -0.021805427968502045, 1.6633070707321167, 0.21247488260269165, -1.0603115558624268, -0.035009145736694336, 1.637758731842041, -1.6437913179397583, -2.095831871032715, 0.9156194925308228, -0.062568299472332, 0.4837956130504608, -0.4418620467185974, -0.7632626295089722, -1.0102483034133911, -1.3742297887802124, -1.120652675628662, 1.1022566556930542, 1.006695032119751, -0.10028854012489319, 0.4259355962276459, -0.21753084659576416, 0.6946454644203186, 1.2265700101852417, 0.43363022804260254, 0.10151305794715881, 1.663769006729126, 0.7964121103286743, -1.7295782566070557, 0.3613467514514923, -0.13090930879116058, -0.7295312285423279, 1.458625078201294, -0.21339797973632812, -0.4685728847980499, 1.0979571342468262, 1.0145875215530396, -0.23448796570301056, -0.6277384757995605, 0.019455045461654663, -0.19490614533424377, -0.8179081082344055], [0.6586976647377014, 0.5076766014099121, -1.5961689949035645, -0.10538601130247116, 0.23689255118370056, 0.7480973601341248, -2.188178300857544, 1.3628349304199219, 0.06432077288627625, 0.5911117196083069, -2.1827242374420166, -0.8566039800643921, 0.3184157907962799, 1.0885967016220093, -0.4865599572658539, 0.5393257141113281, 0.8807601928710938, 0.3914957344532013, 1.2648379802703857, -0.49698272347450256, 1.4165327548980713, -0.3977668583393097, 0.6046157479286194, -0.19099211692810059, -0.018315935507416725, 0.2475304901599884, 1.1162198781967163, -0.6068607568740845, -1.077773928642273, 2.114628553390503, -0.0691559687256813, -0.9888858795166016, 1.5313307046890259, -0.5743128657341003, -0.8041290044784546, 1.2823892831802368, 0.07199376821517944, 0.34418976306915283, 0.1579175442457199, -0.3991999626159668, 0.70172518491745, -0.5430030226707458, 0.21883925795555115, 2.0714495182037354, -0.9892962574958801, 0.007894786074757576, 0.7406242489814758, -0.3949592113494873, 0.4400269091129303, -1.3578532934188843, 0.2986627221107483, 0.8643251061439514, 0.9758108854293823, -0.9887392520904541, -0.32280653715133667, -0.45415738224983215, -0.15503093600273132, -0.6152939796447754, 0.8986137509346008, -0.41918742656707764, 0.39448922872543335, 0.8094500303268433, -0.6808376908302307, -1.2729685306549072], [0.9554258584976196, 0.7218186855316162, 0.3878454864025116, 0.5674481987953186, -0.20710228383541107, -0.684158980846405, -1.0612519979476929, 0.009091629646718502, 1.6394416093826294, 0.12867727875709534, -0.5535383224487305, -0.7225134968757629, 0.6318437457084656, -0.7233878970146179, 0.010171583853662014, -0.9474541544914246, 0.4938201904296875, 2.1902589797973633, 1.0045390129089355, 0.1296909749507904, -0.2320563644170761, 0.487494558095932, -0.49416065216064453, -0.45064279437065125, 0.31944897770881653, 0.6061961054801941, 0.14490365982055664, 0.24506793916225433, 0.43484577536582947, -0.6022742390632629, -0.41842585802078247, -1.8128583431243896, 0.8119798302650452, -0.5039026141166687, -0.8639131188392639, -0.5077354311943054, -0.34911245107650757, 1.310017704963684, 0.36152297258377075, -0.2379390150308609, 0.9722415804862976, 0.01554049551486969, 1.5929914712905884, -0.9945359826087952, -0.2594333589076996, -1.6700396537780762, -1.0598641633987427, -0.8640904426574707, -0.4579814374446869, 1.417523741722107, -0.7859243750572205, -2.197402000427246, 0.7478390336036682, 1.148601770401001, 0.5343409776687622, -0.9195059537887573, -0.4414218068122864, -0.5412255525588989, -0.8877911567687988, 1.243301272392273, -0.38517990708351135, 0.8753104209899902, 0.3308844566345215, 0.7478671073913574], [0.5246310234069824, -1.7291629314422607, 0.7894242405891418, -0.0816906988620758, -0.2102249711751938, 1.2231075763702393, 0.7438154816627502, 0.7252418994903564, 1.1536107063293457, -1.2888044118881226, -0.3185899257659912, 1.216440200805664, 1.1161473989486694, 1.569963812828064, 2.970968008041382, 0.8511086702346802, -0.04028857499361038, 1.455220103263855, 1.777077078819275, 0.9923047423362732, -0.9649589657783508, 2.5930190086364746, -0.8374435901641846, -0.1313873827457428, -0.6153111457824707, -1.4521735906600952, 0.22394491732120514, -0.2784549593925476, -0.05734919384121895, -2.1312084197998047, -0.3496246635913849, 0.2562413513660431, -1.1985061168670654, 0.8465922474861145, 0.9051325917243958, 0.7208672761917114, 0.03120686672627926, 0.20195774734020233, 0.012591136619448662, 0.08623664081096649, -0.8592473268508911, 0.10107069462537766, 0.15204942226409912, 0.9617483615875244, 1.338673710823059, -0.7983174920082092, 0.229350283741951, -0.6457409858703613, 0.21640408039093018, -0.16682256758213043, -0.10322856158018112, 0.3389036953449249, 2.1813831329345703, 0.0012989359674975276, 0.07247921079397202, 0.17652449011802673, -2.4265310764312744, -0.5066548585891724, -0.13251850008964539, -0.5376797318458557, 0.40253040194511414, 1.906605839729309, 1.6061105728149414, -0.2758277654647827], [-2.558269500732422, 1.4759763479232788, -0.4591422379016876, -0.2834911048412323, 0.9449697732925415, 0.10027065128087997, -0.44453996419906616, -2.0238122940063477, 0.26563167572021484, -0.3759533762931824, 1.7100979089736938, -0.8717086315155029, 1.088407039642334, -0.751186728477478, 0.7807389497756958, 1.8847929239273071, 0.22751930356025696, 0.9997928738594055, 0.8857879638671875, -2.2392148971557617, 1.0070873498916626, 0.1864939033985138, 0.5765594244003296, 0.41438692808151245, 1.487412929534912, -0.5143530964851379, 0.6653387546539307, 0.12435968965291977, 1.444725751876831, -0.6494404077529907, 0.08096820116043091, -0.2672420144081116, -0.8411993980407715, -0.9432703852653503, -1.2173289060592651, 1.323121428489685, -1.2435638904571533, 0.1554163098335266, -0.8080368041992188, -0.360666424036026, -0.4835117757320404, 0.08623089641332626, -2.1361606121063232, 0.014530928805470467, -1.4280219078063965, -0.4583938717842102, -0.9108262658119202, -0.5671964287757874, -1.8854526281356812, -2.263535737991333, -0.6283469200134277, 1.1320533752441406, 2.349515914916992, -1.7451603412628174, 0.00240227859467268, 1.2584058046340942, -0.9510751366615295, 0.3310129940509796, -0.7545868158340454, 1.579474925994873, -2.051766872406006, 0.45513179898262024, 1.236817479133606, -1.001766562461853], [-1.5353412628173828, -0.013240644708275795, -1.1156907081604004, -1.1433733701705933, -0.2974686622619629, -0.8516147136688232, -0.6817123889923096, -0.1765352487564087, -0.6244698762893677, 1.6365784406661987, 2.3178536891937256, -1.494749903678894, -2.3566079139709473, 0.9699144959449768, -1.9831628799438477, 0.24231652915477753, -1.2422022819519043, -1.157912015914917, 0.24294352531433105, -2.616438388824463, -0.7442213892936707, 0.2801334857940674, 0.37432289123535156, -0.634353518486023, -2.3068196773529053, -1.4138424396514893, -0.14154288172721863, 0.9012701511383057, 0.0979139655828476, 0.6762714385986328, -0.14532925188541412, -1.227473497390747, -0.731806218624115, -0.6749463677406311, -1.7691930532455444, -2.210054397583008, -0.5446205735206604, 1.0745983123779297, -0.46157991886138916, -0.2719908356666565, -1.098766803741455, 1.115495204925537, 0.7942264080047607, 0.947394609451294, 1.1843323707580566, 0.05360901355743408, 0.45690053701400757, -1.7500213384628296, 0.4417535364627838, 0.9802421927452087, 0.7088385820388794, -1.2814130783081055, -1.1602091789245605, -1.815014123916626, -0.9826515316963196, 0.7772361636161804, -0.5035111308097839, 0.450759619474411, 2.034396171569824, -1.3879152536392212, -1.1210989952087402, -0.6490171551704407, -0.19533294439315796, -0.03670923411846161], [0.6136307120323181, -0.4972066581249237, 1.8323572874069214, 1.1357825994491577, 0.1296825259923935, 0.6974550485610962, -0.45675763487815857, -0.1620151400566101, 0.08083276450634003, 2.461146593093872, 0.013037207536399364, 0.4862147867679596, 1.9803273677825928, 0.04679206758737564, 0.1812298446893692, 0.48922330141067505, 1.3364770412445068, 0.960088849067688, 1.485949993133545, 0.6562344431877136, 0.710289478302002, 0.562183141708374, 0.3760901391506195, -1.1304303407669067, -2.049015760421753, 1.9953510761260986, -0.14901958405971527, 1.127774715423584, -0.06382862478494644, 0.3991154730319977, 1.437666654586792, -0.402447372674942, -0.46005794405937195, 0.346348375082016, -1.3990808725357056, -0.46826601028442383, -0.21722254157066345, -1.5146942138671875, 0.21639692783355713, 0.9519915580749512, -0.7759919166564941, -0.056531719863414764, 0.2651371359825134, -0.5744219422340393, 0.31568512320518494, 1.570939064025879, -0.8669142723083496, -0.5317729115486145, -0.6903690695762634, 0.6105175018310547, 1.322974681854248, -0.6208911538124084, -1.218732476234436, -0.28310659527778625, -1.4164223670959473, -2.9645304679870605, 0.6238594055175781, 0.4042391777038574, 1.5814673900604248, -0.02550435997545719, -1.6314711570739746, -1.4912357330322266, 0.1076713353395462, -0.7025889158248901], [1.494365930557251, 0.25351643562316895, -0.6859242916107178, -1.1710913181304932, -0.8770400285720825, 0.3599795401096344, -0.4625297784805298, -0.8795292973518372, -0.5325934290885925, 0.8043172955513, -0.3308594524860382, 0.6418257355690002, -1.195037603378296, -0.10635776817798615, 0.6112141609191895, -0.8519030213356018, -1.5608032941818237, -0.4760870337486267, -0.5768786668777466, 1.5428861379623413, -0.2076931893825531, -1.180060625076294, 1.4074956178665161, 1.5309072732925415, -1.137660264968872, -0.6346685886383057, 0.19632019102573395, -0.07393535226583481, -0.3386208117008209, -1.4838532209396362, 0.5193088054656982, 0.6951013803482056, 0.019985433667898178, 1.0394617319107056, 1.7209707498550415, 1.2202192544937134, -0.934386134147644, -0.6049176454544067, -0.09482179582118988, 1.3257395029067993, 1.0016374588012695, 0.2520125210285187, 0.14422249794006348, -1.0477172136306763, 1.02118980884552, -1.1148244142532349, -1.0755616426467896, -0.41117021441459656, 0.47022807598114014, -1.5634444952011108, 0.29603174328804016, -1.5836894512176514, -1.103071689605713, 1.8824976682662964, -0.18427687883377075, -2.466975688934326, -0.8673586249351501, 0.777944028377533, 1.0819729566574097, 0.43106499314308167, -0.26929548382759094, -0.4829266667366028, 0.9446232318878174, -0.5753227472305298], [-1.4945822954177856, -0.5871797204017639, 0.36959096789360046, 0.3445030450820923, -1.1862388849258423, 0.43590420484542847, 0.704009473323822, -0.922248363494873, -1.2438243627548218, -0.04201611131429672, 1.3968191146850586, 1.2611799240112305, 1.3977832794189453, 0.2956945598125458, -0.4037332832813263, -0.9525532126426697, 1.522441029548645, 1.9024279117584229, 0.8962158560752869, 0.062160421162843704, -1.116620421409607, -1.2601020336151123, -0.3568952679634094, 1.590489387512207, -1.6164379119873047, 0.2459847331047058, -1.8851860761642456, -1.0988141298294067, 0.9276825785636902, 0.1026792898774147, -1.4708075523376465, 1.3002431392669678, -0.10329987853765488, 0.5261312127113342, -0.7921659350395203, -0.15905001759529114, -0.1185380145907402, -0.7351235151290894, -0.22171662747859955, 0.6603543162345886, 0.16032874584197998, 0.08887916803359985, -1.2239391803741455, -1.7359687089920044, -3.09879469871521, -0.8898813724517822, 0.13132734596729279, -0.6758841872215271, -0.3895149230957031, 2.191577911376953, -1.0060392618179321, -1.9882844686508179, 0.47750595211982727, -0.06262164562940598, -0.33124032616615295, -0.2907254695892334, -1.6202940940856934, -1.6679843664169312, -0.32791149616241455, -0.11495061963796616, 0.4677882492542267, 0.251412570476532, 1.06084406375885, 0.29576170444488525], [0.9703254699707031, -1.2591904401779175, 0.2881930470466614, -0.09653632342815399, 0.51100093126297, 0.05157991871237755, 1.447244644165039, -1.7394911050796509, 0.15908955037593842, 0.8933999538421631, -1.1285423040390015, -0.9460441470146179, 0.1043664887547493, 0.9242297410964966, 0.2940548360347748, -0.4582779109477997, 0.5861697793006897, -0.8372438549995422, -1.8457896709442139, -0.40183451771736145, 0.03305329382419586, 0.8675949573516846, -0.2807236611843109, -0.10195150971412659, -0.8252266645431519, 0.7155243754386902, -0.835516631603241, -1.4582359790802002, -0.37600061297416687, -1.2792972326278687, 0.4889136254787445, -2.2080814838409424, 0.8273008465766907, 1.860133171081543, 2.2339189052581787, 0.09857494384050369, -0.5104355216026306, 1.084187388420105, 1.8458296060562134, -0.2601564824581146, -0.2587509751319885, -0.03696179389953613, 0.8267810940742493, -0.5857177972793579, 1.5253710746765137, -0.9318233132362366, -0.408875048160553, -0.4630023241043091, -0.4373016059398651, 0.7925490140914917, -0.08539615571498871, 0.0022190995514392853, -0.9114797711372375, -0.8988449573516846, 0.7507110834121704, 1.4538545608520508, 0.062088873237371445, -0.7540982961654663, 0.34058713912963867, 1.3409146070480347, -0.7009813785552979, -0.43936917185783386, 1.330500602722168, -2.3653311729431152], [1.1190851926803589, -0.37412479519844055, 0.3097412884235382, -0.4260581433773041, -0.972798228263855, -0.5600208044052124, -1.3964546918869019, 0.3160697817802429, 1.307270884513855, -0.18157339096069336, -0.4913652241230011, -0.3007318675518036, -0.3818780481815338, 0.2530613839626312, 1.3554388284683228, 0.3089849054813385, -0.4113505184650421, -1.1885597705841064, -0.09838689118623734, -0.1819300800561905, -0.9776239991188049, 0.5094172954559326, 1.914574384689331, -0.21800914406776428, -0.6303850412368774, 0.3113636076450348, -0.20279796421527863, 1.1060227155685425, 1.1527037620544434, -1.2131582498550415, -1.2152122259140015, 0.10351710021495819, 0.9678579568862915, 0.5382035374641418, -0.6516728401184082, -0.7462381720542908, 0.6091113686561584, 0.8891127705574036, 2.478273391723633, 1.022500991821289, -0.5948218703269958, -0.42896780371665955, -0.9230107069015503, 0.13518157601356506, -0.7671810984611511, 0.04167100787162781, -1.1348172426223755, 0.9152907133102417, 0.2726907730102539, 0.8642694354057312, -1.3594313859939575, -0.38683080673217773, 0.8208231925964355, 0.6487563848495483, 0.0648527592420578, 0.2231423556804657, -1.186166524887085, 0.07207559049129486, -0.693020761013031, 0.28697487711906433, 0.5358653664588928, 0.7207179069519043, -0.3338034749031067, 0.0991709977388382], [0.017410578206181526, -1.9415563344955444, 1.2666064500808716, -1.315321683883667, 0.13335035741329193, 1.4960922002792358, -1.1432619094848633, 0.14133892953395844, -0.40861353278160095, -1.2520462274551392, 1.455488920211792, -0.5822508931159973, 0.5468370914459229, -0.6040695309638977, -0.6453508734703064, 0.4178001880645752, -0.715015709400177, 0.2792993485927582, 0.9826818108558655, -0.01330346055328846, -1.3432561159133911, 0.938113272190094, 1.3255313634872437, -1.0979174375534058, -0.028483035042881966, 0.3626692593097687, 0.9377351999282837, -0.44124269485473633, 0.08764442056417465, 0.6953432559967041, 0.9030336141586304, 0.20042270421981812, 0.9882715940475464, 0.7095589637756348, -0.7325198650360107, 0.3020547926425934, -0.04846624284982681, -0.35399699211120605, 0.1350417584180832, 0.7164359092712402, 0.5651354193687439, -0.7431235313415527, 1.123164415359497, -0.8992565870285034, -0.9793148636817932, -0.1704513281583786, -1.1636073589324951, -0.9752108454704285, 1.0872281789779663, 0.22017601132392883, -2.4176902770996094, 1.0104882717132568, -1.3531502485275269, 0.6088430285453796, 3.563298225402832, 0.8517555594444275, -0.4850134253501892, 0.23821303248405457, -0.41850805282592773, -1.325626254081726, 0.7267922759056091, 1.1647554636001587, -0.6151017546653748, 0.7668958306312561], [-1.339133381843567, -1.127770185470581, 1.3789180517196655, -1.6966569423675537, 0.8983229994773865, -1.552392840385437, -0.6047202944755554, 0.45134201645851135, -0.9304059147834778, -0.2233319878578186, -0.04435897618532181, 0.36769694089889526, -0.7387341260910034, -0.002265071962028742, -0.49437010288238525, 0.43185341358184814, -1.796675682067871, 1.240312099456787, 1.8390215635299683, 0.11034388840198517, -2.6633312702178955, 0.14628809690475464, -1.792946696281433, -0.023040322586894035, -1.4813190698623657, 0.7222578525543213, -0.8374294638633728, -0.5298418402671814, -0.7881914973258972, 1.0922316312789917, -1.2275418043136597, -1.2821789979934692, -1.7561088800430298, -0.4374868869781494, 2.583237648010254, -0.28252890706062317, 1.269591212272644, 1.950961709022522, 0.9387320876121521, -1.0427571535110474, 1.0164674520492554, -1.3119257688522339, 1.4918309450149536, 0.44859060645103455, 0.2448863536119461, -3.1053059101104736, -1.4616849422454834, 0.13157658278942108, -0.01909545063972473, 1.0302220582962036, 0.12756936252117157, 1.0024493932724, -0.36717158555984497, 1.8563889265060425, 0.949830949306488, 0.3945610225200653, 0.5505208969116211, -1.5235519409179688, -0.4396449029445648, 0.8257302641868591, -0.6825883388519287, 0.03174485266208649, 0.23462006449699402, -0.13582603633403778], [-0.6403135657310486, 0.7236483693122864, -1.4739190340042114, -0.09251436591148376, -0.4918181598186493, -1.041038990020752, 0.23877355456352234, -0.6857211589813232, 0.08489415794610977, -1.5462781190872192, 0.03992754966020584, 1.2012457847595215, -1.9393256902694702, 0.2286248803138733, 0.7139834761619568, 0.9222221970558167, -1.5641100406646729, -0.8726211190223694, -1.9730743169784546, 1.6083965301513672, 2.1998233795166016, -1.6217007637023926, -0.09823252260684967, -2.2529568672180176, 0.6765457391738892, 0.76004958152771, 0.6385728120803833, -0.9505932927131653, 0.3407512903213501, -2.2884328365325928, -0.8061491847038269, -0.729926586151123, -0.029074518010020256, 1.7808033227920532, 0.7035468816757202, -0.048897720873355865, 1.8234503269195557, 0.9877452850341797, -0.10986661911010742, 0.48402711749076843, 1.1092535257339478, 0.33971694111824036, 0.3459322452545166, 1.0830693244934082, 1.2907251119613647, -0.10652477294206619, -1.4716285467147827, 0.9673910737037659, 0.607519268989563, -2.156918525695801, 0.38068047165870667, -0.16466820240020752, -0.6692163348197937, -1.7182183265686035, -1.6927876472473145, -0.04022981598973274, 0.9274881482124329, -2.834602117538452, -0.15532584488391876, 0.015898263081908226, 0.17337901890277863, -0.8073902130126953, 0.5346651077270508, 0.10684617608785629], [-0.36200037598609924, 0.4756447672843933, -0.12562070786952972, 0.6657434105873108, 2.117896318435669, 1.2759385108947754, 0.6882261633872986, 0.8102511763572693, -0.6191263198852539, -0.11772065609693527, 0.10061267018318176, -0.45043936371803284, -0.2706186771392822, -0.5424455404281616, -0.38205093145370483, 0.5821750164031982, 1.4067883491516113, -1.1538082361221313, -1.2194558382034302, -1.5473889112472534, -1.7679386138916016, 0.07194868475198746, 0.32115840911865234, 0.8131577968597412, 0.7064183950424194, -0.4648844599723816, 0.12492170184850693, -0.639820396900177, -0.2344415783882141, 0.6799390912055969, -0.3852047622203827, 0.40107157826423645, 0.2534950077533722, -1.1994032859802246, 0.0368322879076004, 1.230759859085083, -0.12896761298179626, -0.13326673209667206, 0.8777616024017334, 0.8427267670631409, -0.4605601727962494, -0.29489365220069885, 0.3403257131576538, 0.8345542550086975, 0.06015590578317642, -0.6032898426055908, 0.7339748740196228, -1.231221079826355, -2.464831829071045, -0.2931259274482727, -0.91008460521698, 1.130163311958313, -0.8183422088623047, 2.000383138656616, 0.3487931787967682, -1.5782973766326904, -0.1481567919254303, 0.8113420009613037, 0.07502425462007523, -1.0439766645431519, 0.6321101188659668, -0.29842087626457214, -0.6818456649780273, -0.9430207014083862], [0.6909022927284241, -1.151097297668457, 1.0913655757904053, 0.05454370751976967, 1.154932975769043, 1.7062774896621704, -0.08775756508111954, 2.148977518081665, 0.9043327569961548, 0.21148231625556946, -0.8511046767234802, -1.3434103727340698, -1.1248379945755005, 0.8305144906044006, -0.22394582629203796, -0.13951098918914795, 0.7288343906402588, 0.29970884323120117, -0.22949755191802979, 0.7326810956001282, -1.0564184188842773, -2.0524168014526367, 2.112454414367676, 0.9166340231895447, 0.6991832852363586, 0.5418962240219116, 0.4615686535835266, -0.7663716673851013, -0.4014061987400055, 0.18415388464927673, 0.4848991632461548, 0.11248844861984253, -0.5133402347564697, 0.2578565776348114, 0.6401119232177734, -1.9943397045135498, -0.996695339679718, -0.2075616717338562, 1.5539265871047974, 0.21236921846866608, 1.7539442777633667, -0.22762563824653625, 0.1072925478219986, -0.06412634253501892, -1.6980342864990234, -0.399217814207077, -0.6436705589294434, 1.7662363052368164, -1.7101333141326904, -0.144461527466774, 0.002672482980415225, 0.6545385718345642, 0.9733860492706299, 0.009843943640589714, -0.05120568722486496, -0.5692254900932312, -0.8234906792640686, -0.371684193611145, -1.075295090675354, -0.3167405128479004, -0.30569660663604736, -0.15358257293701172, -1.1438515186309814, 0.1773575395345688], [0.8993356227874756, -2.2454750537872314, 0.17375719547271729, -1.1645617485046387, -0.774941623210907, -1.9293212890625, 0.3833174705505371, -0.5179665088653564, -0.24656984210014343, 0.8776394724845886, -0.6384986042976379, 0.08650703728199005, 0.534800112247467, -0.96439129114151, 0.44720879197120667, 0.8311170339584351, 0.7648867964744568, -1.222969651222229, -1.3033612966537476, 0.2484724223613739, -2.084948778152466, 0.5233405232429504, -0.01061192061752081, -1.255744218826294, 1.1151520013809204, 1.5209969282150269, 0.04279348999261856, 1.2306078672409058, 1.1215636730194092, 0.40973392128944397, -0.41770437359809875, 1.5171936750411987, 1.834403157234192, 0.003710675286129117, -1.3064908981323242, 0.8300361633300781, -0.2065039873123169, 0.9928106069564819, 0.34063130617141724, 2.0899717807769775, 0.6694806218147278, 0.7245655655860901, 2.093668222427368, -1.2746319770812988, 1.4137508869171143, -0.7045793533325195, 0.07269831001758575, -1.823956847190857, 1.657670021057129, 0.41816744208335876, -0.04392004758119583, 1.7192281484603882, 0.454222708940506, 0.5807315707206726, 2.6270344257354736, -0.6765251159667969, -0.3723835051059723, -0.14082327485084534, -1.1419671773910522, 0.9022573232650757, -1.1334599256515503, 0.3084877133369446, 0.28193625807762146, 0.8785650134086609], [-0.986848771572113, 0.17502057552337646, -0.6511379480361938, 0.9531631469726562, 2.13230562210083, 1.7610541582107544, -0.30538854002952576, 0.6146741509437561, 0.07291946560144424, 2.142606735229492, -0.06595558673143387, -1.3707393407821655, -0.03687746077775955, -1.3908461332321167, 1.068036675453186, -0.7039746046066284, 0.5806168913841248, -0.34200412034988403, -0.468979150056839, 0.5079293847084045, -1.9199737310409546, 0.2769964933395386, 0.5304145216941833, 0.1477547436952591, -0.670414388179779, -0.4043193459510803, 0.30709534883499146, 1.4097703695297241, 0.33226141333580017, -0.12326642125844955, 0.4540065824985504, -0.05625048279762268, 1.673601746559143, 1.5860525369644165, -0.6730167269706726, 3.7575411796569824, 1.108787178993225, 0.831153392791748, -0.35331013798713684, 0.9459696412086487, -0.676331102848053, 0.7632038593292236, -0.47250956296920776, 0.6428699493408203, 0.15094105899333954, 0.6333852410316467, -2.344944715499878, -0.3850262761116028, -0.8716546893119812, 1.680561900138855, -0.8426980972290039, 1.4566140174865723, -1.137001872062683, 2.2699813842773438, -0.1337510347366333, 1.2131667137145996, -1.845357894897461, 0.15598678588867188, 1.3774865865707397, 1.350619912147522, 0.7635881900787354, -0.15620721876621246, -1.5950977802276611, 0.06626737117767334], [-0.10213501006364822, 0.36982715129852295, 0.7755193114280701, -2.605194568634033, 0.5126007199287415, 0.6014499068260193, 0.37029939889907837, -2.6837375164031982, -0.187511146068573, -0.23554906249046326, -1.65879225730896, -2.541722536087036, -0.23443321883678436, -0.06986145675182343, 1.0667312145233154, 1.8731147050857544, -1.5306833982467651, -1.1053526401519775, 0.03609301894903183, -2.182305097579956, 0.8787534236907959, -0.8159721493721008, -2.3519225120544434, -1.6343077421188354, -3.7679977416992188, 2.213106870651245, 0.2576027512550354, 0.7234137654304504, -0.8036428689956665, 0.3602060079574585, 2.6646270751953125, 1.162052869796753, 0.49277228116989136, -1.7977465391159058, 0.06121606379747391, -0.46699193120002747, 0.6029279828071594, 0.08828262239694595, 0.20970883965492249, 0.9011909365653992, -0.07376173138618469, 0.6924785375595093, 1.3616018295288086, -0.5165280103683472, 1.046862244606018, -1.212041974067688, -0.14216892421245575, 0.5621654987335205, -1.6895188093185425, -0.46954694390296936, -0.9130293726921082, -0.6041373014450073, -1.0501372814178467, 0.7363042831420898, -0.3148338496685028, 0.40207353234291077, 1.100740909576416, -0.7522332072257996, -0.6555546522140503, 1.3031965494155884, 2.633904218673706, -1.8959981203079224, -2.4353907108306885, 0.7994179725646973], [-0.03492661938071251, -0.05980798602104187, -0.5232162475585938, -0.13056007027626038, 1.0842838287353516, -2.00620174407959, -2.07387638092041, 1.754376769065857, 0.09801522642374039, 1.0518031120300293, -0.3902221918106079, 0.28454267978668213, -0.8537653088569641, -1.0823297500610352, 2.111229181289673, -0.7294390797615051, 0.6830989122390747, -0.7955606579780579, 1.4470247030258179, 0.8615733981132507, 1.5946288108825684, -0.8666795492172241, 0.04478298872709274, -0.27313435077667236, 0.5549207329750061, -1.053531289100647, -0.8271039724349976, -0.719792366027832, 1.5453014373779297, -0.18213045597076416, -0.6805050373077393, -1.5830533504486084, -0.5848450660705566, 0.6933899521827698, -1.7836315631866455, -0.8483969569206238, -0.5478240847587585, 0.16622860729694366, 0.37985625863075256, 0.3307592272758484, 1.2833938598632812, 1.2822744846343994, -1.2418612241744995, -0.7929400205612183, 1.0392628908157349, 0.29239121079444885, 0.0062080384232103825, -0.16995283961296082, -0.7895626425743103, 0.4291965663433075, -0.025842798873782158, 1.9719364643096924, 1.0033860206604004, 0.08203806728124619, -1.1405142545700073, 0.10512244701385498, 0.8810153603553772, -1.3620445728302002, 0.12731966376304626, 0.5110377669334412, -1.082214117050171, -0.39409494400024414, -0.29843002557754517, 0.933002233505249], [-0.1748647689819336, -1.0016217231750488, -0.18898096680641174, 0.9154669642448425, -0.4711441099643707, -0.6632086634635925, 0.7120949625968933, 1.4886592626571655, -0.8479596972465515, -0.08785352855920792, 0.703824520111084, 0.9492698311805725, -1.1263427734375, 0.9213810563087463, 0.8928000926971436, 0.04164654389023781, -0.8719853758811951, 0.33235424757003784, -0.892238974571228, -2.32364821434021, -1.158696174621582, -0.9094231724739075, -0.860512375831604, -0.60235196352005, 1.4494202136993408, -0.2759469151496887, 3.2502903938293457, -0.5610424876213074, -0.4230853319168091, 0.3457046449184418, 1.0918060541152954, 0.13167443871498108, -0.5493894815444946, 0.07299965620040894, 1.8410627841949463, -1.5228526592254639, -0.5212636590003967, 0.6299760341644287, -0.3012240529060364, 0.9071002006530762, 1.4656051397323608, 0.33860987424850464, -0.8002734184265137, -0.2462347447872162, -0.32319456338882446, -0.18179109692573547, 0.05107325315475464, -0.26935964822769165, -0.9290140271186829, -1.074721336364746, 0.5816428065299988, 0.8945356607437134, 0.04216624051332474, 0.43421801924705505, 0.2304186224937439, 0.2715143859386444, -0.002623653970658779, -0.37569186091423035, 0.14919936656951904, 0.03950054571032524, -0.5668436884880066, 0.6452283263206482, 1.6595933437347412, -0.7612165808677673], [0.9459331631660461, 0.05884239077568054, 0.8250610828399658, 1.0347535610198975, 0.3590772747993469, -1.1056885719299316, 0.045047517865896225, 0.06065455451607704, 0.14680016040802002, 0.8981180191040039, -0.19587136805057526, -1.7157375812530518, 1.00685453414917, 1.228393316268921, 0.4409791827201843, -0.9582239389419556, -1.8474117517471313, 0.5915045738220215, 0.5959594249725342, -1.1983821392059326, -0.12082263082265854, -0.8983151912689209, 1.9200011491775513, 0.1792914867401123, -0.27025195956230164, 0.8472574949264526, -0.19153210520744324, -0.35913681983947754, -1.2981587648391724, -0.3708389401435852, -1.6112881898880005, -0.47474050521850586, 0.47072291374206543, 0.7408111691474915, -0.7834659218788147, -0.9330482482910156, -0.23756884038448334, 0.2295009344816208, 0.20893658697605133, -0.9317687749862671, -0.450529545545578, -0.7426949143409729, -1.8122563362121582, 0.3125182092189789, -1.2166028022766113, 1.3543646335601807, -0.24638426303863525, 1.415971040725708, 1.508525013923645, -0.4903208911418915, 1.0986531972885132, -0.7342209219932556, 0.03060969151556492, -0.44385215640068054, 0.3848130404949188, 0.38285601139068604, 0.6124134063720703, -0.7001377940177917, 0.41699886322021484, 0.8826654553413391, 0.2964678704738617, -0.627346932888031, -0.4622820019721985, -0.06740135699510574], [-0.2716423571109772, -0.36620163917541504, 0.5654955506324768, -0.14629866182804108, -0.23570898175239563, 0.07043773680925369, -1.1478885412216187, 0.7225856184959412, -0.6895408630371094, 0.17291632294654846, -2.3835530281066895, -0.006869330070912838, -0.1887790411710739, -1.919921875, -2.1375954151153564, 1.4264137744903564, -0.663332462310791, 0.462802916765213, -1.5505070686340332, -0.8048321008682251, 0.4983322024345398, 0.3509356677532196, -1.3756797313690186, 1.3425698280334473, 0.739774227142334, 0.1256166249513626, 0.21993862092494965, -0.9751121401786804, 0.4916646182537079, 0.8817909955978394, -0.2455417662858963, -1.6676690578460693, -0.44195806980133057, -0.06671296805143356, 0.21497222781181335, 0.9225545525550842, -0.9419698715209961, 1.2608602046966553, -0.7905332446098328, -1.2943469285964966, -1.0701922178268433, 0.7297540903091431, 0.03704371303319931, 0.3748677670955658, 0.29989761114120483, -2.4340851306915283, 0.44922059774398804, 1.1519259214401245, 1.4036284685134888, 0.8734135627746582, 0.39145514369010925, -1.778987169265747, -0.23344601690769196, -1.0568619966506958, 0.6951789259910583, -0.07153027504682541, -1.4372907876968384, 1.2189351320266724, 0.801291823387146, 0.6325737237930298, 1.0823205709457397, -0.3807252049446106, -0.47850871086120605, 0.4980207085609436], [-0.0700131207704544, 0.1824212670326233, 1.880777359008789, 1.2095682621002197, 0.3278779685497284, -0.2917822599411011, -0.06779567152261734, -1.5005122423171997, -0.7100067138671875, 0.061453528702259064, 0.12895779311656952, -1.038689136505127, 0.7323111295700073, 0.04241737723350525, -0.001538573415018618, 0.5558668971061707, -0.5502601861953735, -1.1798439025878906, 1.156083583831787, -0.4721190929412842, 0.27485665678977966, 0.5026132464408875, 0.008779442869126797, 1.0641388893127441, 0.03226511552929878, 1.580712914466858, 0.18383772671222687, 0.5336808562278748, 0.02014278434216976, -1.5168603658676147, -0.6864802837371826, 0.7157323360443115, 0.0907803401350975, -2.286343812942505, -0.2512907087802887, -0.4808100163936615, -1.0419690608978271, 0.591141402721405, 0.7114876508712769, 0.7911126613616943, -0.6440037488937378, 0.6514515280723572, -1.5730395317077637, -0.11998848617076874, 0.5807323455810547, -0.20719781517982483, -2.09222412109375, -0.933907151222229, -0.23755072057247162, 0.1788935512304306, 0.7380170226097107, 0.3683503568172455, -0.7699293494224548, -1.516505479812622, -0.4257126748561859, -0.6734410524368286, -0.7699332237243652, -1.3860772848129272, 3.087733030319214, -0.5793702006340027, -0.6341980695724487, 2.6497926712036133, -0.4227825105190277, -0.1485130339860916], [-1.4837714433670044, 0.5660898685455322, 0.6875735521316528, 1.728361964225769, -0.6708052158355713, 1.2332061529159546, -1.2625044584274292, 1.131286859512329, 0.29836875200271606, -0.12160354107618332, 0.7925776243209839, -0.6974465250968933, 0.6601270437240601, -0.3029714822769165, -0.65567547082901, -0.7786097526550293, 1.2292487621307373, 0.05416422709822655, 0.45588386058807373, 0.09090957790613174, 0.45786553621292114, -0.7427237033843994, -2.1717987060546875, 0.1138862892985344, 1.3589082956314087, 1.057334303855896, 0.16326306760311127, -1.0024164915084839, -1.212384581565857, 0.7180557250976562, -1.2117727994918823, 1.5018305778503418, 0.26928067207336426, -0.6706362962722778, 2.9512500762939453, 0.10179721564054489, -1.16730535030365, -0.43176138401031494, 0.7298285961151123, -1.8438796997070312, 0.34168320894241333, 1.1964656114578247, 0.574286937713623, -0.9997764229774475, 0.5006057024002075, 0.9852440357208252, -0.670089602470398, -0.5798577070236206, -0.5631338357925415, -1.1942930221557617, -1.4038331508636475, 0.2274673730134964, 2.883819580078125, -1.4657427072525024, -0.8432270288467407, -2.2633697986602783, 1.3551331758499146, 1.0451960563659668, -1.967136263847351, -0.7923598885536194, -1.0165845155715942, -0.31007444858551025, 0.2984481751918793, 0.1385861039161682], [0.14615555107593536, -0.04666045308113098, 0.003970490302890539, -0.27340182662010193, -0.4871984124183655, 0.18745777010917664, 1.1089059114456177, -1.0014513731002808, -1.6988108158111572, -0.8075142502784729, -0.2523709535598755, 0.058501001447439194, -0.7505186796188354, 0.6972513794898987, -1.6248083114624023, -0.794025719165802, -1.1766968965530396, 0.24310925602912903, 0.5596024394035339, -0.9109734892845154, 1.1695462465286255, 0.13776817917823792, 0.7402613162994385, 0.5967855453491211, -0.12282413244247437, -0.02894216775894165, -1.5419727563858032, -0.16312481462955475, -1.024172306060791, -0.8634173274040222, 1.2038301229476929, 0.26352810859680176, 0.4881858825683594, -0.4070431590080261, 0.8258538842201233, -2.2259693145751953, -0.29786044359207153, -1.450405478477478, -0.2809576988220215, 0.9963704347610474, -0.6756145358085632, -0.17519479990005493, 0.4278097450733185, -1.0564124584197998, -0.3911826014518738, 0.45120495557785034, 1.245070219039917, 1.9327162504196167, -0.5158030986785889, -0.9327466487884521, -1.3520005941390991, -1.1464083194732666, 0.508497953414917, -0.7875732779502869, 1.296546220779419, -1.562956690788269, 0.8439428806304932, -1.5135915279388428, -0.9586904048919678, 0.2497047483921051, 1.6083532571792603, -2.7115371227264404, -1.1699106693267822, -0.20327775180339813], [-0.9387549757957458, -0.2123861461877823, 1.0332063436508179, 1.104597806930542, -0.7413039207458496, 0.8555373549461365, 1.073300838470459, -0.3416120111942291, -0.7515841722488403, -0.8923971652984619, 1.3760493993759155, -0.37954795360565186, -0.27323904633522034, -0.19374139606952667, 0.02314361184835434, 0.9583896994590759, -1.116884469985962, -1.1842577457427979, -0.34148871898651123, 0.6957644820213318, -1.6934912204742432, 1.419980764389038, -0.9781771898269653, -1.7611325979232788, 0.38222771883010864, 0.4695548713207245, -2.391817808151245, 0.0009374450310133398, -1.1282484531402588, -1.9698121547698975, -0.2591373324394226, -1.7888318300247192, 2.25488018989563, -0.18183347582817078, 0.8299887776374817, 0.05433781072497368, -1.102216362953186, -1.0189845561981201, 0.27354779839515686, -0.644330620765686, 0.29671505093574524, 0.6640900373458862, -1.0498722791671753, 0.5960147380828857, 1.3174676895141602, -0.08513346314430237, -0.2361678183078766, 0.6194162368774414, 1.3873507976531982, -0.25638964772224426, 0.6645359992980957, 0.31683114171028137, -0.6910054087638855, -1.8867067098617554, -0.21474522352218628, -0.6956251859664917, -1.1965917348861694, -1.0728355646133423, 1.6990491151809692, -0.09346086531877518, -0.047822579741477966, -0.5048575401306152, 0.10567401349544525, -0.16777074337005615], [1.8856043815612793, 0.5310394167900085, -0.04454539716243744, 0.06376875191926956, -2.210026502609253, 0.997447669506073, 0.0848100483417511, -0.5470161437988281, -0.2632159888744354, 2.1207849979400635, 0.3249971270561218, -1.1146408319473267, 1.2058132886886597, 0.8257158994674683, 0.45823681354522705, -1.3961421251296997, -0.22315442562103271, -0.4464147686958313, 1.210231065750122, 0.40681755542755127, 0.5562804937362671, -0.4051627516746521, 0.5742835402488708, -0.44235482811927795, 2.450981855392456, -0.11845706403255463, -1.1703927516937256, 0.031466878950595856, 1.3797791004180908, -1.1364762783050537, -0.28814172744750977, 0.08597050607204437, -1.3602641820907593, 1.3550323247909546, -0.8573771715164185, -0.26724621653556824, 0.43141257762908936, -0.739018440246582, 1.4109400510787964, 0.819264829158783, 1.5783675909042358, -1.2036164999008179, -0.5534213781356812, 0.26356688141822815, 0.3667182922363281, -0.28626638650894165, -0.1035064160823822, 0.38547512888908386, -1.198851227760315, -1.0826427936553955, -1.2096714973449707, 2.335076093673706, 0.7385743856430054, -0.7557993531227112, -0.9033794403076172, -1.6425645351409912, 0.6047784686088562, -1.2743911743164062, -1.6908495426177979, -0.39633703231811523, 0.24267315864562988, -0.8735126852989197, -1.0472159385681152, 0.8190354704856873], [0.14862462878227234, 0.26019051671028137, 0.06852097064256668, 1.0828369855880737, -0.8632765412330627, -1.104976773262024, -1.7510160207748413, 0.9327576160430908, -0.23131847381591797, -0.4519121050834656, 0.14536908268928528, 1.9081367254257202, -0.8728693127632141, -0.05092581734061241, -0.6269470453262329, 0.2524065375328064, 0.9783003926277161, 0.6097335815429688, 0.7127923965454102, 0.7599347233772278, 1.2203669548034668, 0.258024662733078, 0.8983065485954285, -0.14458729326725006, -2.2131237983703613, -1.211267352104187, -0.36263731122016907, -1.7540545463562012, 0.9831665754318237, -0.6883548498153687, -1.6071385145187378, -0.31659382581710815, -1.5005815029144287, -0.14667661488056183, -1.9652012586593628, 0.09602277725934982, 0.9966529011726379, -0.30841341614723206, -0.5502831935882568, -1.5665993690490723, 0.8688259124755859, 0.5920833945274353, -1.0351301431655884, -0.41538000106811523, 0.17916399240493774, -0.6989208459854126, 3.6640148162841797, 1.1952109336853027, -1.6062389612197876, -0.38776034116744995, -0.6510354280471802, 0.9457080960273743, 0.2378004491329193, 2.044092893600464, 0.6301528811454773, -0.4042893350124359, -1.3623746633529663, 0.33978497982025146, -2.1583290100097656, -1.2053470611572266, 2.807063102722168, -0.9590675830841064, -1.6841188669204712, -0.521034300327301], [0.1838734894990921, -0.8776059150695801, 0.13934198021888733, -1.1844961643218994, 0.270134299993515, -0.14799921214580536, -2.063983678817749, 0.22310537099838257, 0.47007840871810913, -0.1141229122877121, -0.38748422265052795, -0.15841223299503326, -0.08649203181266785, 0.7098743915557861, 0.7082387208938599, -0.9657363891601562, -2.2504019737243652, 1.0225083827972412, -1.7253141403198242, 0.3173638880252838, 0.9696736335754395, -0.9395707249641418, -0.9302676916122437, -0.1922607570886612, -0.2488706111907959, 1.9116370677947998, 1.0515587329864502, 0.6035351157188416, -0.7361054420471191, 0.9740656018257141, -1.43028724193573, -1.4124858379364014, -0.09589898586273193, -0.1749134659767151, 1.6658973693847656, 0.20396873354911804, 0.7729189991950989, 0.9905819296836853, 0.779044508934021, -0.5940564274787903, 0.4775187373161316, 0.6605721712112427, -0.4824362099170685, -1.13937246799469, 1.5232495069503784, 0.02041679620742798, 1.7282503843307495, -0.3759394586086273, -1.487395167350769, -1.165462851524353, 0.2867007851600647, -1.8271437883377075, 0.0003857812553178519, 0.359360009431839, 1.6115275621414185, 0.6252009272575378, -0.6742733716964722, 1.122406244277954, 1.2315696477890015, 0.43532949686050415, -1.290223240852356, 0.5231752395629883, -0.37310791015625, -1.1536626815795898], [0.0986730232834816, -0.10265259444713593, -0.41248008608818054, 0.7466009259223938, -1.2500736713409424, 0.47859784960746765, 0.0727948397397995, 0.24521410465240479, 1.118277668952942, -0.699027955532074, -1.5404058694839478, 1.0266318321228027, 0.8504368662834167, -0.9867958426475525, -1.1235982179641724, 0.37706106901168823, -0.026549089699983597, 1.207926630973816, 1.3332270383834839, -0.3694393038749695, 0.059353966265916824, -2.0066866874694824, 1.3348839282989502, 0.21417218446731567, 1.4983537197113037, 0.05762818083167076, 1.8410060405731201, -2.1358070373535156, 0.0031751771457493305, 1.4669350385665894, -1.4036364555358887, 0.8543019890785217, -0.5117193460464478, -0.5882591605186462, -0.9953504800796509, 0.20479841530323029, -0.6717371344566345, -0.6460029482841492, -0.04133983701467514, 1.2286591529846191, -0.6283763647079468, 1.4935624599456787, 0.6159270405769348, -1.6378142833709717, 0.6737160682678223, -1.4590871334075928, 0.6571133732795715, -0.4196530282497406, -0.3538319766521454, -1.1282427310943604, 0.8583133816719055, -0.22689183056354523, -0.17077453434467316, -2.387854814529419, -0.6332997679710388, 0.31482192873954773, -0.2198168784379959, 0.6892522573471069, 0.6144797205924988, -0.5452427268028259, -1.3550776243209839, -0.3524523973464966, -0.11294993758201599, 2.5530474185943604], [-1.1710097789764404, 0.6619710326194763, 0.26471850275993347, 1.2563724517822266, -1.0433697700500488, 0.2995857894420624, -1.1467972993850708, 1.0738639831542969, 0.05540956184267998, -1.239908218383789, -1.0179729461669922, -0.6831233501434326, 1.2942299842834473, 1.4491692781448364, -0.7026557922363281, 1.1542261838912964, 0.9600598812103271, -0.9957178831100464, 2.0510940551757812, 1.2629344463348389, 0.48526284098625183, -0.3979945182800293, 1.5149152278900146, 0.5368334054946899, -1.6479604244232178, 0.5327285528182983, 1.0539880990982056, -0.12762746214866638, 0.6935985088348389, 0.17532455921173096, -0.5248169898986816, 1.2984063625335693, -1.2149391174316406, -0.3224141299724579, -0.12151817977428436, -1.0090302228927612, 1.8373103141784668, -0.14006198942661285, 0.4400462508201599, 1.493628740310669, 1.4007364511489868, 1.4716242551803589, -0.3318521976470947, 0.1507745087146759, 0.8948745131492615, -0.18408291041851044, -0.1402338296175003, -0.07429429143667221, 1.117097020149231, -0.4040403664112091, 0.8945786952972412, -0.2992511987686157, -0.5855147838592529, -0.345485657453537, 0.30823302268981934, 0.14000007510185242, 0.5312091708183289, -0.6707316040992737, 0.6308344602584839, 0.4893653690814972, 0.6651382446289062, 0.10181336104869843, 1.5016872882843018, 1.7250263690948486], [-0.4353877604007721, 0.566676676273346, 0.1666335165500641, 1.1375545263290405, 1.3556078672409058, -0.657784104347229, 2.0426647663116455, -1.2151941061019897, 0.024942995980381966, 0.6202710866928101, -0.575279176235199, 0.8704742789268494, -0.7061967849731445, -1.5946506261825562, 1.1051366329193115, 0.02713397517800331, 1.7288378477096558, -0.8028660416603088, 0.5522165298461914, -0.25047004222869873, 0.48054903745651245, -0.7039204835891724, 0.594433069229126, -0.021568486467003822, 0.09928475320339203, -2.755430221557617, 2.069675922393799, 0.11102542281150818, 0.6855688095092773, -1.044979214668274, 0.34404894709587097, 0.13837644457817078, 0.24449960887432098, -0.980094850063324, -0.5166004300117493, -0.013549759984016418, -0.37766000628471375, -0.21028642356395721, -0.5696076154708862, 1.3109318017959595, -1.2984215021133423, 0.3876299262046814, -0.7905750870704651, 1.005835771560669, -0.33569005131721497, -0.34965381026268005, -0.6737102270126343, 0.9007154703140259, -1.6440178155899048, -0.0850614458322525, -0.3836454749107361, -0.45797455310821533, -0.4461878538131714, 0.15020103752613068, 0.4004955291748047, 0.4763847589492798, 0.4965271055698395, -0.6173789501190186, -1.1030566692352295, -0.3822953999042511, -1.4036744832992554, -1.3948580026626587, -1.5569384098052979, 0.9327877163887024], [0.11089904606342316, -0.32494738698005676, 0.17127445340156555, 0.09971880912780762, -1.7961822748184204, 0.4879797101020813, -1.507251262664795, 0.4829792380332947, -0.12039241939783096, -0.8065798282623291, -1.827839732170105, 0.5657032132148743, -1.2057045698165894, -1.4568476676940918, 0.5595276951789856, 0.5405804514884949, 0.4192456305027008, -0.08736041188240051, -0.1679176539182663, -0.330195814371109, 1.360451579093933, 1.2575139999389648, -1.1432583332061768, 0.4227941632270813, 0.8983843326568604, -0.9666861295700073, 1.0463762283325195, 0.15819939970970154, 1.1869772672653198, 0.7143205404281616, -0.3734819293022156, 0.8985907435417175, -0.23208093643188477, 1.8164191246032715, 1.321007251739502, -0.9637280106544495, 1.1194566488265991, 1.0718525648117065, 2.109607219696045, 0.5686963200569153, -0.0011815647594630718, 1.2174279689788818, 0.013681449927389622, 0.1357908993959427, -0.41660773754119873, 0.7216296195983887, 0.07283665984869003, 0.6696605086326599, 1.0789886713027954, 1.4715311527252197, 0.468523234128952, -0.3293705880641937, -1.7479370832443237, 0.34058114886283875, -2.2941722869873047, -0.15480248630046844, 1.0624805688858032, 0.3471987843513489, -2.0396950244903564, -0.3119625151157379, 0.42101186513900757, -0.5918375849723816, -0.06332091242074966, -1.0466697216033936], [0.9117034077644348, -0.16475775837898254, -0.9556545615196228, -1.9823657274246216, -0.24617260694503784, -1.1764971017837524, -1.5780147314071655, 0.9331173896789551, 0.04569206386804581, -1.3927572965621948, 0.600877583026886, -0.8921152353286743, -0.03805288299918175, 0.3270825743675232, 0.4839227795600891, 0.3851805031299591, 0.7869210243225098, -0.5841209292411804, 0.2850044369697571, 0.4560188353061676, -0.1990768164396286, -0.8063749074935913, 0.09890557825565338, -0.5910213589668274, -1.7217707633972168, -0.509304404258728, -0.6795795559883118, 0.5185714960098267, 2.457927942276001, 2.689962387084961, -0.2872104346752167, 1.06361985206604, 0.1697874367237091, 1.2524516582489014, -1.113547682762146, -0.08468464016914368, -1.077149510383606, -0.19503161311149597, -0.6122036576271057, -1.4720616340637207, 0.5408336520195007, 0.552365243434906, -1.450927734375, -1.0524604320526123, -1.5355099439620972, -0.14063112437725067, -0.7320036888122559, -0.7281674742698669, -0.5433033108711243, -0.7339051961898804, 2.0307819843292236, -0.3317406475543976, 1.7832345962524414, 0.625632643699646, 0.7246740460395813, 0.6526604890823364, -0.5517519116401672, -0.306484580039978, -1.2315764427185059, 0.8756542205810547, 0.8094975352287292, -2.262822151184082, 0.0554133802652359, -2.0998568534851074], [0.21966463327407837, -1.0546441078186035, 0.5732484459877014, 0.2861904799938202, 0.4725686311721802, -0.6216272711753845, 0.9184601306915283, 2.1121885776519775, -0.08779995143413544, -1.1554834842681885, -1.0819159746170044, -1.1389124393463135, -0.8950231075286865, 0.964439868927002, -0.34300899505615234, -0.8401992917060852, -0.2834038734436035, -1.1596986055374146, 2.28482723236084, 1.0703057050704956, -0.1440541297197342, -0.11140891164541245, -0.5026213526725769, -1.8227453231811523, -0.3000392019748688, -0.4227880835533142, -0.7563943266868591, 1.39644193649292, -0.897288978099823, -2.578665018081665, 0.43879565596580505, 0.7507938742637634, -0.6839438676834106, -0.1938246488571167, -0.9945765137672424, 1.1313281059265137, -0.11799504607915878, -0.5414747595787048, 1.0780926942825317, 0.6178896427154541, 1.2541484832763672, 0.2945398688316345, -0.21188265085220337, 1.5208725929260254, 1.6363356113433838, -1.7219278812408447, 0.3420828878879547, 1.30608332157135, -0.3179700970649719, 1.182477593421936, 0.37245655059814453, 0.8416281342506409, 2.2117466926574707, -1.5233278274536133, -1.342728614807129, -1.1937259435653687, 1.724762201309204, -1.5812734365463257, 3.1737704277038574, 1.624958872795105, 0.39794817566871643, 2.156470537185669, 1.7240574359893799, -0.685957133769989], [-1.3220744132995605, 0.22098210453987122, 0.10141777247190475, -2.318258762359619, 0.6061264872550964, 0.5223340392112732, 2.0231704711914062, 1.2370816469192505, 2.937380313873291, -0.1550513058900833, -1.6162034273147583, 0.5938212275505066, 0.7048787474632263, 0.2709067463874817, 0.7244430184364319, -0.4251210391521454, -0.2535204291343689, -0.19880369305610657, -0.2683456540107727, 0.8246799111366272, -1.8188444375991821, -0.12228339910507202, -0.23091302812099457, -0.5585258603096008, 0.912118673324585, 0.14325475692749023, 0.34195154905319214, -1.8108388185501099, -0.2037564069032669, 1.376423954963684, -0.8108862638473511, 1.4183677434921265, -0.663449227809906, -1.971239686012268, -0.15878714621067047, -0.6971340775489807, 1.8300775289535522, -0.37144431471824646, -0.7593865990638733, -0.18978016078472137, -1.8811681270599365, 1.5597779750823975, -0.1774272620677948, -1.9510256052017212, -1.6590888500213623, 0.797718346118927, -0.5979408621788025, -0.20198164880275726, 0.12425066530704498, -2.1911840438842773, 1.060547113418579, -0.7243439555168152, 1.2401021718978882, 0.3228168189525604, -1.0246773958206177, 0.5352431535720825, 2.162310838699341, -0.4740085303783417, 1.0972310304641724, -0.2284456342458725, 1.660935401916504, -1.2465171813964844, 1.817244291305542, 0.5229571461677551], [-0.21426066756248474, 1.0684579610824585, -0.15393570065498352, 0.14128269255161285, 0.3723907768726349, 0.6654309630393982, -1.669063687324524, -1.0399430990219116, -0.7223324775695801, 0.8607157468795776, 0.0171206071972847, -0.031848639249801636, 0.6489371061325073, -2.6652159690856934, -1.2352527379989624, -0.7389296293258667, 0.006409502122551203, 0.039243679493665695, -1.4254059791564941, -0.31268805265426636, -0.0011682405602186918, -0.274550199508667, 0.03197353705763817, 0.5273871421813965, 0.6295017600059509, -0.44659146666526794, -0.46421507000923157, -0.990541398525238, -0.743502140045166, 0.37085410952568054, 0.005336306523531675, 1.023157000541687, -2.0876529216766357, 0.605668306350708, -1.2645045518875122, -1.0856798887252808, 0.022798944264650345, -0.3351280987262726, -1.1546369791030884, -0.18092814087867737, -1.575905442237854, 0.4132893681526184, -1.429725170135498, -0.7427549958229065, 1.0900300741195679, -0.3235187232494354, 0.013463586568832397, 2.9270763397216797, -0.05559340864419937, 0.12064320594072342, -1.0211269855499268, -0.15906254947185516, -1.005082607269287, 0.5766950845718384, 0.48517483472824097, -0.4138876497745514, -0.11051683127880096, -0.45739102363586426, -0.651327908039093, 0.007414609659463167, 0.7019261717796326, -0.5837334394454956, 1.4335006475448608, 0.7834556102752686], [-0.13816185295581818, 0.4071141481399536, -0.1486535668373108, 0.20135077834129333, 0.2877083122730255, -1.0732537508010864, 0.803791880607605, 0.9409905672073364, 0.9342005848884583, -0.5249208211898804, -0.780884325504303, -1.4403527975082397, -1.7708303928375244, -1.3865383863449097, -0.05574558302760124, -1.179717779159546, -0.9464609622955322, 1.016602635383606, 1.4899109601974487, 0.8544406294822693, -0.4306592345237732, 2.056575059890747, -0.26873883605003357, 0.8045322299003601, 1.4861860275268555, -1.4629899263381958, -0.21407008171081543, -2.3956949710845947, 1.3265676498413086, -1.2945469617843628, -0.3300354480743408, 0.6626275181770325, -0.697304904460907, -0.6217864155769348, 0.705176055431366, 1.4179487228393555, -0.06551290303468704, 0.5114478468894958, 0.01904722861945629, -0.3438883423805237, -0.5601462125778198, 0.7978946566581726, 0.21854358911514282, -0.7636708617210388, -1.398013949394226, 0.8392775654792786, -0.7292788028717041, -0.41142794489860535, -2.264462471008301, -1.871583342552185, -1.100699782371521, -0.8317772150039673, -0.6470393538475037, 0.1084413230419159, 0.04159156605601311, -1.4445794820785522, -1.5263540744781494, -0.36191651225090027, 2.589353084564209, -1.369726538658142, -0.013973316177725792, 0.38781803846359253, -1.1260862350463867, -0.6687882542610168], [-1.6847881078720093, -0.46759316325187683, -0.6692723631858826, 0.6472941637039185, -0.07896576821804047, 0.2159944325685501, 0.38384389877319336, -1.5432658195495605, 0.9290662407875061, 2.4977715015411377, -0.8399583697319031, -1.1494170427322388, -0.24275313317775726, 0.24557794630527496, 0.08569443225860596, -2.665703773498535, 0.11732052266597748, 0.0021594504360109568, 0.6456810235977173, 0.32552021741867065, 1.2280662059783936, 0.0007191899348981678, 0.25820693373680115, -1.885867714881897, -1.713024616241455, 0.46694308519363403, -1.2966785430908203, -0.8946470618247986, 0.11827308684587479, -0.20865505933761597, -0.12913525104522705, 1.6546343564987183, -0.6779983043670654, -0.4065493941307068, -0.6546556949615479, 1.6853034496307373, 0.6136327385902405, 0.474945992231369, -0.20769155025482178, 1.7818210124969482, 2.255281686782837, -0.16306520998477936, -0.12850825488567352, 0.19022785127162933, 0.7475332021713257, 0.9606361985206604, -0.1334851235151291, 0.8790708184242249, 2.429957389831543, -0.5847625136375427, 1.4606682062149048, -1.1626076698303223, 1.494524359703064, -0.06159384548664093, 1.73343825340271, 1.2144170999526978, -1.0760961771011353, -0.9897106289863586, -0.12645192444324493, 0.40484797954559326, 1.2127724885940552, 0.5623561143875122, -0.15105916559696198, 0.0832347422838211], [0.5395525693893433, 0.6242401003837585, -0.6993617415428162, -0.5949511528015137, -1.1026699542999268, 0.5570297241210938, -0.4328260123729706, -0.42526742815971375, 0.6426178812980652, -1.2541661262512207, -0.5029340982437134, -1.4370230436325073, -0.8262097835540771, 0.06523028761148453, -0.26980888843536377, 1.4356073141098022, 0.23467959463596344, -1.1832267045974731, -0.0008382191881537437, -1.805362582206726, -0.7029686570167542, -1.0769264698028564, -0.5280998945236206, -1.4262117147445679, 0.6913756728172302, 0.6687488555908203, 0.5747382044792175, 0.647798478603363, 1.0294450521469116, -0.8598510026931763, 0.7253822684288025, -0.8233789205551147, 0.030023908242583275, 0.3026559054851532, -2.190046787261963, -0.3449119031429291, -1.609268307685852, 0.37564346194267273, -0.6423287391662598, 1.6354472637176514, -0.04707985743880272, -0.3453797996044159, 0.948599636554718, -0.7258796095848083, 0.7482397556304932, 1.2364064455032349, -0.1413317620754242, -1.9834808111190796, -0.2926662266254425, 0.5013070106506348, 0.5242165327072144, -1.3342163562774658, 0.8221380710601807, -0.36275380849838257, -0.7444857954978943, 0.4054468870162964, 0.3452169895172119, 0.03456631675362587, -0.733984112739563, -0.05271105840802193, -0.23943153023719788, -1.9372867345809937, 3.2155938148498535, 1.0543309450149536], [0.7637683749198914, -0.004906778689473867, 1.8623509407043457, 0.6877020597457886, -2.212771415710449, 0.9360799789428711, -1.3933830261230469, -0.36796969175338745, -0.10218202322721481, 1.0243734121322632, 1.6197550296783447, 0.8036302924156189, -0.033612336963415146, 0.12412471324205399, 1.271958589553833, -0.491498202085495, 0.9794695973396301, 1.138392448425293, 0.5424531102180481, -0.005000210367143154, 0.44393065571784973, -1.0662426948547363, 1.431689977645874, -0.617684543132782, -0.4565344452857971, -0.3058018386363983, 1.3843786716461182, -0.41940346360206604, -0.1681685447692871, -0.11789493262767792, 0.7893970012664795, 2.005817413330078, -0.8261248469352722, -0.5929405689239502, -1.2402119636535645, 0.7436985373497009, 1.218510389328003, 0.5163120627403259, -0.7379605770111084, -0.14776656031608582, 0.5321341753005981, 0.6241661906242371, -0.5846222639083862, -2.1682193279266357, 0.7281690835952759, 1.118633508682251, -1.2083656787872314, -0.4311593174934387, -0.7138704061508179, 0.0446026585996151, 0.8691370487213135, -0.44414588809013367, 0.4370344877243042, -0.5115762948989868, 0.9491186141967773, -1.911102533340454, -0.3347579538822174, 1.0068877935409546, -0.4447003901004791, 1.1864213943481445, -0.2233610898256302, -0.35438302159309387, 1.6254605054855347, 0.41502445936203003], [-0.9124293923377991, 0.6311420798301697, 0.17674580216407776, -0.3357521891593933, 1.6808733940124512, 0.6457073092460632, -0.7584786415100098, 0.6471886038780212, 0.5072893500328064, -1.7396317720413208, -0.6615617871284485, 0.5587061643600464, 1.1155413389205933, -0.2938699424266815, 1.0944594144821167, 0.46894386410713196, 1.7882970571517944, -0.7801547646522522, -0.5704626441001892, -0.2121913731098175, 0.25271859765052795, -0.7396999597549438, -2.526808261871338, -0.2348073571920395, -0.6052491068840027, -0.2474314272403717, -2.5175297260284424, 0.3882126212120056, 0.037038106471300125, -0.24957609176635742, -0.237039715051651, -0.6204504370689392, 0.9288634657859802, 0.21755218505859375, 0.9216005206108093, 1.4514081478118896, -0.6036399006843567, 0.4124102294445038, -0.5655580759048462, 0.25841113924980164, 1.566293716430664, -0.8171179294586182, -0.9605821371078491, 0.9184992909431458, -0.6074103116989136, -1.210412859916687, -0.1230081245303154, -1.5148568153381348, -0.918107807636261, -0.4075547456741333, -1.6984246969223022, 0.037974096834659576, -1.1000499725341797, 0.2390671968460083, -1.7668023109436035, -1.024153709411621, 0.7947940230369568, 0.9901683330535889, -1.6352204084396362, -1.0409959554672241, 0.5575894713401794, 0.42990803718566895, -0.3911444842815399, 0.0663740485906601], [2.1853322982788086, -1.2471394538879395, 0.7380908131599426, -1.1458567380905151, -0.41940876841545105, 0.16403627395629883, -0.4709814190864563, -0.48888903856277466, 1.085374116897583, -0.16639791429042816, 0.4144236445426941, -0.7483652234077454, 0.43447861075401306, -0.2669796049594879, 0.18767233192920685, 0.4630524814128876, 0.33955255150794983, 0.9778569340705872, 1.505360722541809, -0.5081697702407837, -0.3569774925708771, -1.038934588432312, -0.6436015963554382, 1.9817605018615723, 1.447505235671997, 0.20040200650691986, 0.5084304213523865, 0.35430118441581726, 1.4155890941619873, 0.28290244936943054, 1.173269271850586, 0.06315172463655472, -0.021958010271191597, -0.82145094871521, -0.22152233123779297, 0.09272009134292603, -0.3058144748210907, -0.27548131346702576, 0.984368622303009, -0.04556020721793175, -1.4594324827194214, 1.9362022876739502, 0.8891125321388245, -0.9415436387062073, -0.2798879146575928, -0.6867432594299316, -0.8350988030433655, -1.0679436922073364, 0.22878092527389526, 0.949201762676239, -2.638676881790161, -0.18865545094013214, -1.0477491617202759, -0.1434325873851776, -0.984320342540741, 1.3246814012527466, -0.7388173341751099, -1.7632108926773071, 0.8739778399467468, 0.6146268844604492, -1.5181576013565063, -0.3002009987831116, -0.781146228313446, 1.68538236618042], [0.324837327003479, -0.4167402982711792, 0.6799888610839844, 0.17348985373973846, 0.10577099025249481, -1.0509146451950073, 0.5679681301116943, -0.1507345587015152, -0.8222247958183289, 0.15475785732269287, 0.2150653451681137, 1.8650603294372559, 0.8978976011276245, 0.13816572725772858, 0.004210630431771278, -0.6732829809188843, 0.4407752454280853, -0.05609746649861336, 0.4919479489326477, -0.4858408570289612, 0.9649531245231628, 0.08246397227048874, 0.1617695391178131, -0.45795929431915283, -1.737714409828186, 0.878034770488739, -0.2783489227294922, 2.3599650859832764, -0.024147426709532738, -0.784021258354187, 1.6224768161773682, -0.23119528591632843, -0.2878284156322479, 0.1995057612657547, -0.2380335032939911, 1.586503267288208, 1.687220573425293, 0.5331630110740662, 0.16269102692604065, -0.9596010446548462, -0.47837722301483154, 1.5046889781951904, 0.7549342513084412, 0.2763286530971527, 0.9788053035736084, 0.33344703912734985, -0.42145299911499023, 0.7250745892524719, 0.11786015331745148, -1.700259804725647, 0.03080044686794281, 2.063314437866211, 1.255324125289917, -0.8470759391784668, -0.82405024766922, 0.4732552170753479, -0.14604303240776062, -2.043210029602051, 0.16411912441253662, 1.8834606409072876, -0.28715214133262634, -0.39242637157440186, 0.43410637974739075, -1.211496353149414], [-0.5884791612625122, 0.7652712464332581, -0.7537494897842407, 2.262974739074707, -1.0259913206100464, -1.6835135221481323, 0.5609595775604248, 0.34696587920188904, 1.181255578994751, -1.0757285356521606, -0.13213010132312775, -0.2549581527709961, 0.0027777496725320816, 1.2594645023345947, 0.2315317541360855, 0.44945430755615234, 0.38020560145378113, -0.6341299414634705, -0.5281388163566589, -0.6500990986824036, 0.04803653806447983, -0.6776779294013977, -1.284970998764038, 1.5475995540618896, 0.3383203148841858, -0.9595992565155029, -1.240092396736145, -0.23136983811855316, 0.12740236520767212, -1.3176707029342651, -0.9495989680290222, 0.6624128818511963, 1.7334332466125488, 0.9547842144966125, -2.119309425354004, -0.17085234820842743, 0.6324081420898438, 0.41746777296066284, 0.2727546691894531, -0.03289930522441864, 0.10580122470855713, -1.3194129467010498, 0.1392548829317093, -0.780536413192749, -0.5724311470985413, -0.0530235581099987, -0.2895812690258026, -1.6210922002792358, -0.5116540789604187, -0.13947871327400208, -0.4149455726146698, 0.6440221667289734, 0.11983102560043335, -0.9222463965415955, 1.4308844804763794, -2.2607622146606445, -0.8924309611320496, -0.7302117943763733, 0.8902482390403748, 0.6883020401000977, -0.45258164405822754, -0.8695212602615356, -0.9875398874282837, -1.3490861654281616], [1.9458446502685547, 1.1058533191680908, -0.14892086386680603, 0.35219985246658325, 0.02224990352988243, -0.8572655320167542, -0.37571558356285095, 0.015107192099094391, 0.27646568417549133, 0.035045016556978226, -0.21591006219387054, 0.7426502704620361, -2.6910126209259033, 0.6779926419258118, 0.853624165058136, 0.6006954908370972, -1.8028666973114014, 1.9264453649520874, -1.7758952379226685, -0.5463030934333801, 0.5708579421043396, -0.7065669298171997, -0.412663072347641, 1.3176418542861938, 1.1490925550460815, 1.4995898008346558, 0.1603882610797882, -0.4936714768409729, -1.256752848625183, 0.09295748174190521, 0.5134251117706299, 1.2157474756240845, 1.1394572257995605, 0.22981497645378113, -0.05773083493113518, 0.26527172327041626, -0.7054689526557922, -0.5865136384963989, 0.5281720161437988, 1.037282109260559, 0.6754146218299866, 1.2019692659378052, -0.902070164680481, 1.4110639095306396, 0.8041585683822632, 0.5973663926124573, -1.3793940544128418, 0.9958165287971497, -0.8582608699798584, 1.0736174583435059, -0.2504291832447052, -0.8066678643226624, -0.935347855091095, -2.0242156982421875, 0.49168938398361206, -0.23188894987106323, -0.702032208442688, 2.3583998680114746, -0.5627276301383972, 0.7080949544906616, 0.380522221326828, 0.3480885922908783, 1.0202765464782715, 0.16668039560317993], [-0.5886527895927429, -0.7801048755645752, -0.5416708588600159, 0.47316688299179077, -1.428452730178833, -0.6963451504707336, -2.0546727180480957, 0.31262239813804626, 1.5324430465698242, 0.11347280442714691, -0.5756670236587524, 1.556613564491272, -2.9569592475891113, -0.5075498819351196, 0.34284618496894836, -0.36795246601104736, 0.478577196598053, -0.4241088330745697, -1.4390945434570312, 0.00952125247567892, -0.1743210107088089, -1.1987769603729248, -0.2437162697315216, -0.8948923349380493, -1.3965381383895874, -0.7835356593132019, -0.8913378119468689, -0.36463090777397156, 1.732814908027649, 0.030073143541812897, 0.5040676593780518, 0.7406305074691772, 0.4025708734989166, -0.6596660614013672, -0.02946457266807556, -1.000540018081665, -0.6047933101654053, -1.3342688083648682, 0.5240306854248047, -0.8466551899909973, 0.2063751518726349, -1.2878085374832153, -0.7359170913696289, -1.5974417924880981, -1.0203309059143066, -1.2463345527648926, 0.35791879892349243, 0.7441652417182922, 1.7566680908203125, -0.7368273735046387, -0.5746155381202698, 2.288508415222168, -2.0659537315368652, 0.7371552586555481, 0.06905114650726318, 1.0528916120529175, -1.77492094039917, -0.12064532935619354, 1.3324308395385742, -0.23436854779720306, -0.6722116470336914, -0.5723089575767517, -0.6201590895652771, 1.3427386283874512], [0.14952750504016876, -0.5019945502281189, 0.517824649810791, 1.6408905982971191, -0.8158507943153381, -0.4555371403694153, -0.6836041212081909, -0.5860698819160461, 0.19275467097759247, -0.22673399746418, 0.8215908408164978, -0.8623237609863281, -0.31769460439682007, 0.6025964021682739, 0.24331901967525482, -0.6248364448547363, 0.034354954957962036, -2.276127815246582, -1.0993258953094482, 1.1016736030578613, -0.3089892566204071, 0.6441663503646851, -1.4966198205947876, 0.8774781823158264, 0.7470375299453735, -0.08539485931396484, -0.08792746812105179, 1.022636890411377, -0.658704400062561, 0.5392067432403564, -1.1526801586151123, -0.21093013882637024, -0.8080047965049744, -0.5865942239761353, -1.432288646697998, -0.9464293718338013, -0.3377457559108734, -0.5657451152801514, 1.1009448766708374, -0.47087839245796204, -0.665755569934845, -0.14486850798130035, 0.5266828536987305, 1.974958062171936, -0.9856090545654297, 0.2534125745296478, 0.24459724128246307, -0.7882323265075684, -0.5813036561012268, -0.28714582324028015, 1.0549793243408203, 1.3564568758010864, 1.5308603048324585, 1.1880767345428467, -0.40329599380493164, -0.2823069095611572, 1.6046111583709717, 1.317313313484192, -1.6777002811431885, 0.7710458040237427, -0.46946296095848083, -0.4769570827484131, 0.8611266016960144, 1.1687078475952148], [-1.812504529953003, 1.8491978645324707, 0.35509219765663147, 1.1046124696731567, 0.19532310962677002, 0.25345197319984436, -0.5735498070716858, 0.03495031222701073, 0.225307896733284, 1.1841027736663818, 0.47491052746772766, -0.6465769410133362, 0.744182825088501, 0.2775305509567261, 0.4315861761569977, 0.41569629311561584, -0.8846963047981262, -0.9432900547981262, 0.5244581699371338, -0.7539150714874268, -1.7813185453414917, -0.10321833193302155, -0.6292728185653687, 0.6532900333404541, -0.4334375262260437, 0.3021692931652069, 0.02707221917808056, -1.0653650760650635, 0.33684584498405457, 1.8996367454528809, 0.4949263334274292, -1.0813395977020264, 0.6093148589134216, -1.6826391220092773, 1.2479020357131958, -0.14191733300685883, -0.37432658672332764, -0.4520595967769623, 0.5318252444267273, 1.8749223947525024, -0.14942653477191925, 0.6767037510871887, -0.6730678677558899, -1.0856789350509644, 1.569303035736084, -0.7202317118644714, -0.8562648892402649, 0.5373652577400208, 0.4918835759162903, 1.184221625328064, 0.47744378447532654, -1.171634554862976, -0.41651588678359985, -0.581394374370575, -0.4204418957233429, -0.3435870409011841, 0.7742415070533752, 0.5067875385284424, 1.0164928436279297, -1.3887121677398682, -0.11020923405885696, 1.754491925239563, -1.8285696506500244, 0.35855376720428467], [-1.097822904586792, 0.3650195896625519, -0.9670684933662415, 1.102184772491455, 1.770680546760559, 0.5250223278999329, -1.2105238437652588, 1.0742604732513428, 0.2398947924375534, -0.18521659076213837, -1.1652508974075317, -1.0332574844360352, -0.3085881173610687, -0.9836982488632202, 1.1104165315628052, 0.13297678530216217, 0.4098007082939148, -0.6659168601036072, 0.12036565691232681, 2.0666449069976807, 0.34780484437942505, -0.6512730121612549, -0.04020804166793823, -0.860852837562561, -1.8461668491363525, 0.8105026483535767, 0.7074143886566162, -0.2703433930873871, -0.21729344129562378, -0.545200526714325, 0.49130168557167053, 2.304866075515747, 2.4937400817871094, -0.4429577589035034, -0.7588814496994019, -1.1170077323913574, 0.31795868277549744, -1.2742491960525513, -1.5854336023330688, 0.4546196162700653, 0.33964693546295166, -0.31237009167671204, 1.1058584451675415, 2.157703399658203, 2.149243116378784, 1.14622962474823, -1.0982460975646973, -0.6139528155326843, -0.2022688090801239, -0.7876341342926025, 1.7973991632461548, 1.4285507202148438, -1.2884187698364258, 1.6014389991760254, 0.3097444176673889, -0.09265919029712677, 0.32492026686668396, -0.8583232760429382, 0.8641127943992615, 1.5176633596420288, 0.49035611748695374, -1.9854826927185059, 0.9328784942626953, -0.20234571397304535], [1.750251054763794, -0.153121218085289, 0.9322242140769958, -1.4408499002456665, 0.49967917799949646, 1.9980356693267822, 0.29197072982788086, 0.7831284403800964, 1.2878522872924805, 1.355492115020752, 1.6954865455627441, -0.14799988269805908, 0.17100951075553894, 0.8962817788124084, 0.5473667979240417, -0.5198304653167725, -1.2382882833480835, 0.8874530792236328, -0.5530344843864441, -1.006325364112854, -0.19395321607589722, 0.7108118534088135, -1.8384084701538086, -0.2434271275997162, 0.980688750743866, -0.8883257508277893, 0.9234904050827026, 0.9643120169639587, -0.16970239579677582, 0.9804509878158569, -2.070441246032715, -1.3469252586364746, 0.7076113820075989, -1.0463193655014038, 0.7944391965866089, -0.6841689348220825, 1.0969336032867432, -2.032310724258423, -0.352181613445282, -1.516983151435852, -0.5366119742393494, 0.9981086254119873, 0.020072191953659058, 1.1058357954025269, 1.5257282257080078, -0.24326181411743164, 0.6396048069000244, 0.6588292121887207, -0.8058778047561646, 1.2922176122665405, -0.9096495509147644, -1.152898907661438, -0.05240415036678314, 1.100539207458496, -0.2689228057861328, 0.8146039247512817, -1.5936734676361084, -1.231028437614441, -0.6487624049186707, 2.326855182647705, -1.3400346040725708, -0.41845500469207764, 0.2900281250476837, -1.1258273124694824], [-1.0016318559646606, 0.48895737528800964, 0.9030817151069641, -0.7911708354949951, 2.0179903507232666, -1.6953502893447876, 1.272192120552063, 0.41479969024658203, -0.29720813035964966, -0.9944339990615845, 1.3738999366760254, -1.8309932947158813, -1.5078818798065186, 0.11050381511449814, -0.4876408576965332, 0.898234486579895, 0.8156172633171082, -0.33722952008247375, 0.21207955479621887, 1.6236298084259033, -1.9013041257858276, 0.6076623797416687, -0.06322100013494492, -0.5453047752380371, -1.8626978397369385, 0.16308173537254333, 1.2984203100204468, -0.2583121657371521, -1.0838935375213623, -0.39754199981689453, 0.8422613739967346, -0.08770667761564255, -0.393223375082016, -0.03405516967177391, 1.2127660512924194, -0.8874569535255432, -0.917602002620697, -0.16981783509254456, 0.7004758715629578, 0.754109799861908, 0.055525023490190506, -0.19530817866325378, -1.0594167709350586, 2.17228627204895, -0.3930453658103943, 0.5368741154670715, 1.3979591131210327, 0.23751311004161835, -0.63057941198349, 0.058138009160757065, -0.3227140009403229, -0.026333967223763466, -1.3184007406234741, -0.05674423649907112, 1.4489576816558838, 0.04438291862607002, -1.4467658996582031, -0.6065053343772888, -0.7091285586357117, 0.04452645033597946, -0.5221797823905945, -0.0695427879691124, 1.707410216331482, -0.6164645552635193], [-0.8418654203414917, -0.5500261783599854, -1.3903950452804565, 2.235787868499756, 0.27029886841773987, -0.9134610295295715, 0.665062665939331, 0.3535030484199524, 1.4777028560638428, 0.23443466424942017, 1.534897804260254, 1.3821425437927246, 0.8876871466636658, -0.34924063086509705, 0.5577390193939209, -1.3164219856262207, -0.6123212575912476, -0.6951429843902588, 0.985340416431427, -0.6658770442008972, -0.3555911183357239, 1.2074064016342163, 1.3774735927581787, -0.31026092171669006, 1.383813738822937, -0.8294118642807007, -0.22021719813346863, -0.526416003704071, -0.7002350687980652, -1.0288605690002441, -1.4758999347686768, 0.7094663381576538, -0.9662273526191711, -1.2811641693115234, 1.0010355710983276, 1.2730120420455933, -0.03657042235136032, 0.6690875291824341, 1.3182944059371948, 1.6816319227218628, -0.44056177139282227, -1.8152176141738892, 1.3594410419464111, 1.866078495979309, -0.5309750437736511, -1.0494822263717651, -1.053088903427124, 0.7906549572944641, 1.4461710453033447, -0.423669695854187, -0.13451696932315826, 0.33161717653274536, 0.5026088356971741, -1.0440618991851807, 0.7856079936027527, 0.511604905128479, -0.2679276466369629, 0.18404312431812286, 0.565910816192627, 0.3372933566570282, 0.10261697322130203, 0.7170159816741943, -1.014646053314209, -0.3361782431602478], [-0.26036927103996277, -1.6647868156433105, 0.3607134521007538, 0.2224586308002472, 0.20965175330638885, -1.7069451808929443, -0.41499340534210205, -1.6817735433578491, 0.2831805348396301, -0.09827530384063721, 1.7814605236053467, 0.7643731236457825, -2.4846179485321045, 0.37133851647377014, -0.9523491263389587, 1.0312918424606323, -1.0886752605438232, -0.7948446273803711, 0.0844932273030281, -0.0837690532207489, -0.545335590839386, -0.9303717017173767, -0.0024771252647042274, -0.6048600077629089, -0.711910605430603, 1.4996343851089478, -0.45979249477386475, 0.09918398410081863, -1.8287628889083862, 1.2147682905197144, -0.8901575207710266, -0.5715884566307068, -1.1197113990783691, 0.644148051738739, 0.2142678052186966, -1.107324481010437, 0.5233070850372314, -0.008969045244157314, 0.05420364439487457, -0.14924365282058716, -0.5111770033836365, 0.9754126071929932, 1.607681155204773, -0.24190212786197662, 0.9312905669212341, 1.4175360202789307, 0.10936755686998367, -2.3296992778778076, -1.6421838998794556, -0.8614294528961182, -0.42088234424591064, 0.6162164211273193, -1.2561591863632202, -0.37988579273223877, 0.9471412301063538, 1.0389039516448975, 0.08887839317321777, -0.5671141147613525, -1.2900338172912598, 0.28869467973709106, -1.3498848676681519, -0.5426803231239319, -0.049045808613300323, 0.3565721809864044], [-0.8247355222702026, 0.026143142953515053, 0.2591469883918762, -0.8280426263809204, 1.4535316228866577, -1.1620583534240723, -1.666552186012268, 1.1966789960861206, -1.604684829711914, -0.6036983132362366, -0.7402556538581848, -0.0799730122089386, -1.3491706848144531, -2.9414453506469727, 1.203368902206421, 0.10628420859575272, -0.35756537318229675, -1.3862978219985962, 1.2224406003952026, -0.4862793982028961, 0.8513015508651733, 1.4292367696762085, 1.507949948310852, -1.192702054977417, -0.2723172605037689, -0.0625474750995636, -1.5067737102508545, -0.35272854566574097, 0.43022847175598145, -1.8796896934509277, -1.911316990852356, 1.368421196937561, -1.2541718482971191, 1.1679718494415283, 1.2218976020812988, 0.6400001645088196, 0.9182479381561279, 0.10773415863513947, 0.18721118569374084, -0.6934044361114502, -1.7284820079803467, -0.1876583695411682, 0.4355831444263458, -0.9003875851631165, 0.5511982440948486, 0.043993156403303146, 1.4218318462371826, -2.031126022338867, -0.1111140251159668, 0.6368367075920105, 2.856851577758789, -0.5590810179710388, -0.2778390049934387, -1.9199150800704956, 0.8131369948387146, -0.05748613923788071, 0.486747682094574, -0.17289474606513977, -0.3599904179573059, -0.8523188829421997, 1.2300807237625122, -0.15873974561691284, -0.7862211465835571, -0.6747019290924072], [-1.2520310878753662, -0.34950289130210876, -1.7361066341400146, 1.261252999305725, 0.012672136537730694, -1.446288824081421, 0.4583747088909149, 1.6724965572357178, -0.2553133964538574, 0.30331557989120483, 0.13519427180290222, -0.4333830177783966, -1.546278476715088, 1.601850152015686, 2.287093162536621, -1.1473209857940674, -0.10970030725002289, -0.1582811176776886, -0.9929238557815552, 1.8097050189971924, -0.4159545302391052, 0.7266244888305664, 1.1397091150283813, -0.20955362915992737, 0.39477482438087463, 1.5083374977111816, -0.505081057548523, 0.3935942053794861, 1.3552838563919067, 0.8849246501922607, -0.33437785506248474, -1.3708343505859375, -0.29387161135673523, 2.295997142791748, -0.44252803921699524, -0.628736138343811, -0.8860645294189453, -1.0828536748886108, -0.22967472672462463, 0.39501163363456726, 0.2210964858531952, 1.437214970588684, -0.22123484313488007, 1.5390464067459106, 0.11713720858097076, -0.9838204383850098, 0.9678647518157959, 0.589658260345459, -1.1836652755737305, 1.6361660957336426, 1.0292963981628418, 0.5057956576347351, -2.145725727081299, 0.06974037736654282, -0.792937695980072, -1.5582164525985718, -0.9904703497886658, 1.0806645154953003, -0.7019692063331604, -1.365804672241211, 0.6810116171836853, 0.1625450849533081, 0.8087397217750549, -0.6791474223136902], [-1.0220783948898315, -0.6110222935676575, -1.160036325454712, 0.3924678862094879, -1.4806346893310547, -0.9352262616157532, 1.1336168050765991, 1.3022451400756836, 1.4359402656555176, -0.6957701444625854, 0.4030706584453583, 0.6291009187698364, 1.0564754009246826, -0.31375640630722046, 0.15321217477321625, 0.41652318835258484, -1.2354501485824585, 0.3939249813556671, -0.16473734378814697, -0.3744311034679413, 0.22912654280662537, 0.1793835312128067, -0.21432091295719147, -0.6233299970626831, 0.7907900214195251, 1.039313793182373, -0.5878749489784241, -0.6569238305091858, -1.6214100122451782, -0.7431023120880127, -0.47482815384864807, 0.525062620639801, -2.061673641204834, 0.49946463108062744, -0.8713173270225525, -0.2108902782201767, -1.6849993467330933, 0.0085885189473629, 0.7112058401107788, 1.0393571853637695, -0.8771241307258606, -0.3460627794265747, -0.18188869953155518, 0.47608843445777893, 0.0029402594082057476, 0.758858323097229, 2.4593923091888428, -0.17000220715999603, 1.154997706413269, -2.243959665298462, 0.5245506167411804, 0.30211424827575684, -0.5164003968238831, 0.2909139096736908, -0.48119235038757324, 0.6185095906257629, -0.08090400695800781, 0.6885601282119751, 1.5109970569610596, -1.8711243867874146, 1.7957606315612793, -1.6427712440490723, -0.13845643401145935, 0.557927131652832], [-0.17003101110458374, 0.007791154086589813, 0.5330931544303894, -1.9293192625045776, 0.5149244070053101, -0.9528447389602661, 0.20790615677833557, -1.2942326068878174, -0.03268507122993469, 0.5019524097442627, -0.7339342832565308, -1.0556741952896118, -1.4004573822021484, -0.44220975041389465, 0.9807662963867188, -0.7588562369346619, -2.0281107425689697, 0.4637182354927063, 0.8481938242912292, 0.36108261346817017, -0.3120248019695282, 0.8136462569236755, 0.515099823474884, 0.29834872484207153, 0.9467736482620239, 0.3828728497028351, 1.0580514669418335, 1.5015417337417603, -0.4875483512878418, 0.02508961781859398, 0.8722385764122009, -2.228431463241577, -1.3096446990966797, 1.1549549102783203, -0.7309157848358154, -0.49769070744514465, 0.3372024595737457, 0.701532781124115, 1.1726067066192627, -0.21611429750919342, -0.04530470445752144, -1.0855785608291626, -0.3594989478588104, -1.2149338722229004, 1.3937196731567383, 1.5793057680130005, -0.4806312620639801, -1.1054574251174927, 0.9427019357681274, 2.631810426712036, -0.5371926426887512, -0.30958321690559387, -1.1357921361923218, -0.1565067023038864, -0.580651044845581, 1.544312834739685, -0.6087045669555664, -0.010809788480401039, -0.8248934149742126, 0.7854153513908386, -0.05276034027338028, -0.35976698994636536, 1.3661569356918335, -0.3058668076992035], [1.5131309032440186, -0.8798443078994751, -1.2110098600387573, -0.6627941131591797, 0.3988657593727112, -1.2973151206970215, 0.163664311170578, -0.6232863664627075, -1.3366377353668213, -1.223759651184082, 0.2864648103713989, -0.8429441452026367, 1.209554672241211, -0.9816311597824097, 1.265201210975647, 0.03826719522476196, 1.2981054782867432, -0.09862518310546875, -0.3639611303806305, -1.3231927156448364, -1.2175706624984741, 0.7081722617149353, 1.0319112539291382, -0.6292323470115662, -0.8245117664337158, -0.23036427795886993, 0.08533702045679092, -1.5121046304702759, 0.4143708050251007, 0.021814610809087753, -0.4562698304653168, 1.952713966369629, 0.196504145860672, 0.6597863435745239, 0.6863226890563965, -1.1219767332077026, 1.5494998693466187, -0.0630626529455185, 0.04680094122886658, -0.2467990219593048, -0.7823072671890259, -0.2759855091571808, 0.037390079349279404, -1.7279694080352783, 0.49271219968795776, -0.11655687540769577, -0.6072132587432861, 1.569218397140503, 0.07050982862710953, -0.3393356502056122, -0.9079551696777344, 0.27419760823249817, -0.3676931858062744, 1.2828341722488403, 0.9371213316917419, -0.3276195824146271, 0.5146785974502563, 0.06210758909583092, -0.3169247806072235, 0.2953707277774811, 0.23544643819332123, 1.1381419897079468, 1.684255838394165, -1.5166490077972412], [0.7696735262870789, -0.4935612678527832, -1.9539152383804321, -1.1360650062561035, 0.18802908062934875, -0.749412477016449, 0.3044619560241699, -0.6558471322059631, 0.47181636095046997, 0.004288239870220423, -0.5419684648513794, -0.1958383321762085, -0.1911104917526245, 0.30627667903900146, -0.5547460913658142, -0.15688541531562805, -1.284261703491211, -1.7092335224151611, -0.14604413509368896, -0.30176061391830444, 0.024887073785066605, -0.9408215880393982, -0.6079912185668945, 0.7884424924850464, -1.489797830581665, 0.3595101237297058, 1.4125248193740845, -2.145592451095581, 0.5146179795265198, 0.12624585628509521, -1.37643563747406, 2.638108253479004, 0.8062432408332825, -0.4566473662853241, 0.22883771359920502, 2.0365638732910156, 0.5091332793235779, -0.73561030626297, 0.3065390884876251, -2.6200971603393555, -1.2890735864639282, 0.49442726373672485, 0.5025942325592041, -0.4448949098587036, -0.02808060124516487, 1.9637420177459717, -1.1947648525238037, -1.9500060081481934, 0.578978955745697, -0.09503275156021118, -2.0850069522857666, 0.30397137999534607, 0.9510999917984009, -1.1116100549697876, 2.05790114402771, 0.17975029349327087, 1.2874619960784912, 0.30858495831489563, -0.24441945552825928, 1.9025086164474487, -1.2966738939285278, 1.0746616125106812, 0.3427085876464844, -0.06123563274741173], [2.263970136642456, -1.8657894134521484, 0.03853512182831764, -1.2008289098739624, 0.45602163672447205, 0.6836749911308289, -0.13469940423965454, 0.23779083788394928, -0.1753959208726883, -0.4606814682483673, 1.4307308197021484, 0.5132637619972229, -1.5221372842788696, 0.7132523059844971, 0.854436457157135, -0.6990015506744385, -0.659424364566803, -0.5752314925193787, -2.2999157905578613, -1.089979887008667, -0.41345661878585815, -1.330755352973938, 2.072019100189209, -0.5981305241584778, -0.34852996468544006, -0.34171175956726074, 0.29294946789741516, -1.065731167793274, -0.3156066834926605, -1.771342396736145, 1.4179246425628662, -0.9800573587417603, -1.014363169670105, -0.27927330136299133, -0.5854735374450684, 0.4585055410861969, -0.3024965524673462, 0.7569798231124878, -0.7644605040550232, -1.9792057275772095, -0.862220048904419, -2.136962652206421, 0.2918994724750519, 0.5574769973754883, -0.20819970965385437, -0.6975983381271362, 0.06972446292638779, 2.072145462036133, -1.533183217048645, -0.8179579973220825, -1.6557585000991821, 0.18316225707530975, 0.861781895160675, -1.8374346494674683, -1.8183590173721313, 0.21151147782802582, 1.0321069955825806, -1.0192981958389282, -0.03825821354985237, -1.9221235513687134, 2.13960337638855, 0.8566325902938843, 0.7393924593925476, -0.41388365626335144]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.44280508160591125, 1.4303609132766724, 0.2963597774505615, -1.6049634218215942, -0.16352947056293488, 0.20739614963531494, 0.482186883687973, -1.051658034324646, 0.9139016270637512, 0.9496085047721863, -1.9995919466018677, -1.4915637969970703, -0.6397563815116882, 0.6146972179412842, -0.2695263624191284, 0.5772497057914734, 0.768519937992096, -0.5960761308670044, -1.3519046306610107, 0.8407791256904602, 0.20258431136608124, 0.6788392663002014, -0.4202357232570648, -0.8186793923377991, -0.13317088782787323, -0.24104364216327667, -0.11921615153551102, 0.3768041431903839, -1.7257896661758423, -1.0477110147476196, -0.3662351071834564, -0.7992873787879944, 0.7005767822265625, 0.7140751481056213, 1.8259481191635132, 1.44326913356781, -1.6470386981964111, 0.8437207341194153, -0.029386445879936218, -0.28822967410087585, -0.744920015335083, 0.41289836168289185, 0.5271305441856384, -0.4549640417098999, -0.5876159071922302, -1.9939687252044678, 1.3093149662017822, -0.03998443856835365, 0.06919459253549576, -1.1868267059326172, -1.057979702949524, -0.7116740345954895, -1.036529541015625, -0.13342086970806122, 0.3180595934391022, 0.32170528173446655, 0.45481234788894653, -0.9656554460525513, -0.13994990289211273, -0.051144346594810486, 0.003995598293840885, -0.276848703622818, 0.04516400769352913, -2.386679172515869, -0.8349555730819702, -0.0004781889438163489, -0.7096040844917297, -1.3220257759094238, -0.509190559387207, -1.9548219442367554], [-0.3269571363925934, -0.5775059461593628, -0.6575673818588257, 1.0341880321502686, 1.2508761882781982, -0.46780216693878174, -1.3991048336029053, 0.6038889288902283, 0.7962014675140381, -1.9853712320327759, -0.653784990310669, -0.9695145487785339, 2.1755552291870117, -0.6427586674690247, 1.5768868923187256, 0.6725166440010071, 1.3221638202667236, -0.32253751158714294, 0.695327877998352, 0.6436320543289185, -0.6524344086647034, -0.33590078353881836, -1.1896487474441528, 0.5052012205123901, -2.3389458656311035, -0.7501124739646912, 0.36563462018966675, 0.44065961241722107, -0.7214556932449341, 0.9552919268608093, 0.08952955901622772, 0.655512273311615, 0.9831815361976624, -0.6653455495834351, 0.5016297101974487, -1.7170302867889404, -1.0428667068481445, 1.2984980344772339, 1.0741047859191895, 2.1831443309783936, -0.4692382514476776, -0.08406099677085876, -0.0939297303557396, -1.0629802942276, 0.5848077535629272, -0.07820317149162292, 0.5378762483596802, -0.09526724368333817, 1.0348715782165527, 0.1035667210817337, -0.143172949552536, -0.32504501938819885, -0.2998296320438385, -0.5883328914642334, 0.43580400943756104, -1.2040092945098877, 0.6574526429176331, 0.45168066024780273, 0.5213439464569092, -1.308923602104187, -0.06038028001785278, -0.9933180809020996, 0.932930052280426, -0.7105221152305603, -0.7694498896598816, 0.2853214740753174, -0.12134234607219696, -0.8893519043922424, -1.8012206554412842, -0.915924072265625], [0.5503101944923401, 0.0347757525742054, -0.6521821618080139, -1.6184197664260864, -0.3056657016277313, 0.24076202511787415, 1.2973673343658447, 1.5656354427337646, -1.0067064762115479, -0.7638749480247498, 1.0487165451049805, 0.4238404333591461, -0.5194843411445618, -0.7078648805618286, -1.531586766242981, 0.7571381330490112, -1.1508220434188843, -0.009252703748643398, -0.8307551741600037, 0.7709883451461792, 0.14813627302646637, 0.15589065849781036, -1.0210883617401123, 0.8576659560203552, -0.8891359567642212, 0.002860646927729249, -1.5042762756347656, 1.358158826828003, 0.968702495098114, -0.4805198907852173, 0.08254794031381607, -0.5731602311134338, 1.6095243692398071, -0.014022227376699448, -0.11691784113645554, 0.1461201012134552, -0.44051575660705566, -0.2448863387107849, -0.26302433013916016, 0.47125348448753357, 0.10232546925544739, -1.3578596115112305, -1.75978422164917, -1.2530014514923096, -0.3272581994533539, -0.029248028993606567, -0.9310663342475891, -0.10413903743028641, 1.1752604246139526, 0.7514644861221313, -0.7268681526184082, -1.6504814624786377, 0.10705269128084183, 2.842708110809326, 0.8277092576026917, -0.43716323375701904, -0.5140575170516968, 0.7431458234786987, -0.7322806119918823, -0.7001209855079651, -1.672414779663086, -0.2502804398536682, 0.5541940331459045, 0.24066466093063354, -1.0132958889007568, -1.2005659341812134, 1.2048346996307373, 2.0394437313079834, -0.19672048091888428, -0.15594099462032318], [-2.3966851234436035, 0.7022680640220642, 0.4630788266658783, 1.7880549430847168, 0.08623840659856796, -0.8958799839019775, -0.18807265162467957, 1.4568904638290405, 0.10398590564727783, 0.13824066519737244, 1.0937821865081787, 0.5546332597732544, 0.49395355582237244, -0.4891314208507538, 0.8432779312133789, 0.3045929968357086, -0.08861952275037766, -1.0188319683074951, 1.381442666053772, 0.4536643624305725, -0.8233689665794373, 0.2561010718345642, -0.332714706659317, -0.12107150256633759, -0.1754635125398636, 2.07251238822937, 0.818844735622406, -0.4182963967323303, 1.6149795055389404, 0.3843151926994324, -2.135021209716797, -0.5053561925888062, -0.13369451463222504, 0.7428860664367676, -1.3842723369598389, 0.08880794793367386, 1.9102952480316162, 1.0803035497665405, 0.8498811721801758, -1.2814507484436035, 0.950386643409729, 0.5974435210227966, -0.0949757769703865, 0.6172016263008118, 0.03261159732937813, 1.551149845123291, 0.2565300464630127, -0.4790799915790558, 0.6019493341445923, -2.5323486328125, 0.890527606010437, 1.9034582376480103, -0.38839423656463623, 0.003434923477470875, -0.1970464140176773, -0.5614762902259827, 0.014710926450788975, 1.5724496841430664, -1.0381510257720947, -1.4914911985397339, -1.047613501548767, -1.5570505857467651, -0.9614403247833252, -0.094561368227005, 1.8061909675598145, 1.159663200378418, 0.28131425380706787, -0.9146504998207092, 1.0471019744873047, -0.12017960846424103], [0.4743921458721161, -1.4445693492889404, 0.5174800753593445, 0.6184548139572144, 1.0363788604736328, 0.3666855990886688, 0.9899292588233948, 0.15871688723564148, -0.1153831034898758, 0.5556033849716187, -0.07883309572935104, 1.3630274534225464, -0.8533170223236084, -0.01614494062960148, -0.7510843276977539, -0.34902632236480713, 0.866935670375824, -0.36092904210090637, -1.260677456855774, 0.5738793611526489, 0.23505324125289917, -0.9842745661735535, 1.9724491834640503, -0.566318690776825, 0.48875656723976135, 0.008625525049865246, 0.8355092406272888, 0.8810750842094421, -0.5710356831550598, -1.2825847864151, 0.7306497693061829, -0.6461235880851746, -1.6412079334259033, -2.161285400390625, 1.6899752616882324, 1.3248885869979858, 0.48294082283973694, 0.09542959183454514, -0.813988208770752, -1.0022016763687134, 0.2187187820672989, 1.3051912784576416, 0.923140287399292, 0.8372040390968323, -1.556792974472046, 1.3683680295944214, 0.45409631729125977, -0.9841788411140442, 1.0207520723342896, 2.010296583175659, 1.946933388710022, -1.191525936126709, 0.12144388258457184, -0.3847058415412903, 1.677195429801941, -0.06525666266679764, -0.6026773452758789, -0.17228065431118011, 0.1553896963596344, -0.8302900791168213, -1.3458906412124634, 0.36025020480155945, -1.6290380954742432, 0.9183518886566162, 0.23417431116104126, 0.23794154822826385, 1.3983896970748901, 0.4288792014122009, -1.2457090616226196, -0.8532958626747131], [1.7401264905929565, 0.7271178364753723, 0.1980314552783966, 1.1418461799621582, 1.4576762914657593, -0.19083155691623688, 0.37601813673973083, -0.22260625660419464, -1.8103649616241455, -3.0281593799591064, 1.4617379903793335, -0.6878136396408081, 1.7843877077102661, -2.1313414573669434, 1.1297353506088257, 2.7384490966796875, -0.7565467357635498, 0.23373563587665558, -0.7796242833137512, 0.04767397791147232, -0.9482946395874023, -1.8571900129318237, 0.06039995700120926, 0.7954912781715393, -0.7729319930076599, 1.3143620491027832, -1.5312530994415283, 0.3862897753715515, -1.0801112651824951, 0.5026304125785828, 0.8133854269981384, -1.5848621129989624, 0.4279917776584625, -0.576248824596405, 0.008467976935207844, -0.09487579762935638, 2.308452844619751, -0.8342383503913879, -0.4542168080806732, -0.304206520318985, -1.2202787399291992, 1.9567086696624756, -0.15225525200366974, 0.7217538356781006, -1.7488386631011963, -0.2122252732515335, 0.09164931625127792, -1.1075609922409058, 0.38315868377685547, 0.10845569521188736, -1.4854000806808472, 0.9723572134971619, -0.4380717873573303, 0.7899146676063538, 0.3178020715713501, 0.4178259074687958, 0.7108185887336731, 1.7293399572372437, 0.5223357677459717, 0.7960121631622314, -0.9197636246681213, -0.7194663286209106, 1.4351952075958252, 0.7083175778388977, -1.7628153562545776, 0.9686563014984131, -0.6364453434944153, 0.6932663321495056, 1.0115070343017578, -0.19261635839939117], [0.5670584440231323, -1.2659229040145874, -0.3728887736797333, -0.41530683636665344, 0.10583879798650742, -0.5033165812492371, -1.1701397895812988, 1.3310825824737549, 0.11083976924419403, -1.3167635202407837, 2.617859363555908, -0.47602495551109314, -0.822422444820404, 1.304918885231018, -1.2999851703643799, -0.887698769569397, -0.725159764289856, 0.749086856842041, -0.39542320370674133, -0.14151355624198914, -2.060767412185669, 0.3990132212638855, 0.15459652245044708, -0.6434407830238342, 0.5917026400566101, 0.9288092851638794, 1.1518789529800415, -1.5077553987503052, 0.6674280166625977, 2.2548470497131348, -0.3592239022254944, -0.6954652667045593, -0.19648176431655884, -1.0615947246551514, 1.1506835222244263, 0.6308456659317017, 2.0116541385650635, -0.35697075724601746, -0.18446899950504303, 0.16862830519676208, -1.2299748659133911, 0.2621655762195587, -1.188506841659546, -0.2963516116142273, 0.8089778423309326, 0.1366206258535385, 0.775034487247467, -1.1565546989440918, -0.40124234557151794, -1.2941137552261353, -0.3726158142089844, -0.2086125910282135, 0.6616613268852234, -0.42989587783813477, -0.37607550621032715, -0.2518135607242584, -0.8244867324829102, 0.2844659984111786, -0.43555834889411926, -1.681104302406311, 0.7066280841827393, 0.22794048488140106, 1.4414403438568115, 0.5683965086936951, -1.1068216562271118, 1.0641433000564575, -2.694533586502075, 0.16583208739757538, 0.7456055879592896, -0.3519720137119293], [0.15752941370010376, -1.557235598564148, -0.33581310510635376, 0.3904436230659485, -0.4694134294986725, -0.06800797581672668, -0.5793206095695496, -0.06994400173425674, 1.0885459184646606, -0.6118496060371399, 1.2366031408309937, 0.0822744220495224, -0.07052911072969437, 0.6485756039619446, 1.3115166425704956, 0.5755025744438171, -0.47775134444236755, -0.6518181562423706, -1.1798397302627563, 1.910982608795166, -0.5838778018951416, -0.8370386958122253, 0.46694278717041016, 0.4839010536670685, 1.0025559663772583, 1.1668211221694946, -1.5279299020767212, 1.2453728914260864, -1.0903639793395996, 1.036863088607788, -0.34084081649780273, 0.2250257283449173, -0.5477524399757385, -0.12080924212932587, -0.9417667984962463, 0.08330743759870529, 1.0047798156738281, -0.006559513974934816, -1.5431814193725586, 0.7628589272499084, 0.0981869325041771, 0.533287525177002, 0.562391996383667, 0.31462225317955017, -0.5623667240142822, 0.32490283250808716, 0.5011147260665894, 0.9540308117866516, -0.10108482092618942, -1.0221225023269653, 1.7369346618652344, 1.3488627672195435, -0.17356035113334656, 3.0140111446380615, -0.8578866124153137, -1.5688849687576294, 0.711506724357605, 0.38983824849128723, 0.7501474618911743, 1.9755651950836182, 0.12367844581604004, 0.2970261871814728, -0.30914101004600525, 1.2589048147201538, 0.010770917870104313, 0.6505719423294067, 0.909869372844696, -1.6717945337295532, 0.7017760872840881, 0.2631687819957733], [2.066472291946411, -0.33636578917503357, 0.902146577835083, 0.26850220561027527, -1.192186713218689, -0.5218479633331299, 1.649485468864441, -1.06048583984375, -0.4264789819717407, 1.5862222909927368, -2.2255513668060303, -1.3067318201065063, -0.25600484013557434, 1.2563016414642334, -2.290797710418701, -1.018538236618042, -1.2047263383865356, -1.6912115812301636, -1.036863088607788, 1.7973803281784058, 0.6871635913848877, 0.23915651440620422, 0.1811959594488144, -1.8825175762176514, -0.7858487963676453, -1.2904843091964722, -0.5032013654708862, -1.341043472290039, -1.6197562217712402, -0.7272545695304871, 0.8484418392181396, 0.9283809661865234, 0.3860040605068207, 0.6363003253936768, -0.2280847281217575, 1.7586228847503662, 1.0209691524505615, 1.8335715532302856, 0.34192904829978943, 0.27075666189193726, -1.5049656629562378, 0.8648931980133057, -1.2477365732192993, -0.6454417705535889, -0.19006940722465515, -0.7814934849739075, 0.926879346370697, -0.12671571969985962, -1.1090741157531738, 1.3095349073410034, -0.6505703330039978, -0.2630775272846222, 1.4689581394195557, 0.1161932572722435, 1.1769404411315918, 1.7771846055984497, 1.4838130474090576, 0.43447425961494446, 0.2059205323457718, 0.9841245412826538, -0.5013439655303955, 0.08484815061092377, 1.5882457494735718, 0.6942358613014221, 0.8744193315505981, -0.5601195693016052, -0.35173919796943665, -2.3475799560546875, 1.229046106338501, 1.2342643737792969], [0.7767070531845093, -1.4913909435272217, 0.33965328335762024, 1.6807154417037964, -0.3146990239620209, 0.5839623212814331, -0.20747512578964233, 0.3123409152030945, 0.2743520140647888, -0.06353042274713516, -1.1337568759918213, 0.6076449155807495, -0.34570562839508057, 0.7355226874351501, 0.4321286082267761, 0.28177958726882935, -0.25079023838043213, 0.7118085026741028, -2.2047157287597656, -1.1806689500808716, 0.8340710401535034, 0.09810991585254669, 0.5057129859924316, 1.540401816368103, 0.7136369347572327, -1.3451181650161743, 0.34612223505973816, -0.705748438835144, -1.830642580986023, 0.7921937108039856, 1.2689977884292603, 1.1398954391479492, 2.0905792713165283, -0.538127064704895, -0.2322266399860382, 0.41054850816726685, -0.015103469602763653, -0.2324559986591339, 0.259591668844223, 0.6100406050682068, 1.2558308839797974, -0.358769953250885, 0.14323392510414124, 1.4904165267944336, 0.05966435745358467, -0.9364458918571472, 1.1530667543411255, 0.13803648948669434, 1.849117636680603, 1.377917766571045, -0.8752222061157227, 0.8383851647377014, 0.39399364590644836, 0.47437381744384766, 1.458461880683899, -0.27394193410873413, -0.09934087097644806, -1.1215381622314453, -0.08458264172077179, 0.4205075204372406, 0.07508087903261185, -1.1281366348266602, -0.4799598455429077, -2.6924729347229004, 0.9171205163002014, -0.556195855140686, 0.7716796398162842, -0.589608371257782, -0.721470057964325, 1.7573795318603516], [0.04891211539506912, 2.338364362716675, 0.6240823268890381, -0.9215217232704163, -0.3671850562095642, 0.09364904463291168, -1.907318353652954, -1.0797979831695557, 1.7528631687164307, 1.27618408203125, -0.4690597355365753, 0.6435851454734802, 1.7241528034210205, -1.2294820547103882, 0.29648348689079285, 1.0383546352386475, -0.07750599086284637, 0.7704665064811707, 0.9282940030097961, -1.5538727045059204, 1.0284162759780884, 1.263260006904602, 0.24031466245651245, -1.3415066003799438, 0.44855907559394836, -0.4814506769180298, 0.9866454601287842, 0.28834059834480286, 1.1126511096954346, 1.350921630859375, -0.4852193593978882, -0.1380699723958969, 0.18598362803459167, -1.0382897853851318, -0.8868938684463501, -0.42145195603370667, 1.622083306312561, 0.9782228469848633, 1.5666061639785767, -2.3014678955078125, 0.10071117430925369, 1.2496060132980347, 0.5529727339744568, -1.7267848253250122, 0.3862934112548828, 0.34255465865135193, 2.481147050857544, -0.03191803768277168, -2.4534785747528076, -0.2891044020652771, -0.5157498121261597, 1.4799367189407349, -0.22204825282096863, -0.5719646215438843, -0.2737775146961212, -0.6374572515487671, 1.9322909116744995, -0.034715138375759125, 0.6586096286773682, 0.6063557863235474, 1.0808720588684082, -0.911363959312439, 0.608424961566925, 1.9794782400131226, -0.35513314604759216, 0.03177337348461151, -0.24576327204704285, -1.5609474182128906, 0.07532462477684021, 1.826335072517395], [-0.9749610424041748, -0.4069940447807312, 0.8444125056266785, -0.38245120644569397, -0.1157383844256401, -0.7863340973854065, 1.0368752479553223, -1.0021882057189941, -0.8868524432182312, -0.5149103999137878, 1.3784652948379517, 1.878701090812683, -0.16797253489494324, 0.8601852059364319, 2.5827383995056152, 1.214883804321289, 0.5528185963630676, -1.071024775505066, -1.0935488939285278, -0.09742359071969986, 0.7226366400718689, 0.753433346748352, 0.6489998698234558, -1.3269141912460327, -0.5559860467910767, 0.9026747941970825, 1.847939372062683, -0.5997872948646545, 0.20651446282863617, 1.3144304752349854, 1.0922012329101562, 0.264759361743927, -0.7171360850334167, -0.004576528910547495, 0.13683414459228516, -1.5669602155685425, -2.1762242317199707, -0.326617568731308, -1.3346483707427979, -0.8993481397628784, -0.08944518119096756, 0.5200627446174622, -0.574501633644104, 0.2835707366466522, 1.2602039575576782, 0.386200487613678, 0.41911551356315613, -0.7378063797950745, 0.2028297781944275, 1.2370309829711914, -1.3371933698654175, 0.09444739669561386, 0.364641934633255, -0.17873765528202057, 0.5096559524536133, -1.8402833938598633, -1.189488410949707, 2.167576313018799, -0.30452844500541687, -1.0174833536148071, 0.4608880281448364, 0.9778062105178833, 0.4891843795776367, 1.2553244829177856, 0.9295943975448608, 1.1281052827835083, 0.15780286490917206, -0.9241790175437927, -0.495524138212204, 1.3890223503112793], [-0.04685007780790329, 0.46792763471603394, -0.1703261286020279, 1.387450933456421, 0.5502406358718872, -2.067652940750122, -1.148400902748108, -0.8921011090278625, 1.1618155241012573, 1.6713365316390991, -0.4224424660205841, 0.04649841785430908, 0.463575541973114, -0.08872223645448685, -0.7259487509727478, -0.5107312798500061, 2.2371718883514404, -0.5495654344558716, -0.5428544282913208, 0.9850358963012695, 0.6661985516548157, 0.35585588216781616, -0.4162273108959198, -0.632625162601471, 0.9105641841888428, -0.12171360850334167, 1.0821584463119507, 0.00039026146987453103, 1.0364190340042114, -0.5021300315856934, 0.6839936375617981, 0.25070905685424805, 1.5640913248062134, 1.5301566123962402, 0.5457803010940552, 0.518609881401062, 0.0025572918821126223, -0.920051634311676, 0.5226525068283081, -0.6019607186317444, -2.018411874771118, -1.1955890655517578, 0.536644458770752, -1.3477920293807983, -1.0384467840194702, 1.5973769426345825, -0.5827511548995972, -0.9586367011070251, 0.856432318687439, 0.6474496126174927, -0.35612916946411133, -0.684002697467804, -0.640083372592926, -1.0240933895111084, 1.1350356340408325, 0.6382095217704773, 0.4162405729293823, 2.903069019317627, 0.18538792431354523, -0.7199545502662659, 1.043853998184204, -1.9387528896331787, 0.025453481823205948, 0.451205849647522, 1.0444865226745605, 0.2001810222864151, 0.8677982091903687, -1.2258752584457397, 0.3305976986885071, 0.6916252374649048], [-1.5128178596496582, 0.05040311813354492, -0.6119596362113953, -0.004065188579261303, -0.28125864267349243, -1.353285551071167, -0.30672019720077515, 0.3294171690940857, -0.7563528418540955, -0.4417758882045746, -0.3730677664279938, 0.9236100912094116, 0.7939465641975403, -1.1596095561981201, -0.03551016375422478, 0.1489894539117813, -0.23167604207992554, 0.039582252502441406, -0.9694807529449463, -0.2913375198841095, 0.8576592206954956, -0.39089834690093994, 0.2086479663848877, 0.738278865814209, 0.6081814765930176, 1.726683497428894, -0.4169676899909973, 0.12993383407592773, -1.10029137134552, -0.6305108666419983, 1.2446571588516235, -0.7890781760215759, 2.5220906734466553, 0.22886155545711517, 0.4459245502948761, -0.8366793990135193, -0.5561423301696777, -0.9123261570930481, -0.8632451295852661, 0.446188360452652, -0.7774487733840942, 2.7997756004333496, 0.4499501585960388, -0.21370014548301697, 0.9596097469329834, -0.7746561765670776, 1.0693519115447998, 1.2830722332000732, -0.27606329321861267, -0.0825953483581543, -2.387059450149536, 1.2728887796401978, 0.3665566146373749, 0.8389058709144592, -0.8234984874725342, 0.7866398096084595, 1.2587435245513916, 0.3172808289527893, 0.1914137303829193, 1.3412957191467285, -0.35898661613464355, 0.3322286307811737, -0.7395948767662048, -0.34829819202423096, 0.2906559109687805, 0.7327268123626709, -0.43339914083480835, -0.032088130712509155, -0.9819590449333191, -0.6183879971504211], [-0.2813457250595093, 0.019394321367144585, -0.0032459774520248175, -0.4133109748363495, 0.9081853032112122, -0.45726868510246277, -0.584888219833374, 0.7654758095741272, -0.6389152407646179, -0.9001184105873108, -1.4736191034317017, -0.8791859745979309, -0.4903890788555145, 1.0700184106826782, 2.0850980281829834, 2.6727638244628906, 0.3668134808540344, -1.3328737020492554, -1.975158452987671, 1.872833251953125, -1.8000736236572266, -1.2157427072525024, -1.076509952545166, 0.8232683539390564, 0.10612107068300247, -1.9528816938400269, -0.393399178981781, -0.5055912733078003, 0.5888087749481201, 0.05895645543932915, -1.874473214149475, -0.18908511102199554, 0.8319835066795349, -0.0227140374481678, -0.1145421713590622, 0.16763941943645477, -0.8132068514823914, 0.6073718070983887, -0.3210306167602539, -1.6400835514068604, 0.7457373738288879, 0.8078919649124146, -1.5524039268493652, -0.29919424653053284, 0.3256857991218567, -0.32016968727111816, 0.9807774424552917, 1.763764500617981, -0.14336107671260834, 0.2381313443183899, 0.1929374486207962, -0.59837806224823, 0.4647541344165802, -0.8200318813323975, -0.6082965135574341, -1.1691399812698364, -0.21548520028591156, 1.2865101099014282, -1.249795913696289, -1.543552279472351, -0.2839665412902832, 0.18454323709011078, 0.3528023064136505, -2.187344551086426, 1.154404878616333, 0.8393645882606506, -1.3790169954299927, -0.38396984338760376, 0.9257561564445496, 1.5330836772918701], [0.031803201884031296, 0.11724888533353806, 0.40849265456199646, 0.6744706630706787, 0.9322047829627991, 0.21576137840747833, -2.0696938037872314, -0.0740073025226593, -1.8670004606246948, -2.2225489616394043, 2.154747486114502, 0.7767680287361145, 1.355428695678711, -0.5032809376716614, 1.3775016069412231, 0.4537794888019562, -0.07391732931137085, -0.0027761515229940414, -2.5370566844940186, 0.5079435706138611, -0.3319561183452606, -0.38803938031196594, 0.021959858015179634, 0.39355191588401794, -0.9190229773521423, 0.14420592784881592, -0.8836538195610046, -0.018996719270944595, 0.34692028164863586, 0.6577188372612, -1.8649884462356567, 0.7496073842048645, -0.23683977127075195, -0.43828853964805603, 0.745760977268219, 0.2134813368320465, 0.9349930882453918, -0.7231748700141907, -0.8711258769035339, -0.06866888701915741, -1.0149362087249756, 0.3288643956184387, 0.8170731067657471, -0.8747732043266296, -0.373084157705307, 1.2636908292770386, -0.13157138228416443, -0.337820440530777, 1.0569486618041992, -0.6279898881912231, 0.9932927489280701, -1.1451181173324585, -2.1122334003448486, -0.4192636013031006, 0.5147877931594849, 1.828912377357483, 0.02068641223013401, -0.7683727741241455, -0.07290423661470413, -0.3093319237232208, -0.26388436555862427, -0.513763964176178, -0.36411261558532715, -1.2846505641937256, 0.2885182201862335, 0.35197004675865173, -0.761770486831665, -0.1736295521259308, -0.39090949296951294, -0.9428280591964722], [-1.359370231628418, -0.5196812152862549, -0.5272858142852783, 1.1005467176437378, 0.17424888908863068, 1.3651138544082642, -0.8838122487068176, 0.05320873111486435, -0.49330997467041016, 0.39334559440612793, -0.41013064980506897, 0.744485080242157, -1.6125855445861816, 0.14383380115032196, 0.4464554786682129, 0.8242036700248718, 0.8512179255485535, 0.8740065097808838, -0.9134808778762817, -0.16869518160820007, 0.5192636251449585, -1.023606300354004, 2.190321922302246, 1.14448082447052, 0.5350333452224731, -0.9667453169822693, -1.0580638647079468, -2.2010903358459473, 0.07798468321561813, 1.8597149848937988, -1.3805193901062012, 1.0879616737365723, -0.8143323659896851, 1.9490737915039062, 0.21196547150611877, -1.3072513341903687, -0.8261590600013733, 1.6460539102554321, 1.095367431640625, 0.09473760426044464, 1.4668688774108887, -0.005540103185921907, -0.3776092529296875, -0.0564529225230217, -2.1081724166870117, -0.6760526299476624, -0.11911032348871231, 0.02380002662539482, -1.1922467947006226, 1.8536449670791626, -0.8499791622161865, -1.2753762006759644, 0.6010007262229919, -0.5244473814964294, 0.016032187268137932, 0.15967923402786255, 0.0355568565428257, -2.1901443004608154, 0.2843003273010254, -0.5774217844009399, 0.1639580875635147, 1.2886680364608765, -0.1921708881855011, 0.5237479209899902, 1.2354230880737305, 1.3563247919082642, 1.365385890007019, -1.0583962202072144, 0.27658501267433167, 0.8486860990524292], [-3.467604637145996, -0.5924422144889832, -1.9868556261062622, 2.156107187271118, 0.4803948402404785, -1.826351523399353, 0.02019844576716423, 0.6968889832496643, 0.023632800206542015, -0.302543044090271, -0.32653045654296875, 0.5859342813491821, -0.8288165330886841, 0.2267707884311676, 2.226658821105957, -1.605089783668518, 0.08036268502473831, 1.9836400747299194, -0.0530085563659668, -1.310105800628662, 0.36093443632125854, 0.11409059911966324, -1.73443603515625, -1.0760666131973267, -1.4950202703475952, -0.5012762546539307, 2.1305816173553467, 1.3989518880844116, -1.1373882293701172, -0.22216686606407166, -2.3960001468658447, -0.19529388844966888, -0.6102783679962158, 0.08616728335618973, 1.2731072902679443, -0.1745033860206604, 0.16704805195331573, -0.10755216330289841, 0.013703415170311928, 1.8950449228286743, 0.3510226011276245, 0.048544567078351974, -0.11732892692089081, -0.5809670686721802, -0.01846781000494957, -0.34581202268600464, 0.6244943737983704, 0.0072934431955218315, -0.854189932346344, 0.7242521047592163, -2.129606246948242, 0.5472646355628967, -0.34681475162506104, -0.7433812022209167, 1.0443171262741089, 0.9320998787879944, -0.08689213544130325, -0.22881434857845306, -1.0376205444335938, 0.846889078617096, -2.121450901031494, -0.5761076211929321, -2.379081964492798, -0.12556162476539612, -0.6995493173599243, 1.5498688220977783, 0.18638651072978973, 0.30179670453071594, 0.804990291595459, -1.1689027547836304], [1.1097909212112427, 1.3019022941589355, 0.2532607614994049, 0.9389157891273499, 0.07110065221786499, -2.488090991973877, 1.3930020332336426, 0.7555659413337708, -1.2545355558395386, -0.1253974735736847, 0.02849642001092434, -1.2646883726119995, 1.2747331857681274, -0.6356675028800964, 1.351024866104126, 0.4419185519218445, 1.3215892314910889, -0.4457889795303345, 1.3230561017990112, 0.2882055640220642, 0.8752288222312927, -0.018561724573373795, -0.44052669405937195, 1.1700682640075684, 0.4812484681606293, -0.7489339709281921, -0.1536409854888916, 0.9044121503829956, 0.1388629823923111, -0.31486931443214417, 0.6563006043434143, -1.726969838142395, -0.7915902137756348, -1.1388881206512451, -0.9209718704223633, -0.20167399942874908, 1.3545230627059937, -0.5080669522285461, -1.0542163848876953, -0.9672793745994568, -0.5869001150131226, -0.5214324593544006, 0.7549601197242737, -1.4764807224273682, -0.550322949886322, 0.13942185044288635, 1.2102090120315552, 2.091555595397949, -0.4610937535762787, 0.35064610838890076, -0.3627181947231293, -1.5216481685638428, 1.055195689201355, 0.1891162097454071, -0.9747977256774902, -0.8400007486343384, -1.1813147068023682, -0.10593412816524506, 1.4403115510940552, 0.6467636823654175, -0.6159198880195618, -0.9748803973197937, -0.2895357012748718, -0.3510175049304962, -0.9283972978591919, -2.2037129402160645, -0.3960105776786804, 0.7686980962753296, 1.6738855838775635, -0.42049428820610046], [-0.03958633914589882, 0.7715886831283569, -0.5561782121658325, 0.7823039889335632, -1.5745174884796143, -1.2202110290527344, -0.8752236366271973, 0.062218137085437775, -0.30849355459213257, 0.328181654214859, -0.03964166343212128, 1.0172048807144165, 1.3897860050201416, 0.6965205669403076, -0.41320106387138367, 0.6417480111122131, 0.8038228154182434, 0.048151396214962006, -0.12069276720285416, 0.4979976713657379, 0.29296064376831055, -1.7260934114456177, -1.397423505783081, -1.8475488424301147, 0.6875718235969543, 0.7161420583724976, -0.6102203130722046, 0.6980676054954529, 0.17975136637687683, 0.5254415273666382, 0.5096674561500549, -0.11407601833343506, 0.6303573250770569, -0.1585003137588501, 0.4303087592124939, 0.8165830969810486, -0.31781721115112305, -0.09671612083911896, -0.35507625341415405, -0.649226725101471, 0.13535068929195404, -1.7858892679214478, -2.0493874549865723, 1.7938439846038818, -0.9502183198928833, 0.7404763698577881, -0.9561804533004761, -0.22176308929920197, 2.4788551330566406, -1.9021482467651367, 0.8061060905456543, 0.5315218567848206, 0.8200269341468811, -0.22452548146247864, 0.18059778213500977, 0.2501384913921356, -0.8308793902397156, 0.9407126307487488, -1.0608420372009277, -2.222834348678589, 0.4422459006309509, -0.5792306661605835, 1.3595319986343384, -0.15013886988162994, -2.015510320663452, -0.7601553201675415, 0.46063899993896484, -1.2026278972625732, -1.0211831331253052, -0.10145614296197891], [1.6367931365966797, -1.2364848852157593, 1.2654300928115845, 0.8289097547531128, 0.2386673390865326, -0.9513953328132629, 1.4042999744415283, 0.4677097201347351, 1.14752197265625, 0.21006512641906738, 0.013359091244637966, 0.20079956948757172, -0.652015209197998, -1.2317901849746704, -0.7738243937492371, -2.149587869644165, 1.9284344911575317, 0.2781268358230591, -1.5921083688735962, 0.29261305928230286, -0.7054240703582764, 0.9347184896469116, 1.2246809005737305, -1.1098086833953857, 0.8649235367774963, 1.2565076351165771, 1.3990942239761353, 0.3965991139411926, -0.42187973856925964, -0.35499435663223267, -0.7307186126708984, -0.4291057288646698, -0.9675933122634888, 0.447100430727005, -1.3265180587768555, -2.6558127403259277, -1.3451647758483887, -0.038454569876194, 0.539569616317749, -0.2072206735610962, 0.6081919074058533, -0.4204123318195343, 0.7286632061004639, -0.7176520228385925, -0.7419347167015076, -0.15919972956180573, -1.1769062280654907, -0.4954063594341278, -0.20434680581092834, 0.5588781833648682, -0.16039647161960602, 0.35775262117385864, -1.006560206413269, -0.6368533968925476, -0.24407000839710236, -1.6096160411834717, 1.5302581787109375, -2.0818238258361816, -0.07056284695863724, -0.413961797952652, -0.5289332270622253, -0.23131144046783447, 1.2929526567459106, 0.6182065606117249, -0.4897776246070862, -0.1393800973892212, 1.8085201978683472, 0.6220486760139465, 1.032950758934021, 1.3460092544555664], [-1.2928622961044312, 1.186232089996338, 1.2781156301498413, -1.766940951347351, 0.9406410455703735, 0.6955577731132507, -0.7860601544380188, -1.5319355726242065, 1.064923882484436, 0.7014485597610474, -0.20528803765773773, 1.0955358743667603, 1.308656930923462, -0.8783389925956726, 0.6448330283164978, 0.7327595949172974, 0.9820383787155151, -1.5926709175109863, -0.18976998329162598, -2.1149697303771973, 0.11593158543109894, -0.42151758074760437, 0.05774121358990669, -1.5339505672454834, 0.36580023169517517, -0.35113903880119324, -0.30835652351379395, 1.448643684387207, -0.5980699062347412, 1.8098667860031128, 0.4675021469593048, -0.28778889775276184, -1.4930334091186523, -0.6240761876106262, 0.7440525889396667, -0.13558782637119293, -0.3464633822441101, -1.5475008487701416, -1.2276469469070435, 0.18657661974430084, 1.367075800895691, -0.3081885576248169, -1.3142844438552856, 0.35740432143211365, 1.043432593345642, 1.397399663925171, 0.27179625630378723, -2.0742526054382324, -1.8981562852859497, 2.838268280029297, 0.683628261089325, 0.41719913482666016, -0.4127011001110077, 0.3952673077583313, -0.5046959519386292, -0.7047301530838013, -1.6730629205703735, -0.7127963304519653, 0.5007957220077515, -0.6949547529220581, 0.6166484355926514, 0.14676032960414886, -0.23575738072395325, 0.11054803431034088, -0.031940020620822906, -0.022563129663467407, -1.7154089212417603, -0.3411325216293335, 0.7000875473022461, 2.0129880905151367], [1.0699135065078735, -1.967200756072998, -0.15714780986309052, -0.8753332495689392, -1.0806586742401123, 0.8091142773628235, -0.2833382785320282, -0.6733196377754211, 1.8472988605499268, 2.2328968048095703, 0.14081119000911713, -0.5095331072807312, -1.8094154596328735, 1.0969955921173096, -0.39383187890052795, 0.8384184837341309, 0.7005000710487366, -0.6042508482933044, 0.14003023505210876, 0.854315996170044, -0.2955067455768585, 2.0858957767486572, 0.16115006804466248, 0.23296497762203217, 1.2026854753494263, -0.8811586499214172, -0.6491857171058655, -0.6035438179969788, 0.25287091732025146, -1.0871864557266235, -1.1090511083602905, -0.8663045763969421, 2.2877037525177, -1.360094428062439, -0.13344749808311462, 0.017439477145671844, -1.8361090421676636, -1.4295333623886108, 0.8745023608207703, -1.740095615386963, -2.8446156978607178, 0.18584004044532776, 1.076222538948059, 1.4809939861297607, 1.044053077697754, -0.06772755831480026, 0.731387197971344, 0.0012689536670222878, -1.0019729137420654, -1.3828635215759277, 1.4857234954833984, 2.372230291366577, -1.1484885215759277, 0.21108564734458923, 0.23627126216888428, -2.4533677101135254, 1.5296213626861572, -1.2025896310806274, 0.5845800042152405, 0.14016835391521454, -1.0278605222702026, -0.4816436171531677, -0.1831725686788559, -0.45607703924179077, 0.41328155994415283, 0.036625321954488754, -1.212277889251709, -0.31561392545700073, -1.64320707321167, 0.49899548292160034], [0.8642928600311279, -0.6844402551651001, -0.10132422298192978, -0.5059205889701843, 1.1837739944458008, 0.8151512145996094, -0.3458191454410553, 0.5330387949943542, -0.15408895909786224, -0.4135763347148895, 0.5008264183998108, -0.8057782649993896, 0.5142449140548706, 1.0426764488220215, 0.5127596259117126, -1.2017489671707153, 0.7229970693588257, 0.8211948275566101, 0.9599484801292419, 0.5496016144752502, -1.062652826309204, 0.2879432737827301, 0.12725333869457245, 0.1714269369840622, -0.15207624435424805, 0.007268228568136692, -2.047072649002075, 0.10115447640419006, 0.2269585132598877, -1.0355783700942993, -1.2654038667678833, 0.28528833389282227, 0.41993096470832825, -0.2060178965330124, -0.6502509713172913, 1.5688457489013672, 1.1230862140655518, 0.5951850414276123, 0.8497157692909241, -0.5868121385574341, 0.8080899119377136, 0.38101017475128174, -0.8768452405929565, -0.21919018030166626, 0.26526790857315063, -1.215962290763855, 0.2394786775112152, 0.06995191425085068, 0.266370564699173, -0.5412341952323914, 1.1735153198242188, 2.8225724697113037, 0.7196919322013855, 0.20374520123004913, -0.7841286063194275, 0.3705730438232422, -0.16245409846305847, 0.10330215841531754, 1.1738697290420532, -0.24721047282218933, -0.12085007131099701, -0.4139300286769867, 0.31758585572242737, 0.3470650315284729, -0.47694674134254456, -0.18511086702346802, 0.5330585241317749, -1.499718189239502, -0.855362057685852, 1.7436904907226562], [2.445511817932129, -1.388648271560669, -1.33440101146698, -1.631497859954834, -0.982694685459137, 0.15229591727256775, 0.018442116677761078, 0.7919120192527771, -0.29383641481399536, -1.2623523473739624, 0.21276652812957764, 1.8547102212905884, 0.40489745140075684, -0.36171984672546387, -1.111391544342041, 1.7399797439575195, 1.70579195022583, -0.3597234785556793, -1.1644713878631592, 0.13186873495578766, 0.4068291485309601, 0.5515672564506531, -0.2844773828983307, 1.3498626947402954, 0.4398581385612488, 0.564218282699585, -0.3484295606613159, 0.21691463887691498, -0.5585595965385437, -1.6047255992889404, 1.3881269693374634, 1.2066398859024048, -0.3715479373931885, 1.6871544122695923, -0.35336989164352417, 0.051080476492643356, -0.024962861090898514, -0.6507844924926758, 1.8332369327545166, -0.067175954580307, 1.3004206418991089, -1.0599461793899536, -1.2316213846206665, 1.1888339519500732, -0.4319838583469391, 0.8091236352920532, -1.7251646518707275, 1.7963933944702148, 1.6185619831085205, 0.8129748106002808, -0.12960337102413177, -1.8326106071472168, -0.6797130107879639, 0.8956472277641296, 1.758044958114624, -0.35186144709587097, 0.008265255019068718, 0.3679424226284027, -0.08503264933824539, 2.0626440048217773, -0.0037014561239629984, 1.5316542387008667, -0.2714602053165436, 1.1085506677627563, 0.33795514702796936, -0.9392845034599304, 0.3358834981918335, 2.2750375270843506, 0.6724454164505005, -0.40251556038856506], [-0.23360300064086914, 0.2410593181848526, -0.15750686824321747, -0.6917137503623962, -1.161821961402893, -1.2548906803131104, 2.480722427368164, 0.2739003002643585, -2.3441336154937744, -1.4346213340759277, 0.5810766220092773, -1.9163459539413452, -0.5919109582901001, 0.170929953455925, 0.2760136127471924, -0.3984447121620178, -0.1184661015868187, -0.9499393701553345, 1.2105165719985962, -1.626306176185608, -0.9348533749580383, -0.7881023287773132, 0.3968943953514099, 0.3457278609275818, -1.2533220052719116, -0.7878350615501404, -1.3115222454071045, 0.1591726541519165, -0.38672012090682983, 0.6603729128837585, -0.7048208117485046, -0.24566389620304108, 0.09780246019363403, 1.8303014039993286, 0.39419567584991455, 0.7767348289489746, -0.37408143281936646, 0.5416340231895447, -0.9731485843658447, 0.48466479778289795, 0.20590521395206451, 0.5483015775680542, 0.23035088181495667, -0.560158371925354, 0.5837802886962891, 0.3141638934612274, -0.8258086442947388, -1.4860731363296509, 0.33076027035713196, -0.457311749458313, 0.05944797396659851, 0.555932879447937, -0.3585776090621948, -0.40757548809051514, -0.23279564082622528, -1.5462478399276733, 0.8542478084564209, -0.07099601626396179, 0.8178693056106567, 0.29092955589294434, -0.3497565984725952, 1.1652796268463135, 0.4701234996318817, 0.46837183833122253, -0.4853198230266571, 0.52093505859375, -1.4238739013671875, 1.4190378189086914, 2.5224626064300537, -1.0963730812072754], [0.15621578693389893, 0.6982500553131104, -1.8039746284484863, 0.20225924253463745, 0.33694055676460266, 0.19296976923942566, 0.09625782817602158, 0.24697963893413544, -0.6870889663696289, -0.8146517872810364, 0.7177279591560364, 0.786690354347229, 0.21649393439292908, -0.7802314758300781, 0.28943026065826416, 0.5572245717048645, 0.954452395439148, 0.7735992670059204, -1.314761996269226, -0.93638014793396, 0.049034491181373596, -0.27890077233314514, 1.5404211282730103, 0.07808845490217209, 0.0868656262755394, 0.05312669277191162, 0.6456570625305176, 0.027791501954197884, -2.3148481845855713, 2.0349485874176025, 0.23966829478740692, 0.42835018038749695, -0.6051558256149292, -0.0038612426724284887, 0.4194839894771576, -0.8096935749053955, 0.9061692357063293, 0.34585434198379517, -3.2650675773620605, -0.6528272032737732, -0.1539628654718399, -0.4163820445537567, 0.7206942439079285, -0.3033151626586914, -0.1901128888130188, -0.8473642468452454, -0.3509562313556671, -0.05822982266545296, 0.31801092624664307, 2.025122880935669, 1.4632713794708252, 1.0967451333999634, 0.5562528967857361, 0.6618943810462952, -0.12408585101366043, 2.8831145763397217, 0.2890775501728058, -0.522932231426239, 1.0620992183685303, -0.3692452013492584, 1.0750157833099365, 0.6828508377075195, 1.5610642433166504, -1.4352010488510132, -0.26361480355262756, 0.11447491496801376, -0.052481915801763535, 0.4994382858276367, 0.7039594650268555, 0.2937205731868744], [0.7063030004501343, -1.4193435907363892, -0.4110919237136841, 1.7407938241958618, 0.027225598692893982, -1.0186245441436768, 1.0310698747634888, -0.7134324908256531, -0.9132281541824341, 0.962261438369751, 0.8858211636543274, -0.2538401484489441, 1.7809805870056152, 0.21704646944999695, 0.49769166111946106, 0.14644575119018555, 0.8683226704597473, 0.5347753763198853, 0.6045018434524536, 0.9449782967567444, -0.41262009739875793, -0.37936997413635254, 3.3844683170318604, -0.04727819934487343, 0.23254601657390594, -0.5442861318588257, 0.9605587124824524, 1.3005222082138062, 0.6727029085159302, -0.5361359119415283, 2.1609458923339844, -0.2739124000072479, -0.12664546072483063, -0.33568477630615234, -0.7400486469268799, -0.6552751660346985, 1.6334185600280762, 0.7532209753990173, -0.3351599872112274, 0.3841952085494995, -0.9331059455871582, 0.29098376631736755, 0.016071751713752747, 2.4093379974365234, -0.4387708008289337, -0.4841739535331726, -1.3081748485565186, 0.39270228147506714, -0.19507189095020294, 0.061372555792331696, -1.8018254041671753, 1.5362253189086914, -0.34923285245895386, 0.9069385528564453, 0.40931040048599243, 0.42970359325408936, 0.660942792892456, 0.6753584146499634, 1.2904350757598877, 0.6973778605461121, -2.2147159576416016, 0.03603136166930199, -1.179612398147583, 0.5501375794410706, 1.9984681606292725, -0.4585285484790802, -1.6222989559173584, -0.622413694858551, -2.0964701175689697, 0.8194953203201294], [-0.6315962076187134, -1.3845382928848267, 0.4468449354171753, 0.09535059332847595, 0.3875683844089508, 0.5680106282234192, 0.5095880031585693, 1.7758762836456299, -0.3950616419315338, 1.2547121047973633, -0.33301252126693726, 0.9645686149597168, -0.8052039742469788, -0.32508739829063416, -2.3609931468963623, -0.28935110569000244, -0.35256075859069824, -0.13472773134708405, -0.9918180704116821, -0.43689319491386414, -0.47242841124534607, -1.040303111076355, -1.1281064748764038, -0.6255194544792175, 0.895254909992218, 1.6666253805160522, -0.0795164406299591, -2.5551235675811768, -0.026164861395955086, 0.601041853427887, -1.3783951997756958, 0.1744520664215088, 0.19216963648796082, 1.2862948179244995, -0.7525246739387512, 1.6358627080917358, 0.4422389268875122, -0.6266348958015442, -0.6869161128997803, 0.12049148976802826, 0.9294725060462952, -0.10347910225391388, -0.7835367321968079, -0.9964900016784668, -1.2607636451721191, 0.887509286403656, 0.9954971671104431, -0.9502460956573486, 0.011576410382986069, -2.2391223907470703, 2.5003085136413574, 1.1774200201034546, 0.2238454669713974, 1.019740104675293, -0.19390814006328583, 0.5418007969856262, -0.055931609123945236, -0.5964322686195374, -1.3390192985534668, 1.4117034673690796, 0.47758424282073975, -0.3889204263687134, 0.6106300354003906, -0.4562484323978424, -0.38643428683280945, -0.5748535990715027, 0.4056434631347656, -0.5990669131278992, -0.9185624122619629, -1.8013907670974731], [-0.9520278573036194, 0.32421791553497314, -0.40875062346458435, 1.4599610567092896, 1.1979167461395264, -0.8049061298370361, -1.8020983934402466, 1.182795524597168, 0.7273520827293396, 0.012033252976834774, -1.1566340923309326, 0.2548290193080902, 0.4152505397796631, -0.17475122213363647, -0.03973911330103874, -0.10920204222202301, -1.4839515686035156, 0.9470715522766113, 1.1201486587524414, 0.2444034069776535, -0.6712763905525208, -1.3282897472381592, 3.192432165145874, -0.5392500162124634, -0.3495247960090637, -0.6480036973953247, -0.1075221598148346, 2.622624158859253, -0.32713890075683594, 0.46045994758605957, -0.3198500871658325, -0.622113823890686, -0.546775221824646, -0.14415860176086426, 1.2115232944488525, 0.03903697431087494, 0.19101770222187042, 0.33918875455856323, 0.42132681608200073, 0.2732911705970764, 0.5550609827041626, 1.0864765644073486, 0.36792999505996704, -0.7352631688117981, -0.5909485816955566, -1.913561463356018, 1.1378284692764282, 0.4762137234210968, -0.3792227506637573, -1.3636873960494995, -0.06228348985314369, -0.5244929194450378, 0.12489921599626541, -0.6314524412155151, -0.2974832355976105, -0.08534348756074905, -0.979587197303772, -0.581879734992981, -0.2733287215232849, 0.546602189540863, 0.6315075755119324, -1.2371339797973633, -0.7291518449783325, -1.5682286024093628, -2.639827251434326, -0.9921295642852783, 0.6288851499557495, 1.9062002897262573, 1.9182089567184448, 0.3737029433250427], [1.0577747821807861, 1.024393916130066, 0.31028661131858826, -0.29837971925735474, 1.093001365661621, -0.3551842272281647, 0.2071915864944458, 0.05026599392294884, -0.22419989109039307, -0.5026038885116577, 0.1772446483373642, -0.390140563249588, -0.1809779703617096, 1.1866328716278076, 2.258291244506836, 2.084818124771118, -0.18759846687316895, -1.554081916809082, -1.010676383972168, -0.7212377786636353, 0.5685429573059082, 1.3371546268463135, -0.5673259496688843, -0.4228360950946808, -0.33073440194129944, -0.4391044080257416, 0.11206324398517609, -2.2055504322052, 0.05033137649297714, -2.3813161849975586, -3.5429675579071045, 0.5098833441734314, 1.6506197452545166, 1.7737298011779785, -0.9215187430381775, -0.30532804131507874, 1.4006541967391968, 1.192572832107544, 0.35457679629325867, 1.5741479396820068, 1.336687445640564, -0.14985468983650208, 0.2708500325679779, -0.005228765774518251, 0.3567679226398468, -0.23374386131763458, -0.08167829364538193, 0.08645781874656677, 1.1564966440200806, 0.1905835121870041, 1.6894603967666626, -0.3209328055381775, -0.529151439666748, -0.8376772999763489, 0.5622466802597046, 1.02858567237854, 2.4938387870788574, -0.03523929789662361, 1.1997123956680298, 2.3267345428466797, 0.9619269371032715, -0.07346519827842712, -0.39196300506591797, -2.3698954582214355, 0.4960106611251831, 0.3935980796813965, -0.08900552988052368, 0.5945647954940796, 0.4221624433994293, 0.23348668217658997], [-0.011260943487286568, -1.120560884475708, 0.07257089763879776, 0.5039306282997131, -0.38201457262039185, 0.2358212023973465, 0.22947779297828674, 1.6445151567459106, -0.545268177986145, 2.5012242794036865, 0.5262479186058044, 1.300431251525879, -0.39257073402404785, -2.154600143432617, 1.33756685256958, -0.6842649579048157, 0.008463931269943714, -1.498746395111084, 0.47545313835144043, 1.9191614389419556, 0.28416141867637634, -1.4522253274917603, -1.116978406906128, -0.021894706413149834, 0.9942949414253235, -0.36599424481391907, 1.3564225435256958, -1.527300238609314, 0.3591661751270294, -1.2054948806762695, -0.691754937171936, -0.34775400161743164, 0.4914230704307556, -1.1264828443527222, 1.4063698053359985, -1.099577784538269, -0.23702624440193176, -0.18977361917495728, 1.356138825416565, 0.6011810898780823, 0.9531124234199524, 0.3494047522544861, -0.5745034217834473, -0.7130845785140991, -0.838853657245636, 1.127086877822876, -1.233534574508667, -0.8642869591712952, -0.3226979672908783, -0.4645378589630127, -0.3750862181186676, 1.2429739236831665, -0.6500008702278137, -0.41570910811424255, -0.7678437829017639, -0.5812945365905762, 1.3052688837051392, 0.2139451950788498, -1.0491095781326294, 0.8559850454330444, 1.5544297695159912, 0.3976323902606964, -0.2642357349395752, -0.7426775693893433, 0.9963453412055969, -1.4302819967269897, 0.7706234455108643, 1.704546332359314, -0.07270032167434692, 1.1278681755065918], [-0.18128594756126404, 0.7679879665374756, -0.08983474969863892, -1.2742445468902588, -0.031408537179231644, 0.42140448093414307, 0.7220390439033508, 0.6712164282798767, -0.09799414873123169, 0.1767456978559494, 0.8972131013870239, -0.8681837320327759, 0.08687976747751236, 1.6838151216506958, 0.04049953073263168, 0.6089691519737244, -1.9031401872634888, 2.003865957260132, -0.8130072355270386, -0.4460652470588684, 1.760499358177185, -0.8937281370162964, -0.30819350481033325, -2.1570277214050293, -1.1853773593902588, -0.10120408236980438, 0.04907361418008804, 0.2929381728172302, -0.7143484950065613, -1.839002013206482, 1.5416916608810425, -0.4096807539463043, -0.7374477386474609, 0.5746737718582153, -0.8173245787620544, -1.0269825458526611, 1.845281958580017, 0.6782819628715515, -0.3018801212310791, -0.6630101799964905, -0.11614498496055603, 0.3345932960510254, 1.1702576875686646, -0.7446957230567932, 0.10483676940202713, -1.4219449758529663, -0.03476995229721069, 0.20071114599704742, -0.14202100038528442, -0.49817201495170593, -0.4962832033634186, -0.4739629030227661, -0.28572964668273926, -1.857115626335144, 1.9394346475601196, 0.00511568458750844, 1.5743829011917114, -0.09239626675844193, -0.7926651835441589, -0.8436810374259949, 1.9090908765792847, -1.6296186447143555, 0.7964950203895569, -1.595828652381897, -0.19486117362976074, 1.1135172843933105, -0.008394564501941204, -0.6965034008026123, -0.7378952503204346, -0.7460854053497314]], "dtype": "float32"}], "e": {"__nd__": [[-0.44280508160591125, -0.3269571363925934, 0.5503101944923401, -2.3966851234436035, 0.4743921458721161, 1.7401264905929565, 0.5670584440231323, 0.15752941370010376, 2.066472291946411, 0.7767070531845093, 0.04891211539506912, -0.9749610424041748, -0.04685007780790329, -1.5128178596496582, -0.2813457250595093, 0.031803201884031296, -1.359370231628418, -3.467604637145996, 1.1097909212112427, -0.03958633914589882, 1.6367931365966797, -1.2928622961044312, 1.0699135065078735, 0.8642928600311279, 2.445511817932129, -0.23360300064086914, 0.15621578693389893, 0.7063030004501343, -0.6315962076187134, -0.9520278573036194, 1.0577747821807861, -0.011260943487286568, -0.18128594756126404], [1.4303609132766724, -0.5775059461593628, 0.0347757525742054, 0.7022680640220642, -1.4445693492889404, 0.7271178364753723, -1.2659229040145874, -1.557235598564148, -0.33636578917503357, -1.4913909435272217, 2.338364362716675, -0.4069940447807312, 0.46792763471603394, 0.05040311813354492, 0.019394321367144585, 0.11724888533353806, -0.5196812152862549, -0.5924422144889832, 1.3019022941589355, 0.7715886831283569, -1.2364848852157593, 1.186232089996338, -1.967200756072998, -0.6844402551651001, -1.388648271560669, 0.2410593181848526, 0.6982500553131104, -1.4193435907363892, -1.3845382928848267, 0.32421791553497314, 1.024393916130066, -1.120560884475708, 0.7679879665374756], [0.2963597774505615, -0.6575673818588257, -0.6521821618080139, 0.4630788266658783, 0.5174800753593445, 0.1980314552783966, -0.3728887736797333, -0.33581310510635376, 0.902146577835083, 0.33965328335762024, 0.6240823268890381, 0.8444125056266785, -0.1703261286020279, -0.6119596362113953, -0.0032459774520248175, 0.40849265456199646, -0.5272858142852783, -1.9868556261062622, 0.2532607614994049, -0.5561782121658325, 1.2654300928115845, 1.2781156301498413, -0.15714780986309052, -0.10132422298192978, -1.33440101146698, -0.15750686824321747, -1.8039746284484863, -0.4110919237136841, 0.4468449354171753, -0.40875062346458435, 0.31028661131858826, 0.07257089763879776, -0.08983474969863892], [-1.6049634218215942, 1.0341880321502686, -1.6184197664260864, 1.7880549430847168, 0.6184548139572144, 1.1418461799621582, -0.41530683636665344, 0.3904436230659485, 0.26850220561027527, 1.6807154417037964, -0.9215217232704163, -0.38245120644569397, 1.387450933456421, -0.004065188579261303, -0.4133109748363495, 0.6744706630706787, 1.1005467176437378, 2.156107187271118, 0.9389157891273499, 0.7823039889335632, 0.8289097547531128, -1.766940951347351, -0.8753332495689392, -0.5059205889701843, -1.631497859954834, -0.6917137503623962, 0.20225924253463745, 1.7407938241958618, 0.09535059332847595, 1.4599610567092896, -0.29837971925735474, 0.5039306282997131, -1.2742445468902588], [-0.16352947056293488, 1.2508761882781982, -0.3056657016277313, 0.08623840659856796, 1.0363788604736328, 1.4576762914657593, 0.10583879798650742, -0.4694134294986725, -1.192186713218689, -0.3146990239620209, -0.3671850562095642, -0.1157383844256401, 0.5502406358718872, -0.28125864267349243, 0.9081853032112122, 0.9322047829627991, 0.17424888908863068, 0.4803948402404785, 0.07110065221786499, -1.5745174884796143, 0.2386673390865326, 0.9406410455703735, -1.0806586742401123, 1.1837739944458008, -0.982694685459137, -1.161821961402893, 0.33694055676460266, 0.027225598692893982, 0.3875683844089508, 1.1979167461395264, 1.093001365661621, -0.38201457262039185, -0.031408537179231644], [0.20739614963531494, -0.46780216693878174, 0.24076202511787415, -0.8958799839019775, 0.3666855990886688, -0.19083155691623688, -0.5033165812492371, -0.06800797581672668, -0.5218479633331299, 0.5839623212814331, 0.09364904463291168, -0.7863340973854065, -2.067652940750122, -1.353285551071167, -0.45726868510246277, 0.21576137840747833, 1.3651138544082642, -1.826351523399353, -2.488090991973877, -1.2202110290527344, -0.9513953328132629, 0.6955577731132507, 0.8091142773628235, 0.8151512145996094, 0.15229591727256775, -1.2548906803131104, 0.19296976923942566, -1.0186245441436768, 0.5680106282234192, -0.8049061298370361, -0.3551842272281647, 0.2358212023973465, 0.42140448093414307], [0.482186883687973, -1.3991048336029053, 1.2973673343658447, -0.18807265162467957, 0.9899292588233948, 0.37601813673973083, -1.1701397895812988, -0.5793206095695496, 1.649485468864441, -0.20747512578964233, -1.907318353652954, 1.0368752479553223, -1.148400902748108, -0.30672019720077515, -0.584888219833374, -2.0696938037872314, -0.8838122487068176, 0.02019844576716423, 1.3930020332336426, -0.8752236366271973, 1.4042999744415283, -0.7860601544380188, -0.2833382785320282, -0.3458191454410553, 0.018442116677761078, 2.480722427368164, 0.09625782817602158, 1.0310698747634888, 0.5095880031585693, -1.8020983934402466, 0.2071915864944458, 0.22947779297828674, 0.7220390439033508], [-1.051658034324646, 0.6038889288902283, 1.5656354427337646, 1.4568904638290405, 0.15871688723564148, -0.22260625660419464, 1.3310825824737549, -0.06994400173425674, -1.06048583984375, 0.3123409152030945, -1.0797979831695557, -1.0021882057189941, -0.8921011090278625, 0.3294171690940857, 0.7654758095741272, -0.0740073025226593, 0.05320873111486435, 0.6968889832496643, 0.7555659413337708, 0.062218137085437775, 0.4677097201347351, -1.5319355726242065, -0.6733196377754211, 0.5330387949943542, 0.7919120192527771, 0.2739003002643585, 0.24697963893413544, -0.7134324908256531, 1.7758762836456299, 1.182795524597168, 0.05026599392294884, 1.6445151567459106, 0.6712164282798767], [0.9139016270637512, 0.7962014675140381, -1.0067064762115479, 0.10398590564727783, -0.1153831034898758, -1.8103649616241455, 0.11083976924419403, 1.0885459184646606, -0.4264789819717407, 0.2743520140647888, 1.7528631687164307, -0.8868524432182312, 1.1618155241012573, -0.7563528418540955, -0.6389152407646179, -1.8670004606246948, -0.49330997467041016, 0.023632800206542015, -1.2545355558395386, -0.30849355459213257, 1.14752197265625, 1.064923882484436, 1.8472988605499268, -0.15408895909786224, -0.29383641481399536, -2.3441336154937744, -0.6870889663696289, -0.9132281541824341, -0.3950616419315338, 0.7273520827293396, -0.22419989109039307, -0.545268177986145, -0.09799414873123169], [0.9496085047721863, -1.9853712320327759, -0.7638749480247498, 0.13824066519737244, 0.5556033849716187, -3.0281593799591064, -1.3167635202407837, -0.6118496060371399, 1.5862222909927368, -0.06353042274713516, 1.27618408203125, -0.5149103999137878, 1.6713365316390991, -0.4417758882045746, -0.9001184105873108, -2.2225489616394043, 0.39334559440612793, -0.302543044090271, -0.1253974735736847, 0.328181654214859, 0.21006512641906738, 0.7014485597610474, 2.2328968048095703, -0.4135763347148895, -1.2623523473739624, -1.4346213340759277, -0.8146517872810364, 0.962261438369751, 1.2547121047973633, 0.012033252976834774, -0.5026038885116577, 2.5012242794036865, 0.1767456978559494], [-1.9995919466018677, -0.653784990310669, 1.0487165451049805, 1.0937821865081787, -0.07883309572935104, 1.4617379903793335, 2.617859363555908, 1.2366031408309937, -2.2255513668060303, -1.1337568759918213, -0.4690597355365753, 1.3784652948379517, -0.4224424660205841, -0.3730677664279938, -1.4736191034317017, 2.154747486114502, -0.41013064980506897, -0.32653045654296875, 0.02849642001092434, -0.03964166343212128, 0.013359091244637966, -0.20528803765773773, 0.14081119000911713, 0.5008264183998108, 0.21276652812957764, 0.5810766220092773, 0.7177279591560364, 0.8858211636543274, -0.33301252126693726, -1.1566340923309326, 0.1772446483373642, 0.5262479186058044, 0.8972131013870239], [-1.4915637969970703, -0.9695145487785339, 0.4238404333591461, 0.5546332597732544, 1.3630274534225464, -0.6878136396408081, -0.47602495551109314, 0.0822744220495224, -1.3067318201065063, 0.6076449155807495, 0.6435851454734802, 1.878701090812683, 0.04649841785430908, 0.9236100912094116, -0.8791859745979309, 0.7767680287361145, 0.744485080242157, 0.5859342813491821, -1.2646883726119995, 1.0172048807144165, 0.20079956948757172, 1.0955358743667603, -0.5095331072807312, -0.8057782649993896, 1.8547102212905884, -1.9163459539413452, 0.786690354347229, -0.2538401484489441, 0.9645686149597168, 0.2548290193080902, -0.390140563249588, 1.300431251525879, -0.8681837320327759], [-0.6397563815116882, 2.1755552291870117, -0.5194843411445618, 0.49395355582237244, -0.8533170223236084, 1.7843877077102661, -0.822422444820404, -0.07052911072969437, -0.25600484013557434, -0.34570562839508057, 1.7241528034210205, -0.16797253489494324, 0.463575541973114, 0.7939465641975403, -0.4903890788555145, 1.355428695678711, -1.6125855445861816, -0.8288165330886841, 1.2747331857681274, 1.3897860050201416, -0.652015209197998, 1.308656930923462, -1.8094154596328735, 0.5142449140548706, 0.40489745140075684, -0.5919109582901001, 0.21649393439292908, 1.7809805870056152, -0.8052039742469788, 0.4152505397796631, -0.1809779703617096, -0.39257073402404785, 0.08687976747751236], [0.6146972179412842, -0.6427586674690247, -0.7078648805618286, -0.4891314208507538, -0.01614494062960148, -2.1313414573669434, 1.304918885231018, 0.6485756039619446, 1.2563016414642334, 0.7355226874351501, -1.2294820547103882, 0.8601852059364319, -0.08872223645448685, -1.1596095561981201, 1.0700184106826782, -0.5032809376716614, 0.14383380115032196, 0.2267707884311676, -0.6356675028800964, 0.6965205669403076, -1.2317901849746704, -0.8783389925956726, 1.0969955921173096, 1.0426764488220215, -0.36171984672546387, 0.170929953455925, -0.7802314758300781, 0.21704646944999695, -0.32508739829063416, -0.17475122213363647, 1.1866328716278076, -2.154600143432617, 1.6838151216506958], [-0.2695263624191284, 1.5768868923187256, -1.531586766242981, 0.8432779312133789, -0.7510843276977539, 1.1297353506088257, -1.2999851703643799, 1.3115166425704956, -2.290797710418701, 0.4321286082267761, 0.29648348689079285, 2.5827383995056152, -0.7259487509727478, -0.03551016375422478, 2.0850980281829834, 1.3775016069412231, 0.4464554786682129, 2.226658821105957, 1.351024866104126, -0.41320106387138367, -0.7738243937492371, 0.6448330283164978, -0.39383187890052795, 0.5127596259117126, -1.111391544342041, 0.2760136127471924, 0.28943026065826416, 0.49769166111946106, -2.3609931468963623, -0.03973911330103874, 2.258291244506836, 1.33756685256958, 0.04049953073263168], [0.5772497057914734, 0.6725166440010071, 0.7571381330490112, 0.3045929968357086, -0.34902632236480713, 2.7384490966796875, -0.887698769569397, 0.5755025744438171, -1.018538236618042, 0.28177958726882935, 1.0383546352386475, 1.214883804321289, -0.5107312798500061, 0.1489894539117813, 2.6727638244628906, 0.4537794888019562, 0.8242036700248718, -1.605089783668518, 0.4419185519218445, 0.6417480111122131, -2.149587869644165, 0.7327595949172974, 0.8384184837341309, -1.2017489671707153, 1.7399797439575195, -0.3984447121620178, 0.5572245717048645, 0.14644575119018555, -0.28935110569000244, -0.10920204222202301, 2.084818124771118, -0.6842649579048157, 0.6089691519737244], [0.768519937992096, 1.3221638202667236, -1.1508220434188843, -0.08861952275037766, 0.866935670375824, -0.7565467357635498, -0.725159764289856, -0.47775134444236755, -1.2047263383865356, -0.25079023838043213, -0.07750599086284637, 0.5528185963630676, 2.2371718883514404, -0.23167604207992554, 0.3668134808540344, -0.07391732931137085, 0.8512179255485535, 0.08036268502473831, 1.3215892314910889, 0.8038228154182434, 1.9284344911575317, 0.9820383787155151, 0.7005000710487366, 0.7229970693588257, 1.70579195022583, -0.1184661015868187, 0.954452395439148, 0.8683226704597473, -0.35256075859069824, -1.4839515686035156, -0.18759846687316895, 0.008463931269943714, -1.9031401872634888], [-0.5960761308670044, -0.32253751158714294, -0.009252703748643398, -1.0188319683074951, -0.36092904210090637, 0.23373563587665558, 0.749086856842041, -0.6518181562423706, -1.6912115812301636, 0.7118085026741028, 0.7704665064811707, -1.071024775505066, -0.5495654344558716, 0.039582252502441406, -1.3328737020492554, -0.0027761515229940414, 0.8740065097808838, 1.9836400747299194, -0.4457889795303345, 0.048151396214962006, 0.2781268358230591, -1.5926709175109863, -0.6042508482933044, 0.8211948275566101, -0.3597234785556793, -0.9499393701553345, 0.7735992670059204, 0.5347753763198853, -0.13472773134708405, 0.9470715522766113, -1.554081916809082, -1.498746395111084, 2.003865957260132], [-1.3519046306610107, 0.695327877998352, -0.8307551741600037, 1.381442666053772, -1.260677456855774, -0.7796242833137512, -0.39542320370674133, -1.1798397302627563, -1.036863088607788, -2.2047157287597656, 0.9282940030097961, -1.0935488939285278, -0.5428544282913208, -0.9694807529449463, -1.975158452987671, -2.5370566844940186, -0.9134808778762817, -0.0530085563659668, 1.3230561017990112, -0.12069276720285416, -1.5921083688735962, -0.18976998329162598, 0.14003023505210876, 0.9599484801292419, -1.1644713878631592, 1.2105165719985962, -1.314761996269226, 0.6045018434524536, -0.9918180704116821, 1.1201486587524414, -1.010676383972168, 0.47545313835144043, -0.8130072355270386], [0.8407791256904602, 0.6436320543289185, 0.7709883451461792, 0.4536643624305725, 0.5738793611526489, 0.04767397791147232, -0.14151355624198914, 1.910982608795166, 1.7973803281784058, -1.1806689500808716, -1.5538727045059204, -0.09742359071969986, 0.9850358963012695, -0.2913375198841095, 1.872833251953125, 0.5079435706138611, -0.16869518160820007, -1.310105800628662, 0.2882055640220642, 0.4979976713657379, 0.29261305928230286, -2.1149697303771973, 0.854315996170044, 0.5496016144752502, 0.13186873495578766, -1.626306176185608, -0.93638014793396, 0.9449782967567444, -0.43689319491386414, 0.2444034069776535, -0.7212377786636353, 1.9191614389419556, -0.4460652470588684], [0.20258431136608124, -0.6524344086647034, 0.14813627302646637, -0.8233689665794373, 0.23505324125289917, -0.9482946395874023, -2.060767412185669, -0.5838778018951416, 0.6871635913848877, 0.8340710401535034, 1.0284162759780884, 0.7226366400718689, 0.6661985516548157, 0.8576592206954956, -1.8000736236572266, -0.3319561183452606, 0.5192636251449585, 0.36093443632125854, 0.8752288222312927, 0.29296064376831055, -0.7054240703582764, 0.11593158543109894, -0.2955067455768585, -1.062652826309204, 0.4068291485309601, -0.9348533749580383, 0.049034491181373596, -0.41262009739875793, -0.47242841124534607, -0.6712763905525208, 0.5685429573059082, 0.28416141867637634, 1.760499358177185], [0.6788392663002014, -0.33590078353881836, 0.15589065849781036, 0.2561010718345642, -0.9842745661735535, -1.8571900129318237, 0.3990132212638855, -0.8370386958122253, 0.23915651440620422, 0.09810991585254669, 1.263260006904602, 0.753433346748352, 0.35585588216781616, -0.39089834690093994, -1.2157427072525024, -0.38803938031196594, -1.023606300354004, 0.11409059911966324, -0.018561724573373795, -1.7260934114456177, 0.9347184896469116, -0.42151758074760437, 2.0858957767486572, 0.2879432737827301, 0.5515672564506531, -0.7881023287773132, -0.27890077233314514, -0.37936997413635254, -1.040303111076355, -1.3282897472381592, 1.3371546268463135, -1.4522253274917603, -0.8937281370162964], [-0.4202357232570648, -1.1896487474441528, -1.0210883617401123, -0.332714706659317, 1.9724491834640503, 0.06039995700120926, 0.15459652245044708, 0.46694278717041016, 0.1811959594488144, 0.5057129859924316, 0.24031466245651245, 0.6489998698234558, -0.4162273108959198, 0.2086479663848877, -1.076509952545166, 0.021959858015179634, 2.190321922302246, -1.73443603515625, -0.44052669405937195, -1.397423505783081, 1.2246809005737305, 0.05774121358990669, 0.16115006804466248, 0.12725333869457245, -0.2844773828983307, 0.3968943953514099, 1.5404211282730103, 3.3844683170318604, -1.1281064748764038, 3.192432165145874, -0.5673259496688843, -1.116978406906128, -0.30819350481033325], [-0.8186793923377991, 0.5052012205123901, 0.8576659560203552, -0.12107150256633759, -0.566318690776825, 0.7954912781715393, -0.6434407830238342, 0.4839010536670685, -1.8825175762176514, 1.540401816368103, -1.3415066003799438, -1.3269141912460327, -0.632625162601471, 0.738278865814209, 0.8232683539390564, 0.39355191588401794, 1.14448082447052, -1.0760666131973267, 1.1700682640075684, -1.8475488424301147, -1.1098086833953857, -1.5339505672454834, 0.23296497762203217, 0.1714269369840622, 1.3498626947402954, 0.3457278609275818, 0.07808845490217209, -0.04727819934487343, -0.6255194544792175, -0.5392500162124634, -0.4228360950946808, -0.021894706413149834, -2.1570277214050293], [-0.13317088782787323, -2.3389458656311035, -0.8891359567642212, -0.1754635125398636, 0.48875656723976135, -0.7729319930076599, 0.5917026400566101, 1.0025559663772583, -0.7858487963676453, 0.7136369347572327, 0.44855907559394836, -0.5559860467910767, 0.9105641841888428, 0.6081814765930176, 0.10612107068300247, -0.9190229773521423, 0.5350333452224731, -1.4950202703475952, 0.4812484681606293, 0.6875718235969543, 0.8649235367774963, 0.36580023169517517, 1.2026854753494263, -0.15207624435424805, 0.4398581385612488, -1.2533220052719116, 0.0868656262755394, 0.23254601657390594, 0.895254909992218, -0.3495247960090637, -0.33073440194129944, 0.9942949414253235, -1.1853773593902588], [-0.24104364216327667, -0.7501124739646912, 0.002860646927729249, 2.07251238822937, 0.008625525049865246, 1.3143620491027832, 0.9288092851638794, 1.1668211221694946, -1.2904843091964722, -1.3451181650161743, -0.4814506769180298, 0.9026747941970825, -0.12171360850334167, 1.726683497428894, -1.9528816938400269, 0.14420592784881592, -0.9667453169822693, -0.5012762546539307, -0.7489339709281921, 0.7161420583724976, 1.2565076351165771, -0.35113903880119324, -0.8811586499214172, 0.007268228568136692, 0.564218282699585, -0.7878350615501404, 0.05312669277191162, -0.5442861318588257, 1.6666253805160522, -0.6480036973953247, -0.4391044080257416, -0.36599424481391907, -0.10120408236980438], [-0.11921615153551102, 0.36563462018966675, -1.5042762756347656, 0.818844735622406, 0.8355092406272888, -1.5312530994415283, 1.1518789529800415, -1.5279299020767212, -0.5032013654708862, 0.34612223505973816, 0.9866454601287842, 1.847939372062683, 1.0821584463119507, -0.4169676899909973, -0.393399178981781, -0.8836538195610046, -1.0580638647079468, 2.1305816173553467, -0.1536409854888916, -0.6102203130722046, 1.3990942239761353, -0.30835652351379395, -0.6491857171058655, -2.047072649002075, -0.3484295606613159, -1.3115222454071045, 0.6456570625305176, 0.9605587124824524, -0.0795164406299591, -0.1075221598148346, 0.11206324398517609, 1.3564225435256958, 0.04907361418008804], [0.3768041431903839, 0.44065961241722107, 1.358158826828003, -0.4182963967323303, 0.8810750842094421, 0.3862897753715515, -1.5077553987503052, 1.2453728914260864, -1.341043472290039, -0.705748438835144, 0.28834059834480286, -0.5997872948646545, 0.00039026146987453103, 0.12993383407592773, -0.5055912733078003, -0.018996719270944595, -2.2010903358459473, 1.3989518880844116, 0.9044121503829956, 0.6980676054954529, 0.3965991139411926, 1.448643684387207, -0.6035438179969788, 0.10115447640419006, 0.21691463887691498, 0.1591726541519165, 0.027791501954197884, 1.3005222082138062, -2.5551235675811768, 2.622624158859253, -2.2055504322052, -1.527300238609314, 0.2929381728172302], [-1.7257896661758423, -0.7214556932449341, 0.968702495098114, 1.6149795055389404, -0.5710356831550598, -1.0801112651824951, 0.6674280166625977, -1.0903639793395996, -1.6197562217712402, -1.830642580986023, 1.1126511096954346, 0.20651446282863617, 1.0364190340042114, -1.10029137134552, 0.5888087749481201, 0.34692028164863586, 0.07798468321561813, -1.1373882293701172, 0.1388629823923111, 0.17975136637687683, -0.42187973856925964, -0.5980699062347412, 0.25287091732025146, 0.2269585132598877, -0.5585595965385437, -0.38672012090682983, -2.3148481845855713, 0.6727029085159302, -0.026164861395955086, -0.32713890075683594, 0.05033137649297714, 0.3591661751270294, -0.7143484950065613], [-1.0477110147476196, 0.9552919268608093, -0.4805198907852173, 0.3843151926994324, -1.2825847864151, 0.5026304125785828, 2.2548470497131348, 1.036863088607788, -0.7272545695304871, 0.7921937108039856, 1.350921630859375, 1.3144304752349854, -0.5021300315856934, -0.6305108666419983, 0.05895645543932915, 0.6577188372612, 1.8597149848937988, -0.22216686606407166, -0.31486931443214417, 0.5254415273666382, -0.35499435663223267, 1.8098667860031128, -1.0871864557266235, -1.0355783700942993, -1.6047255992889404, 0.6603729128837585, 2.0349485874176025, -0.5361359119415283, 0.601041853427887, 0.46045994758605957, -2.3813161849975586, -1.2054948806762695, -1.839002013206482], [-0.3662351071834564, 0.08952955901622772, 0.08254794031381607, -2.135021209716797, 0.7306497693061829, 0.8133854269981384, -0.3592239022254944, -0.34084081649780273, 0.8484418392181396, 1.2689977884292603, -0.4852193593978882, 1.0922012329101562, 0.6839936375617981, 1.2446571588516235, -1.874473214149475, -1.8649884462356567, -1.3805193901062012, -2.3960001468658447, 0.6563006043434143, 0.5096674561500549, -0.7307186126708984, 0.4675021469593048, -1.1090511083602905, -1.2654038667678833, 1.3881269693374634, -0.7048208117485046, 0.23966829478740692, 2.1609458923339844, -1.3783951997756958, -0.3198500871658325, -3.5429675579071045, -0.691754937171936, 1.5416916608810425], [-0.7992873787879944, 0.655512273311615, -0.5731602311134338, -0.5053561925888062, -0.6461235880851746, -1.5848621129989624, -0.6954652667045593, 0.2250257283449173, 0.9283809661865234, 1.1398954391479492, -0.1380699723958969, 0.264759361743927, 0.25070905685424805, -0.7890781760215759, -0.18908511102199554, 0.7496073842048645, 1.0879616737365723, -0.19529388844966888, -1.726969838142395, -0.11407601833343506, -0.4291057288646698, -0.28778889775276184, -0.8663045763969421, 0.28528833389282227, 1.2066398859024048, -0.24566389620304108, 0.42835018038749695, -0.2739124000072479, 0.1744520664215088, -0.622113823890686, 0.5098833441734314, -0.34775400161743164, -0.4096807539463043], [0.7005767822265625, 0.9831815361976624, 1.6095243692398071, -0.13369451463222504, -1.6412079334259033, 0.4279917776584625, -0.19648176431655884, -0.5477524399757385, 0.3860040605068207, 2.0905792713165283, 0.18598362803459167, -0.7171360850334167, 1.5640913248062134, 2.5220906734466553, 0.8319835066795349, -0.23683977127075195, -0.8143323659896851, -0.6102783679962158, -0.7915902137756348, 0.6303573250770569, -0.9675933122634888, -1.4930334091186523, 2.2877037525177, 0.41993096470832825, -0.3715479373931885, 0.09780246019363403, -0.6051558256149292, -0.12664546072483063, 0.19216963648796082, -0.546775221824646, 1.6506197452545166, 0.4914230704307556, -0.7374477386474609], [0.7140751481056213, -0.6653455495834351, -0.014022227376699448, 0.7428860664367676, -2.161285400390625, -0.576248824596405, -1.0615947246551514, -0.12080924212932587, 0.6363003253936768, -0.538127064704895, -1.0382897853851318, -0.004576528910547495, 1.5301566123962402, 0.22886155545711517, -0.0227140374481678, -0.43828853964805603, 1.9490737915039062, 0.08616728335618973, -1.1388881206512451, -0.1585003137588501, 0.447100430727005, -0.6240761876106262, -1.360094428062439, -0.2060178965330124, 1.6871544122695923, 1.8303014039993286, -0.0038612426724284887, -0.33568477630615234, 1.2862948179244995, -0.14415860176086426, 1.7737298011779785, -1.1264828443527222, 0.5746737718582153], [1.8259481191635132, 0.5016297101974487, -0.11691784113645554, -1.3842723369598389, 1.6899752616882324, 0.008467976935207844, 1.1506835222244263, -0.9417667984962463, -0.2280847281217575, -0.2322266399860382, -0.8868938684463501, 0.13683414459228516, 0.5457803010940552, 0.4459245502948761, -0.1145421713590622, 0.745760977268219, 0.21196547150611877, 1.2731072902679443, -0.9209718704223633, 0.4303087592124939, -1.3265180587768555, 0.7440525889396667, -0.13344749808311462, -0.6502509713172913, -0.35336989164352417, 0.39419567584991455, 0.4194839894771576, -0.7400486469268799, -0.7525246739387512, 1.2115232944488525, -0.9215187430381775, 1.4063698053359985, -0.8173245787620544], [1.44326913356781, -1.7170302867889404, 0.1461201012134552, 0.08880794793367386, 1.3248885869979858, -0.09487579762935638, 0.6308456659317017, 0.08330743759870529, 1.7586228847503662, 0.41054850816726685, -0.42145195603370667, -1.5669602155685425, 0.518609881401062, -0.8366793990135193, 0.16763941943645477, 0.2134813368320465, -1.3072513341903687, -0.1745033860206604, -0.20167399942874908, 0.8165830969810486, -2.6558127403259277, -0.13558782637119293, 0.017439477145671844, 1.5688457489013672, 0.051080476492643356, 0.7767348289489746, -0.8096935749053955, -0.6552751660346985, 1.6358627080917358, 0.03903697431087494, -0.30532804131507874, -1.099577784538269, -1.0269825458526611], [-1.6470386981964111, -1.0428667068481445, -0.44051575660705566, 1.9102952480316162, 0.48294082283973694, 2.308452844619751, 2.0116541385650635, 1.0047798156738281, 1.0209691524505615, -0.015103469602763653, 1.622083306312561, -2.1762242317199707, 0.0025572918821126223, -0.5561423301696777, -0.8132068514823914, 0.9349930882453918, -0.8261590600013733, 0.16704805195331573, 1.3545230627059937, -0.31781721115112305, -1.3451647758483887, -0.3464633822441101, -1.8361090421676636, 1.1230862140655518, -0.024962861090898514, -0.37408143281936646, 0.9061692357063293, 1.6334185600280762, 0.4422389268875122, 0.19101770222187042, 1.4006541967391968, -0.23702624440193176, 1.845281958580017], [0.8437207341194153, 1.2984980344772339, -0.2448863387107849, 1.0803035497665405, 0.09542959183454514, -0.8342383503913879, -0.35697075724601746, -0.006559513974934816, 1.8335715532302856, -0.2324559986591339, 0.9782228469848633, -0.326617568731308, -0.920051634311676, -0.9123261570930481, 0.6073718070983887, -0.7231748700141907, 1.6460539102554321, -0.10755216330289841, -0.5080669522285461, -0.09671612083911896, -0.038454569876194, -1.5475008487701416, -1.4295333623886108, 0.5951850414276123, -0.6507844924926758, 0.5416340231895447, 0.34585434198379517, 0.7532209753990173, -0.6266348958015442, 0.33918875455856323, 1.192572832107544, -0.18977361917495728, 0.6782819628715515], [-0.029386445879936218, 1.0741047859191895, -0.26302433013916016, 0.8498811721801758, -0.813988208770752, -0.4542168080806732, -0.18446899950504303, -1.5431814193725586, 0.34192904829978943, 0.259591668844223, 1.5666061639785767, -1.3346483707427979, 0.5226525068283081, -0.8632451295852661, -0.3210306167602539, -0.8711258769035339, 1.095367431640625, 0.013703415170311928, -1.0542163848876953, -0.35507625341415405, 0.539569616317749, -1.2276469469070435, 0.8745023608207703, 0.8497157692909241, 1.8332369327545166, -0.9731485843658447, -3.2650675773620605, -0.3351599872112274, -0.6869161128997803, 0.42132681608200073, 0.35457679629325867, 1.356138825416565, -0.3018801212310791], [-0.28822967410087585, 2.1831443309783936, 0.47125348448753357, -1.2814507484436035, -1.0022016763687134, -0.304206520318985, 0.16862830519676208, 0.7628589272499084, 0.27075666189193726, 0.6100406050682068, -2.3014678955078125, -0.8993481397628784, -0.6019607186317444, 0.446188360452652, -1.6400835514068604, -0.06866888701915741, 0.09473760426044464, 1.8950449228286743, -0.9672793745994568, -0.649226725101471, -0.2072206735610962, 0.18657661974430084, -1.740095615386963, -0.5868121385574341, -0.067175954580307, 0.48466479778289795, -0.6528272032737732, 0.3841952085494995, 0.12049148976802826, 0.2732911705970764, 1.5741479396820068, 0.6011810898780823, -0.6630101799964905], [-0.744920015335083, -0.4692382514476776, 0.10232546925544739, 0.950386643409729, 0.2187187820672989, -1.2202787399291992, -1.2299748659133911, 0.0981869325041771, -1.5049656629562378, 1.2558308839797974, 0.10071117430925369, -0.08944518119096756, -2.018411874771118, -0.7774487733840942, 0.7457373738288879, -1.0149362087249756, 1.4668688774108887, 0.3510226011276245, -0.5869001150131226, 0.13535068929195404, 0.6081919074058533, 1.367075800895691, -2.8446156978607178, 0.8080899119377136, 1.3004206418991089, 0.20590521395206451, -0.1539628654718399, -0.9331059455871582, 0.9294725060462952, 0.5550609827041626, 1.336687445640564, 0.9531124234199524, -0.11614498496055603], [0.41289836168289185, -0.08406099677085876, -1.3578596115112305, 0.5974435210227966, 1.3051912784576416, 1.9567086696624756, 0.2621655762195587, 0.533287525177002, 0.8648931980133057, -0.358769953250885, 1.2496060132980347, 0.5200627446174622, -1.1955890655517578, 2.7997756004333496, 0.8078919649124146, 0.3288643956184387, -0.005540103185921907, 0.048544567078351974, -0.5214324593544006, -1.7858892679214478, -0.4204123318195343, -0.3081885576248169, 0.18584004044532776, 0.38101017475128174, -1.0599461793899536, 0.5483015775680542, -0.4163820445537567, 0.29098376631736755, -0.10347910225391388, 1.0864765644073486, -0.14985468983650208, 0.3494047522544861, 0.3345932960510254], [0.5271305441856384, -0.0939297303557396, -1.75978422164917, -0.0949757769703865, 0.923140287399292, -0.15225525200366974, -1.188506841659546, 0.562391996383667, -1.2477365732192993, 0.14323392510414124, 0.5529727339744568, -0.574501633644104, 0.536644458770752, 0.4499501585960388, -1.5524039268493652, 0.8170731067657471, -0.3776092529296875, -0.11732892692089081, 0.7549601197242737, -2.0493874549865723, 0.7286632061004639, -1.3142844438552856, 1.076222538948059, -0.8768452405929565, -1.2316213846206665, 0.23035088181495667, 0.7206942439079285, 0.016071751713752747, -0.7835367321968079, 0.36792999505996704, 0.2708500325679779, -0.5745034217834473, 1.1702576875686646], [-0.4549640417098999, -1.0629802942276, -1.2530014514923096, 0.6172016263008118, 0.8372040390968323, 0.7217538356781006, -0.2963516116142273, 0.31462225317955017, -0.6454417705535889, 1.4904165267944336, -1.7267848253250122, 0.2835707366466522, -1.3477920293807983, -0.21370014548301697, -0.29919424653053284, -0.8747732043266296, -0.0564529225230217, -0.5809670686721802, -1.4764807224273682, 1.7938439846038818, -0.7176520228385925, 0.35740432143211365, 1.4809939861297607, -0.21919018030166626, 1.1888339519500732, -0.560158371925354, -0.3033151626586914, 2.4093379974365234, -0.9964900016784668, -0.7352631688117981, -0.005228765774518251, -0.7130845785140991, -0.7446957230567932], [-0.5876159071922302, 0.5848077535629272, -0.3272581994533539, 0.03261159732937813, -1.556792974472046, -1.7488386631011963, 0.8089778423309326, -0.5623667240142822, -0.19006940722465515, 0.05966435745358467, 0.3862934112548828, 1.2602039575576782, -1.0384467840194702, 0.9596097469329834, 0.3256857991218567, -0.373084157705307, -2.1081724166870117, -0.01846781000494957, -0.550322949886322, -0.9502183198928833, -0.7419347167015076, 1.043432593345642, 1.044053077697754, 0.26526790857315063, -0.4319838583469391, 0.5837802886962891, -0.1901128888130188, -0.4387708008289337, -1.2607636451721191, -0.5909485816955566, 0.3567679226398468, -0.838853657245636, 0.10483676940202713], [-1.9939687252044678, -0.07820317149162292, -0.029248028993606567, 1.551149845123291, 1.3683680295944214, -0.2122252732515335, 0.1366206258535385, 0.32490283250808716, -0.7814934849739075, -0.9364458918571472, 0.34255465865135193, 0.386200487613678, 1.5973769426345825, -0.7746561765670776, -0.32016968727111816, 1.2636908292770386, -0.6760526299476624, -0.34581202268600464, 0.13942185044288635, 0.7404763698577881, -0.15919972956180573, 1.397399663925171, -0.06772755831480026, -1.215962290763855, 0.8091236352920532, 0.3141638934612274, -0.8473642468452454, -0.4841739535331726, 0.887509286403656, -1.913561463356018, -0.23374386131763458, 1.127086877822876, -1.4219449758529663], [1.3093149662017822, 0.5378762483596802, -0.9310663342475891, 0.2565300464630127, 0.45409631729125977, 0.09164931625127792, 0.775034487247467, 0.5011147260665894, 0.926879346370697, 1.1530667543411255, 2.481147050857544, 0.41911551356315613, -0.5827511548995972, 1.0693519115447998, 0.9807774424552917, -0.13157138228416443, -0.11911032348871231, 0.6244943737983704, 1.2102090120315552, -0.9561804533004761, -1.1769062280654907, 0.27179625630378723, 0.731387197971344, 0.2394786775112152, -1.7251646518707275, -0.8258086442947388, -0.3509562313556671, -1.3081748485565186, 0.9954971671104431, 1.1378284692764282, -0.08167829364538193, -1.233534574508667, -0.03476995229721069], [-0.03998443856835365, -0.09526724368333817, -0.10413903743028641, -0.4790799915790558, -0.9841788411140442, -1.1075609922409058, -1.1565546989440918, 0.9540308117866516, -0.12671571969985962, 0.13803648948669434, -0.03191803768277168, -0.7378063797950745, -0.9586367011070251, 1.2830722332000732, 1.763764500617981, -0.337820440530777, 0.02380002662539482, 0.0072934431955218315, 2.091555595397949, -0.22176308929920197, -0.4954063594341278, -2.0742526054382324, 0.0012689536670222878, 0.06995191425085068, 1.7963933944702148, -1.4860731363296509, -0.05822982266545296, 0.39270228147506714, -0.9502460956573486, 0.4762137234210968, 0.08645781874656677, -0.8642869591712952, 0.20071114599704742], [0.06919459253549576, 1.0348715782165527, 1.1752604246139526, 0.6019493341445923, 1.0207520723342896, 0.38315868377685547, -0.40124234557151794, -0.10108482092618942, -1.1090741157531738, 1.849117636680603, -2.4534785747528076, 0.2028297781944275, 0.856432318687439, -0.27606329321861267, -0.14336107671260834, 1.0569486618041992, -1.1922467947006226, -0.854189932346344, -0.4610937535762787, 2.4788551330566406, -0.20434680581092834, -1.8981562852859497, -1.0019729137420654, 0.266370564699173, 1.6185619831085205, 0.33076027035713196, 0.31801092624664307, -0.19507189095020294, 0.011576410382986069, -0.3792227506637573, 1.1564966440200806, -0.3226979672908783, -0.14202100038528442], [-1.1868267059326172, 0.1035667210817337, 0.7514644861221313, -2.5323486328125, 2.010296583175659, 0.10845569521188736, -1.2941137552261353, -1.0221225023269653, 1.3095349073410034, 1.377917766571045, -0.2891044020652771, 1.2370309829711914, 0.6474496126174927, -0.0825953483581543, 0.2381313443183899, -0.6279898881912231, 1.8536449670791626, 0.7242521047592163, 0.35064610838890076, -1.9021482467651367, 0.5588781833648682, 2.838268280029297, -1.3828635215759277, -0.5412341952323914, 0.8129748106002808, -0.457311749458313, 2.025122880935669, 0.061372555792331696, -2.2391223907470703, -1.3636873960494995, 0.1905835121870041, -0.4645378589630127, -0.49817201495170593], [-1.057979702949524, -0.143172949552536, -0.7268681526184082, 0.890527606010437, 1.946933388710022, -1.4854000806808472, -0.3726158142089844, 1.7369346618652344, -0.6505703330039978, -0.8752222061157227, -0.5157498121261597, -1.3371933698654175, -0.35612916946411133, -2.387059450149536, 0.1929374486207962, 0.9932927489280701, -0.8499791622161865, -2.129606246948242, -0.3627181947231293, 0.8061060905456543, -0.16039647161960602, 0.683628261089325, 1.4857234954833984, 1.1735153198242188, -0.12960337102413177, 0.05944797396659851, 1.4632713794708252, -1.8018254041671753, 2.5003085136413574, -0.06228348985314369, 1.6894603967666626, -0.3750862181186676, -0.4962832033634186], [-0.7116740345954895, -0.32504501938819885, -1.6504814624786377, 1.9034582376480103, -1.191525936126709, 0.9723572134971619, -0.2086125910282135, 1.3488627672195435, -0.2630775272846222, 0.8383851647377014, 1.4799367189407349, 0.09444739669561386, -0.684002697467804, 1.2728887796401978, -0.59837806224823, -1.1451181173324585, -1.2753762006759644, 0.5472646355628967, -1.5216481685638428, 0.5315218567848206, 0.35775262117385864, 0.41719913482666016, 2.372230291366577, 2.8225724697113037, -1.8326106071472168, 0.555932879447937, 1.0967451333999634, 1.5362253189086914, 1.1774200201034546, -0.5244929194450378, -0.3209328055381775, 1.2429739236831665, -0.4739629030227661], [-1.036529541015625, -0.2998296320438385, 0.10705269128084183, -0.38839423656463623, 0.12144388258457184, -0.4380717873573303, 0.6616613268852234, -0.17356035113334656, 1.4689581394195557, 0.39399364590644836, -0.22204825282096863, 0.364641934633255, -0.640083372592926, 0.3665566146373749, 0.4647541344165802, -2.1122334003448486, 0.6010007262229919, -0.34681475162506104, 1.055195689201355, 0.8200269341468811, -1.006560206413269, -0.4127011001110077, -1.1484885215759277, 0.7196919322013855, -0.6797130107879639, -0.3585776090621948, 0.5562528967857361, -0.34923285245895386, 0.2238454669713974, 0.12489921599626541, -0.529151439666748, -0.6500008702278137, -0.28572964668273926], [-0.13342086970806122, -0.5883328914642334, 2.842708110809326, 0.003434923477470875, -0.3847058415412903, 0.7899146676063538, -0.42989587783813477, 3.0140111446380615, 0.1161932572722435, 0.47437381744384766, -0.5719646215438843, -0.17873765528202057, -1.0240933895111084, 0.8389058709144592, -0.8200318813323975, -0.4192636013031006, -0.5244473814964294, -0.7433812022209167, 0.1891162097454071, -0.22452548146247864, -0.6368533968925476, 0.3952673077583313, 0.21108564734458923, 0.20374520123004913, 0.8956472277641296, -0.40757548809051514, 0.6618943810462952, 0.9069385528564453, 1.019740104675293, -0.6314524412155151, -0.8376772999763489, -0.41570910811424255, -1.857115626335144], [0.3180595934391022, 0.43580400943756104, 0.8277092576026917, -0.1970464140176773, 1.677195429801941, 0.3178020715713501, -0.37607550621032715, -0.8578866124153137, 1.1769404411315918, 1.458461880683899, -0.2737775146961212, 0.5096559524536133, 1.1350356340408325, -0.8234984874725342, -0.6082965135574341, 0.5147877931594849, 0.016032187268137932, 1.0443171262741089, -0.9747977256774902, 0.18059778213500977, -0.24407000839710236, -0.5046959519386292, 0.23627126216888428, -0.7841286063194275, 1.758044958114624, -0.23279564082622528, -0.12408585101366043, 0.40931040048599243, -0.19390814006328583, -0.2974832355976105, 0.5622466802597046, -0.7678437829017639, 1.9394346475601196], [0.32170528173446655, -1.2040092945098877, -0.43716323375701904, -0.5614762902259827, -0.06525666266679764, 0.4178259074687958, -0.2518135607242584, -1.5688849687576294, 1.7771846055984497, -0.27394193410873413, -0.6374572515487671, -1.8402833938598633, 0.6382095217704773, 0.7866398096084595, -1.1691399812698364, 1.828912377357483, 0.15967923402786255, 0.9320998787879944, -0.8400007486343384, 0.2501384913921356, -1.6096160411834717, -0.7047301530838013, -2.4533677101135254, 0.3705730438232422, -0.35186144709587097, -1.5462478399276733, 2.8831145763397217, 0.42970359325408936, 0.5418007969856262, -0.08534348756074905, 1.02858567237854, -0.5812945365905762, 0.00511568458750844], [0.45481234788894653, 0.6574526429176331, -0.5140575170516968, 0.014710926450788975, -0.6026773452758789, 0.7108185887336731, -0.8244867324829102, 0.711506724357605, 1.4838130474090576, -0.09934087097644806, 1.9322909116744995, -1.189488410949707, 0.4162405729293823, 1.2587435245513916, -0.21548520028591156, 0.02068641223013401, 0.0355568565428257, -0.08689213544130325, -1.1813147068023682, -0.8308793902397156, 1.5302581787109375, -1.6730629205703735, 1.5296213626861572, -0.16245409846305847, 0.008265255019068718, 0.8542478084564209, 0.2890775501728058, 0.660942792892456, -0.055931609123945236, -0.979587197303772, 2.4938387870788574, 1.3052688837051392, 1.5743829011917114], [-0.9656554460525513, 0.45168066024780273, 0.7431458234786987, 1.5724496841430664, -0.17228065431118011, 1.7293399572372437, 0.2844659984111786, 0.38983824849128723, 0.43447425961494446, -1.1215381622314453, -0.034715138375759125, 2.167576313018799, 2.903069019317627, 0.3172808289527893, 1.2865101099014282, -0.7683727741241455, -2.1901443004608154, -0.22881434857845306, -0.10593412816524506, 0.9407126307487488, -2.0818238258361816, -0.7127963304519653, -1.2025896310806274, 0.10330215841531754, 0.3679424226284027, -0.07099601626396179, -0.522932231426239, 0.6753584146499634, -0.5964322686195374, -0.581879734992981, -0.03523929789662361, 0.2139451950788498, -0.09239626675844193], [-0.13994990289211273, 0.5213439464569092, -0.7322806119918823, -1.0381510257720947, 0.1553896963596344, 0.5223357677459717, -0.43555834889411926, 0.7501474618911743, 0.2059205323457718, -0.08458264172077179, 0.6586096286773682, -0.30452844500541687, 0.18538792431354523, 0.1914137303829193, -1.249795913696289, -0.07290423661470413, 0.2843003273010254, -1.0376205444335938, 1.4403115510940552, -1.0608420372009277, -0.07056284695863724, 0.5007957220077515, 0.5845800042152405, 1.1738697290420532, -0.08503264933824539, 0.8178693056106567, 1.0620992183685303, 1.2904350757598877, -1.3390192985534668, -0.2733287215232849, 1.1997123956680298, -1.0491095781326294, -0.7926651835441589], [-0.051144346594810486, -1.308923602104187, -0.7001209855079651, -1.4914911985397339, -0.8302900791168213, 0.7960121631622314, -1.681104302406311, 1.9755651950836182, 0.9841245412826538, 0.4205075204372406, 0.6063557863235474, -1.0174833536148071, -0.7199545502662659, 1.3412957191467285, -1.543552279472351, -0.3093319237232208, -0.5774217844009399, 0.846889078617096, 0.6467636823654175, -2.222834348678589, -0.413961797952652, -0.6949547529220581, 0.14016835391521454, -0.24721047282218933, 2.0626440048217773, 0.29092955589294434, -0.3692452013492584, 0.6973778605461121, 1.4117034673690796, 0.546602189540863, 2.3267345428466797, 0.8559850454330444, -0.8436810374259949], [0.003995598293840885, -0.06038028001785278, -1.672414779663086, -1.047613501548767, -1.3458906412124634, -0.9197636246681213, 0.7066280841827393, 0.12367844581604004, -0.5013439655303955, 0.07508087903261185, 1.0808720588684082, 0.4608880281448364, 1.043853998184204, -0.35898661613464355, -0.2839665412902832, -0.26388436555862427, 0.1639580875635147, -2.121450901031494, -0.6159198880195618, 0.4422459006309509, -0.5289332270622253, 0.6166484355926514, -1.0278605222702026, -0.12085007131099701, -0.0037014561239629984, -0.3497565984725952, 1.0750157833099365, -2.2147159576416016, 0.47758424282073975, 0.6315075755119324, 0.9619269371032715, 1.5544297695159912, 1.9090908765792847], [-0.276848703622818, -0.9933180809020996, -0.2502804398536682, -1.5570505857467651, 0.36025020480155945, -0.7194663286209106, 0.22794048488140106, 0.2970261871814728, 0.08484815061092377, -1.1281366348266602, -0.911363959312439, 0.9778062105178833, -1.9387528896331787, 0.3322286307811737, 0.18454323709011078, -0.513763964176178, 1.2886680364608765, -0.5761076211929321, -0.9748803973197937, -0.5792306661605835, -0.23131144046783447, 0.14676032960414886, -0.4816436171531677, -0.4139300286769867, 1.5316542387008667, 1.1652796268463135, 0.6828508377075195, 0.03603136166930199, -0.3889204263687134, -1.2371339797973633, -0.07346519827842712, 0.3976323902606964, -1.6296186447143555], [0.04516400769352913, 0.932930052280426, 0.5541940331459045, -0.9614403247833252, -1.6290380954742432, 1.4351952075958252, 1.4414403438568115, -0.30914101004600525, 1.5882457494735718, -0.4799598455429077, 0.608424961566925, 0.4891843795776367, 0.025453481823205948, -0.7395948767662048, 0.3528023064136505, -0.36411261558532715, -0.1921708881855011, -2.379081964492798, -0.2895357012748718, 1.3595319986343384, 1.2929526567459106, -0.23575738072395325, -0.1831725686788559, 0.31758585572242737, -0.2714602053165436, 0.4701234996318817, 1.5610642433166504, -1.179612398147583, 0.6106300354003906, -0.7291518449783325, -0.39196300506591797, -0.2642357349395752, 0.7964950203895569], [-2.386679172515869, -0.7105221152305603, 0.24066466093063354, -0.094561368227005, 0.9183518886566162, 0.7083175778388977, 0.5683965086936951, 1.2589048147201538, 0.6942358613014221, -2.6924729347229004, 1.9794782400131226, 1.2553244829177856, 0.451205849647522, -0.34829819202423096, -2.187344551086426, -1.2846505641937256, 0.5237479209899902, -0.12556162476539612, -0.3510175049304962, -0.15013886988162994, 0.6182065606117249, 0.11054803431034088, -0.45607703924179077, 0.3470650315284729, 1.1085506677627563, 0.46837183833122253, -1.4352010488510132, 0.5501375794410706, -0.4562484323978424, -1.5682286024093628, -2.3698954582214355, -0.7426775693893433, -1.595828652381897], [-0.8349555730819702, -0.7694498896598816, -1.0132958889007568, 1.8061909675598145, 0.23417431116104126, -1.7628153562545776, -1.1068216562271118, 0.010770917870104313, 0.8744193315505981, 0.9171205163002014, -0.35513314604759216, 0.9295943975448608, 1.0444865226745605, 0.2906559109687805, 1.154404878616333, 0.2885182201862335, 1.2354230880737305, -0.6995493173599243, -0.9283972978591919, -2.015510320663452, -0.4897776246070862, -0.031940020620822906, 0.41328155994415283, -0.47694674134254456, 0.33795514702796936, -0.4853198230266571, -0.26361480355262756, 1.9984681606292725, -0.38643428683280945, -2.639827251434326, 0.4960106611251831, 0.9963453412055969, -0.19486117362976074], [-0.0004781889438163489, 0.2853214740753174, -1.2005659341812134, 1.159663200378418, 0.23794154822826385, 0.9686563014984131, 1.0641433000564575, 0.6505719423294067, -0.5601195693016052, -0.556195855140686, 0.03177337348461151, 1.1281052827835083, 0.2001810222864151, 0.7327268123626709, 0.8393645882606506, 0.35197004675865173, 1.3563247919082642, 1.5498688220977783, -2.2037129402160645, -0.7601553201675415, -0.1393800973892212, -0.022563129663467407, 0.036625321954488754, -0.18511086702346802, -0.9392845034599304, 0.52093505859375, 0.11447491496801376, -0.4585285484790802, -0.5748535990715027, -0.9921295642852783, 0.3935980796813965, -1.4302819967269897, 1.1135172843933105], [-0.7096040844917297, -0.12134234607219696, 1.2048346996307373, 0.28131425380706787, 1.3983896970748901, -0.6364453434944153, -2.694533586502075, 0.909869372844696, -0.35173919796943665, 0.7716796398162842, -0.24576327204704285, 0.15780286490917206, 0.8677982091903687, -0.43339914083480835, -1.3790169954299927, -0.761770486831665, 1.365385890007019, 0.18638651072978973, -0.3960105776786804, 0.46063899993896484, 1.8085201978683472, -1.7154089212417603, -1.212277889251709, 0.5330585241317749, 0.3358834981918335, -1.4238739013671875, -0.052481915801763535, -1.6222989559173584, 0.4056434631347656, 0.6288851499557495, -0.08900552988052368, 0.7706234455108643, -0.008394564501941204], [-1.3220257759094238, -0.8893519043922424, 2.0394437313079834, -0.9146504998207092, 0.4288792014122009, 0.6932663321495056, 0.16583208739757538, -1.6717945337295532, -2.3475799560546875, -0.589608371257782, -1.5609474182128906, -0.9241790175437927, -1.2258752584457397, -0.032088130712509155, -0.38396984338760376, -0.1736295521259308, -1.0583962202072144, 0.30179670453071594, 0.7686980962753296, -1.2026278972625732, 0.6220486760139465, -0.3411325216293335, -0.31561392545700073, -1.499718189239502, 2.2750375270843506, 1.4190378189086914, 0.4994382858276367, -0.622413694858551, -0.5990669131278992, 1.9062002897262573, 0.5945647954940796, 1.704546332359314, -0.6965034008026123], [-0.509190559387207, -1.8012206554412842, -0.19672048091888428, 1.0471019744873047, -1.2457090616226196, 1.0115070343017578, 0.7456055879592896, 0.7017760872840881, 1.229046106338501, -0.721470057964325, 0.07532462477684021, -0.495524138212204, 0.3305976986885071, -0.9819590449333191, 0.9257561564445496, -0.39090949296951294, 0.27658501267433167, 0.804990291595459, 1.6738855838775635, -1.0211831331253052, 1.032950758934021, 0.7000875473022461, -1.64320707321167, -0.855362057685852, 0.6724454164505005, 2.5224626064300537, 0.7039594650268555, -2.0964701175689697, -0.9185624122619629, 1.9182089567184448, 0.4221624433994293, -0.07270032167434692, -0.7378952503204346], [-1.9548219442367554, -0.915924072265625, -0.15594099462032318, -0.12017960846424103, -0.8532958626747131, -0.19261635839939117, -0.3519720137119293, 0.2631687819957733, 1.2342643737792969, 1.7573795318603516, 1.826335072517395, 1.3890223503112793, 0.6916252374649048, -0.6183879971504211, 1.5330836772918701, -0.9428280591964722, 0.8486860990524292, -1.1689027547836304, -0.42049428820610046, -0.10145614296197891, 1.3460092544555664, 2.0129880905151367, 0.49899548292160034, 1.7436904907226562, -0.40251556038856506, -1.0963730812072754, 0.2937205731868744, 0.8194953203201294, -1.8013907670974731, 0.3737029433250427, 0.23348668217658997, 1.1278681755065918, -0.7460854053497314]], "dtype": "float32"}}, {"a": [{"__nd__": [[742, 552, -293], [735, 998, -5], [-604, 913, -718], [-252, -852, 589], [-877, -565, 424], [-224, 469, 555], [-555, 780, 238], [-354, -178, 541], [-721, 503, -817], [730, -90, 765], [114, -675, 503], [-545, -379, -184], [-817, 702, 302], [403, 558, 246], [402, 739, -92], [718, -981, 493], [113, 276, 356], [-441, -493, -696], [893, -177, -501], [-7, -31, 7], [-439, -753, -864], [996, 971, -557], [863, 688, -893], [48, -809, -171], [695, 938, 759], [-726, -920, 162], [905, -18, 681], [430, -139, -562], [-934, 889, -934], [-651, -911, -219], [-362, -782, -925], [483, -435, -252], [101, 25, 724], [826, 79, 350], [970, -302, -474], [302, -346, 619], [-715, 796, -386], [446, -718, -921], [-207, -877, -248], [565, -1000, 665], [48, 915, -437], [-213, 350, -8], [990, 891, 548], [845, 44, -235], [-706, -970, -84], [-857, 825, -519], [-146, -476, -385], [-895, -293, -531], [256, 351, -588], [625, -644, 1], [-816, 966, -804], [968, -290, -230], [513, 941, -324], [476, -787, 93], [253, 229, -414], [-156, 811, 755], [-976, 414, -283], [-655, 176, -742], [160, -384, -535], [521, -263, 422], [986, 63, -887], [458, 3, 320], [-574, 755, 688], [-912, -158, -281], [930, 773, -438], [181, -1, 711], [-379, -670, -114], [66, 718, 223], [394, -213, -644], [-102, -671, 868], [234, 766, -311], [871, 705, 402], [743, -368, 411], [-397, -987, 647], [162, -222, 147], [835, -790, -749], [-708, -933, -601], [-208, 543, -420], [-155, 876, 799], [-609, 949, 884], [799, 598, 228], [187, -685, 590], [-229, 693, -335], [915, -114, -399], [854, -231, 595], [87, 235, 477], [-535, -15, 944], [545, -773, 211], [-354, -890, 49], [-584, -679, -775], [943, -952, 374], [664, -92, 200], [853, -480, -337], [650, -872, -331], [13, -378, 955], [682, -231, 399], [865, -385, 871], [722, 568, 431], [592, -600, -126], [-157, -415, 897]], "dtype": "int32"}], "e": {"__nd__": [[742, 735, -604, -252, -877, -224, -555, -354, -721, 730, 114, -545, -817, 403, 402, 718, 113, -441, 893, -7, -439, 996, 863, 48, 695, -726, 905, 430, -934, -651, -362, 483, 101, 826, 970, 302, -715, 446, -207, 565, 48, -213, 990, 845, -706, -857, -146, -895, 256, 625, -816, 968, 513, 476, 253, -156, -976, -655, 160, 521, 986, 458, -574, -912, 930, 181, -379, 66, 394, -102, 234, 871, 743, -397, 162, 835, -708, -208, -155, -609, 799, 187, -229, 915, 854, 87, -535, 545, -354, -584, 943, 664, 853, 650, 13, 682, 865, 722, 592, -157], [552, 998, 913, -852, -565, 469, 780, -178, 503, -90, -675, -379, 702, 558, 739, -981, 276, -493, -177, -31, -753, 971, 688, -809, 938, -920, -18, -139, 889, -911, -782, -435, 25, 79, -302, -346, 796, -718, -877, -1000, 915, 350, 891, 44, -970, 825, -476, -293, 351, -644, 966, -290, 941, -787, 229, 811, 414, 176, -384, -263, 63, 3, 755, -158, 773, -1, -670, 718, -213, -671, 766, 705, -368, -987, -222, -790, -933, 543, 876, 949, 598, -685, 693, -114, -231, 235, -15, -773, -890, -679, -952, -92, -480, -872, -378, -231, -385, 568, -600, -415], [-293, -5, -718, 589, 424, 555, 238, 541, -817, 765, 503, -184, 302, 246, -92, 493, 356, -696, -501, 7, -864, -557, -893, -171, 759, 162, 681, -562, -934, -219, -925, -252, 724, 350, -474, 619, -386, -921, -248, 665, -437, -8, 548, -235, -84, -519, -385, -531, -588, 1, -804, -230, -324, 93, -414, 755, -283, -742, -535, 422, -887, 320, 688, -281, -438, 711, -114, 223, -644, 868, -311, 402, 411, 647, 147, -749, -601, -420, 799, 884, 228, 590, -335, -399, 595, 477, 944, 211, 49, -775, 374, 200, -337, -331, 955, 399, 871, 431, -126, 897]], "dtype": "int32"}}, {"a": [{"__nd__": [[1.8314611911773682, -0.21212846040725708, -2.394015073776245, -0.4111219346523285, 0.9101552963256836, -0.8629969358444214, -0.08548121899366379, -0.5782530307769775, 0.30402544140815735, 0.7049210071563721, -0.8063345551490784, 0.33550742268562317, 0.4934980869293213, -1.6836026906967163, 0.5531863570213318, -0.8320940732955933, -0.48952868580818176, 0.5290991067886353, -0.0333014540374279, 1.0293927192687988, 0.6947815418243408, -2.2108235359191895, 0.272339791059494, -1.7585190534591675, 0.4660826027393341, -0.5149959921836853, 0.6004643440246582, 0.30938220024108887, -0.1668969839811325, -2.379852771759033, -0.934755802154541], [-0.14517533779144287, 1.0117796659469604, -0.4555515646934509, -0.504378080368042, -1.4339739084243774, 2.1576459407806396, -0.2929539680480957, -0.5601922273635864, -0.4764561653137207, -0.10892939567565918, -0.9054820537567139, 0.34343111515045166, -2.679811477661133, -1.1872543096542358, 0.20690880715847015, 1.0696624517440796, 0.5078275203704834, -0.09926766902208328, 1.600055456161499, -0.20570023357868195, -0.20481842756271362, -0.26238125562667847, -0.419754296541214, 0.04709396883845329, -0.8050892949104309, 1.0817383527755737, -1.1671409606933594, -1.297170877456665, 1.3912088871002197, 0.25007858872413635, 0.7883578538894653], [-0.990673303604126, 1.246144413948059, -1.3939985036849976, 0.7515562772750854, 0.9735942482948303, -0.1892968863248825, -1.9053304195404053, -0.18174073100090027, 1.4591599702835083, -0.020416859537363052, -1.0898692607879639, 1.1625113487243652, 2.4017951488494873, 1.6441901922225952, 0.5885197520256042, -0.09249625355005264, -0.11928823590278625, 0.8446840047836304, 0.8051104545593262, 0.4376782476902008, -0.42220234870910645, 0.6280560493469238, 0.4606426954269409, 0.8519623279571533, -0.9667774438858032, 2.4725000858306885, 1.7189282178878784, -1.474492073059082, -0.3568890690803528, 0.24704964458942413, 0.03379594534635544], [-1.0598353147506714, 0.4696548879146576, 0.6294484734535217, -0.06258724629878998, -0.9508728384971619, 0.07408116012811661, 0.8823204636573792, 1.9756748676300049, 0.24186304211616516, 0.8613426089286804, 0.7670099139213562, 1.001837968826294, -1.2338956594467163, 1.7421764135360718, 0.6976189613342285, -1.1226422786712646, 0.27291956543922424, -0.5575341582298279, -0.2906922996044159, 0.6675100922584534, -1.1138683557510376, -0.007336026057600975, 0.15236559510231018, -0.48446527123451233, -0.48722973465919495, 0.790945827960968, -0.679418683052063, 0.8629192113876343, -0.08342543244361877, 0.9864001870155334, 0.7462831139564514], [0.5810195207595825, -0.19313795864582062, 1.2244408130645752, 2.4011125564575195, -1.2702523469924927, -1.32418692111969, 0.043983399868011475, 2.5173091888427734, -1.8272812366485596, -0.5172780156135559, -0.9411382079124451, 0.8316216468811035, -0.17097976803779602, 0.10073570162057877, -0.5711592435836792, -1.2858612537384033, 0.31414636969566345, -0.4680486023426056, 1.336600422859192, -0.7687689065933228, 0.6147221922874451, -0.16935226321220398, 0.14809001982212067, -0.10860789567232132, 1.0769340991973877, -1.6823523044586182, -0.5788229703903198, 0.8050779104232788, -1.0657830238342285, 0.1178336814045906, 0.6880485415458679], [1.0754780769348145, -0.1387188583612442, 0.15714557468891144, -0.41369470953941345, 0.5552850365638733, -0.6433817744255066, -0.08671879023313522, -0.2120666801929474, -0.09002033621072769, 1.7304271459579468, -0.6005355715751648, -0.11141137033700943, 1.4105685949325562, -0.6935920715332031, 0.08270400017499924, -0.0857856273651123, -1.2546610832214355, -0.46917274594306946, -1.7624337673187256, 0.5339840054512024, 0.6288599371910095, 0.21194815635681152, -0.8013015389442444, -0.29011455178260803, 0.5883454084396362, 1.5655858516693115, 0.40832263231277466, -2.020433187484741, -0.3188422918319702, 1.1343690156936646, -0.8923438787460327], [0.006932623218744993, 1.0116032361984253, 0.7404350638389587, -1.698319435119629, -0.689485490322113, 1.203761100769043, -0.2578622102737427, -0.6563969254493713, -0.2915245592594147, 0.24298593401908875, 0.3128099739551544, 1.051524043083191, 0.37820640206336975, -0.6528393030166626, 0.6684989929199219, 0.6036516427993774, 0.2879334092140198, 0.6441166400909424, -0.508413553237915, -0.912264347076416, -0.2651650905609131, 1.317600131034851, 1.1075704097747803, 1.1395986080169678, 1.1421340703964233, -0.6621925234794617, 0.510866641998291, -0.44834133982658386, -0.39990147948265076, -0.4432697594165802, 1.4361281394958496], [-0.201649010181427, 1.042862892150879, -0.5600792169570923, -2.7840306758880615, -0.1932140588760376, -2.5527307987213135, 0.33472198247909546, -0.1668413132429123, -0.5693341493606567, 0.15144960582256317, -0.9127938747406006, -2.4038259983062744, -0.3789249658584595, -1.7637548446655273, -0.6227508783340454, -0.18206770718097687, -0.2809329032897949, -0.7952303290367126, 0.6290260553359985, 0.6906354427337646, -0.5424806475639343, 0.5121368169784546, 0.37611812353134155, -1.0136951208114624, -0.5207625031471252, -0.40320825576782227, -0.8802111744880676, -0.019711386412382126, 0.43310534954071045, 0.4528869390487671, -0.2768307328224182], [0.3850143849849701, -2.182095527648926, 0.043105531483888626, 0.5284362435340881, 0.0782979428768158, -0.06815361231565475, -0.4106152355670929, -0.5546054840087891, -0.17821529507637024, -0.4238491356372833, -2.0083730220794678, -1.768474817276001, -0.5471705198287964, -0.6218112111091614, 0.38724493980407715, 0.4772305488586426, 0.16943879425525665, 0.7475669980049133, 0.889835774898529, 1.9359766244888306, 0.6142483353614807, -1.3484138250350952, -1.2357081174850464, 2.5169284343719482, 0.25539830327033997, 0.7937415242195129, -0.4008444547653198, 0.4045107364654541, -1.4360703229904175, -1.601118564605713, 0.34475743770599365], [1.2330012321472168, -0.23561686277389526, 0.32682961225509644, -0.3881165087223053, 1.2328521013259888, -0.10824783891439438, -1.3588223457336426, 2.2728195190429688, -0.9097006916999817, 1.3241336345672607, 0.042633213102817535, 0.1822144240140915, 0.4353739619255066, -0.14502887427806854, -0.6753343939781189, -0.5628121495246887, -0.022623280063271523, -0.7760487198829651, 1.177900791168213, -1.42164945602417, 0.6728270649909973, -0.6471058130264282, 0.5255551338195801, -0.517952561378479, 0.8071964979171753, -1.1356168985366821, 1.5000650882720947, -0.2728368043899536, -1.0578091144561768, 0.6304627060890198, -1.1237024068832397], [-0.5460718274116516, 1.3200799226760864, 0.6684293150901794, -0.2990837097167969, -0.699491024017334, 0.523946225643158, -0.9087852239608765, -1.2663458585739136, -1.6101115942001343, 0.91238933801651, 0.4573710262775421, 1.1058157682418823, 0.05605659261345863, 1.3217910528182983, -1.2328081130981445, -0.5053418278694153, 1.1566417217254639, 1.1285005807876587, 0.28367599844932556, 0.5985116958618164, -0.212117537856102, 0.0437912717461586, 1.0643181800842285, -1.7637207508087158, -0.3966217637062073, -0.0628429725766182, 0.33314183354377747, -0.9063357710838318, 2.0369913578033447, -1.1061519384384155, 0.06519834697246552], [0.6842378377914429, 0.7526978254318237, 0.9526932835578918, -0.6658565402030945, 0.06648173183202744, 0.36821192502975464, -0.06835149973630905, -0.06076599285006523, -0.3804909884929657, -0.21753908693790436, -0.7286021709442139, 1.0072438716888428, -0.9548628926277161, -0.2399837076663971, 0.3776617646217346, -0.4989333152770996, -0.24854716658592224, 1.6066550016403198, 0.20647092163562775, 1.1743139028549194, 0.1475559026002884, 2.2796247005462646, 0.5564283728599548, 0.5056384205818176, 0.2245904505252838, 0.6813902258872986, -0.18801477551460266, -1.8589617013931274, -1.0215167999267578, -2.8348183631896973, -2.0644733905792236], [2.0344114303588867, 1.1598665714263916, 2.5011487007141113, -0.7682463526725769, -1.1100612878799438, 0.4468945264816284, 0.27833592891693115, 0.5219970345497131, -0.9131054282188416, -1.536738634109497, 1.0190349817276, 0.8302682638168335, 2.4334323406219482, 0.116155244410038, 3.296583890914917, 0.04085089638829231, 0.14003099501132965, 0.3295910656452179, 0.6420074105262756, -1.5967093706130981, -0.885112464427948, -0.6535990834236145, 2.4378762245178223, -1.0908697843551636, -0.6392540335655212, 0.8886602520942688, 0.942479133605957, -1.0680162906646729, 1.9644780158996582, 0.11881304532289505, 1.105741024017334], [0.991860032081604, 0.5347906947135925, -0.5409600138664246, -1.5906994342803955, -0.8863663673400879, 0.3145420253276825, -1.184556007385254, 0.5981714725494385, -0.9303840398788452, 0.7018967270851135, -0.2000168263912201, -0.5177063345909119, -0.005377970170229673, -0.1496669352054596, 0.4575062692165375, -0.21806295216083527, -1.5416975021362305, 0.7798487544059753, -0.3161563575267792, -2.9996724128723145, 1.1771154403686523, 1.1720691919326782, -1.42130708694458, -0.6114386320114136, -0.018857182934880257, 1.4498953819274902, 0.8993534445762634, -0.9685867428779602, -0.28276732563972473, 0.7314795255661011, -1.6019634008407593], [1.385502576828003, -0.5689979791641235, 0.2854747772216797, 1.2966066598892212, -0.8381891250610352, 0.17500261962413788, 0.006027581170201302, 0.8720093965530396, -1.0846573114395142, 0.1683686077594757, 0.32276177406311035, -0.38410481810569763, 0.5920903086662292, 1.5063270330429077, 0.6080256104469299, 0.7853367924690247, -1.582865595817566, -0.7175677418708801, -1.3542646169662476, 1.1495567560195923, 0.3745271563529968, 0.03599359840154648, 0.6191831827163696, -1.2620059251785278, 0.4026302695274353, -0.527283787727356, -1.026094675064087, 0.6835550665855408, 0.5356741547584534, -1.8444877862930298, -0.23532740771770477], [1.5992300510406494, 1.2757439613342285, 0.21113061904907227, 0.21292538940906525, 1.1811269521713257, -1.621106743812561, 0.6169980764389038, -0.2825905978679657, 0.0328470803797245, -1.2628110647201538, 1.7319976091384888, 0.952045202255249, -1.4987531900405884, -0.4840964078903198, -0.196431502699852, -0.7327600121498108, -0.5087536573410034, 0.3281940221786499, -0.41925248503685, -0.4190903306007385, -0.9821982979774475, 0.7112870812416077, -0.04864524304866791, 0.36918187141418457, -0.41753289103507996, -1.664416790008545, -1.1104389429092407, 0.9294427037239075, 0.09185943752527237, 0.9925876259803772, -1.2020431756973267], [-0.4934542775154114, -0.6199666261672974, -0.9893078207969666, -0.9172034859657288, -0.32199716567993164, -0.3881458342075348, -0.47009795904159546, -0.36108601093292236, -1.3968324661254883, 1.4736343622207642, -0.0741904228925705, -0.43372154235839844, -0.6972594857215881, 1.1095844507217407, 0.03319726511836052, -1.0517191886901855, -2.4062070846557617, 1.9903373718261719, 1.1485300064086914, -0.14587071537971497, -0.4262557923793793, -0.07704517990350723, -0.028758326545357704, -1.3511680364608765, -0.2957921326160431, -1.987646222114563, -0.298019140958786, 0.11449426412582397, 1.1766167879104614, -1.5876508951187134, 1.323006272315979], [-0.3718750774860382, -0.2654281258583069, -0.09851600229740143, -0.7546308636665344, -0.08641711622476578, 0.2749064564704895, -0.3840329945087433, -1.658310890197754, -0.8580965995788574, 1.0519920587539673, -1.467100977897644, 0.3593783974647522, 1.594077229499817, -0.08761713653802872, 0.19688762724399567, -0.342146098613739, 0.6674664616584778, 0.3407857418060303, -0.366608589887619, -0.29141268134117126, 2.143644332885742, 1.6157501935958862, -0.3560062348842621, -0.9199876189231873, 0.7677261829376221, 1.5042898654937744, -0.08333103358745575, -1.5493336915969849, -0.1944812685251236, 0.11702635139226913, 0.45875877141952515], [0.5397542119026184, -2.1401402950286865, 0.6504780054092407, 0.7611957788467407, 0.8778930902481079, -0.65944904088974, -0.003328672144562006, 1.0077170133590698, 1.1040465831756592, 2.0367271900177, -0.475725919008255, -1.8536057472229004, 0.49536243081092834, 0.6998342275619507, -0.6383841037750244, 2.352841854095459, 0.7070732712745667, 1.918355941772461, 0.28254786133766174, 0.181104376912117, -1.066089153289795, 1.0335346460342407, 0.7301533222198486, -2.228980541229248, -1.1030865907669067, -2.1233315467834473, 0.05767300724983215, -1.1426228284835815, 1.4985899925231934, -0.18096663057804108, -0.5074804425239563], [2.3407552242279053, -1.5281121730804443, -0.09656616300344467, 0.3691624701023102, -0.3930843472480774, 0.18619076907634735, 0.8205493688583374, -0.6782922744750977, -0.45622631907463074, -0.7214857339859009, 0.014197789132595062, 1.049493432044983, 0.6707140207290649, 1.3234734535217285, 1.0261937379837036, 1.2632052898406982, -0.945482075214386, -0.10428670793771744, 0.34891393780708313, -0.3430003523826599, -1.2466931343078613, 1.6020289659500122, 0.5775207281112671, 0.8452780842781067, -0.08740496635437012, -0.9747108817100525, -0.30536237359046936, 1.929614543914795, 0.6016175150871277, 2.298915386199951, -0.1934422254562378], [0.41781482100486755, -0.08016592264175415, -0.935230016708374, 1.150066614151001, -0.7145572900772095, -1.8518192768096924, -2.3199501037597656, 2.846038341522217, 2.47430419921875, 2.144737720489502, 0.7553079724311829, 0.48922958970069885, 1.0311627388000488, 0.7393538355827332, 1.3591899871826172, -1.1574101448059082, -0.8072368502616882, -0.41575512290000916, -1.7197027206420898, -1.1508558988571167, 1.841310977935791, 1.8541069030761719, 0.29674065113067627, -1.5691885948181152, -0.4027461111545563, -1.5721129179000854, 1.1750625371932983, -2.4129810333251953, -2.0849103927612305, 1.7131956815719604, 0.15231648087501526], [-0.4651757776737213, 1.706803321838379, -0.8817654848098755, 0.1421533077955246, 1.2613776922225952, 1.1758129596710205, 0.13567321002483368, 1.4719443321228027, 1.7200630903244019, 1.6434890031814575, 0.1649996042251587, -1.906849980354309, 0.7457695603370667, -0.535515308380127, 1.231110692024231, -1.6356066465377808, -0.7918795943260193, -1.042510986328125, -1.44212806224823, -1.7224966287612915, -0.13305003941059113, -0.30615219473838806, -1.3683863878250122, -1.845118761062622, 0.37490352988243103, -0.24637296795845032, -0.189091756939888, 0.4671521484851837, -0.5512995719909668, -1.0935777425765991, -0.722172737121582], [1.274505853652954, 1.0718426704406738, -1.4877359867095947, 0.38264453411102295, -1.8261682987213135, -0.04167108237743378, -2.153646945953369, -0.2049959897994995, -1.3005794286727905, -1.1254481077194214, 0.04774310812354088, -1.2163439989089966, 0.9300553798675537, -1.5894544124603271, -0.4297384023666382, -3.31866192817688, -0.5428176522254944, 1.0267412662506104, 0.5845681428909302, -0.021868843585252762, -2.073150396347046, 0.18572546541690826, 0.11695560812950134, 0.18210679292678833, 0.39689961075782776, -0.9673024415969849, 0.4656801223754883, 0.9441999197006226, -0.27983149886131287, -0.45341140031814575, -1.5953530073165894], [1.0715177059173584, -0.12060504406690598, 0.39353683590888977, -0.06882255524396896, 1.0891073942184448, 0.3741447925567627, -0.7695006728172302, 0.3206258714199066, 1.2124370336532593, -1.0211156606674194, 0.07376037538051605, 0.5109211802482605, -0.22015637159347534, -1.9678125381469727, -0.31839001178741455, -0.3794640600681305, -2.0860862731933594, -0.1072680801153183, 1.2370827198028564, -0.018792303279042244, -1.2330862283706665, 1.2479811906814575, -0.09932219982147217, 0.23706082999706268, 0.5839447975158691, -1.368751883506775, -1.2576172351837158, 0.27972733974456787, 0.17707638442516327, -0.057150375097990036, 1.1795189380645752], [0.08150725811719894, 0.3324219584465027, -1.4584101438522339, 0.27845773100852966, 0.647488534450531, -0.7814022302627563, 1.8085064888000488, -1.2725934982299805, 1.0041025876998901, 0.24639791250228882, -1.0585336685180664, 0.7890924215316772, 0.7194098234176636, 0.5231953859329224, 1.3285479545593262, 0.2778262794017792, 0.4277343451976776, 0.5301883816719055, 0.6954048275947571, 1.122246503829956, 0.5208264589309692, -0.7264543771743774, 0.09232562780380249, -0.8818853497505188, 0.8273704051971436, -0.9278144240379333, 0.34638160467147827, -1.4764500856399536, 0.758579671382904, 0.14331857860088348, -0.49674686789512634], [-1.508273959159851, -0.40597179532051086, 0.2410571426153183, 0.9164498448371887, -0.08942605555057526, 0.6071053147315979, -0.272225558757782, -0.03998695686459541, -1.4240984916687012, 1.2999202013015747, 0.8563091158866882, -0.1813359260559082, 0.9492132067680359, -0.2406189739704132, 1.438356637954712, -0.7024730443954468, -1.7546781301498413, 0.7500275373458862, 0.1041160374879837, 0.4227138161659241, 0.6476534008979797, -0.878829836845398, 0.7897257804870605, 1.3265550136566162, 0.23966409265995026, 0.3275817632675171, 1.014807105064392, -1.1565945148468018, 1.2346611022949219, -1.0933159589767456, 1.6302120685577393], [-0.17337459325790405, -0.2848508059978485, -0.1719411164522171, -0.056986674666404724, 1.717917561531067, 0.1160164624452591, -0.7337120771408081, 1.9104218482971191, 0.011295527219772339, -0.199677973985672, 1.094290852546692, 0.7045323252677917, -0.19077366590499878, 0.21981391310691833, 1.7948367595672607, 0.5166721343994141, -0.31168174743652344, 0.1780727207660675, 0.8508645296096802, -0.012376775965094566, -0.4462644159793854, 1.3086360692977905, -0.7247142791748047, 0.8466644883155823, -2.6653454303741455, -1.0006901025772095, -0.31510624289512634, -0.5575198531150818, 0.3401992917060852, 2.064213752746582, 0.7625582814216614], [-0.3253912329673767, 0.09942755103111267, -1.3024741411209106, 0.5082759857177734, -1.1463775634765625, -0.1630144715309143, -0.24461308121681213, -1.5851385593414307, 0.31294456124305725, -0.05619794875383377, 2.385061264038086, -1.5681160688400269, 1.8931511640548706, 0.6656240224838257, -1.6675878763198853, -0.33275237679481506, -2.9697630405426025, 1.2764793634414673, -1.9474986791610718, -0.37121105194091797, 0.4916924834251404, 0.8561903834342957, -0.4538915157318115, 0.4492330253124237, -0.9812260270118713, 1.291412353515625, 0.6067607402801514, -1.3116984367370605, -0.7351610660552979, -0.284533828496933, 2.0309839248657227], [0.9643575549125671, -1.8399949073791504, 1.837459683418274, -0.732742190361023, 0.3860560953617096, -0.15426769852638245, 0.4534689784049988, -0.28595614433288574, -0.8778231143951416, -0.006109577603638172, 0.03348139300942421, 1.2406398057937622, -0.9813918471336365, -1.0527762174606323, 1.1841530799865723, -0.1349211484193802, -2.0920557975769043, -1.061203956604004, -1.7381771802902222, -1.2321058511734009, -1.4646575450897217, 1.761276125907898, -0.8852013349533081, -1.1146186590194702, 2.6289920806884766, -0.49063751101493835, 0.419974148273468, -0.5169654488563538, -0.9442557692527771, 0.6283194422721863, 1.9375513792037964], [-0.6144128441810608, -1.0179402828216553, 0.8296866416931152, 1.6951051950454712, -0.3631415367126465, 0.4591749310493469, -0.29004141688346863, -1.2635046243667603, -1.3431103229522705, 0.4486519992351532, 0.7034425735473633, 0.9123783707618713, -0.05411975458264351, -1.333357572555542, -0.11542432755231857, 0.816146731376648, -0.38661760091781616, -0.4496946930885315, 0.834000825881958, -0.9491202235221863, -0.7023788094520569, -1.0025341510772705, 1.0377601385116577, -0.8048997521400452, -0.26946204900741577, 1.1457351446151733, -0.778425395488739, 1.1954505443572998, -0.6128268837928772, 0.11370459944009781, -1.3487194776535034], [0.7749868035316467, -1.55853271484375, -0.4154419004917145, -1.167047142982483, -0.6111412644386292, -0.8320674896240234, -0.480337530374527, -0.3052077293395996, -1.253352403640747, 1.8503278493881226, 0.1309313327074051, -0.8756036758422852, 0.25498202443122864, -0.45500174164772034, -1.537900447845459, -1.3020150661468506, 1.5371111631393433, 0.694943904876709, -0.47583794593811035, 2.3250982761383057, -0.3908231854438782, 0.5628311634063721, 1.0576192140579224, 0.07884260267019272, -1.343044638633728, 0.019969239830970764, 0.4734455347061157, -0.6000180244445801, 0.5139465928077698, -0.5913829803466797, -0.10209953784942627], [1.1275430917739868, -0.9911660552024841, 0.5884270668029785, -0.6454065442085266, 0.9478386044502258, 0.7278352975845337, -0.77677983045578, 2.362229585647583, 0.6374277472496033, 0.48092028498649597, 0.297683447599411, -0.4190966784954071, -0.8967857360839844, 0.6297850608825684, 1.0206648111343384, -0.11123280227184296, 1.382651448249817, 0.23791290819644928, -0.48433545231819153, -1.0472806692123413, -1.3668467998504639, 0.6519150137901306, 0.5540313124656677, -1.7215654850006104, 0.4473518133163452, -0.12331268936395645, -0.9495335817337036, -2.163816213607788, 0.6140503883361816, 0.45586222410202026, 1.1774406433105469], [-0.14428037405014038, 0.9348194003105164, 2.7604928016662598, -0.7984036803245544, -0.7818652391433716, 2.1959071159362793, 0.9356593489646912, -0.7877044081687927, -1.2982878684997559, -0.7553954124450684, -1.0601835250854492, 0.47903725504875183, 0.6573916673660278, -1.8584061861038208, 0.4527602195739746, 0.6982101798057556, 0.6773503422737122, 0.31275486946105957, 1.6923208236694336, 0.14804059267044067, 0.15789489448070526, -0.8072845935821533, -1.6362837553024292, 1.0412927865982056, -0.10539046674966812, -0.3820647895336151, 0.1995851993560791, -0.2670189440250397, 0.5713584423065186, -0.09005743265151978, -0.5359631776809692], [0.2071048468351364, 0.37369269132614136, -0.6401896476745605, 1.8969919681549072, 0.25342652201652527, 0.11338088661432266, 0.41908931732177734, -1.5317137241363525, 0.6405224800109863, -0.021210668608546257, -0.5670639872550964, -0.8593438863754272, -0.13155154883861542, -2.009763717651367, 0.6383091807365417, -0.05452568084001541, -0.7905638217926025, 1.1853781938552856, -0.8660032153129578, -1.1514426469802856, -0.3950519859790802, -0.510940432548523, -1.07916259765625, 0.26003411412239075, 1.0983896255493164, -1.369622826576233, -0.9987757802009583, -0.037695303559303284, -0.928203821182251, 0.5687654614448547, 1.1224277019500732], [-1.5949245691299438, 0.9421014189720154, -0.7616243958473206, 0.45742735266685486, 1.5198205709457397, -0.8618493676185608, -0.7337480783462524, 0.11037804186344147, -2.397109270095825, -0.07785163819789886, 0.15084350109100342, -0.7562969923019409, -1.5702989101409912, 1.061653971672058, -0.9157474040985107, 0.5490899085998535, 0.8685299158096313, -1.722694993019104, -0.14554716646671295, -0.4665551483631134, 0.30858293175697327, 1.8024929761886597, -0.49789807200431824, 0.42459043860435486, 0.16828685998916626, -0.5908886790275574, 1.1485223770141602, 0.1867581158876419, 0.3462722599506378, 0.17227865755558014, -1.4120701551437378], [-0.839078426361084, 0.03868807107210159, 0.5169333219528198, -1.670350432395935, -0.8458612561225891, -0.19895727932453156, -0.3093803822994232, -0.5626093149185181, -0.7902058959007263, -0.5753894448280334, 0.042829059064388275, -0.02773940935730934, -0.6294649243354797, -0.15410351753234863, -1.3870915174484253, 0.30029627680778503, 1.0713344812393188, -0.20922274887561798, -0.5400494337081909, -0.4969332218170166, 0.27515190839767456, -0.8263818621635437, -0.499021053314209, -0.47083979845046997, 0.07497306168079376, -0.5243374109268188, 1.750773549079895, 0.1447359323501587, 0.23474179208278656, -0.9278759360313416, 0.605278491973877], [0.375457227230072, 0.47584712505340576, 0.8866193294525146, -0.3925503194332123, 1.0818092823028564, -0.05804560333490372, 0.07236815243959427, -0.9884893298149109, 0.428257554769516, 0.7341750264167786, 0.4841792583465576, -0.06215016916394234, -0.028649775311350822, 1.1281628608703613, 0.9963968396186829, -0.5090944766998291, -1.029170274734497, 1.0327637195587158, 1.4395607709884644, -0.43852075934410095, -0.9306878447532654, -0.7996318340301514, 0.8226134181022644, -0.42137280106544495, 0.39485353231430054, -2.4669575691223145, 0.7463653683662415, -0.12409964948892593, 1.3669153451919556, 0.40624499320983887, 0.006589402444660664], [0.7110879421234131, 0.1195206567645073, -0.8238638043403625, -0.3845605254173279, 0.28626126050949097, 1.1153188943862915, -0.9247187972068787, -0.29399344325065613, 1.7048697471618652, 0.32468464970588684, -0.7818419933319092, -0.7779103517532349, -1.7214186191558838, -0.20177890360355377, 0.1685919463634491, -2.299051284790039, -1.2453912496566772, -0.1739024817943573, -0.11598765105009079, 0.1548151969909668, -2.9967920780181885, 0.8579975962638855, -1.2464616298675537, -0.49855226278305054, 0.15581871569156647, 1.1388543844223022, 1.3313292264938354, -0.25807201862335205, -0.7262607216835022, -0.42380091547966003, 0.2542918920516968], [0.2344033569097519, 1.7522231340408325, 0.005713304039090872, 1.1399048566818237, 0.009918268769979477, 0.11584564298391342, 1.5636317729949951, -0.4049302041530609, -0.3821682929992676, -1.152870535850525, 0.0675937831401825, 1.2693647146224976, 0.23799115419387817, -1.5189564228057861, -0.8888655304908752, 0.9143125414848328, -1.3566995859146118, 0.21817295253276825, -1.3863744735717773, 1.5542210340499878, 1.1671525239944458, -0.35616979002952576, 0.7663818001747131, -1.144040822982788, 0.3704150915145874, -1.633056402206421, -0.1673170030117035, 0.36408865451812744, 0.1120203509926796, -1.142240285873413, -1.0189927816390991], [0.4709406793117523, -1.0527565479278564, -0.0965019166469574, -0.09140557795763016, 0.6747688055038452, 1.06868577003479, -0.7935543656349182, -0.04123418778181076, 1.2700631618499756, -0.08953435719013214, 1.2833468914031982, -0.8221092820167542, -1.4305740594863892, -0.5539798736572266, 0.32257533073425293, -0.022160528227686882, -0.6301864385604858, -1.8576724529266357, 0.3340699076652527, -0.639860212802887, -0.04294991120696068, -0.5864004492759705, 0.1600489467382431, -2.219802141189575, -1.699904203414917, -0.565126359462738, -1.094895362854004, 0.14202599227428436, 1.1512762308120728, -0.48379454016685486, 0.5357025861740112], [1.097241759300232, 0.4727626144886017, 0.22292709350585938, 1.5896059274673462, 0.16931575536727905, 2.0047061443328857, -0.6567842364311218, -1.235592246055603, 0.44807103276252747, -1.2022422552108765, -0.7937237620353699, 0.03620556369423866, 0.18992897868156433, -0.2672426104545593, -0.11100775003433228, 0.1405043751001358, 0.47785496711730957, 0.5370975136756897, -0.4650322496891022, 0.05897559970617294, -0.7235539555549622, 0.68526691198349, -1.0691277980804443, 0.4324953556060791, -0.0627380758523941, -1.3089455366134644, -0.5718945860862732, -0.9570289254188538, -0.32899364829063416, 0.875920295715332, -0.5589437484741211], [-0.5411231517791748, 0.6497879028320312, 1.1681050062179565, 0.5969669222831726, -0.4875859022140503, 0.4788987934589386, -0.4299287796020508, -1.2664819955825806, 0.9415683150291443, 1.2912487983703613, 0.5939888954162598, 0.42518723011016846, -0.2624174654483795, 1.1246728897094727, 0.6684484481811523, 1.0251718759536743, 2.341144323348999, -0.22655509412288666, 0.3817678391933441, -1.0719630718231201, 0.38390472531318665, -0.6674644947052002, -0.16893114149570465, 1.311453104019165, -0.7929648160934448, -0.6958339214324951, -0.46568456292152405, -1.2517460584640503, -1.27717125415802, -1.3701927661895752, 1.3642830848693848], [-1.0100924968719482, -0.45644253492355347, -0.02263643406331539, 0.39339402318000793, 0.8018644452095032, -0.6908888220787048, -1.2194432020187378, -0.39568567276000977, 0.845304548740387, -0.018860939890146255, 0.8548625111579895, -0.9734957218170166, -0.8929722905158997, 1.3862168788909912, -0.15174219012260437, 0.4078148305416107, 0.13085125386714935, -0.5798578858375549, -0.10315442085266113, -0.8918983340263367, -1.7591888904571533, 0.12373191863298416, 1.932847023010254, 1.152708649635315, 1.741357684135437, -0.8313584327697754, -1.8616597652435303, -0.24237248301506042, 0.8344513177871704, 0.04479898512363434, -1.5920791625976562], [-0.3854620158672333, -0.8553760647773743, 0.4414701461791992, -0.6386474967002869, 0.25379717350006104, 0.8181407451629639, -0.03251338750123978, -1.1060199737548828, -0.7068474292755127, 0.06944452226161957, 0.07622247189283371, -1.6537500619888306, -0.4274130165576935, 0.8562620878219604, -1.0109565258026123, -0.3658285140991211, 1.2741838693618774, 1.1216728687286377, 0.3507866859436035, 1.0892198085784912, 1.8637112379074097, -1.532790184020996, -1.2994399070739746, -1.8560892343521118, -0.668485164642334, -1.1751433610916138, 0.08281990885734558, -1.9162863492965698, -0.608836829662323, -0.9929649829864502, -1.2117431163787842], [-0.24460576474666595, 0.979509711265564, 0.5811188817024231, 0.27953267097473145, 1.097720980644226, -0.036493152379989624, 0.8669057488441467, -1.2854284048080444, -0.4269864559173584, 0.021518591791391373, -0.13033731281757355, -0.14107897877693176, -0.5136457085609436, 0.013372328132390976, -0.6026788353919983, -0.11207947880029678, 0.07180707156658173, -0.33738335967063904, 0.2780008614063263, -1.9354324340820312, -0.3215479850769043, -0.1744881570339203, -1.6148300170898438, -0.43283775448799133, 2.1487977504730225, 0.23135894536972046, -0.05572584643959999, 0.06974141299724579, 1.13506019115448, -0.8590519428253174, 0.5592306852340698], [0.7376121282577515, 0.7269995808601379, -0.5098750591278076, 1.0249706506729126, 0.965287446975708, -2.6137256622314453, -0.014858915470540524, -0.22996832430362701, 0.7362406849861145, 0.6185839176177979, -1.360425353050232, -0.0999433621764183, -0.9970879554748535, -0.5900167226791382, -0.5924866795539856, 0.20837675034999847, -1.2420120239257812, 0.3980320692062378, -1.1078174114227295, -0.6900467276573181, -1.2513986825942993, 0.3869415819644928, -1.2487812042236328, 0.7676643133163452, -0.7035082578659058, -0.6674354672431946, 1.297192931175232, -1.7618216276168823, 0.5524778962135315, -0.5349332690238953, 0.21677926182746887], [0.8255569338798523, -0.5247205495834351, -0.16517877578735352, -1.373883843421936, 0.397774875164032, 0.23881791532039642, 0.5839869379997253, 0.3442881405353546, 0.5964995622634888, -1.450986385345459, 1.272254228591919, 1.1521461009979248, -1.3647252321243286, -0.6701651215553284, -1.2062016725540161, 0.2445482313632965, 0.321230411529541, -1.9317991733551025, 0.6712756752967834, -0.17382927238941193, -0.22331330180168152, -1.153573989868164, -1.0190579891204834, -0.1792765110731125, 0.9607180953025818, -0.07020413130521774, 1.307187795639038, -0.8685872554779053, -0.5753854513168335, -1.609528660774231, 1.0080933570861816], [-1.6022512912750244, 0.03820056468248367, 0.36820071935653687, -0.49221065640449524, 0.3276534080505371, -0.38920363783836365, -0.329581081867218, -0.4508781135082245, -0.45299211144447327, 0.6590209007263184, 0.3803202211856842, 0.20435695350170135, -0.2577323615550995, 0.1496373862028122, -0.06361223012208939, -0.8096859455108643, 1.1646249294281006, 0.3269283175468445, 0.42841771245002747, 0.9001417756080627, 0.32641443610191345, -1.830308198928833, -0.49467751383781433, -1.8678653240203857, -0.33775365352630615, 1.0921822786331177, 0.7090123891830444, -1.2443784475326538, -0.5147417783737183, 0.9310961365699768, -1.0233380794525146], [0.4808003604412079, -0.8171783685684204, -1.401787519454956, 0.8078743815422058, -0.3187079429626465, -0.4712984263896942, 0.5496507287025452, -0.21708977222442627, 0.053550176322460175, -0.6521735787391663, 0.3188130855560303, -0.3922572135925293, 0.1975812017917633, 1.0207500457763672, -0.9698506593704224, -0.8899236917495728, 0.9914593696594238, -0.884381115436554, 0.4708156883716583, -0.0712578296661377, -0.09241536259651184, 0.7797442674636841, 0.9750664830207825, 0.3017346262931824, 0.7010027170181274, -0.8835291862487793, -0.3151910901069641, -0.8221455812454224, -1.038490891456604, -0.6038610339164734, -1.093428134918213], [-1.6798175573349, -0.13903498649597168, 0.12929752469062805, 1.634770393371582, 2.455183506011963, 1.3633126020431519, -1.2903530597686768, -0.5800521373748779, 0.6181479692459106, 1.2227787971496582, -1.4895954132080078, -1.1120442152023315, -0.7288857102394104, -0.24124419689178467, 0.9562284350395203, -1.7427161931991577, 1.4941238164901733, -1.5135977268218994, 0.5880029797554016, 2.6004533767700195, 0.6025485396385193, -0.9412236213684082, 0.5810254216194153, 0.5610688328742981, -0.5131603479385376, -1.461044430732727, -0.6704360842704773, -0.09542573243379593, 0.12191596627235413, 0.5961014032363892, -0.5739267468452454], [0.4973251223564148, 0.8283689022064209, -0.8741931319236755, 0.21678972244262695, -0.02883736602962017, 1.3365956544876099, 0.38391682505607605, 0.6554083228111267, 0.1764838546514511, 1.1928634643554688, -0.8768047094345093, -0.7914597392082214, -1.0341547727584839, 0.0414743646979332, -0.6787872910499573, -1.3030438423156738, 0.6601332426071167, 0.6454576253890991, -2.2209510803222656, -1.7704156637191772, -0.13675114512443542, 1.2243057489395142, 0.8880713582038879, 0.09256661683320999, -1.110275387763977, 1.4795215129852295, 0.8998793959617615, -0.4364912807941437, -0.6822623610496521, -1.16097092628479, 0.986305296421051], [-0.48466581106185913, 0.9610456824302673, 0.061736635863780975, -0.7031830549240112, 0.5857332348823547, -0.5855982899665833, -1.6612281799316406, -1.5429965257644653, -0.6281660795211792, -1.2945692539215088, -0.830669105052948, 0.1047571450471878, -0.22933721542358398, 1.6027297973632812, -0.5975714325904846, -0.004361962433904409, 0.3962728977203369, -0.18533839285373688, 0.27623000741004944, 0.4166838228702545, 1.9626625776290894, 0.7530367374420166, -0.40445390343666077, 1.193289041519165, -0.7597149014472961, 0.3226364254951477, -1.6019439697265625, -0.4433860778808594, 0.7056628465652466, -0.2699902355670929, 0.027541177347302437], [-0.512828528881073, -1.6209216117858887, 1.6270769834518433, -1.3286300897598267, 1.0354430675506592, 0.14734092354774475, -0.5264033079147339, -1.9635816812515259, 0.8490915894508362, 0.17715269327163696, -0.2051781862974167, 0.08175981789827347, 0.8089094758033752, 1.450427770614624, -1.4393497705459595, 0.11866585165262222, -2.201629400253296, -0.8549988269805908, 1.0540684461593628, 0.6068093180656433, 0.2586899399757385, 0.5126243233680725, 0.846413791179657, -0.7999319434165955, -0.7071583271026611, 1.2612510919570923, -0.8369940519332886, 0.09397516399621964, -1.422494888305664, 0.4839118421077728, -1.793463945388794], [0.20669396221637726, 1.753823161125183, -1.1639994382858276, 0.35284221172332764, -0.08218705654144287, -0.1997276395559311, -0.5749725103378296, 1.0222584009170532, 0.2770999073982239, 0.8785882592201233, 0.29082030057907104, 0.48208266496658325, -0.5405625700950623, -0.43295347690582275, -2.6252193450927734, 0.3591744899749756, 0.9073582291603088, -0.27401983737945557, -1.6551364660263062, 1.7557569742202759, 1.9878394603729248, -0.2722161114215851, 0.15771980583667755, 0.018919093534350395, 0.9745885729789734, 0.8800183534622192, 0.28560909628868103, 1.0652880668640137, 0.07693855464458466, -1.0547523498535156, -0.02105814218521118], [0.8958985805511475, -0.7475829720497131, -0.5021466612815857, 0.12750910222530365, -0.47867274284362793, -0.3127177655696869, -0.9397487044334412, -1.3850632905960083, -0.39268022775650024, 0.07733870297670364, -0.0513063482940197, 1.5170745849609375, 0.31005600094795227, 0.31708812713623047, 0.3053382337093353, -0.29828840494155884, -0.5187252163887024, 0.4547334909439087, -0.0472324900329113, 1.9176992177963257, 0.2375006079673767, -1.5615051984786987, -0.37418845295906067, 1.8645480871200562, -1.137711763381958, -0.3690042495727539, -0.35921627283096313, 0.20699059963226318, 0.3336663544178009, -0.8075066804885864, -0.757714569568634], [0.36263614892959595, 1.634285569190979, 0.25425416231155396, -1.526407241821289, -0.6597191691398621, -0.07230282574892044, -0.8125987648963928, -0.07784027606248856, -0.3542684018611908, 0.02437490224838257, 0.5017791986465454, -2.197801113128662, 0.8541887998580933, 1.1723909378051758, 0.7364023327827454, -0.01421385444700718, 0.2598029375076294, 2.1610562801361084, 1.4093194007873535, -0.018822714686393738, 0.8257951140403748, 0.31065675616264343, -1.2925808429718018, -0.7970237135887146, 0.4129348695278168, -1.3076626062393188, 0.6887454986572266, 0.575278103351593, 1.6092519760131836, 0.017384227365255356, -1.6939457654953003], [-0.3782947361469269, -0.5909169912338257, -0.5143182873725891, -0.06261099129915237, -1.7867445945739746, 1.0981532335281372, 0.49595123529434204, -0.4646267294883728, -0.1828591376543045, 0.061147790402173996, -0.07329436391592026, -1.0181059837341309, -0.921370267868042, -1.3447450399398804, 0.4849357306957245, -0.7990272641181946, -0.5053874850273132, -1.5548467636108398, -0.03281436488032341, -0.9718114733695984, -0.8821970820426941, -2.134857416152954, 0.9522486925125122, 0.10819532722234726, 0.2360609471797943, -2.0840182304382324, -1.884204387664795, 1.4873034954071045, -0.4974699020385742, -0.1673143059015274, -0.05120202153921127], [0.11194189637899399, -0.00649813748896122, -1.1482971906661987, 0.8178497552871704, -0.6458467841148376, -0.20798125863075256, -1.4420146942138672, -1.7256686687469482, 0.8306668996810913, -1.4130992889404297, -1.721554160118103, -1.0723246335983276, 0.378785640001297, -0.2912886142730713, -0.6760950088500977, -1.060102939605713, -0.33190762996673584, -0.9864497184753418, -0.8168679475784302, -0.5591674447059631, -1.0157755613327026, -0.567794919013977, 0.801035463809967, -0.45461276173591614, 1.6623010635375977, -0.5870884656906128, -2.6277379989624023, -0.08844684064388275, 0.6719110012054443, -0.06473632156848907, -1.892642855644226], [-0.3231038749217987, -1.7665256261825562, 0.8580392599105835, 0.5304247736930847, 0.27292314171791077, 1.0685604810714722, -0.8255398273468018, 0.013615889474749565, -0.3631298840045929, -0.9504479765892029, -0.3816438317298889, -1.524263620376587, -1.7544747591018677, -0.9611214995384216, -0.037163496017456055, 0.25953802466392517, -0.08661375194787979, 2.2346205711364746, 0.9776848554611206, -0.47701531648635864, 0.8263320922851562, -1.005403757095337, 0.48044243454933167, -0.5254747867584229, 0.3102215826511383, -1.1315577030181885, 1.3476336002349854, -0.2011861652135849, 0.6842817068099976, 0.38579297065734863, -2.6614348888397217], [0.3738093376159668, 0.30389368534088135, -1.6337145566940308, 1.4113495349884033, 0.5116433501243591, -1.0374218225479126, 0.7313273549079895, -0.443685919046402, -0.42894884943962097, 0.7047721147537231, -0.22293011844158173, -1.1107780933380127, 0.03494589403271675, 0.3720880150794983, 0.7571253776550293, 0.3908412456512451, -0.3699364960193634, 0.70328688621521, -1.519095778465271, -0.44140464067459106, -0.28969183564186096, 0.7011076807975769, 0.16139622032642365, -0.0009976079454645514, 0.6124628186225891, -0.9646650552749634, -0.6641655564308167, 0.23817497491836548, -0.09632843732833862, 0.08275580406188965, -0.618524968624115], [-0.7715355753898621, -0.3684723973274231, 1.311316728591919, 0.36316415667533875, 0.3344014883041382, -1.5983766317367554, -0.8099855780601501, -1.1323156356811523, -0.40909427404403687, -1.1415095329284668, 0.5460498332977295, 0.08516452461481094, -0.09639070183038712, 0.7722142338752747, -0.44583842158317566, -1.3297133445739746, 0.14644959568977356, -0.8800665140151978, -1.446388602256775, -1.00503671169281, -0.6300265789031982, -0.6009039282798767, 0.16456544399261475, -0.3272649645805359, -0.14538390934467316, -0.9480281472206116, -1.0480141639709473, -0.07915555685758591, 1.4011805057525635, -0.7422923445701599, 2.6297638416290283], [-0.23762765526771545, -1.2253801822662354, -0.40424445271492004, -0.02154519222676754, 0.9008948802947998, 0.9159940481185913, -0.7349756956100464, -0.8297791481018066, -0.21886250376701355, -0.25814884901046753, 0.6681212186813354, 0.05786646902561188, 0.6159993410110474, -0.7964404821395874, 0.8199850916862488, -0.3245046138763428, -0.9445955157279968, -1.9501609802246094, -2.895263433456421, -0.5398381352424622, -0.3965272903442383, -1.6360639333724976, -1.9447120428085327, -0.36738428473472595, 0.6118091940879822, 0.19062650203704834, 2.0047478675842285, -0.43785762786865234, -1.7406421899795532, 0.538936197757721, -0.14846794307231903], [1.5749108791351318, -0.8808534145355225, 1.3619471788406372, -0.8278377056121826, -1.2051900625228882, -0.6168670654296875, -1.7284255027770996, -1.0300081968307495, 0.28483468294143677, -1.757996916770935, -0.6577789187431335, -0.29920482635498047, 0.6305592060089111, 1.144449234008789, 0.17744538187980652, 0.5562980771064758, 0.9354936480522156, 0.8785611391067505, -0.10730026662349701, 1.9299907684326172, 1.0134861469268799, 0.5678386092185974, 0.012979025952517986, -1.5775936841964722, 0.0356585793197155, -0.9259161353111267, 0.9289919137954712, 0.28423553705215454, 0.1848224401473999, -0.7898216843605042, -0.4235180914402008], [-1.2881503105163574, -0.7301387190818787, 0.2970917522907257, 0.8495899438858032, -0.5864861607551575, -0.8259982466697693, 1.145329236984253, -1.9043083190917969, -0.2887876033782959, 0.455424964427948, 0.7462840676307678, 0.46460315585136414, -0.9796565771102905, -0.9529938101768494, 0.25970447063446045, -0.7160426378250122, 1.1458837985992432, 1.0537950992584229, -0.9392752647399902, 1.1262152194976807, -0.616151750087738, 0.6606099009513855, 1.7268471717834473, -0.28790006041526794, -1.0076006650924683, -0.4122888147830963, 1.051368236541748, 0.20174413919448853, -2.5675110816955566, -0.4417691230773926, 0.15324419736862183], [1.499774694442749, 1.775849461555481, 1.2691822052001953, -0.3019898533821106, -0.8200145363807678, 1.0948666334152222, -0.14749178290367126, -1.4842333793640137, -1.1059318780899048, 1.5301949977874756, -1.1198011636734009, -1.6462138891220093, -1.9737364053726196, -1.4022648334503174, -0.4298378527164459, 1.0236833095550537, 1.3535614013671875, -0.7117483615875244, -1.1920320987701416, -0.687292754650116, 0.28548330068588257, 0.1351311206817627, 0.9049732685089111, -2.017061471939087, 2.018040895462036, -2.282590627670288, -0.408812552690506, -1.0735775232315063, 0.8066277503967285, 1.4877370595932007, -0.05463465675711632], [0.27823618054389954, 1.5529102087020874, 1.736811637878418, -1.3333450555801392, -0.44910168647766113, -0.023348985239863396, -0.6466782093048096, -2.6458709239959717, 0.10769779980182648, 0.09165707230567932, -0.4344644546508789, -0.041253793984651566, -0.8418391346931458, -1.6691166162490845, 1.1918271780014038, 1.0731401443481445, 2.884631872177124, 0.01808822900056839, -0.00015621018246747553, 0.26914462447166443, 0.975104808807373, -0.5858946442604065, -0.14099663496017456, 0.16775226593017578, 1.0768150091171265, -1.0798190832138062, -0.014544883742928505, -1.6455893516540527, 2.260863780975342, 1.2614243030548096, -0.614548921585083], [0.31900858879089355, 0.2313554286956787, -1.3015786409378052, 1.4724149703979492, 0.5477567911148071, -0.036055419594049454, 0.19059410691261292, -1.731704592704773, 0.13600072264671326, 1.8337385654449463, 1.4603376388549805, 2.5786287784576416, -0.7414077520370483, 1.4722613096237183, -0.0532732717692852, 0.5175474882125854, 0.8043010830879211, -0.4411078989505768, 0.05610746890306473, -1.2795578241348267, -0.8814493417739868, -0.41398561000823975, 0.9874811768531799, -0.6580967307090759, 0.45072993636131287, 0.7737000584602356, -0.797280490398407, 0.36721935868263245, -0.4266416132450104, -0.7556228041648865, -0.9666985273361206], [-1.0557652711868286, -0.08353358507156372, 1.7234890460968018, -0.5066438913345337, -1.9465192556381226, -0.9320257902145386, 0.6610256433486938, -1.7496752738952637, 1.6782317161560059, -0.8539604544639587, 1.3994266986846924, 0.2618565857410431, -1.7265405654907227, -0.6558769941329956, 0.6340780258178711, 1.3476145267486572, 1.752793312072754, 0.18517877161502838, -1.0347909927368164, -1.892870545387268, 0.4603668451309204, 0.5405884981155396, 0.19218307733535767, -0.872097373008728, -0.2120080590248108, 2.010612726211548, -0.42413902282714844, -1.0623432397842407, -1.1574524641036987, -0.7847273945808411, 1.7627439498901367], [0.18572326004505157, -0.9710010886192322, -0.15757150948047638, -1.4154078960418701, -0.9389414191246033, 0.8586532473564148, -0.9536478519439697, 1.4026331901550293, -0.801718533039093, -0.22788755595684052, 0.8638091683387756, -0.14773938059806824, 1.3940098285675049, -1.521504521369934, 0.9079343676567078, -0.5736343264579773, 0.040092457085847855, 0.9808877110481262, -1.023071050643921, 0.9823772311210632, -1.4210011959075928, 1.0840511322021484, 0.5765699148178101, 0.05162734165787697, -0.23036813735961914, 0.44424912333488464, -1.2768160104751587, -1.4111088514328003, -0.40309855341911316, -0.16664613783359528, 0.4229128956794739], [-1.3721333742141724, -0.04431099444627762, 0.7740611433982849, -0.053824882954359055, 0.5431603193283081, 0.3523256182670593, 1.5468273162841797, 0.8732019662857056, -0.3203423023223877, -0.9601350426673889, 2.357797622680664, -1.2296950817108154, 0.16502217948436737, 0.2509765923023224, 1.9139312505722046, 0.8976900577545166, -0.863908052444458, -0.6548640131950378, 0.15273533761501312, 0.913069486618042, 1.1760611534118652, -1.5295088291168213, -0.3901248574256897, -0.722743034362793, -2.0653278827667236, 0.4868890047073364, -1.1638554334640503, 0.059346266090869904, -0.20238618552684784, 0.3858749568462372, -0.23874981701374054], [-0.9418673515319824, -0.5103363990783691, -0.22561690211296082, -1.7408713102340698, -1.496909260749817, 0.4453689455986023, 0.6625651121139526, -0.4433211088180542, 0.44764384627342224, 0.4197987914085388, 1.061331868171692, 0.16334712505340576, 0.446850448846817, 1.266608476638794, -0.6259452700614929, -0.258556991815567, -1.4347718954086304, -1.3279855251312256, -1.2335529327392578, -1.3729219436645508, -0.08100682497024536, -0.3216901421546936, 1.9436538219451904, 1.2808692455291748, -0.11205561459064484, 1.0009570121765137, -0.21549224853515625, -0.04315852001309395, 1.2050033807754517, 1.0537605285644531, -0.9417451620101929], [-0.2791809141635895, -0.9524105787277222, -0.8650596141815186, 0.29248708486557007, 1.209139108657837, -0.5326158404350281, 0.11807545274496078, -1.3260328769683838, -0.34483739733695984, 0.4229625463485718, -1.2176827192306519, 0.6003639101982117, -0.5505265593528748, -1.3170374631881714, -1.0288807153701782, -0.10862318426370621, -0.47064757347106934, -1.3058183193206787, 0.6148233413696289, 0.342324435710907, -0.2907104194164276, -0.11936210840940475, 0.8187763690948486, -0.5408757925033569, 0.00835030060261488, 0.26851433515548706, -1.4107890129089355, -0.4524790942668915, 1.5564494132995605, 1.4695909023284912, 0.5362366437911987], [-0.5271303653717041, -0.263479620218277, 0.016773030161857605, 0.31537503004074097, 0.17650234699249268, -0.5869859457015991, 1.067976474761963, -0.8607656359672546, 0.7631015777587891, 1.4559338092803955, 0.4287804067134857, -0.28625938296318054, -2.6311051845550537, 1.1936184167861938, 1.4053318500518799, -0.056618768721818924, 0.33227023482322693, 0.5392254590988159, 0.3061976134777069, -0.7595224380493164, -0.3928467631340027, -1.270014762878418, 0.9001995921134949, 0.6659714579582214, 1.98585045337677, -1.182450532913208, -0.19620610773563385, -0.11143781244754791, -0.3337741196155548, 0.13646583259105682, -0.11269709467887878], [-0.6348387598991394, -0.21566730737686157, -0.5881375670433044, 1.2117100954055786, -0.49229657649993896, -1.3204714059829712, -0.2260860949754715, -1.197218656539917, 0.07512439787387848, 0.12357188761234283, -0.7278842926025391, 1.0523900985717773, -0.7627328038215637, 0.5464094281196594, 0.49573925137519836, -0.9952153563499451, 0.19806016981601715, 0.5484898090362549, -0.288287490606308, 0.5082020163536072, 0.36439135670661926, -0.7580965161323547, 0.9314829707145691, -0.8097376823425293, -0.17494820058345795, 0.2446538805961609, -0.9209794402122498, -0.3994826376438141, 1.0761418342590332, -0.7174903750419617, 1.4020599126815796], [-1.466352939605713, 0.010600385256111622, 1.4958144426345825, -1.1131680011749268, 0.7792298197746277, -0.29731810092926025, -0.3659686744213104, -0.4261714816093445, -1.8619122505187988, -0.09007216989994049, -1.6904629468917847, -1.2163262367248535, 0.08176849782466888, -2.140324592590332, 0.7139079570770264, -0.8798079490661621, 0.3279481530189514, 0.10442864894866943, 1.2034924030303955, -0.08480630069971085, -0.27476558089256287, 0.845099687576294, -1.1445338726043701, -0.4713309109210968, -1.6362913846969604, -1.4373526573181152, -1.1232725381851196, 2.2524099349975586, 2.4082489013671875, 0.1320929378271103, -0.3303682804107666], [-0.6551176309585571, 1.8003506660461426, 0.8380928039550781, -1.2607603073120117, 0.15172454714775085, 2.142789363861084, -0.5768862962722778, -0.22168096899986267, -0.27530795335769653, 1.6438074111938477, -0.18021364510059357, -0.6019604206085205, 0.8629665970802307, -0.8141658306121826, 1.6626849174499512, 0.3920193910598755, 0.4021841585636139, 1.1462870836257935, 1.168611764907837, 0.5211985111236572, -0.18360960483551025, 1.1529978513717651, -1.6535656452178955, -0.07286595553159714, -0.6018948554992676, -3.1850826740264893, -0.6950860023498535, 0.3605163097381592, 0.6465437412261963, 0.6029210090637207, -1.8374927043914795], [-0.6065263152122498, 0.4224570095539093, -0.09379478543996811, 1.9483762979507446, -1.1695209741592407, -0.25443825125694275, 1.040144920349121, 0.405944287776947, 0.40857386589050293, -1.4146863222122192, -1.2453469038009644, -0.6436789631843567, -0.3571300506591797, -0.3116327226161957, -0.04493436962366104, 0.726892352104187, 0.37684422731399536, 0.4646076261997223, -0.5617304444313049, -0.573874294757843, -0.3196546137332916, 0.10583218932151794, 0.06260602921247482, 1.303484559059143, -0.7509085536003113, -0.4367200434207916, 0.20892246067523956, -0.19696733355522156, -0.25846704840660095, 0.8166622519493103, 0.728489339351654], [0.6935450434684753, -0.007912646979093552, 0.48240184783935547, 0.6600342392921448, 0.3006336987018585, -0.9930435419082642, -1.017773985862732, -0.008633793331682682, -1.0099542140960693, -0.8188064694404602, 0.43640458583831787, 0.6591635346412659, -1.6171598434448242, 0.13217675685882568, -0.035406142473220825, 0.25676429271698, -0.45860958099365234, -0.39318400621414185, 0.5316866636276245, -1.047277808189392, 0.5257636308670044, 0.4961681365966797, -0.39551419019699097, -0.615029513835907, 0.038718659430742264, 0.645556628704071, 0.3238248825073242, 1.1675302982330322, 1.5667304992675781, 1.5668644905090332, -0.966923177242279], [0.31754443049430847, -1.0868337154388428, 0.27661633491516113, -0.580205500125885, 1.0159653425216675, 0.6279154419898987, 1.283399224281311, 2.8670599460601807, -2.8346750736236572, -0.12539105117321014, -0.253776490688324, 0.3483615219593048, 0.11433950811624527, 0.2705041766166687, 0.8714864253997803, -0.230600506067276, -0.1721847802400589, -0.683174729347229, 0.06825036555528641, 0.5454875230789185, -0.7256373763084412, -0.05606833100318909, -0.2901877462863922, -1.0747114419937134, 0.5802562832832336, 0.7854791283607483, -0.02701530046761036, 1.241782784461975, 0.2694283425807953, -0.7085946798324585, -2.631322145462036], [-0.4152401089668274, -0.44639503955841064, 0.34006771445274353, 0.2747352421283722, -1.5715162754058838, 0.39663010835647583, -1.066361427307129, -2.8237221240997314, 0.9209019541740417, 0.1327550709247589, -0.40475085377693176, -0.3800571858882904, 1.0418877601623535, -1.3605760335922241, -1.027356743812561, -1.1096727848052979, 1.1022136211395264, 0.5081177353858948, -0.11489449441432953, 0.7061768174171448, -0.6569194793701172, -1.0423524379730225, 0.9599167108535767, -1.072861671447754, -0.8256291151046753, -1.56143057346344, 1.1486451625823975, -0.5887597799301147, 0.7597559690475464, 0.33708420395851135, -0.6303848624229431], [0.9393934011459351, -0.06450194120407104, 0.724471926689148, 0.4152720272541046, -0.30458810925483704, -0.6735740900039673, -1.116087555885315, 0.7620155811309814, -0.04319137707352638, -0.9804452061653137, 1.2467771768569946, -0.16092677414417267, -0.5149791836738586, 0.38398829102516174, 1.5483970642089844, -0.7364452481269836, -1.8531826734542847, 0.5751078724861145, 0.019955476745963097, -0.3801290988922119, 0.5607262849807739, -1.5232127904891968, -0.30101388692855835, 0.22821247577667236, -0.8121591806411743, 0.8979605436325073, -0.0027319358196109533, -1.3947466611862183, -0.39520934224128723, -0.4326056241989136, 0.007151959929615259], [0.4038235545158386, 0.38131090998649597, 1.9164105653762817, 2.0418660640716553, 1.0462286472320557, -0.3980411887168884, -1.320983648300171, 1.403955101966858, 0.05671094357967377, 0.6523199081420898, -0.43254324793815613, 1.103031873703003, -0.5704027414321899, -0.34546783566474915, -0.2589680254459381, -0.2021854817867279, 0.25235509872436523, 0.19566094875335693, -0.39799046516418457, -0.26634615659713745, -0.4144361913204193, 0.8477921485900879, 1.0543828010559082, -0.0009899898432195187, -0.28664812445640564, -1.1506948471069336, -1.7135777473449707, 0.19022251665592194, -0.17822612822055817, -1.1200000047683716, -0.5354771614074707], [-0.0035796973388642073, 0.13033874332904816, 1.776381254196167, -0.021896036341786385, -1.1090670824050903, -1.2661736011505127, 0.27640411257743835, -0.2969399094581604, -2.1907455921173096, 1.2212927341461182, 0.21036195755004883, 0.8236870765686035, -0.7108004689216614, -0.025189513340592384, 1.7727519273757935, 1.1724658012390137, 1.4003496170043945, 1.5639631748199463, 0.9819134473800659, 0.051106978207826614, -0.766471266746521, 0.26446112990379333, 4.528099060058594, -0.5473648309707642, -0.06171491742134094, 0.24994495511054993, 0.9005752205848694, -0.6621305346488953, -0.04632720723748207, 2.378188133239746, 1.3570541143417358], [0.2627449631690979, -1.2042906284332275, 0.42776286602020264, 1.4653527736663818, -0.07582750171422958, -0.2316378951072693, 0.32459092140197754, 1.543845534324646, 0.7959599494934082, -1.1510162353515625, -0.40200650691986084, -0.6927193403244019, -1.2378662824630737, -0.7157668471336365, -0.581606924533844, 0.254505455493927, -0.5901006460189819, -0.5468810200691223, -0.02249983511865139, 0.7334126830101013, -0.39492714405059814, -1.3086826801300049, -0.7732113003730774, 0.3780534267425537, 0.9857895970344543, -0.8240052461624146, 1.2957459688186646, 1.7460805177688599, 0.9840703010559082, -0.1761072874069214, -0.9585814476013184], [-0.5982276201248169, -0.6098280549049377, -1.0165977478027344, -0.4497460722923279, -0.8595421314239502, -0.07068663835525513, 0.5270912647247314, -1.6199394464492798, -0.596868097782135, -0.6890312433242798, 0.16227658092975616, 1.0465278625488281, 0.5762686133384705, 0.31326207518577576, 0.5726742744445801, -0.9676131010055542, 0.2345767468214035, 0.20687884092330933, -1.0116175413131714, 0.2280246913433075, 0.49844977259635925, 1.9910709857940674, -0.2894696295261383, 0.26758506894111633, -0.23370207846164703, -0.2631383240222931, -1.0487736463546753, -0.4914109706878662, -1.4870761632919312, -0.8239482641220093, 1.6464641094207764], [0.5925680994987488, -0.36356088519096375, 0.8135951161384583, -1.0852253437042236, -0.02828109636902809, 2.066141366958618, -1.2257006168365479, 0.43271902203559875, -0.3017316162586212, 0.14545831084251404, -0.14128045737743378, -1.5727953910827637, 1.9212511777877808, -0.5274868011474609, -0.23876842856407166, 0.5538238286972046, 0.644307017326355, -1.68035888671875, -0.795757532119751, -0.1702987253665924, -2.42583966255188, 0.11270911991596222, -0.04508213698863983, 0.5975645780563354, 0.2077649086713791, 0.17120292782783508, 0.27548912167549133, -1.4312478303909302, -1.3671998977661133, -1.2518949508666992, -2.3404572010040283], [0.7697360515594482, -2.268510103225708, 0.19192321598529816, -1.6960968971252441, 0.7666382789611816, 0.0496959462761879, 0.02926958166062832, 1.2790285348892212, 0.2374524176120758, -1.320900559425354, 1.3521322011947632, -0.3056032955646515, 1.609988808631897, -0.7447879314422607, -0.25868573784828186, 0.6760613918304443, 0.8054176568984985, 1.2031936645507812, 0.8589191436767578, 1.5883809328079224, 1.55233633518219, 0.9857364296913147, 0.5131049752235413, 0.5103688836097717, 0.7834703922271729, -1.6245561838150024, -0.688304603099823, -1.347267746925354, -0.39729881286621094, -0.22568486630916595, 1.0327671766281128], [0.10954615473747253, -0.5932667851448059, -0.7667669653892517, 0.5646756291389465, -0.031397540122270584, -0.8650144338607788, 0.6216845512390137, -0.3678467571735382, 1.1159766912460327, -0.1379466950893402, -0.010189197957515717, 0.44085752964019775, 1.5182859897613525, -1.6728456020355225, -1.5846326351165771, -0.4072133004665375, 1.2781420946121216, 0.013818939216434956, 0.17249426245689392, -0.10687470436096191, -1.397630214691162, 0.7108827233314514, -0.19342520833015442, 0.5130883455276489, -0.7150583863258362, -0.7587121725082397, 1.180458426475525, -0.44047024846076965, 0.6454337239265442, 0.9139595031738281, -1.4365310668945312], [1.1323449611663818, 0.08160267025232315, -2.3133351802825928, -0.3252706825733185, -0.7016269564628601, 1.7096823453903198, 1.1657869815826416, 1.9240986108779907, 1.2308735847473145, 1.130040168762207, -0.7534153461456299, -0.8588849306106567, -2.6376194953918457, 0.14204610884189606, -0.06236693635582924, -0.29289761185646057, -0.7266058325767517, 1.5947542190551758, 0.5286194682121277, -0.578622579574585, 0.5938574075698853, 0.20746169984340668, -0.4606676697731018, 0.45059284567832947, 0.28030094504356384, -0.8169374465942383, -0.13353581726551056, -0.04518749937415123, 1.6148849725723267, -0.10625304281711578, 1.0483051538467407], [0.4438502788543701, 1.5384266376495361, 0.09901917725801468, 2.8182144165039062, -1.3372184038162231, -0.23375080525875092, 0.4733431339263916, 0.7891275882720947, -0.5157313942909241, 0.24417509138584137, 0.6511634588241577, 1.5706489086151123, 0.8293654918670654, 1.441846489906311, 0.4678781032562256, -0.9702507853507996, -0.46921300888061523, 0.6181798577308655, 0.001064202981069684, -0.005273886024951935, 0.24928295612335205, 0.16694758832454681, 0.15995587408542633, 0.25664207339286804, 0.5045489072799683, -1.6649513244628906, 0.3379359841346741, -0.6776567697525024, -1.2506482601165771, -0.015952954068779945, 0.9629379510879517], [-2.316134452819824, -0.21030667424201965, 0.13430264592170715, -2.2390902042388916, -0.8470133543014526, -0.39377760887145996, -0.5199034214019775, 0.875892698764801, -0.07336746901273727, 0.4961390197277069, 0.575797975063324, 0.44057875871658325, 1.2908276319503784, 1.9311217069625854, 0.8626510500907898, 0.17046146094799042, -0.45738255977630615, -0.9842987060546875, -0.6740337014198303, -0.34648749232292175, 1.1383068561553955, -0.3182818591594696, 0.12041434645652771, 1.176568627357483, 0.7174041867256165, 0.09099540114402771, 0.18193483352661133, 0.8152327537536621, -0.8307231664657593, 0.2477046102285385, -0.7200594544410706], [0.031063780188560486, -0.5863063931465149, 1.2949328422546387, 0.36272719502449036, -0.3385457396507263, -1.0989502668380737, 0.07461177557706833, 0.9054735898971558, 0.06830354779958725, -2.0592141151428223, -0.35544338822364807, -0.3413890600204468, 0.2581612765789032, 0.1978921890258789, 0.5082757472991943, 0.26519525051116943, 0.8206202983856201, -1.2698469161987305, 0.01689152605831623, -0.7054096460342407, -0.057969823479652405, -0.29669177532196045, 0.47772231698036194, 0.641867458820343, -1.7030041217803955, 0.06932715326547623, -2.137014389038086, -0.917722761631012, 0.46996092796325684, 0.6133231520652771, 0.14839911460876465], [-0.5928117036819458, -0.7043001651763916, 0.7719364166259766, -0.05457274615764618, -1.2621556520462036, -2.6704177856445312, -0.1883394420146942, 0.30255264043807983, -2.459839344024658, 1.5824111700057983, 0.6304231286048889, 1.0463393926620483, -0.6699048280715942, 0.1208510622382164, 0.6011061072349548, -1.0443376302719116, -0.33984947204589844, -1.6634916067123413, 0.3492553234100342, 0.42279529571533203, 2.2306737899780273, -0.5033565163612366, -0.7289785146713257, 0.5852304100990295, -1.5231233835220337, -0.9493889212608337, 0.47508320212364197, 0.4013362526893616, 1.8221406936645508, -1.0684322118759155, -1.0598770380020142], [0.2975975275039673, -1.181586742401123, -2.131183385848999, -0.6026066541671753, 0.8607448935508728, 0.9299806952476501, -0.7226256132125854, -0.9741036295890808, 0.1067657619714737, -1.8645374774932861, -0.9446126818656921, 1.1431149244308472, -0.2206648290157318, -1.747001051902771, -1.0233287811279297, -0.5070479512214661, -0.3817223310470581, -0.08060560375452042, -0.902556300163269, 0.14401020109653473, 0.4588204622268677, -1.1145586967468262, 0.1932554841041565, 1.4042904376983643, 0.28483474254608154, -0.6666746735572815, 0.256344199180603, -1.0694612264633179, -0.41615718603134155, 0.01076308824121952, -0.4219094514846802], [-1.081505298614502, 1.1075447797775269, -1.5874502658843994, 1.4638559818267822, 1.1752634048461914, 0.17994001507759094, 0.21534018218517303, -0.13418473303318024, 0.8428825736045837, 0.44397449493408203, 0.4810653626918793, 0.26529422402381897, -1.5711283683776855, -0.4666357934474945, 0.4322364032268524, 0.33944761753082275, 1.0232292413711548, -0.223502516746521, -0.180852472782135, -1.317579984664917, -0.4328678548336029, 1.0945720672607422, -0.7846588492393494, -0.132017582654953, -2.191601037979126, 0.25805339217185974, 0.2400965690612793, -0.1861032247543335, -1.6458015441894531, -1.057261347770691, 0.805824875831604], [-0.5786218047142029, -0.38615643978118896, 0.8670237064361572, -0.5172384977340698, 0.9025017619132996, 0.07657209038734436, -1.1158514022827148, 0.1262367069721222, -1.1927168369293213, -0.15967202186584473, -0.7704101204872131, 1.009202480316162, -0.13661538064479828, 0.44154515862464905, 0.23095828294754028, -1.1420791149139404, 0.5493685007095337, -2.312056303024292, -0.7030104398727417, 1.1318517923355103, 0.2872443199157715, -1.8689714670181274, 1.2503944635391235, -0.8083896040916443, -1.0701618194580078, 1.653591513633728, -0.29977676272392273, -0.4107392132282257, -1.1939032077789307, -0.543559730052948, 0.41950297355651855], [1.9177500009536743, -0.3169621229171753, -0.10783857852220535, -0.9298937320709229, -1.6713835000991821, -0.06455688178539276, 0.14402908086776733, -0.6204595565795898, 0.631580114364624, 0.31581535935401917, 0.132600799202919, -0.1888633370399475, 1.0991981029510498, 0.9422498941421509, 2.577348470687866, 1.932466745376587, -0.12694001197814941, -0.3124018609523773, -0.23010489344596863, -0.05776219442486763, -1.3830504417419434, -1.8291946649551392, -1.1685535907745361, 0.7142876982688904, -2.675248146057129, -0.6650707721710205, -1.1088554859161377, -0.8063421845436096, 0.5647501349449158, -0.8323883414268494, 0.16237857937812805], [-0.4275156557559967, 2.4226670265197754, -0.08581352233886719, 0.5647186040878296, 0.018392767757177353, -0.3385154604911804, 0.5949903726577759, -1.0220005512237549, -0.09552726149559021, -0.06551650911569595, 0.5297324657440186, -0.9726383686065674, -0.9490318894386292, -0.47418496012687683, -1.4360028505325317, 0.051768958568573, 2.3080341815948486, -0.9437879920005798, 0.9449293613433838, 0.2947165071964264, 1.7026537656784058, -0.6035447120666504, -1.779922366142273, -0.6146323680877686, -1.6698201894760132, 0.911136269569397, -0.18406222760677338, -0.46383458375930786, 0.48738667368888855, -1.1114444732666016, 0.4752330482006073], [-0.6573379635810852, 0.1256977766752243, -0.6734502911567688, 0.4054957628250122, 2.621453046798706, 1.2082555294036865, 0.8734456300735474, 1.0366623401641846, 1.35677969455719, 1.2062079906463623, 0.8746020793914795, -1.2704955339431763, 1.5686428546905518, 0.04612761363387108, -0.12808506190776825, -1.3921353816986084, 0.523612380027771, 0.22069592773914337, 0.1892150640487671, 0.3254474401473999, 0.550961434841156, -1.534791350364685, 1.8871583938598633, -0.7574371099472046, 0.08064468204975128, 0.5664638876914978, 0.610823392868042, -0.6060823202133179, -0.18116679787635803, 0.03496994078159332, -2.0373833179473877], [0.8053544163703918, 0.6291490793228149, 2.1288907527923584, 1.7862378358840942, 0.18569785356521606, 2.3733389377593994, 0.05941574275493622, -0.7756116390228271, 0.1571151316165924, 0.36425331234931946, 0.6486042141914368, -0.8676347732543945, -1.1362626552581787, 1.358392357826233, 0.035601671785116196, 1.2920317649841309, 0.033771585673093796, 1.3449469804763794, -0.20131030678749084, -0.0307808555662632, -0.1866472065448761, -1.4923100471496582, 0.7265682816505432, 0.1980307698249817, -0.1554373949766159, -0.24318447709083557, -1.7925734519958496, 0.8214755058288574, -0.2512892186641693, -2.61006760597229, -0.25885820388793945], [-1.9063661098480225, 1.1486340761184692, 0.711338222026825, -2.069706678390503, -0.9075145721435547, 0.012475772760808468, -0.3459470570087433, -1.2003670930862427, -0.5934295654296875, 0.5843662619590759, -1.106019139289856, 0.9351005554199219, 0.717764675617218, 2.0202090740203857, 0.5746579170227051, 1.729636311531067, 1.1420954465866089, 0.8200594782829285, -0.3459573984146118, 0.013601982966065407, -0.5169364809989929, -1.0548053979873657, -0.7038385272026062, 0.18413609266281128, -0.8577828407287598, 0.8157011866569519, -0.5951441526412964, 0.03540642186999321, -0.028180068358778954, -0.06603702902793884, -0.8211561441421509], [-1.0308607816696167, 0.9352630972862244, -0.013513528741896152, 2.364879846572876, -0.01618942618370056, -1.8201481103897095, -0.4495912790298462, -1.6597511768341064, 1.0011589527130127, -0.5890416502952576, 0.7811460494995117, 1.738860845565796, 0.2223903089761734, 0.2145162969827652, 0.2856127917766571, -0.8031604886054993, -0.36986473202705383, 0.80005943775177, -1.8296215534210205, -0.04846058040857315, 1.7682757377624512, 0.24398569762706757, 0.8978433609008789, -1.549232840538025, 0.22126919031143188, -0.8651995062828064, 2.0896565914154053, -1.2357286214828491, 1.0180760622024536, -0.5893309116363525, -1.0195436477661133], [-0.8293412327766418, -0.523572564125061, -0.6660016775131226, 0.5028786659240723, -0.3934696614742279, 0.320615291595459, -1.5459953546524048, -0.28175631165504456, -0.010955951176583767, -0.4522067904472351, 1.4877766370773315, 0.05181273818016052, 1.341505527496338, 2.012573480606079, 0.041926298290491104, -1.0329327583312988, -0.4175379276275635, 0.46691858768463135, -1.5255502462387085, 0.6791440844535828, -1.9195094108581543, 1.0452370643615723, 0.038619063794612885, -0.676798939704895, -0.29244503378868103, 0.112189881503582, 0.29229629039764404, 1.7239694595336914, -2.0120184421539307, 0.1495516300201416, -0.6580609679222107], [1.6216877698898315, -2.792405128479004, 0.11248350888490677, 0.3371530771255493, -0.024509208276867867, -0.06578691303730011, -0.3664630055427551, 1.2656115293502808, 1.3412508964538574, 1.288353681564331, 0.3393727242946625, -1.5057889223098755, 0.003693582257255912, 0.12478924542665482, -0.01622169278562069, 1.4401872158050537, 0.533873975276947, -0.9328035116195679, -0.6116886138916016, -1.453771948814392, 1.7370879650115967, 1.4959826469421387, 1.4371259212493896, -0.21007314324378967, 1.6136314868927002, -0.45301783084869385, 0.285170316696167, 0.01018517930060625, 0.18895994126796722, 0.7704911231994629, 0.770155668258667], [-1.5271821022033691, -0.034512460231781006, -0.4650816023349762, 0.5367650389671326, 1.9927643537521362, 0.0761418417096138, -0.2408500760793686, 0.36749398708343506, -0.5773681402206421, -0.5989954471588135, -0.48365065455436707, 0.18906374275684357, -2.031277894973755, 0.3944627046585083, -0.11658800393342972, -0.31195545196533203, -0.14426790177822113, -0.3442799746990204, 0.4883994460105896, 0.8711374402046204, -1.8540488481521606, 0.2903780937194824, -0.5914223790168762, -0.5852323770523071, 1.8613085746765137, 0.8660931587219238, 0.8840749859809875, 0.40712180733680725, -1.0072407722473145, -1.9476579427719116, 0.051731813699007034], [2.0780999660491943, -1.1102588176727295, 0.29968374967575073, 0.07342828065156937, -0.11833076179027557, -1.6732776165008545, -0.02812301740050316, -0.1311914324760437, -0.4372314214706421, -0.35694220662117004, 0.06123284995555878, 1.054787278175354, -1.0773686170578003, -0.9509587287902832, -1.2195677757263184, 0.6969624161720276, -2.080958604812622, -0.6867640614509583, 1.3693801164627075, 1.0081007480621338, -0.34011805057525635, -0.6594576239585876, 0.4856531620025635, -0.8394131064414978, -0.057695962488651276, -1.251795768737793, -1.663352608680725, -0.2275361865758896, -1.187495231628418, 0.5569999814033508, 0.1843366175889969], [-3.0921356678009033, -1.0134357213974, 1.8508083820343018, 0.7913557291030884, 2.8667619228363037, 0.08017779886722565, -1.2405081987380981, 0.7052721381187439, -0.4899485111236572, -1.3805657625198364, 1.0908246040344238, 0.38605406880378723, 0.2992815375328064, 1.80801260471344, -2.2023444175720215, 0.6539175510406494, 0.6959123015403748, -1.682610034942627, 1.2445719242095947, -1.2920573949813843, -0.17187392711639404, -0.4074011743068695, 1.2073017358779907, -0.7388037443161011, -0.8735748529434204, 0.6701216697692871, 0.3305201530456543, -0.3412814736366272, -0.7473475933074951, -0.14120352268218994, -0.5340683460235596], [0.627963125705719, 0.08823644369840622, 0.021656813099980354, 0.2016616314649582, -0.4915536940097809, -1.153472661972046, -0.7987843751907349, -0.29216083884239197, -1.7722028493881226, -0.04668111726641655, -0.506627082824707, 0.3574577569961548, -0.497010737657547, -0.9403583407402039, 1.0974853038787842, -0.42028263211250305, -1.375827670097351, -0.7709831595420837, 0.3287920355796814, -1.352785587310791, 0.9783014059066772, -1.500785231590271, -1.1990076303482056, 0.3689987063407898, -0.5717056393623352, 0.6098632216453552, 0.10312783718109131, -0.9542528986930847, -1.0936717987060547, 2.284083366394043, -0.828725278377533], [-0.5583553910255432, -1.851714015007019, -0.573697566986084, 1.444739818572998, -0.4658609926700592, 0.044338520616292953, -0.8944250345230103, -0.8483310341835022, 0.9151071906089783, 0.9585349559783936, 0.6949580907821655, 0.596796989440918, 0.26206329464912415, 1.2043590545654297, -1.0487524271011353, 1.6481728553771973, -1.1704126596450806, -1.1543477773666382, 0.8897722363471985, 0.9445759057998657, 1.6488919258117676, 0.09211380034685135, 1.2336442470550537, -0.3974166512489319, -0.44879940152168274, -0.8926771283149719, -1.520471453666687, 1.6326982975006104, -0.07439852505922318, 1.8030482530593872, -0.4621196985244751], [0.18626050651073456, 0.7817714214324951, -1.1229842901229858, 0.3302643895149231, -0.5040808916091919, -0.5131139159202576, 0.21636591851711273, -0.6280289888381958, -1.4368804693222046, 0.3037209212779999, 1.4902020692825317, -1.2857387065887451, 0.23192989826202393, 1.230545997619629, 0.051775697618722916, -0.24838677048683167, -0.3890337646007538, 0.5579243898391724, -2.3219947814941406, -0.15936417877674103, 1.0843549966812134, -0.4027692973613739, -1.116687536239624, 1.6170848608016968, 1.2344564199447632, 0.8454408049583435, 0.3017062246799469, 1.4126099348068237, -0.8046709299087524, -0.7838191986083984, 0.7654950022697449], [1.0885331630706787, 0.2048632651567459, -0.9909752607345581, -0.1180635392665863, -2.060927152633667, 0.5229699015617371, 1.030038595199585, -0.501446008682251, 0.18701577186584473, 1.2256749868392944, 0.8744609355926514, -0.3922298550605774, 0.20336385071277618, 0.3786879777908325, 0.08958959579467773, -0.7634917497634888, 1.0525131225585938, 0.274582177400589, -0.11632551997900009, 0.9587435722351074, -1.6096173524856567, 0.0027499543502926826, -0.8768853545188904, 0.4596981704235077, -0.17403662204742432, -0.7192109227180481, -0.41370129585266113, -2.2362899780273438, -0.45870184898376465, -0.31318363547325134, -0.9580084085464478], [-0.497598797082901, -2.5610694885253906, 0.38056328892707825, 0.2776150107383728, 2.2238643169403076, -0.9068543314933777, 1.4092719554901123, -0.6846181750297546, 1.5876384973526, -0.1894347071647644, -0.8789272904396057, 0.6653140187263489, -3.2759037017822266, -1.1787816286087036, -0.5875270366668701, -0.5605406761169434, 1.2558754682540894, 0.4647124707698822, 0.42709624767303467, -1.9702397584915161, 0.8380990028381348, 0.7724612951278687, -0.24736745655536652, -0.18968795239925385, -1.0222935676574707, 0.7556647062301636, 0.3012895882129669, -0.6170680522918701, 1.5616397857666016, -1.0441811084747314, 0.08137936890125275], [-2.1785192489624023, 0.22714489698410034, -1.747177004814148, -0.24649517238140106, -1.6310417652130127, -1.3935585021972656, 0.35230231285095215, -0.09009313583374023, 0.7155565619468689, 0.14642731845378876, 1.0617965459823608, -1.1381555795669556, 0.6969247460365295, 0.2241160273551941, -1.0821539163589478, -1.6353687047958374, -1.3164033889770508, 0.3410271406173706, -0.8549069166183472, -0.774869441986084, 1.470336675643921, 0.8695546388626099, -1.3834418058395386, -1.5634130239486694, 0.6330335140228271, 1.8228832483291626, 1.133437156677246, -1.1078224182128906, -1.4934149980545044, 1.2237154245376587, 0.43979355692863464], [-0.3632234036922455, -1.108177661895752, -0.9969732761383057, -0.46110498905181885, -0.7588750720024109, -0.670078694820404, -1.0017364025115967, -0.4094180166721344, 0.3356165289878845, -0.17076483368873596, 1.4438811540603638, -1.4379537105560303, -0.8698294758796692, -0.6779597401618958, 1.364478349685669, 1.592569351196289, -0.4812394976615906, 0.8861101865768433, 0.4477650821208954, 1.2428505420684814, 0.09486168622970581, 0.34237873554229736, 1.6716015338897705, 0.6712148785591125, 0.16893362998962402, -1.166555643081665, -0.5703045725822449, 0.9760453701019287, 1.0851472616195679, -0.08303854614496231, 1.2416085004806519], [0.004367851186543703, 0.4460260272026062, -0.5648086667060852, 2.0298707485198975, 0.40782734751701355, 0.1682930290699005, -0.07730137556791306, 0.20123343169689178, 0.7264956831932068, 0.19746403396129608, 1.4930484294891357, 1.921041488647461, 0.6625503301620483, -1.2688038349151611, 0.03652124106884003, 2.0576353073120117, 0.5501890182495117, -0.6545595526695251, 1.4213277101516724, 1.2351298332214355, -0.03851025551557541, 0.6644005179405212, 0.32683172821998596, -1.3655771017074585, 2.552260637283325, -0.4809618294239044, -0.30608081817626953, 0.8892715573310852, 2.3357791900634766, -1.9249118566513062, -0.00332133611664176], [0.4402633607387543, 1.9316577911376953, -0.3182193338871002, -0.8371680378913879, -0.452157199382782, 0.17271865904331207, 1.0412321090698242, 0.5126670002937317, -0.7325825095176697, 1.6188091039657593, -0.39683595299720764, -0.37960338592529297, -0.33556288480758667, 2.2701663970947266, 0.21889442205429077, 0.4249227046966553, -0.5481521487236023, 0.7986307144165039, -0.8616434931755066, 0.3002222776412964, 0.45309978723526, -0.7027101516723633, 0.32526424527168274, 0.1526394635438919, -0.19121481478214264, 0.923373281955719, 0.7629517912864685, 0.5506743788719177, 1.2962563037872314, 0.1742505431175232, 0.24917636811733246], [-1.0390734672546387, -0.460113525390625, -0.16562215983867645, -0.47016727924346924, 1.489797592163086, 0.01852835714817047, -1.6981539726257324, -0.3199811279773712, 0.045768383890390396, -1.9119313955307007, 0.6232866644859314, 1.704653263092041, 1.5249370336532593, -1.1198458671569824, 1.9778846502304077, -0.6250505447387695, -1.5133519172668457, 0.5182169079780579, -1.6401371955871582, -1.5372430086135864, -1.1530251502990723, 0.862864077091217, 0.8059313297271729, -0.20151536166667938, -0.9079384803771973, 1.2392712831497192, 0.739727795124054, -0.6165040731430054, -1.524040699005127, 2.112051010131836, -0.5313743352890015], [0.9199309945106506, 2.77270770072937, -1.2161730527877808, 1.2101370096206665, -0.6100457906723022, 0.5987197160720825, -0.024649545550346375, -0.1020653024315834, -3.1881446838378906, -2.448406219482422, -0.6951296925544739, -0.1762157380580902, 0.6245748400688171, -0.46265316009521484, -0.40802785754203796, -0.19676275551319122, -0.7154894471168518, -1.087436556816101, -0.9596808552742004, -0.455511212348938, -0.5928913354873657, -0.7248556613922119, 0.031064096838235855, 0.7807395458221436, 1.7581331729888916, 0.7599861025810242, 0.4866653084754944, -1.7553269863128662, 0.6983761191368103, 0.9598606824874878, 1.4599127769470215], [0.17593912780284882, -0.9087165594100952, 1.960181474685669, -1.0376667976379395, 1.9519014358520508, -0.018273405730724335, -0.9659863114356995, -0.07067859172821045, -0.8730925917625427, 0.6378980875015259, 1.742813229560852, 0.7417003512382507, 0.6962320804595947, -2.4158308506011963, 1.7804211378097534, 0.2102494090795517, -0.1019933819770813, -1.1110721826553345, -0.6201949715614319, -1.0392552614212036, 1.509294033050537, -0.20597364008426666, 0.5813263058662415, -0.4769374430179596, -0.506543755531311, 0.6241638660430908, 0.30809396505355835, -0.07624661922454834, 0.38720840215682983, 0.7485634684562683, 1.2404961585998535], [-0.26461121439933777, 0.9270989894866943, -0.4288703501224518, -0.4983886480331421, 2.1294119358062744, 0.36645835638046265, -1.2763389348983765, 0.8324966430664062, 0.2713034451007843, -0.992795467376709, -0.3866860568523407, 0.22523394227027893, -1.807186245918274, 0.2050931602716446, 0.002115225652232766, 0.9608064293861389, 1.0856571197509766, 0.399176687002182, 0.9835801720619202, 0.43948692083358765, 0.2873584032058716, 0.3094642460346222, -1.4858214855194092, 0.7891826033592224, -0.21934297680854797, 1.0960969924926758, 0.09995561093091965, -1.211320161819458, -0.3664090633392334, -0.26118186116218567, -0.7984284162521362], [2.894765853881836, 0.6133800745010376, 0.5452911257743835, 0.755517840385437, 1.1525814533233643, 0.5714293718338013, 2.8632709980010986, -0.30070286989212036, -0.515053391456604, -0.2426343709230423, 0.25223785638809204, 0.3118222951889038, 0.07829922437667847, -0.8250125050544739, -0.13436277210712433, 0.32210519909858704, 0.014099306426942348, 0.5704144835472107, -0.014237562194466591, 0.49203065037727356, 1.1120861768722534, 1.9352062940597534, -0.9687658548355103, 0.10756649821996689, -1.7126743793487549, -0.12840856611728668, -0.4526478350162506, 0.42119044065475464, -0.061588749289512634, 1.3362741470336914, 0.6736571192741394], [0.3811626434326172, 1.2221592664718628, -0.7599076628684998, 0.04093950241804123, 0.25967058539390564, 2.3243203163146973, -0.7025310397148132, 0.6549809575080872, -1.401168942451477, -0.256612628698349, -0.13210415840148926, -0.784892201423645, 1.5319602489471436, 2.1225249767303467, 1.1799182891845703, -1.8496705293655396, 2.472831964492798, -0.029309572651982307, 0.9577406048774719, 0.16645516455173492, 0.04565737023949623, -0.3477080762386322, 0.3337518572807312, -1.62258780002594, 0.1289805918931961, 0.3848545551300049, 0.5896903276443481, -0.7715246677398682, 0.2542855143547058, 0.42748698592185974, 0.9163482785224915], [0.8216834664344788, -0.045069269835948944, -0.02945464849472046, 1.3893074989318848, 1.812006950378418, 1.5142086744308472, -1.350914478302002, 0.25219789147377014, 0.5116147398948669, 0.4294796884059906, 0.8573004603385925, -0.0878540426492691, 1.065557599067688, 0.3850100636482239, -0.8003702759742737, 1.1681146621704102, 2.1522037982940674, 0.841168224811554, -0.40308496356010437, -1.8052668571472168, 1.4131314754486084, 0.7085573077201843, 0.49681541323661804, 0.04733016714453697, 0.5550697445869446, 0.9047254323959351, -0.8101711273193359, -0.6454972624778748, -0.22124382853507996, 0.4010276198387146, 0.7166630625724792], [0.6233394742012024, 0.6846868991851807, 0.04322420805692673, -0.6291787028312683, -0.1853201538324356, -1.3060396909713745, -0.8299652338027954, -2.06961727142334, 0.7702249884605408, 0.9661646485328674, 1.0215651988983154, -0.366111695766449, 1.1043024063110352, 0.6171583533287048, -0.35247257351875305, -0.17421624064445496, -0.3607092797756195, -0.03921618312597275, 0.229551762342453, 0.3770058751106262, 0.7106847167015076, -0.11701023578643799, 0.7573708891868591, 0.1840975433588028, -0.5313616991043091, 1.8014789819717407, -0.7273930907249451, -1.108605980873108, 1.517749309539795, -0.9581322073936462, 0.4648233950138092], [-1.212065577507019, 1.5461509227752686, -0.333726167678833, 1.0142567157745361, 0.17957323789596558, -2.65181040763855, 0.8919550180435181, -0.9022703170776367, 0.6024171113967896, -0.45594316720962524, 0.16184158623218536, 0.49888455867767334, 2.8524839878082275, 0.07161015272140503, -0.6843708157539368, -0.8915217518806458, -1.8706547021865845, -0.501434862613678, -0.17760802805423737, -0.02143721655011177, 1.654242753982544, -0.7517386674880981, -0.9822805523872375, 0.30996018648147583, 0.43456006050109863, -2.0658071041107178, -0.08386315405368805, 0.14111219346523285, 0.39730390906333923, -1.0497087240219116, 0.15017005801200867], [0.43639740347862244, 0.7762191295623779, 0.002253386890515685, 0.12211831659078598, 0.8544104099273682, -0.5703039765357971, -1.213854193687439, -1.108174443244934, 2.023015022277832, -1.5523852109909058, -1.2298966646194458, -0.2440527081489563, 0.7579118013381958, 1.086207628250122, 0.5146442651748657, -0.6214272975921631, 0.8652706742286682, -0.7347277998924255, -0.21146278083324432, 0.13351206481456757, 1.2554199695587158, 0.8943196535110474, 1.0669015645980835, -2.4134066104888916, 1.3298207521438599, 0.07098232209682465, -0.3105771541595459, -0.6097124218940735, 0.19383279979228973, -3.0423786640167236, 1.5037579536437988], [1.6938538551330566, 0.8986001014709473, 0.6342250108718872, 1.2521131038665771, -0.9190546870231628, 1.1367467641830444, 1.1682629585266113, 1.0351430177688599, 0.8644383549690247, 0.9182700514793396, -1.2942713499069214, -0.26271557807922363, 1.266007423400879, 1.543901801109314, 1.7293308973312378, 0.4979183077812195, -1.0490574836730957, -0.8666189908981323, 2.7214126586914062, -1.762593388557434, -1.3984555006027222, 0.39151591062545776, 0.2815034091472626, 2.8613040447235107, 1.2054190635681152, -0.9065613150596619, -0.5301028490066528, -0.7490296363830566, -1.3837617635726929, -1.2020949125289917, -1.832297444343567], [-0.37810018658638, 0.9720914363861084, -0.9176777601242065, 0.481893926858902, -0.5274640917778015, -0.3894028961658478, -1.0408098697662354, -0.12663137912750244, 1.285605549812317, 0.629474401473999, -0.06050698831677437, 1.268182396888733, -0.03854791447520256, 0.05957181379199028, -0.4423351287841797, -1.080341100692749, -0.7152531743049622, 0.5576445460319519, -1.6138668060302734, 0.06912606954574585, 0.6278581619262695, 1.1757005453109741, -0.1334577202796936, 1.1399483680725098, 1.0620193481445312, -0.4677916169166565, -0.11774156987667084, 1.3680529594421387, 2.268242120742798, -0.8735920190811157, -2.057271957397461], [-1.5107362270355225, -0.9182109236717224, 0.468440443277359, -0.49752864241600037, -2.2983999252319336, 0.27737659215927124, 0.26365926861763, 0.07593555003404617, -0.268051415681839, 0.9760565757751465, -1.200302004814148, -0.37361952662467957, 1.738953709602356, -1.1523436307907104, -0.14882288873195648, -0.1761784553527832, 0.14792503416538239, 1.3029096126556396, 0.28824833035469055, -0.4126616418361664, 0.2253018021583557, -0.9480178356170654, -0.8388245105743408, -0.6018637418746948, -0.22032153606414795, -1.193625807762146, -0.40553614497184753, -0.9535717964172363, 0.6802579164505005, -0.9459497332572937, 0.167949378490448]], "dtype": "float32"}], "e": {"__nd__": [[1.8314611911773682, -0.14517533779144287, -0.990673303604126, -1.0598353147506714, 0.5810195207595825, 1.0754780769348145, 0.006932623218744993, -0.201649010181427, 0.3850143849849701, 1.2330012321472168, -0.5460718274116516, 0.6842378377914429, 2.0344114303588867, 0.991860032081604, 1.385502576828003, 1.5992300510406494, -0.4934542775154114, -0.3718750774860382, 0.5397542119026184, 2.3407552242279053, 0.41781482100486755, -0.4651757776737213, 1.274505853652954, 1.0715177059173584, 0.08150725811719894, -1.508273959159851, -0.17337459325790405, -0.3253912329673767, 0.9643575549125671, -0.6144128441810608, 0.7749868035316467, 1.1275430917739868, -0.14428037405014038, 0.2071048468351364, -1.5949245691299438, -0.839078426361084, 0.375457227230072, 0.7110879421234131, 0.2344033569097519, 0.4709406793117523, 1.097241759300232, -0.5411231517791748, -1.0100924968719482, -0.3854620158672333, -0.24460576474666595, 0.7376121282577515, 0.8255569338798523, -1.6022512912750244, 0.4808003604412079, -1.6798175573349, 0.4973251223564148, -0.48466581106185913, -0.512828528881073, 0.20669396221637726, 0.8958985805511475, 0.36263614892959595, -0.3782947361469269, 0.11194189637899399, -0.3231038749217987, 0.3738093376159668, -0.7715355753898621, -0.23762765526771545, 1.5749108791351318, -1.2881503105163574, 1.499774694442749, 0.27823618054389954, 0.31900858879089355, -1.0557652711868286, 0.18572326004505157, -1.3721333742141724, -0.9418673515319824, -0.2791809141635895, -0.5271303653717041, -0.6348387598991394, -1.466352939605713, -0.6551176309585571, -0.6065263152122498, 0.6935450434684753, 0.31754443049430847, -0.4152401089668274, 0.9393934011459351, 0.4038235545158386, -0.0035796973388642073, 0.2627449631690979, -0.5982276201248169, 0.5925680994987488, 0.7697360515594482, 0.10954615473747253, 1.1323449611663818, 0.4438502788543701, -2.316134452819824, 0.031063780188560486, -0.5928117036819458, 0.2975975275039673, -1.081505298614502, -0.5786218047142029, 1.9177500009536743, -0.4275156557559967, -0.6573379635810852, 0.8053544163703918, -1.9063661098480225, -1.0308607816696167, -0.8293412327766418, 1.6216877698898315, -1.5271821022033691, 2.0780999660491943, -3.0921356678009033, 0.627963125705719, -0.5583553910255432, 0.18626050651073456, 1.0885331630706787, -0.497598797082901, -2.1785192489624023, -0.3632234036922455, 0.004367851186543703, 0.4402633607387543, -1.0390734672546387, 0.9199309945106506, 0.17593912780284882, -0.26461121439933777, 2.894765853881836, 0.3811626434326172, 0.8216834664344788, 0.6233394742012024, -1.212065577507019, 0.43639740347862244, 1.6938538551330566, -0.37810018658638, -1.5107362270355225], [-0.21212846040725708, 1.0117796659469604, 1.246144413948059, 0.4696548879146576, -0.19313795864582062, -0.1387188583612442, 1.0116032361984253, 1.042862892150879, -2.182095527648926, -0.23561686277389526, 1.3200799226760864, 0.7526978254318237, 1.1598665714263916, 0.5347906947135925, -0.5689979791641235, 1.2757439613342285, -0.6199666261672974, -0.2654281258583069, -2.1401402950286865, -1.5281121730804443, -0.08016592264175415, 1.706803321838379, 1.0718426704406738, -0.12060504406690598, 0.3324219584465027, -0.40597179532051086, -0.2848508059978485, 0.09942755103111267, -1.8399949073791504, -1.0179402828216553, -1.55853271484375, -0.9911660552024841, 0.9348194003105164, 0.37369269132614136, 0.9421014189720154, 0.03868807107210159, 0.47584712505340576, 0.1195206567645073, 1.7522231340408325, -1.0527565479278564, 0.4727626144886017, 0.6497879028320312, -0.45644253492355347, -0.8553760647773743, 0.979509711265564, 0.7269995808601379, -0.5247205495834351, 0.03820056468248367, -0.8171783685684204, -0.13903498649597168, 0.8283689022064209, 0.9610456824302673, -1.6209216117858887, 1.753823161125183, -0.7475829720497131, 1.634285569190979, -0.5909169912338257, -0.00649813748896122, -1.7665256261825562, 0.30389368534088135, -0.3684723973274231, -1.2253801822662354, -0.8808534145355225, -0.7301387190818787, 1.775849461555481, 1.5529102087020874, 0.2313554286956787, -0.08353358507156372, -0.9710010886192322, -0.04431099444627762, -0.5103363990783691, -0.9524105787277222, -0.263479620218277, -0.21566730737686157, 0.010600385256111622, 1.8003506660461426, 0.4224570095539093, -0.007912646979093552, -1.0868337154388428, -0.44639503955841064, -0.06450194120407104, 0.38131090998649597, 0.13033874332904816, -1.2042906284332275, -0.6098280549049377, -0.36356088519096375, -2.268510103225708, -0.5932667851448059, 0.08160267025232315, 1.5384266376495361, -0.21030667424201965, -0.5863063931465149, -0.7043001651763916, -1.181586742401123, 1.1075447797775269, -0.38615643978118896, -0.3169621229171753, 2.4226670265197754, 0.1256977766752243, 0.6291490793228149, 1.1486340761184692, 0.9352630972862244, -0.523572564125061, -2.792405128479004, -0.034512460231781006, -1.1102588176727295, -1.0134357213974, 0.08823644369840622, -1.851714015007019, 0.7817714214324951, 0.2048632651567459, -2.5610694885253906, 0.22714489698410034, -1.108177661895752, 0.4460260272026062, 1.9316577911376953, -0.460113525390625, 2.77270770072937, -0.9087165594100952, 0.9270989894866943, 0.6133800745010376, 1.2221592664718628, -0.045069269835948944, 0.6846868991851807, 1.5461509227752686, 0.7762191295623779, 0.8986001014709473, 0.9720914363861084, -0.9182109236717224], [-2.394015073776245, -0.4555515646934509, -1.3939985036849976, 0.6294484734535217, 1.2244408130645752, 0.15714557468891144, 0.7404350638389587, -0.5600792169570923, 0.043105531483888626, 0.32682961225509644, 0.6684293150901794, 0.9526932835578918, 2.5011487007141113, -0.5409600138664246, 0.2854747772216797, 0.21113061904907227, -0.9893078207969666, -0.09851600229740143, 0.6504780054092407, -0.09656616300344467, -0.935230016708374, -0.8817654848098755, -1.4877359867095947, 0.39353683590888977, -1.4584101438522339, 0.2410571426153183, -0.1719411164522171, -1.3024741411209106, 1.837459683418274, 0.8296866416931152, -0.4154419004917145, 0.5884270668029785, 2.7604928016662598, -0.6401896476745605, -0.7616243958473206, 0.5169333219528198, 0.8866193294525146, -0.8238638043403625, 0.005713304039090872, -0.0965019166469574, 0.22292709350585938, 1.1681050062179565, -0.02263643406331539, 0.4414701461791992, 0.5811188817024231, -0.5098750591278076, -0.16517877578735352, 0.36820071935653687, -1.401787519454956, 0.12929752469062805, -0.8741931319236755, 0.061736635863780975, 1.6270769834518433, -1.1639994382858276, -0.5021466612815857, 0.25425416231155396, -0.5143182873725891, -1.1482971906661987, 0.8580392599105835, -1.6337145566940308, 1.311316728591919, -0.40424445271492004, 1.3619471788406372, 0.2970917522907257, 1.2691822052001953, 1.736811637878418, -1.3015786409378052, 1.7234890460968018, -0.15757150948047638, 0.7740611433982849, -0.22561690211296082, -0.8650596141815186, 0.016773030161857605, -0.5881375670433044, 1.4958144426345825, 0.8380928039550781, -0.09379478543996811, 0.48240184783935547, 0.27661633491516113, 0.34006771445274353, 0.724471926689148, 1.9164105653762817, 1.776381254196167, 0.42776286602020264, -1.0165977478027344, 0.8135951161384583, 0.19192321598529816, -0.7667669653892517, -2.3133351802825928, 0.09901917725801468, 0.13430264592170715, 1.2949328422546387, 0.7719364166259766, -2.131183385848999, -1.5874502658843994, 0.8670237064361572, -0.10783857852220535, -0.08581352233886719, -0.6734502911567688, 2.1288907527923584, 0.711338222026825, -0.013513528741896152, -0.6660016775131226, 0.11248350888490677, -0.4650816023349762, 0.29968374967575073, 1.8508083820343018, 0.021656813099980354, -0.573697566986084, -1.1229842901229858, -0.9909752607345581, 0.38056328892707825, -1.747177004814148, -0.9969732761383057, -0.5648086667060852, -0.3182193338871002, -0.16562215983867645, -1.2161730527877808, 1.960181474685669, -0.4288703501224518, 0.5452911257743835, -0.7599076628684998, -0.02945464849472046, 0.04322420805692673, -0.333726167678833, 0.002253386890515685, 0.6342250108718872, -0.9176777601242065, 0.468440443277359], [-0.4111219346523285, -0.504378080368042, 0.7515562772750854, -0.06258724629878998, 2.4011125564575195, -0.41369470953941345, -1.698319435119629, -2.7840306758880615, 0.5284362435340881, -0.3881165087223053, -0.2990837097167969, -0.6658565402030945, -0.7682463526725769, -1.5906994342803955, 1.2966066598892212, 0.21292538940906525, -0.9172034859657288, -0.7546308636665344, 0.7611957788467407, 0.3691624701023102, 1.150066614151001, 0.1421533077955246, 0.38264453411102295, -0.06882255524396896, 0.27845773100852966, 0.9164498448371887, -0.056986674666404724, 0.5082759857177734, -0.732742190361023, 1.6951051950454712, -1.167047142982483, -0.6454065442085266, -0.7984036803245544, 1.8969919681549072, 0.45742735266685486, -1.670350432395935, -0.3925503194332123, -0.3845605254173279, 1.1399048566818237, -0.09140557795763016, 1.5896059274673462, 0.5969669222831726, 0.39339402318000793, -0.6386474967002869, 0.27953267097473145, 1.0249706506729126, -1.373883843421936, -0.49221065640449524, 0.8078743815422058, 1.634770393371582, 0.21678972244262695, -0.7031830549240112, -1.3286300897598267, 0.35284221172332764, 0.12750910222530365, -1.526407241821289, -0.06261099129915237, 0.8178497552871704, 0.5304247736930847, 1.4113495349884033, 0.36316415667533875, -0.02154519222676754, -0.8278377056121826, 0.8495899438858032, -0.3019898533821106, -1.3333450555801392, 1.4724149703979492, -0.5066438913345337, -1.4154078960418701, -0.053824882954359055, -1.7408713102340698, 0.29248708486557007, 0.31537503004074097, 1.2117100954055786, -1.1131680011749268, -1.2607603073120117, 1.9483762979507446, 0.6600342392921448, -0.580205500125885, 0.2747352421283722, 0.4152720272541046, 2.0418660640716553, -0.021896036341786385, 1.4653527736663818, -0.4497460722923279, -1.0852253437042236, -1.6960968971252441, 0.5646756291389465, -0.3252706825733185, 2.8182144165039062, -2.2390902042388916, 0.36272719502449036, -0.05457274615764618, -0.6026066541671753, 1.4638559818267822, -0.5172384977340698, -0.9298937320709229, 0.5647186040878296, 0.4054957628250122, 1.7862378358840942, -2.069706678390503, 2.364879846572876, 0.5028786659240723, 0.3371530771255493, 0.5367650389671326, 0.07342828065156937, 0.7913557291030884, 0.2016616314649582, 1.444739818572998, 0.3302643895149231, -0.1180635392665863, 0.2776150107383728, -0.24649517238140106, -0.46110498905181885, 2.0298707485198975, -0.8371680378913879, -0.47016727924346924, 1.2101370096206665, -1.0376667976379395, -0.4983886480331421, 0.755517840385437, 0.04093950241804123, 1.3893074989318848, -0.6291787028312683, 1.0142567157745361, 0.12211831659078598, 1.2521131038665771, 0.481893926858902, -0.49752864241600037], [0.9101552963256836, -1.4339739084243774, 0.9735942482948303, -0.9508728384971619, -1.2702523469924927, 0.5552850365638733, -0.689485490322113, -0.1932140588760376, 0.0782979428768158, 1.2328521013259888, -0.699491024017334, 0.06648173183202744, -1.1100612878799438, -0.8863663673400879, -0.8381891250610352, 1.1811269521713257, -0.32199716567993164, -0.08641711622476578, 0.8778930902481079, -0.3930843472480774, -0.7145572900772095, 1.2613776922225952, -1.8261682987213135, 1.0891073942184448, 0.647488534450531, -0.08942605555057526, 1.717917561531067, -1.1463775634765625, 0.3860560953617096, -0.3631415367126465, -0.6111412644386292, 0.9478386044502258, -0.7818652391433716, 0.25342652201652527, 1.5198205709457397, -0.8458612561225891, 1.0818092823028564, 0.28626126050949097, 0.009918268769979477, 0.6747688055038452, 0.16931575536727905, -0.4875859022140503, 0.8018644452095032, 0.25379717350006104, 1.097720980644226, 0.965287446975708, 0.397774875164032, 0.3276534080505371, -0.3187079429626465, 2.455183506011963, -0.02883736602962017, 0.5857332348823547, 1.0354430675506592, -0.08218705654144287, -0.47867274284362793, -0.6597191691398621, -1.7867445945739746, -0.6458467841148376, 0.27292314171791077, 0.5116433501243591, 0.3344014883041382, 0.9008948802947998, -1.2051900625228882, -0.5864861607551575, -0.8200145363807678, -0.44910168647766113, 0.5477567911148071, -1.9465192556381226, -0.9389414191246033, 0.5431603193283081, -1.496909260749817, 1.209139108657837, 0.17650234699249268, -0.49229657649993896, 0.7792298197746277, 0.15172454714775085, -1.1695209741592407, 0.3006336987018585, 1.0159653425216675, -1.5715162754058838, -0.30458810925483704, 1.0462286472320557, -1.1090670824050903, -0.07582750171422958, -0.8595421314239502, -0.02828109636902809, 0.7666382789611816, -0.031397540122270584, -0.7016269564628601, -1.3372184038162231, -0.8470133543014526, -0.3385457396507263, -1.2621556520462036, 0.8607448935508728, 1.1752634048461914, 0.9025017619132996, -1.6713835000991821, 0.018392767757177353, 2.621453046798706, 0.18569785356521606, -0.9075145721435547, -0.01618942618370056, -0.3934696614742279, -0.024509208276867867, 1.9927643537521362, -0.11833076179027557, 2.8667619228363037, -0.4915536940097809, -0.4658609926700592, -0.5040808916091919, -2.060927152633667, 2.2238643169403076, -1.6310417652130127, -0.7588750720024109, 0.40782734751701355, -0.452157199382782, 1.489797592163086, -0.6100457906723022, 1.9519014358520508, 2.1294119358062744, 1.1525814533233643, 0.25967058539390564, 1.812006950378418, -0.1853201538324356, 0.17957323789596558, 0.8544104099273682, -0.9190546870231628, -0.5274640917778015, -2.2983999252319336], [-0.8629969358444214, 2.1576459407806396, -0.1892968863248825, 0.07408116012811661, -1.32418692111969, -0.6433817744255066, 1.203761100769043, -2.5527307987213135, -0.06815361231565475, -0.10824783891439438, 0.523946225643158, 0.36821192502975464, 0.4468945264816284, 0.3145420253276825, 0.17500261962413788, -1.621106743812561, -0.3881458342075348, 0.2749064564704895, -0.65944904088974, 0.18619076907634735, -1.8518192768096924, 1.1758129596710205, -0.04167108237743378, 0.3741447925567627, -0.7814022302627563, 0.6071053147315979, 0.1160164624452591, -0.1630144715309143, -0.15426769852638245, 0.4591749310493469, -0.8320674896240234, 0.7278352975845337, 2.1959071159362793, 0.11338088661432266, -0.8618493676185608, -0.19895727932453156, -0.05804560333490372, 1.1153188943862915, 0.11584564298391342, 1.06868577003479, 2.0047061443328857, 0.4788987934589386, -0.6908888220787048, 0.8181407451629639, -0.036493152379989624, -2.6137256622314453, 0.23881791532039642, -0.38920363783836365, -0.4712984263896942, 1.3633126020431519, 1.3365956544876099, -0.5855982899665833, 0.14734092354774475, -0.1997276395559311, -0.3127177655696869, -0.07230282574892044, 1.0981532335281372, -0.20798125863075256, 1.0685604810714722, -1.0374218225479126, -1.5983766317367554, 0.9159940481185913, -0.6168670654296875, -0.8259982466697693, 1.0948666334152222, -0.023348985239863396, -0.036055419594049454, -0.9320257902145386, 0.8586532473564148, 0.3523256182670593, 0.4453689455986023, -0.5326158404350281, -0.5869859457015991, -1.3204714059829712, -0.29731810092926025, 2.142789363861084, -0.25443825125694275, -0.9930435419082642, 0.6279154419898987, 0.39663010835647583, -0.6735740900039673, -0.3980411887168884, -1.2661736011505127, -0.2316378951072693, -0.07068663835525513, 2.066141366958618, 0.0496959462761879, -0.8650144338607788, 1.7096823453903198, -0.23375080525875092, -0.39377760887145996, -1.0989502668380737, -2.6704177856445312, 0.9299806952476501, 0.17994001507759094, 0.07657209038734436, -0.06455688178539276, -0.3385154604911804, 1.2082555294036865, 2.3733389377593994, 0.012475772760808468, -1.8201481103897095, 0.320615291595459, -0.06578691303730011, 0.0761418417096138, -1.6732776165008545, 0.08017779886722565, -1.153472661972046, 0.044338520616292953, -0.5131139159202576, 0.5229699015617371, -0.9068543314933777, -1.3935585021972656, -0.670078694820404, 0.1682930290699005, 0.17271865904331207, 0.01852835714817047, 0.5987197160720825, -0.018273405730724335, 0.36645835638046265, 0.5714293718338013, 2.3243203163146973, 1.5142086744308472, -1.3060396909713745, -2.65181040763855, -0.5703039765357971, 1.1367467641830444, -0.3894028961658478, 0.27737659215927124], [-0.08548121899366379, -0.2929539680480957, -1.9053304195404053, 0.8823204636573792, 0.043983399868011475, -0.08671879023313522, -0.2578622102737427, 0.33472198247909546, -0.4106152355670929, -1.3588223457336426, -0.9087852239608765, -0.06835149973630905, 0.27833592891693115, -1.184556007385254, 0.006027581170201302, 0.6169980764389038, -0.47009795904159546, -0.3840329945087433, -0.003328672144562006, 0.8205493688583374, -2.3199501037597656, 0.13567321002483368, -2.153646945953369, -0.7695006728172302, 1.8085064888000488, -0.272225558757782, -0.7337120771408081, -0.24461308121681213, 0.4534689784049988, -0.29004141688346863, -0.480337530374527, -0.77677983045578, 0.9356593489646912, 0.41908931732177734, -0.7337480783462524, -0.3093803822994232, 0.07236815243959427, -0.9247187972068787, 1.5636317729949951, -0.7935543656349182, -0.6567842364311218, -0.4299287796020508, -1.2194432020187378, -0.03251338750123978, 0.8669057488441467, -0.014858915470540524, 0.5839869379997253, -0.329581081867218, 0.5496507287025452, -1.2903530597686768, 0.38391682505607605, -1.6612281799316406, -0.5264033079147339, -0.5749725103378296, -0.9397487044334412, -0.8125987648963928, 0.49595123529434204, -1.4420146942138672, -0.8255398273468018, 0.7313273549079895, -0.8099855780601501, -0.7349756956100464, -1.7284255027770996, 1.145329236984253, -0.14749178290367126, -0.6466782093048096, 0.19059410691261292, 0.6610256433486938, -0.9536478519439697, 1.5468273162841797, 0.6625651121139526, 0.11807545274496078, 1.067976474761963, -0.2260860949754715, -0.3659686744213104, -0.5768862962722778, 1.040144920349121, -1.017773985862732, 1.283399224281311, -1.066361427307129, -1.116087555885315, -1.320983648300171, 0.27640411257743835, 0.32459092140197754, 0.5270912647247314, -1.2257006168365479, 0.02926958166062832, 0.6216845512390137, 1.1657869815826416, 0.4733431339263916, -0.5199034214019775, 0.07461177557706833, -0.1883394420146942, -0.7226256132125854, 0.21534018218517303, -1.1158514022827148, 0.14402908086776733, 0.5949903726577759, 0.8734456300735474, 0.05941574275493622, -0.3459470570087433, -0.4495912790298462, -1.5459953546524048, -0.3664630055427551, -0.2408500760793686, -0.02812301740050316, -1.2405081987380981, -0.7987843751907349, -0.8944250345230103, 0.21636591851711273, 1.030038595199585, 1.4092719554901123, 0.35230231285095215, -1.0017364025115967, -0.07730137556791306, 1.0412321090698242, -1.6981539726257324, -0.024649545550346375, -0.9659863114356995, -1.2763389348983765, 2.8632709980010986, -0.7025310397148132, -1.350914478302002, -0.8299652338027954, 0.8919550180435181, -1.213854193687439, 1.1682629585266113, -1.0408098697662354, 0.26365926861763], [-0.5782530307769775, -0.5601922273635864, -0.18174073100090027, 1.9756748676300049, 2.5173091888427734, -0.2120666801929474, -0.6563969254493713, -0.1668413132429123, -0.5546054840087891, 2.2728195190429688, -1.2663458585739136, -0.06076599285006523, 0.5219970345497131, 0.5981714725494385, 0.8720093965530396, -0.2825905978679657, -0.36108601093292236, -1.658310890197754, 1.0077170133590698, -0.6782922744750977, 2.846038341522217, 1.4719443321228027, -0.2049959897994995, 0.3206258714199066, -1.2725934982299805, -0.03998695686459541, 1.9104218482971191, -1.5851385593414307, -0.28595614433288574, -1.2635046243667603, -0.3052077293395996, 2.362229585647583, -0.7877044081687927, -1.5317137241363525, 0.11037804186344147, -0.5626093149185181, -0.9884893298149109, -0.29399344325065613, -0.4049302041530609, -0.04123418778181076, -1.235592246055603, -1.2664819955825806, -0.39568567276000977, -1.1060199737548828, -1.2854284048080444, -0.22996832430362701, 0.3442881405353546, -0.4508781135082245, -0.21708977222442627, -0.5800521373748779, 0.6554083228111267, -1.5429965257644653, -1.9635816812515259, 1.0222584009170532, -1.3850632905960083, -0.07784027606248856, -0.4646267294883728, -1.7256686687469482, 0.013615889474749565, -0.443685919046402, -1.1323156356811523, -0.8297791481018066, -1.0300081968307495, -1.9043083190917969, -1.4842333793640137, -2.6458709239959717, -1.731704592704773, -1.7496752738952637, 1.4026331901550293, 0.8732019662857056, -0.4433211088180542, -1.3260328769683838, -0.8607656359672546, -1.197218656539917, -0.4261714816093445, -0.22168096899986267, 0.405944287776947, -0.008633793331682682, 2.8670599460601807, -2.8237221240997314, 0.7620155811309814, 1.403955101966858, -0.2969399094581604, 1.543845534324646, -1.6199394464492798, 0.43271902203559875, 1.2790285348892212, -0.3678467571735382, 1.9240986108779907, 0.7891275882720947, 0.875892698764801, 0.9054735898971558, 0.30255264043807983, -0.9741036295890808, -0.13418473303318024, 0.1262367069721222, -0.6204595565795898, -1.0220005512237549, 1.0366623401641846, -0.7756116390228271, -1.2003670930862427, -1.6597511768341064, -0.28175631165504456, 1.2656115293502808, 0.36749398708343506, -0.1311914324760437, 0.7052721381187439, -0.29216083884239197, -0.8483310341835022, -0.6280289888381958, -0.501446008682251, -0.6846181750297546, -0.09009313583374023, -0.4094180166721344, 0.20123343169689178, 0.5126670002937317, -0.3199811279773712, -0.1020653024315834, -0.07067859172821045, 0.8324966430664062, -0.30070286989212036, 0.6549809575080872, 0.25219789147377014, -2.06961727142334, -0.9022703170776367, -1.108174443244934, 1.0351430177688599, -0.12663137912750244, 0.07593555003404617], [0.30402544140815735, -0.4764561653137207, 1.4591599702835083, 0.24186304211616516, -1.8272812366485596, -0.09002033621072769, -0.2915245592594147, -0.5693341493606567, -0.17821529507637024, -0.9097006916999817, -1.6101115942001343, -0.3804909884929657, -0.9131054282188416, -0.9303840398788452, -1.0846573114395142, 0.0328470803797245, -1.3968324661254883, -0.8580965995788574, 1.1040465831756592, -0.45622631907463074, 2.47430419921875, 1.7200630903244019, -1.3005794286727905, 1.2124370336532593, 1.0041025876998901, -1.4240984916687012, 0.011295527219772339, 0.31294456124305725, -0.8778231143951416, -1.3431103229522705, -1.253352403640747, 0.6374277472496033, -1.2982878684997559, 0.6405224800109863, -2.397109270095825, -0.7902058959007263, 0.428257554769516, 1.7048697471618652, -0.3821682929992676, 1.2700631618499756, 0.44807103276252747, 0.9415683150291443, 0.845304548740387, -0.7068474292755127, -0.4269864559173584, 0.7362406849861145, 0.5964995622634888, -0.45299211144447327, 0.053550176322460175, 0.6181479692459106, 0.1764838546514511, -0.6281660795211792, 0.8490915894508362, 0.2770999073982239, -0.39268022775650024, -0.3542684018611908, -0.1828591376543045, 0.8306668996810913, -0.3631298840045929, -0.42894884943962097, -0.40909427404403687, -0.21886250376701355, 0.28483468294143677, -0.2887876033782959, -1.1059318780899048, 0.10769779980182648, 0.13600072264671326, 1.6782317161560059, -0.801718533039093, -0.3203423023223877, 0.44764384627342224, -0.34483739733695984, 0.7631015777587891, 0.07512439787387848, -1.8619122505187988, -0.27530795335769653, 0.40857386589050293, -1.0099542140960693, -2.8346750736236572, 0.9209019541740417, -0.04319137707352638, 0.05671094357967377, -2.1907455921173096, 0.7959599494934082, -0.596868097782135, -0.3017316162586212, 0.2374524176120758, 1.1159766912460327, 1.2308735847473145, -0.5157313942909241, -0.07336746901273727, 0.06830354779958725, -2.459839344024658, 0.1067657619714737, 0.8428825736045837, -1.1927168369293213, 0.631580114364624, -0.09552726149559021, 1.35677969455719, 0.1571151316165924, -0.5934295654296875, 1.0011589527130127, -0.010955951176583767, 1.3412508964538574, -0.5773681402206421, -0.4372314214706421, -0.4899485111236572, -1.7722028493881226, 0.9151071906089783, -1.4368804693222046, 0.18701577186584473, 1.5876384973526, 0.7155565619468689, 0.3356165289878845, 0.7264956831932068, -0.7325825095176697, 0.045768383890390396, -3.1881446838378906, -0.8730925917625427, 0.2713034451007843, -0.515053391456604, -1.401168942451477, 0.5116147398948669, 0.7702249884605408, 0.6024171113967896, 2.023015022277832, 0.8644383549690247, 1.285605549812317, -0.268051415681839], [0.7049210071563721, -0.10892939567565918, -0.020416859537363052, 0.8613426089286804, -0.5172780156135559, 1.7304271459579468, 0.24298593401908875, 0.15144960582256317, -0.4238491356372833, 1.3241336345672607, 0.91238933801651, -0.21753908693790436, -1.536738634109497, 0.7018967270851135, 0.1683686077594757, -1.2628110647201538, 1.4736343622207642, 1.0519920587539673, 2.0367271900177, -0.7214857339859009, 2.144737720489502, 1.6434890031814575, -1.1254481077194214, -1.0211156606674194, 0.24639791250228882, 1.2999202013015747, -0.199677973985672, -0.05619794875383377, -0.006109577603638172, 0.4486519992351532, 1.8503278493881226, 0.48092028498649597, -0.7553954124450684, -0.021210668608546257, -0.07785163819789886, -0.5753894448280334, 0.7341750264167786, 0.32468464970588684, -1.152870535850525, -0.08953435719013214, -1.2022422552108765, 1.2912487983703613, -0.018860939890146255, 0.06944452226161957, 0.021518591791391373, 0.6185839176177979, -1.450986385345459, 0.6590209007263184, -0.6521735787391663, 1.2227787971496582, 1.1928634643554688, -1.2945692539215088, 0.17715269327163696, 0.8785882592201233, 0.07733870297670364, 0.02437490224838257, 0.061147790402173996, -1.4130992889404297, -0.9504479765892029, 0.7047721147537231, -1.1415095329284668, -0.25814884901046753, -1.757996916770935, 0.455424964427948, 1.5301949977874756, 0.09165707230567932, 1.8337385654449463, -0.8539604544639587, -0.22788755595684052, -0.9601350426673889, 0.4197987914085388, 0.4229625463485718, 1.4559338092803955, 0.12357188761234283, -0.09007216989994049, 1.6438074111938477, -1.4146863222122192, -0.8188064694404602, -0.12539105117321014, 0.1327550709247589, -0.9804452061653137, 0.6523199081420898, 1.2212927341461182, -1.1510162353515625, -0.6890312433242798, 0.14545831084251404, -1.320900559425354, -0.1379466950893402, 1.130040168762207, 0.24417509138584137, 0.4961390197277069, -2.0592141151428223, 1.5824111700057983, -1.8645374774932861, 0.44397449493408203, -0.15967202186584473, 0.31581535935401917, -0.06551650911569595, 1.2062079906463623, 0.36425331234931946, 0.5843662619590759, -0.5890416502952576, -0.4522067904472351, 1.288353681564331, -0.5989954471588135, -0.35694220662117004, -1.3805657625198364, -0.04668111726641655, 0.9585349559783936, 0.3037209212779999, 1.2256749868392944, -0.1894347071647644, 0.14642731845378876, -0.17076483368873596, 0.19746403396129608, 1.6188091039657593, -1.9119313955307007, -2.448406219482422, 0.6378980875015259, -0.992795467376709, -0.2426343709230423, -0.256612628698349, 0.4294796884059906, 0.9661646485328674, -0.45594316720962524, -1.5523852109909058, 0.9182700514793396, 0.629474401473999, 0.9760565757751465], [-0.8063345551490784, -0.9054820537567139, -1.0898692607879639, 0.7670099139213562, -0.9411382079124451, -0.6005355715751648, 0.3128099739551544, -0.9127938747406006, -2.0083730220794678, 0.042633213102817535, 0.4573710262775421, -0.7286021709442139, 1.0190349817276, -0.2000168263912201, 0.32276177406311035, 1.7319976091384888, -0.0741904228925705, -1.467100977897644, -0.475725919008255, 0.014197789132595062, 0.7553079724311829, 0.1649996042251587, 0.04774310812354088, 0.07376037538051605, -1.0585336685180664, 0.8563091158866882, 1.094290852546692, 2.385061264038086, 0.03348139300942421, 0.7034425735473633, 0.1309313327074051, 0.297683447599411, -1.0601835250854492, -0.5670639872550964, 0.15084350109100342, 0.042829059064388275, 0.4841792583465576, -0.7818419933319092, 0.0675937831401825, 1.2833468914031982, -0.7937237620353699, 0.5939888954162598, 0.8548625111579895, 0.07622247189283371, -0.13033731281757355, -1.360425353050232, 1.272254228591919, 0.3803202211856842, 0.3188130855560303, -1.4895954132080078, -0.8768047094345093, -0.830669105052948, -0.2051781862974167, 0.29082030057907104, -0.0513063482940197, 0.5017791986465454, -0.07329436391592026, -1.721554160118103, -0.3816438317298889, -0.22293011844158173, 0.5460498332977295, 0.6681212186813354, -0.6577789187431335, 0.7462840676307678, -1.1198011636734009, -0.4344644546508789, 1.4603376388549805, 1.3994266986846924, 0.8638091683387756, 2.357797622680664, 1.061331868171692, -1.2176827192306519, 0.4287804067134857, -0.7278842926025391, -1.6904629468917847, -0.18021364510059357, -1.2453469038009644, 0.43640458583831787, -0.253776490688324, -0.40475085377693176, 1.2467771768569946, -0.43254324793815613, 0.21036195755004883, -0.40200650691986084, 0.16227658092975616, -0.14128045737743378, 1.3521322011947632, -0.010189197957515717, -0.7534153461456299, 0.6511634588241577, 0.575797975063324, -0.35544338822364807, 0.6304231286048889, -0.9446126818656921, 0.4810653626918793, -0.7704101204872131, 0.132600799202919, 0.5297324657440186, 0.8746020793914795, 0.6486042141914368, -1.106019139289856, 0.7811460494995117, 1.4877766370773315, 0.3393727242946625, -0.48365065455436707, 0.06123284995555878, 1.0908246040344238, -0.506627082824707, 0.6949580907821655, 1.4902020692825317, 0.8744609355926514, -0.8789272904396057, 1.0617965459823608, 1.4438811540603638, 1.4930484294891357, -0.39683595299720764, 0.6232866644859314, -0.6951296925544739, 1.742813229560852, -0.3866860568523407, 0.25223785638809204, -0.13210415840148926, 0.8573004603385925, 1.0215651988983154, 0.16184158623218536, -1.2298966646194458, -1.2942713499069214, -0.06050698831677437, -1.200302004814148], [0.33550742268562317, 0.34343111515045166, 1.1625113487243652, 1.001837968826294, 0.8316216468811035, -0.11141137033700943, 1.051524043083191, -2.4038259983062744, -1.768474817276001, 0.1822144240140915, 1.1058157682418823, 1.0072438716888428, 0.8302682638168335, -0.5177063345909119, -0.38410481810569763, 0.952045202255249, -0.43372154235839844, 0.3593783974647522, -1.8536057472229004, 1.049493432044983, 0.48922958970069885, -1.906849980354309, -1.2163439989089966, 0.5109211802482605, 0.7890924215316772, -0.1813359260559082, 0.7045323252677917, -1.5681160688400269, 1.2406398057937622, 0.9123783707618713, -0.8756036758422852, -0.4190966784954071, 0.47903725504875183, -0.8593438863754272, -0.7562969923019409, -0.02773940935730934, -0.06215016916394234, -0.7779103517532349, 1.2693647146224976, -0.8221092820167542, 0.03620556369423866, 0.42518723011016846, -0.9734957218170166, -1.6537500619888306, -0.14107897877693176, -0.0999433621764183, 1.1521461009979248, 0.20435695350170135, -0.3922572135925293, -1.1120442152023315, -0.7914597392082214, 0.1047571450471878, 0.08175981789827347, 0.48208266496658325, 1.5170745849609375, -2.197801113128662, -1.0181059837341309, -1.0723246335983276, -1.524263620376587, -1.1107780933380127, 0.08516452461481094, 0.05786646902561188, -0.29920482635498047, 0.46460315585136414, -1.6462138891220093, -0.041253793984651566, 2.5786287784576416, 0.2618565857410431, -0.14773938059806824, -1.2296950817108154, 0.16334712505340576, 0.6003639101982117, -0.28625938296318054, 1.0523900985717773, -1.2163262367248535, -0.6019604206085205, -0.6436789631843567, 0.6591635346412659, 0.3483615219593048, -0.3800571858882904, -0.16092677414417267, 1.103031873703003, 0.8236870765686035, -0.6927193403244019, 1.0465278625488281, -1.5727953910827637, -0.3056032955646515, 0.44085752964019775, -0.8588849306106567, 1.5706489086151123, 0.44057875871658325, -0.3413890600204468, 1.0463393926620483, 1.1431149244308472, 0.26529422402381897, 1.009202480316162, -0.1888633370399475, -0.9726383686065674, -1.2704955339431763, -0.8676347732543945, 0.9351005554199219, 1.738860845565796, 0.05181273818016052, -1.5057889223098755, 0.18906374275684357, 1.054787278175354, 0.38605406880378723, 0.3574577569961548, 0.596796989440918, -1.2857387065887451, -0.3922298550605774, 0.6653140187263489, -1.1381555795669556, -1.4379537105560303, 1.921041488647461, -0.37960338592529297, 1.704653263092041, -0.1762157380580902, 0.7417003512382507, 0.22523394227027893, 0.3118222951889038, -0.784892201423645, -0.0878540426492691, -0.366111695766449, 0.49888455867767334, -0.2440527081489563, -0.26271557807922363, 1.268182396888733, -0.37361952662467957], [0.4934980869293213, -2.679811477661133, 2.4017951488494873, -1.2338956594467163, -0.17097976803779602, 1.4105685949325562, 0.37820640206336975, -0.3789249658584595, -0.5471705198287964, 0.4353739619255066, 0.05605659261345863, -0.9548628926277161, 2.4334323406219482, -0.005377970170229673, 0.5920903086662292, -1.4987531900405884, -0.6972594857215881, 1.594077229499817, 0.49536243081092834, 0.6707140207290649, 1.0311627388000488, 0.7457695603370667, 0.9300553798675537, -0.22015637159347534, 0.7194098234176636, 0.9492132067680359, -0.19077366590499878, 1.8931511640548706, -0.9813918471336365, -0.05411975458264351, 0.25498202443122864, -0.8967857360839844, 0.6573916673660278, -0.13155154883861542, -1.5702989101409912, -0.6294649243354797, -0.028649775311350822, -1.7214186191558838, 0.23799115419387817, -1.4305740594863892, 0.18992897868156433, -0.2624174654483795, -0.8929722905158997, -0.4274130165576935, -0.5136457085609436, -0.9970879554748535, -1.3647252321243286, -0.2577323615550995, 0.1975812017917633, -0.7288857102394104, -1.0341547727584839, -0.22933721542358398, 0.8089094758033752, -0.5405625700950623, 0.31005600094795227, 0.8541887998580933, -0.921370267868042, 0.378785640001297, -1.7544747591018677, 0.03494589403271675, -0.09639070183038712, 0.6159993410110474, 0.6305592060089111, -0.9796565771102905, -1.9737364053726196, -0.8418391346931458, -0.7414077520370483, -1.7265405654907227, 1.3940098285675049, 0.16502217948436737, 0.446850448846817, -0.5505265593528748, -2.6311051845550537, -0.7627328038215637, 0.08176849782466888, 0.8629665970802307, -0.3571300506591797, -1.6171598434448242, 0.11433950811624527, 1.0418877601623535, -0.5149791836738586, -0.5704027414321899, -0.7108004689216614, -1.2378662824630737, 0.5762686133384705, 1.9212511777877808, 1.609988808631897, 1.5182859897613525, -2.6376194953918457, 0.8293654918670654, 1.2908276319503784, 0.2581612765789032, -0.6699048280715942, -0.2206648290157318, -1.5711283683776855, -0.13661538064479828, 1.0991981029510498, -0.9490318894386292, 1.5686428546905518, -1.1362626552581787, 0.717764675617218, 0.2223903089761734, 1.341505527496338, 0.003693582257255912, -2.031277894973755, -1.0773686170578003, 0.2992815375328064, -0.497010737657547, 0.26206329464912415, 0.23192989826202393, 0.20336385071277618, -3.2759037017822266, 0.6969247460365295, -0.8698294758796692, 0.6625503301620483, -0.33556288480758667, 1.5249370336532593, 0.6245748400688171, 0.6962320804595947, -1.807186245918274, 0.07829922437667847, 1.5319602489471436, 1.065557599067688, 1.1043024063110352, 2.8524839878082275, 0.7579118013381958, 1.266007423400879, -0.03854791447520256, 1.738953709602356], [-1.6836026906967163, -1.1872543096542358, 1.6441901922225952, 1.7421764135360718, 0.10073570162057877, -0.6935920715332031, -0.6528393030166626, -1.7637548446655273, -0.6218112111091614, -0.14502887427806854, 1.3217910528182983, -0.2399837076663971, 0.116155244410038, -0.1496669352054596, 1.5063270330429077, -0.4840964078903198, 1.1095844507217407, -0.08761713653802872, 0.6998342275619507, 1.3234734535217285, 0.7393538355827332, -0.535515308380127, -1.5894544124603271, -1.9678125381469727, 0.5231953859329224, -0.2406189739704132, 0.21981391310691833, 0.6656240224838257, -1.0527762174606323, -1.333357572555542, -0.45500174164772034, 0.6297850608825684, -1.8584061861038208, -2.009763717651367, 1.061653971672058, -0.15410351753234863, 1.1281628608703613, -0.20177890360355377, -1.5189564228057861, -0.5539798736572266, -0.2672426104545593, 1.1246728897094727, 1.3862168788909912, 0.8562620878219604, 0.013372328132390976, -0.5900167226791382, -0.6701651215553284, 0.1496373862028122, 1.0207500457763672, -0.24124419689178467, 0.0414743646979332, 1.6027297973632812, 1.450427770614624, -0.43295347690582275, 0.31708812713623047, 1.1723909378051758, -1.3447450399398804, -0.2912886142730713, -0.9611214995384216, 0.3720880150794983, 0.7722142338752747, -0.7964404821395874, 1.144449234008789, -0.9529938101768494, -1.4022648334503174, -1.6691166162490845, 1.4722613096237183, -0.6558769941329956, -1.521504521369934, 0.2509765923023224, 1.266608476638794, -1.3170374631881714, 1.1936184167861938, 0.5464094281196594, -2.140324592590332, -0.8141658306121826, -0.3116327226161957, 0.13217675685882568, 0.2705041766166687, -1.3605760335922241, 0.38398829102516174, -0.34546783566474915, -0.025189513340592384, -0.7157668471336365, 0.31326207518577576, -0.5274868011474609, -0.7447879314422607, -1.6728456020355225, 0.14204610884189606, 1.441846489906311, 1.9311217069625854, 0.1978921890258789, 0.1208510622382164, -1.747001051902771, -0.4666357934474945, 0.44154515862464905, 0.9422498941421509, -0.47418496012687683, 0.04612761363387108, 1.358392357826233, 2.0202090740203857, 0.2145162969827652, 2.012573480606079, 0.12478924542665482, 0.3944627046585083, -0.9509587287902832, 1.80801260471344, -0.9403583407402039, 1.2043590545654297, 1.230545997619629, 0.3786879777908325, -1.1787816286087036, 0.2241160273551941, -0.6779597401618958, -1.2688038349151611, 2.2701663970947266, -1.1198458671569824, -0.46265316009521484, -2.4158308506011963, 0.2050931602716446, -0.8250125050544739, 2.1225249767303467, 0.3850100636482239, 0.6171583533287048, 0.07161015272140503, 1.086207628250122, 1.543901801109314, 0.05957181379199028, -1.1523436307907104], [0.5531863570213318, 0.20690880715847015, 0.5885197520256042, 0.6976189613342285, -0.5711592435836792, 0.08270400017499924, 0.6684989929199219, -0.6227508783340454, 0.38724493980407715, -0.6753343939781189, -1.2328081130981445, 0.3776617646217346, 3.296583890914917, 0.4575062692165375, 0.6080256104469299, -0.196431502699852, 0.03319726511836052, 0.19688762724399567, -0.6383841037750244, 1.0261937379837036, 1.3591899871826172, 1.231110692024231, -0.4297384023666382, -0.31839001178741455, 1.3285479545593262, 1.438356637954712, 1.7948367595672607, -1.6675878763198853, 1.1841530799865723, -0.11542432755231857, -1.537900447845459, 1.0206648111343384, 0.4527602195739746, 0.6383091807365417, -0.9157474040985107, -1.3870915174484253, 0.9963968396186829, 0.1685919463634491, -0.8888655304908752, 0.32257533073425293, -0.11100775003433228, 0.6684484481811523, -0.15174219012260437, -1.0109565258026123, -0.6026788353919983, -0.5924866795539856, -1.2062016725540161, -0.06361223012208939, -0.9698506593704224, 0.9562284350395203, -0.6787872910499573, -0.5975714325904846, -1.4393497705459595, -2.6252193450927734, 0.3053382337093353, 0.7364023327827454, 0.4849357306957245, -0.6760950088500977, -0.037163496017456055, 0.7571253776550293, -0.44583842158317566, 0.8199850916862488, 0.17744538187980652, 0.25970447063446045, -0.4298378527164459, 1.1918271780014038, -0.0532732717692852, 0.6340780258178711, 0.9079343676567078, 1.9139312505722046, -0.6259452700614929, -1.0288807153701782, 1.4053318500518799, 0.49573925137519836, 0.7139079570770264, 1.6626849174499512, -0.04493436962366104, -0.035406142473220825, 0.8714864253997803, -1.027356743812561, 1.5483970642089844, -0.2589680254459381, 1.7727519273757935, -0.581606924533844, 0.5726742744445801, -0.23876842856407166, -0.25868573784828186, -1.5846326351165771, -0.06236693635582924, 0.4678781032562256, 0.8626510500907898, 0.5082757472991943, 0.6011061072349548, -1.0233287811279297, 0.4322364032268524, 0.23095828294754028, 2.577348470687866, -1.4360028505325317, -0.12808506190776825, 0.035601671785116196, 0.5746579170227051, 0.2856127917766571, 0.041926298290491104, -0.01622169278562069, -0.11658800393342972, -1.2195677757263184, -2.2023444175720215, 1.0974853038787842, -1.0487524271011353, 0.051775697618722916, 0.08958959579467773, -0.5875270366668701, -1.0821539163589478, 1.364478349685669, 0.03652124106884003, 0.21889442205429077, 1.9778846502304077, -0.40802785754203796, 1.7804211378097534, 0.002115225652232766, -0.13436277210712433, 1.1799182891845703, -0.8003702759742737, -0.35247257351875305, -0.6843708157539368, 0.5146442651748657, 1.7293308973312378, -0.4423351287841797, -0.14882288873195648], [-0.8320940732955933, 1.0696624517440796, -0.09249625355005264, -1.1226422786712646, -1.2858612537384033, -0.0857856273651123, 0.6036516427993774, -0.18206770718097687, 0.4772305488586426, -0.5628121495246887, -0.5053418278694153, -0.4989333152770996, 0.04085089638829231, -0.21806295216083527, 0.7853367924690247, -0.7327600121498108, -1.0517191886901855, -0.342146098613739, 2.352841854095459, 1.2632052898406982, -1.1574101448059082, -1.6356066465377808, -3.31866192817688, -0.3794640600681305, 0.2778262794017792, -0.7024730443954468, 0.5166721343994141, -0.33275237679481506, -0.1349211484193802, 0.816146731376648, -1.3020150661468506, -0.11123280227184296, 0.6982101798057556, -0.05452568084001541, 0.5490899085998535, 0.30029627680778503, -0.5090944766998291, -2.299051284790039, 0.9143125414848328, -0.022160528227686882, 0.1405043751001358, 1.0251718759536743, 0.4078148305416107, -0.3658285140991211, -0.11207947880029678, 0.20837675034999847, 0.2445482313632965, -0.8096859455108643, -0.8899236917495728, -1.7427161931991577, -1.3030438423156738, -0.004361962433904409, 0.11866585165262222, 0.3591744899749756, -0.29828840494155884, -0.01421385444700718, -0.7990272641181946, -1.060102939605713, 0.25953802466392517, 0.3908412456512451, -1.3297133445739746, -0.3245046138763428, 0.5562980771064758, -0.7160426378250122, 1.0236833095550537, 1.0731401443481445, 0.5175474882125854, 1.3476145267486572, -0.5736343264579773, 0.8976900577545166, -0.258556991815567, -0.10862318426370621, -0.056618768721818924, -0.9952153563499451, -0.8798079490661621, 0.3920193910598755, 0.726892352104187, 0.25676429271698, -0.230600506067276, -1.1096727848052979, -0.7364452481269836, -0.2021854817867279, 1.1724658012390137, 0.254505455493927, -0.9676131010055542, 0.5538238286972046, 0.6760613918304443, -0.4072133004665375, -0.29289761185646057, -0.9702507853507996, 0.17046146094799042, 0.26519525051116943, -1.0443376302719116, -0.5070479512214661, 0.33944761753082275, -1.1420791149139404, 1.932466745376587, 0.051768958568573, -1.3921353816986084, 1.2920317649841309, 1.729636311531067, -0.8031604886054993, -1.0329327583312988, 1.4401872158050537, -0.31195545196533203, 0.6969624161720276, 0.6539175510406494, -0.42028263211250305, 1.6481728553771973, -0.24838677048683167, -0.7634917497634888, -0.5605406761169434, -1.6353687047958374, 1.592569351196289, 2.0576353073120117, 0.4249227046966553, -0.6250505447387695, -0.19676275551319122, 0.2102494090795517, 0.9608064293861389, 0.32210519909858704, -1.8496705293655396, 1.1681146621704102, -0.17421624064445496, -0.8915217518806458, -0.6214272975921631, 0.4979183077812195, -1.080341100692749, -0.1761784553527832], [-0.48952868580818176, 0.5078275203704834, -0.11928823590278625, 0.27291956543922424, 0.31414636969566345, -1.2546610832214355, 0.2879334092140198, -0.2809329032897949, 0.16943879425525665, -0.022623280063271523, 1.1566417217254639, -0.24854716658592224, 0.14003099501132965, -1.5416975021362305, -1.582865595817566, -0.5087536573410034, -2.4062070846557617, 0.6674664616584778, 0.7070732712745667, -0.945482075214386, -0.8072368502616882, -0.7918795943260193, -0.5428176522254944, -2.0860862731933594, 0.4277343451976776, -1.7546781301498413, -0.31168174743652344, -2.9697630405426025, -2.0920557975769043, -0.38661760091781616, 1.5371111631393433, 1.382651448249817, 0.6773503422737122, -0.7905638217926025, 0.8685299158096313, 1.0713344812393188, -1.029170274734497, -1.2453912496566772, -1.3566995859146118, -0.6301864385604858, 0.47785496711730957, 2.341144323348999, 0.13085125386714935, 1.2741838693618774, 0.07180707156658173, -1.2420120239257812, 0.321230411529541, 1.1646249294281006, 0.9914593696594238, 1.4941238164901733, 0.6601332426071167, 0.3962728977203369, -2.201629400253296, 0.9073582291603088, -0.5187252163887024, 0.2598029375076294, -0.5053874850273132, -0.33190762996673584, -0.08661375194787979, -0.3699364960193634, 0.14644959568977356, -0.9445955157279968, 0.9354936480522156, 1.1458837985992432, 1.3535614013671875, 2.884631872177124, 0.8043010830879211, 1.752793312072754, 0.040092457085847855, -0.863908052444458, -1.4347718954086304, -0.47064757347106934, 0.33227023482322693, 0.19806016981601715, 0.3279481530189514, 0.4021841585636139, 0.37684422731399536, -0.45860958099365234, -0.1721847802400589, 1.1022136211395264, -1.8531826734542847, 0.25235509872436523, 1.4003496170043945, -0.5901006460189819, 0.2345767468214035, 0.644307017326355, 0.8054176568984985, 1.2781420946121216, -0.7266058325767517, -0.46921300888061523, -0.45738255977630615, 0.8206202983856201, -0.33984947204589844, -0.3817223310470581, 1.0232292413711548, 0.5493685007095337, -0.12694001197814941, 2.3080341815948486, 0.523612380027771, 0.033771585673093796, 1.1420954465866089, -0.36986473202705383, -0.4175379276275635, 0.533873975276947, -0.14426790177822113, -2.080958604812622, 0.6959123015403748, -1.375827670097351, -1.1704126596450806, -0.3890337646007538, 1.0525131225585938, 1.2558754682540894, -1.3164033889770508, -0.4812394976615906, 0.5501890182495117, -0.5481521487236023, -1.5133519172668457, -0.7154894471168518, -0.1019933819770813, 1.0856571197509766, 0.014099306426942348, 2.472831964492798, 2.1522037982940674, -0.3607092797756195, -1.8706547021865845, 0.8652706742286682, -1.0490574836730957, -0.7152531743049622, 0.14792503416538239], [0.5290991067886353, -0.09926766902208328, 0.8446840047836304, -0.5575341582298279, -0.4680486023426056, -0.46917274594306946, 0.6441166400909424, -0.7952303290367126, 0.7475669980049133, -0.7760487198829651, 1.1285005807876587, 1.6066550016403198, 0.3295910656452179, 0.7798487544059753, -0.7175677418708801, 0.3281940221786499, 1.9903373718261719, 0.3407857418060303, 1.918355941772461, -0.10428670793771744, -0.41575512290000916, -1.042510986328125, 1.0267412662506104, -0.1072680801153183, 0.5301883816719055, 0.7500275373458862, 0.1780727207660675, 1.2764793634414673, -1.061203956604004, -0.4496946930885315, 0.694943904876709, 0.23791290819644928, 0.31275486946105957, 1.1853781938552856, -1.722694993019104, -0.20922274887561798, 1.0327637195587158, -0.1739024817943573, 0.21817295253276825, -1.8576724529266357, 0.5370975136756897, -0.22655509412288666, -0.5798578858375549, 1.1216728687286377, -0.33738335967063904, 0.3980320692062378, -1.9317991733551025, 0.3269283175468445, -0.884381115436554, -1.5135977268218994, 0.6454576253890991, -0.18533839285373688, -0.8549988269805908, -0.27401983737945557, 0.4547334909439087, 2.1610562801361084, -1.5548467636108398, -0.9864497184753418, 2.2346205711364746, 0.70328688621521, -0.8800665140151978, -1.9501609802246094, 0.8785611391067505, 1.0537950992584229, -0.7117483615875244, 0.01808822900056839, -0.4411078989505768, 0.18517877161502838, 0.9808877110481262, -0.6548640131950378, -1.3279855251312256, -1.3058183193206787, 0.5392254590988159, 0.5484898090362549, 0.10442864894866943, 1.1462870836257935, 0.4646076261997223, -0.39318400621414185, -0.683174729347229, 0.5081177353858948, 0.5751078724861145, 0.19566094875335693, 1.5639631748199463, -0.5468810200691223, 0.20687884092330933, -1.68035888671875, 1.2031936645507812, 0.013818939216434956, 1.5947542190551758, 0.6181798577308655, -0.9842987060546875, -1.2698469161987305, -1.6634916067123413, -0.08060560375452042, -0.223502516746521, -2.312056303024292, -0.3124018609523773, -0.9437879920005798, 0.22069592773914337, 1.3449469804763794, 0.8200594782829285, 0.80005943775177, 0.46691858768463135, -0.9328035116195679, -0.3442799746990204, -0.6867640614509583, -1.682610034942627, -0.7709831595420837, -1.1543477773666382, 0.5579243898391724, 0.274582177400589, 0.4647124707698822, 0.3410271406173706, 0.8861101865768433, -0.6545595526695251, 0.7986307144165039, 0.5182169079780579, -1.087436556816101, -1.1110721826553345, 0.399176687002182, 0.5704144835472107, -0.029309572651982307, 0.841168224811554, -0.03921618312597275, -0.501434862613678, -0.7347277998924255, -0.8666189908981323, 0.5576445460319519, 1.3029096126556396], [-0.0333014540374279, 1.600055456161499, 0.8051104545593262, -0.2906922996044159, 1.336600422859192, -1.7624337673187256, -0.508413553237915, 0.6290260553359985, 0.889835774898529, 1.177900791168213, 0.28367599844932556, 0.20647092163562775, 0.6420074105262756, -0.3161563575267792, -1.3542646169662476, -0.41925248503685, 1.1485300064086914, -0.366608589887619, 0.28254786133766174, 0.34891393780708313, -1.7197027206420898, -1.44212806224823, 0.5845681428909302, 1.2370827198028564, 0.6954048275947571, 0.1041160374879837, 0.8508645296096802, -1.9474986791610718, -1.7381771802902222, 0.834000825881958, -0.47583794593811035, -0.48433545231819153, 1.6923208236694336, -0.8660032153129578, -0.14554716646671295, -0.5400494337081909, 1.4395607709884644, -0.11598765105009079, -1.3863744735717773, 0.3340699076652527, -0.4650322496891022, 0.3817678391933441, -0.10315442085266113, 0.3507866859436035, 0.2780008614063263, -1.1078174114227295, 0.6712756752967834, 0.42841771245002747, 0.4708156883716583, 0.5880029797554016, -2.2209510803222656, 0.27623000741004944, 1.0540684461593628, -1.6551364660263062, -0.0472324900329113, 1.4093194007873535, -0.03281436488032341, -0.8168679475784302, 0.9776848554611206, -1.519095778465271, -1.446388602256775, -2.895263433456421, -0.10730026662349701, -0.9392752647399902, -1.1920320987701416, -0.00015621018246747553, 0.05610746890306473, -1.0347909927368164, -1.023071050643921, 0.15273533761501312, -1.2335529327392578, 0.6148233413696289, 0.3061976134777069, -0.288287490606308, 1.2034924030303955, 1.168611764907837, -0.5617304444313049, 0.5316866636276245, 0.06825036555528641, -0.11489449441432953, 0.019955476745963097, -0.39799046516418457, 0.9819134473800659, -0.02249983511865139, -1.0116175413131714, -0.795757532119751, 0.8589191436767578, 0.17249426245689392, 0.5286194682121277, 0.001064202981069684, -0.6740337014198303, 0.01689152605831623, 0.3492553234100342, -0.902556300163269, -0.180852472782135, -0.7030104398727417, -0.23010489344596863, 0.9449293613433838, 0.1892150640487671, -0.20131030678749084, -0.3459573984146118, -1.8296215534210205, -1.5255502462387085, -0.6116886138916016, 0.4883994460105896, 1.3693801164627075, 1.2445719242095947, 0.3287920355796814, 0.8897722363471985, -2.3219947814941406, -0.11632551997900009, 0.42709624767303467, -0.8549069166183472, 0.4477650821208954, 1.4213277101516724, -0.8616434931755066, -1.6401371955871582, -0.9596808552742004, -0.6201949715614319, 0.9835801720619202, -0.014237562194466591, 0.9577406048774719, -0.40308496356010437, 0.229551762342453, -0.17760802805423737, -0.21146278083324432, 2.7214126586914062, -1.6138668060302734, 0.28824833035469055], [1.0293927192687988, -0.20570023357868195, 0.4376782476902008, 0.6675100922584534, -0.7687689065933228, 0.5339840054512024, -0.912264347076416, 0.6906354427337646, 1.9359766244888306, -1.42164945602417, 0.5985116958618164, 1.1743139028549194, -1.5967093706130981, -2.9996724128723145, 1.1495567560195923, -0.4190903306007385, -0.14587071537971497, -0.29141268134117126, 0.181104376912117, -0.3430003523826599, -1.1508558988571167, -1.7224966287612915, -0.021868843585252762, -0.018792303279042244, 1.122246503829956, 0.4227138161659241, -0.012376775965094566, -0.37121105194091797, -1.2321058511734009, -0.9491202235221863, 2.3250982761383057, -1.0472806692123413, 0.14804059267044067, -1.1514426469802856, -0.4665551483631134, -0.4969332218170166, -0.43852075934410095, 0.1548151969909668, 1.5542210340499878, -0.639860212802887, 0.05897559970617294, -1.0719630718231201, -0.8918983340263367, 1.0892198085784912, -1.9354324340820312, -0.6900467276573181, -0.17382927238941193, 0.9001417756080627, -0.0712578296661377, 2.6004533767700195, -1.7704156637191772, 0.4166838228702545, 0.6068093180656433, 1.7557569742202759, 1.9176992177963257, -0.018822714686393738, -0.9718114733695984, -0.5591674447059631, -0.47701531648635864, -0.44140464067459106, -1.00503671169281, -0.5398381352424622, 1.9299907684326172, 1.1262152194976807, -0.687292754650116, 0.26914462447166443, -1.2795578241348267, -1.892870545387268, 0.9823772311210632, 0.913069486618042, -1.3729219436645508, 0.342324435710907, -0.7595224380493164, 0.5082020163536072, -0.08480630069971085, 0.5211985111236572, -0.573874294757843, -1.047277808189392, 0.5454875230789185, 0.7061768174171448, -0.3801290988922119, -0.26634615659713745, 0.051106978207826614, 0.7334126830101013, 0.2280246913433075, -0.1702987253665924, 1.5883809328079224, -0.10687470436096191, -0.578622579574585, -0.005273886024951935, -0.34648749232292175, -0.7054096460342407, 0.42279529571533203, 0.14401020109653473, -1.317579984664917, 1.1318517923355103, -0.05776219442486763, 0.2947165071964264, 0.3254474401473999, -0.0307808555662632, 0.013601982966065407, -0.04846058040857315, 0.6791440844535828, -1.453771948814392, 0.8711374402046204, 1.0081007480621338, -1.2920573949813843, -1.352785587310791, 0.9445759057998657, -0.15936417877674103, 0.9587435722351074, -1.9702397584915161, -0.774869441986084, 1.2428505420684814, 1.2351298332214355, 0.3002222776412964, -1.5372430086135864, -0.455511212348938, -1.0392552614212036, 0.43948692083358765, 0.49203065037727356, 0.16645516455173492, -1.8052668571472168, 0.3770058751106262, -0.02143721655011177, 0.13351206481456757, -1.762593388557434, 0.06912606954574585, -0.4126616418361664], [0.6947815418243408, -0.20481842756271362, -0.42220234870910645, -1.1138683557510376, 0.6147221922874451, 0.6288599371910095, -0.2651650905609131, -0.5424806475639343, 0.6142483353614807, 0.6728270649909973, -0.212117537856102, 0.1475559026002884, -0.885112464427948, 1.1771154403686523, 0.3745271563529968, -0.9821982979774475, -0.4262557923793793, 2.143644332885742, -1.066089153289795, -1.2466931343078613, 1.841310977935791, -0.13305003941059113, -2.073150396347046, -1.2330862283706665, 0.5208264589309692, 0.6476534008979797, -0.4462644159793854, 0.4916924834251404, -1.4646575450897217, -0.7023788094520569, -0.3908231854438782, -1.3668467998504639, 0.15789489448070526, -0.3950519859790802, 0.30858293175697327, 0.27515190839767456, -0.9306878447532654, -2.9967920780181885, 1.1671525239944458, -0.04294991120696068, -0.7235539555549622, 0.38390472531318665, -1.7591888904571533, 1.8637112379074097, -0.3215479850769043, -1.2513986825942993, -0.22331330180168152, 0.32641443610191345, -0.09241536259651184, 0.6025485396385193, -0.13675114512443542, 1.9626625776290894, 0.2586899399757385, 1.9878394603729248, 0.2375006079673767, 0.8257951140403748, -0.8821970820426941, -1.0157755613327026, 0.8263320922851562, -0.28969183564186096, -0.6300265789031982, -0.3965272903442383, 1.0134861469268799, -0.616151750087738, 0.28548330068588257, 0.975104808807373, -0.8814493417739868, 0.4603668451309204, -1.4210011959075928, 1.1760611534118652, -0.08100682497024536, -0.2907104194164276, -0.3928467631340027, 0.36439135670661926, -0.27476558089256287, -0.18360960483551025, -0.3196546137332916, 0.5257636308670044, -0.7256373763084412, -0.6569194793701172, 0.5607262849807739, -0.4144361913204193, -0.766471266746521, -0.39492714405059814, 0.49844977259635925, -2.42583966255188, 1.55233633518219, -1.397630214691162, 0.5938574075698853, 0.24928295612335205, 1.1383068561553955, -0.057969823479652405, 2.2306737899780273, 0.4588204622268677, -0.4328678548336029, 0.2872443199157715, -1.3830504417419434, 1.7026537656784058, 0.550961434841156, -0.1866472065448761, -0.5169364809989929, 1.7682757377624512, -1.9195094108581543, 1.7370879650115967, -1.8540488481521606, -0.34011805057525635, -0.17187392711639404, 0.9783014059066772, 1.6488919258117676, 1.0843549966812134, -1.6096173524856567, 0.8380990028381348, 1.470336675643921, 0.09486168622970581, -0.03851025551557541, 0.45309978723526, -1.1530251502990723, -0.5928913354873657, 1.509294033050537, 0.2873584032058716, 1.1120861768722534, 0.04565737023949623, 1.4131314754486084, 0.7106847167015076, 1.654242753982544, 1.2554199695587158, -1.3984555006027222, 0.6278581619262695, 0.2253018021583557], [-2.2108235359191895, -0.26238125562667847, 0.6280560493469238, -0.007336026057600975, -0.16935226321220398, 0.21194815635681152, 1.317600131034851, 0.5121368169784546, -1.3484138250350952, -0.6471058130264282, 0.0437912717461586, 2.2796247005462646, -0.6535990834236145, 1.1720691919326782, 0.03599359840154648, 0.7112870812416077, -0.07704517990350723, 1.6157501935958862, 1.0335346460342407, 1.6020289659500122, 1.8541069030761719, -0.30615219473838806, 0.18572546541690826, 1.2479811906814575, -0.7264543771743774, -0.878829836845398, 1.3086360692977905, 0.8561903834342957, 1.761276125907898, -1.0025341510772705, 0.5628311634063721, 0.6519150137901306, -0.8072845935821533, -0.510940432548523, 1.8024929761886597, -0.8263818621635437, -0.7996318340301514, 0.8579975962638855, -0.35616979002952576, -0.5864004492759705, 0.68526691198349, -0.6674644947052002, 0.12373191863298416, -1.532790184020996, -0.1744881570339203, 0.3869415819644928, -1.153573989868164, -1.830308198928833, 0.7797442674636841, -0.9412236213684082, 1.2243057489395142, 0.7530367374420166, 0.5126243233680725, -0.2722161114215851, -1.5615051984786987, 0.31065675616264343, -2.134857416152954, -0.567794919013977, -1.005403757095337, 0.7011076807975769, -0.6009039282798767, -1.6360639333724976, 0.5678386092185974, 0.6606099009513855, 0.1351311206817627, -0.5858946442604065, -0.41398561000823975, 0.5405884981155396, 1.0840511322021484, -1.5295088291168213, -0.3216901421546936, -0.11936210840940475, -1.270014762878418, -0.7580965161323547, 0.845099687576294, 1.1529978513717651, 0.10583218932151794, 0.4961681365966797, -0.05606833100318909, -1.0423524379730225, -1.5232127904891968, 0.8477921485900879, 0.26446112990379333, -1.3086826801300049, 1.9910709857940674, 0.11270911991596222, 0.9857364296913147, 0.7108827233314514, 0.20746169984340668, 0.16694758832454681, -0.3182818591594696, -0.29669177532196045, -0.5033565163612366, -1.1145586967468262, 1.0945720672607422, -1.8689714670181274, -1.8291946649551392, -0.6035447120666504, -1.534791350364685, -1.4923100471496582, -1.0548053979873657, 0.24398569762706757, 1.0452370643615723, 1.4959826469421387, 0.2903780937194824, -0.6594576239585876, -0.4074011743068695, -1.500785231590271, 0.09211380034685135, -0.4027692973613739, 0.0027499543502926826, 0.7724612951278687, 0.8695546388626099, 0.34237873554229736, 0.6644005179405212, -0.7027101516723633, 0.862864077091217, -0.7248556613922119, -0.20597364008426666, 0.3094642460346222, 1.9352062940597534, -0.3477080762386322, 0.7085573077201843, -0.11701023578643799, -0.7517386674880981, 0.8943196535110474, 0.39151591062545776, 1.1757005453109741, -0.9480178356170654], [0.272339791059494, -0.419754296541214, 0.4606426954269409, 0.15236559510231018, 0.14809001982212067, -0.8013015389442444, 1.1075704097747803, 0.37611812353134155, -1.2357081174850464, 0.5255551338195801, 1.0643181800842285, 0.5564283728599548, 2.4378762245178223, -1.42130708694458, 0.6191831827163696, -0.04864524304866791, -0.028758326545357704, -0.3560062348842621, 0.7301533222198486, 0.5775207281112671, 0.29674065113067627, -1.3683863878250122, 0.11695560812950134, -0.09932219982147217, 0.09232562780380249, 0.7897257804870605, -0.7247142791748047, -0.4538915157318115, -0.8852013349533081, 1.0377601385116577, 1.0576192140579224, 0.5540313124656677, -1.6362837553024292, -1.07916259765625, -0.49789807200431824, -0.499021053314209, 0.8226134181022644, -1.2464616298675537, 0.7663818001747131, 0.1600489467382431, -1.0691277980804443, -0.16893114149570465, 1.932847023010254, -1.2994399070739746, -1.6148300170898438, -1.2487812042236328, -1.0190579891204834, -0.49467751383781433, 0.9750664830207825, 0.5810254216194153, 0.8880713582038879, -0.40445390343666077, 0.846413791179657, 0.15771980583667755, -0.37418845295906067, -1.2925808429718018, 0.9522486925125122, 0.801035463809967, 0.48044243454933167, 0.16139622032642365, 0.16456544399261475, -1.9447120428085327, 0.012979025952517986, 1.7268471717834473, 0.9049732685089111, -0.14099663496017456, 0.9874811768531799, 0.19218307733535767, 0.5765699148178101, -0.3901248574256897, 1.9436538219451904, 0.8187763690948486, 0.9001995921134949, 0.9314829707145691, -1.1445338726043701, -1.6535656452178955, 0.06260602921247482, -0.39551419019699097, -0.2901877462863922, 0.9599167108535767, -0.30101388692855835, 1.0543828010559082, 4.528099060058594, -0.7732113003730774, -0.2894696295261383, -0.04508213698863983, 0.5131049752235413, -0.19342520833015442, -0.4606676697731018, 0.15995587408542633, 0.12041434645652771, 0.47772231698036194, -0.7289785146713257, 0.1932554841041565, -0.7846588492393494, 1.2503944635391235, -1.1685535907745361, -1.779922366142273, 1.8871583938598633, 0.7265682816505432, -0.7038385272026062, 0.8978433609008789, 0.038619063794612885, 1.4371259212493896, -0.5914223790168762, 0.4856531620025635, 1.2073017358779907, -1.1990076303482056, 1.2336442470550537, -1.116687536239624, -0.8768853545188904, -0.24736745655536652, -1.3834418058395386, 1.6716015338897705, 0.32683172821998596, 0.32526424527168274, 0.8059313297271729, 0.031064096838235855, 0.5813263058662415, -1.4858214855194092, -0.9687658548355103, 0.3337518572807312, 0.49681541323661804, 0.7573708891868591, -0.9822805523872375, 1.0669015645980835, 0.2815034091472626, -0.1334577202796936, -0.8388245105743408], [-1.7585190534591675, 0.04709396883845329, 0.8519623279571533, -0.48446527123451233, -0.10860789567232132, -0.29011455178260803, 1.1395986080169678, -1.0136951208114624, 2.5169284343719482, -0.517952561378479, -1.7637207508087158, 0.5056384205818176, -1.0908697843551636, -0.6114386320114136, -1.2620059251785278, 0.36918187141418457, -1.3511680364608765, -0.9199876189231873, -2.228980541229248, 0.8452780842781067, -1.5691885948181152, -1.845118761062622, 0.18210679292678833, 0.23706082999706268, -0.8818853497505188, 1.3265550136566162, 0.8466644883155823, 0.4492330253124237, -1.1146186590194702, -0.8048997521400452, 0.07884260267019272, -1.7215654850006104, 1.0412927865982056, 0.26003411412239075, 0.42459043860435486, -0.47083979845046997, -0.42137280106544495, -0.49855226278305054, -1.144040822982788, -2.219802141189575, 0.4324953556060791, 1.311453104019165, 1.152708649635315, -1.8560892343521118, -0.43283775448799133, 0.7676643133163452, -0.1792765110731125, -1.8678653240203857, 0.3017346262931824, 0.5610688328742981, 0.09256661683320999, 1.193289041519165, -0.7999319434165955, 0.018919093534350395, 1.8645480871200562, -0.7970237135887146, 0.10819532722234726, -0.45461276173591614, -0.5254747867584229, -0.0009976079454645514, -0.3272649645805359, -0.36738428473472595, -1.5775936841964722, -0.28790006041526794, -2.017061471939087, 0.16775226593017578, -0.6580967307090759, -0.872097373008728, 0.05162734165787697, -0.722743034362793, 1.2808692455291748, -0.5408757925033569, 0.6659714579582214, -0.8097376823425293, -0.4713309109210968, -0.07286595553159714, 1.303484559059143, -0.615029513835907, -1.0747114419937134, -1.072861671447754, 0.22821247577667236, -0.0009899898432195187, -0.5473648309707642, 0.3780534267425537, 0.26758506894111633, 0.5975645780563354, 0.5103688836097717, 0.5130883455276489, 0.45059284567832947, 0.25664207339286804, 1.176568627357483, 0.641867458820343, 0.5852304100990295, 1.4042904376983643, -0.132017582654953, -0.8083896040916443, 0.7142876982688904, -0.6146323680877686, -0.7574371099472046, 0.1980307698249817, 0.18413609266281128, -1.549232840538025, -0.676798939704895, -0.21007314324378967, -0.5852323770523071, -0.8394131064414978, -0.7388037443161011, 0.3689987063407898, -0.3974166512489319, 1.6170848608016968, 0.4596981704235077, -0.18968795239925385, -1.5634130239486694, 0.6712148785591125, -1.3655771017074585, 0.1526394635438919, -0.20151536166667938, 0.7807395458221436, -0.4769374430179596, 0.7891826033592224, 0.10756649821996689, -1.62258780002594, 0.04733016714453697, 0.1840975433588028, 0.30996018648147583, -2.4134066104888916, 2.8613040447235107, 1.1399483680725098, -0.6018637418746948], [0.4660826027393341, -0.8050892949104309, -0.9667774438858032, -0.48722973465919495, 1.0769340991973877, 0.5883454084396362, 1.1421340703964233, -0.5207625031471252, 0.25539830327033997, 0.8071964979171753, -0.3966217637062073, 0.2245904505252838, -0.6392540335655212, -0.018857182934880257, 0.4026302695274353, -0.41753289103507996, -0.2957921326160431, 0.7677261829376221, -1.1030865907669067, -0.08740496635437012, -0.4027461111545563, 0.37490352988243103, 0.39689961075782776, 0.5839447975158691, 0.8273704051971436, 0.23966409265995026, -2.6653454303741455, -0.9812260270118713, 2.6289920806884766, -0.26946204900741577, -1.343044638633728, 0.4473518133163452, -0.10539046674966812, 1.0983896255493164, 0.16828685998916626, 0.07497306168079376, 0.39485353231430054, 0.15581871569156647, 0.3704150915145874, -1.699904203414917, -0.0627380758523941, -0.7929648160934448, 1.741357684135437, -0.668485164642334, 2.1487977504730225, -0.7035082578659058, 0.9607180953025818, -0.33775365352630615, 0.7010027170181274, -0.5131603479385376, -1.110275387763977, -0.7597149014472961, -0.7071583271026611, 0.9745885729789734, -1.137711763381958, 0.4129348695278168, 0.2360609471797943, 1.6623010635375977, 0.3102215826511383, 0.6124628186225891, -0.14538390934467316, 0.6118091940879822, 0.0356585793197155, -1.0076006650924683, 2.018040895462036, 1.0768150091171265, 0.45072993636131287, -0.2120080590248108, -0.23036813735961914, -2.0653278827667236, -0.11205561459064484, 0.00835030060261488, 1.98585045337677, -0.17494820058345795, -1.6362913846969604, -0.6018948554992676, -0.7509085536003113, 0.038718659430742264, 0.5802562832832336, -0.8256291151046753, -0.8121591806411743, -0.28664812445640564, -0.06171491742134094, 0.9857895970344543, -0.23370207846164703, 0.2077649086713791, 0.7834703922271729, -0.7150583863258362, 0.28030094504356384, 0.5045489072799683, 0.7174041867256165, -1.7030041217803955, -1.5231233835220337, 0.28483474254608154, -2.191601037979126, -1.0701618194580078, -2.675248146057129, -1.6698201894760132, 0.08064468204975128, -0.1554373949766159, -0.8577828407287598, 0.22126919031143188, -0.29244503378868103, 1.6136314868927002, 1.8613085746765137, -0.057695962488651276, -0.8735748529434204, -0.5717056393623352, -0.44879940152168274, 1.2344564199447632, -0.17403662204742432, -1.0222935676574707, 0.6330335140228271, 0.16893362998962402, 2.552260637283325, -0.19121481478214264, -0.9079384803771973, 1.7581331729888916, -0.506543755531311, -0.21934297680854797, -1.7126743793487549, 0.1289805918931961, 0.5550697445869446, -0.5313616991043091, 0.43456006050109863, 1.3298207521438599, 1.2054190635681152, 1.0620193481445312, -0.22032153606414795], [-0.5149959921836853, 1.0817383527755737, 2.4725000858306885, 0.790945827960968, -1.6823523044586182, 1.5655858516693115, -0.6621925234794617, -0.40320825576782227, 0.7937415242195129, -1.1356168985366821, -0.0628429725766182, 0.6813902258872986, 0.8886602520942688, 1.4498953819274902, -0.527283787727356, -1.664416790008545, -1.987646222114563, 1.5042898654937744, -2.1233315467834473, -0.9747108817100525, -1.5721129179000854, -0.24637296795845032, -0.9673024415969849, -1.368751883506775, -0.9278144240379333, 0.3275817632675171, -1.0006901025772095, 1.291412353515625, -0.49063751101493835, 1.1457351446151733, 0.019969239830970764, -0.12331268936395645, -0.3820647895336151, -1.369622826576233, -0.5908886790275574, -0.5243374109268188, -2.4669575691223145, 1.1388543844223022, -1.633056402206421, -0.565126359462738, -1.3089455366134644, -0.6958339214324951, -0.8313584327697754, -1.1751433610916138, 0.23135894536972046, -0.6674354672431946, -0.07020413130521774, 1.0921822786331177, -0.8835291862487793, -1.461044430732727, 1.4795215129852295, 0.3226364254951477, 1.2612510919570923, 0.8800183534622192, -0.3690042495727539, -1.3076626062393188, -2.0840182304382324, -0.5870884656906128, -1.1315577030181885, -0.9646650552749634, -0.9480281472206116, 0.19062650203704834, -0.9259161353111267, -0.4122888147830963, -2.282590627670288, -1.0798190832138062, 0.7737000584602356, 2.010612726211548, 0.44424912333488464, 0.4868890047073364, 1.0009570121765137, 0.26851433515548706, -1.182450532913208, 0.2446538805961609, -1.4373526573181152, -3.1850826740264893, -0.4367200434207916, 0.645556628704071, 0.7854791283607483, -1.56143057346344, 0.8979605436325073, -1.1506948471069336, 0.24994495511054993, -0.8240052461624146, -0.2631383240222931, 0.17120292782783508, -1.6245561838150024, -0.7587121725082397, -0.8169374465942383, -1.6649513244628906, 0.09099540114402771, 0.06932715326547623, -0.9493889212608337, -0.6666746735572815, 0.25805339217185974, 1.653591513633728, -0.6650707721710205, 0.911136269569397, 0.5664638876914978, -0.24318447709083557, 0.8157011866569519, -0.8651995062828064, 0.112189881503582, -0.45301783084869385, 0.8660931587219238, -1.251795768737793, 0.6701216697692871, 0.6098632216453552, -0.8926771283149719, 0.8454408049583435, -0.7192109227180481, 0.7556647062301636, 1.8228832483291626, -1.166555643081665, -0.4809618294239044, 0.923373281955719, 1.2392712831497192, 0.7599861025810242, 0.6241638660430908, 1.0960969924926758, -0.12840856611728668, 0.3848545551300049, 0.9047254323959351, 1.8014789819717407, -2.0658071041107178, 0.07098232209682465, -0.9065613150596619, -0.4677916169166565, -1.193625807762146], [0.6004643440246582, -1.1671409606933594, 1.7189282178878784, -0.679418683052063, -0.5788229703903198, 0.40832263231277466, 0.510866641998291, -0.8802111744880676, -0.4008444547653198, 1.5000650882720947, 0.33314183354377747, -0.18801477551460266, 0.942479133605957, 0.8993534445762634, -1.026094675064087, -1.1104389429092407, -0.298019140958786, -0.08333103358745575, 0.05767300724983215, -0.30536237359046936, 1.1750625371932983, -0.189091756939888, 0.4656801223754883, -1.2576172351837158, 0.34638160467147827, 1.014807105064392, -0.31510624289512634, 0.6067607402801514, 0.419974148273468, -0.778425395488739, 0.4734455347061157, -0.9495335817337036, 0.1995851993560791, -0.9987757802009583, 1.1485223770141602, 1.750773549079895, 0.7463653683662415, 1.3313292264938354, -0.1673170030117035, -1.094895362854004, -0.5718945860862732, -0.46568456292152405, -1.8616597652435303, 0.08281990885734558, -0.05572584643959999, 1.297192931175232, 1.307187795639038, 0.7090123891830444, -0.3151910901069641, -0.6704360842704773, 0.8998793959617615, -1.6019439697265625, -0.8369940519332886, 0.28560909628868103, -0.35921627283096313, 0.6887454986572266, -1.884204387664795, -2.6277379989624023, 1.3476336002349854, -0.6641655564308167, -1.0480141639709473, 2.0047478675842285, 0.9289919137954712, 1.051368236541748, -0.408812552690506, -0.014544883742928505, -0.797280490398407, -0.42413902282714844, -1.2768160104751587, -1.1638554334640503, -0.21549224853515625, -1.4107890129089355, -0.19620610773563385, -0.9209794402122498, -1.1232725381851196, -0.6950860023498535, 0.20892246067523956, 0.3238248825073242, -0.02701530046761036, 1.1486451625823975, -0.0027319358196109533, -1.7135777473449707, 0.9005752205848694, 1.2957459688186646, -1.0487736463546753, 0.27548912167549133, -0.688304603099823, 1.180458426475525, -0.13353581726551056, 0.3379359841346741, 0.18193483352661133, -2.137014389038086, 0.47508320212364197, 0.256344199180603, 0.2400965690612793, -0.29977676272392273, -1.1088554859161377, -0.18406222760677338, 0.610823392868042, -1.7925734519958496, -0.5951441526412964, 2.0896565914154053, 0.29229629039764404, 0.285170316696167, 0.8840749859809875, -1.663352608680725, 0.3305201530456543, 0.10312783718109131, -1.520471453666687, 0.3017062246799469, -0.41370129585266113, 0.3012895882129669, 1.133437156677246, -0.5703045725822449, -0.30608081817626953, 0.7629517912864685, 0.739727795124054, 0.4866653084754944, 0.30809396505355835, 0.09995561093091965, -0.4526478350162506, 0.5896903276443481, -0.8101711273193359, -0.7273930907249451, -0.08386315405368805, -0.3105771541595459, -0.5301028490066528, -0.11774156987667084, -0.40553614497184753], [0.30938220024108887, -1.297170877456665, -1.474492073059082, 0.8629192113876343, 0.8050779104232788, -2.020433187484741, -0.44834133982658386, -0.019711386412382126, 0.4045107364654541, -0.2728368043899536, -0.9063357710838318, -1.8589617013931274, -1.0680162906646729, -0.9685867428779602, 0.6835550665855408, 0.9294427037239075, 0.11449426412582397, -1.5493336915969849, -1.1426228284835815, 1.929614543914795, -2.4129810333251953, 0.4671521484851837, 0.9441999197006226, 0.27972733974456787, -1.4764500856399536, -1.1565945148468018, -0.5575198531150818, -1.3116984367370605, -0.5169654488563538, 1.1954505443572998, -0.6000180244445801, -2.163816213607788, -0.2670189440250397, -0.037695303559303284, 0.1867581158876419, 0.1447359323501587, -0.12409964948892593, -0.25807201862335205, 0.36408865451812744, 0.14202599227428436, -0.9570289254188538, -1.2517460584640503, -0.24237248301506042, -1.9162863492965698, 0.06974141299724579, -1.7618216276168823, -0.8685872554779053, -1.2443784475326538, -0.8221455812454224, -0.09542573243379593, -0.4364912807941437, -0.4433860778808594, 0.09397516399621964, 1.0652880668640137, 0.20699059963226318, 0.575278103351593, 1.4873034954071045, -0.08844684064388275, -0.2011861652135849, 0.23817497491836548, -0.07915555685758591, -0.43785762786865234, 0.28423553705215454, 0.20174413919448853, -1.0735775232315063, -1.6455893516540527, 0.36721935868263245, -1.0623432397842407, -1.4111088514328003, 0.059346266090869904, -0.04315852001309395, -0.4524790942668915, -0.11143781244754791, -0.3994826376438141, 2.2524099349975586, 0.3605163097381592, -0.19696733355522156, 1.1675302982330322, 1.241782784461975, -0.5887597799301147, -1.3947466611862183, 0.19022251665592194, -0.6621305346488953, 1.7460805177688599, -0.4914109706878662, -1.4312478303909302, -1.347267746925354, -0.44047024846076965, -0.04518749937415123, -0.6776567697525024, 0.8152327537536621, -0.917722761631012, 0.4013362526893616, -1.0694612264633179, -0.1861032247543335, -0.4107392132282257, -0.8063421845436096, -0.46383458375930786, -0.6060823202133179, 0.8214755058288574, 0.03540642186999321, -1.2357286214828491, 1.7239694595336914, 0.01018517930060625, 0.40712180733680725, -0.2275361865758896, -0.3412814736366272, -0.9542528986930847, 1.6326982975006104, 1.4126099348068237, -2.2362899780273438, -0.6170680522918701, -1.1078224182128906, 0.9760453701019287, 0.8892715573310852, 0.5506743788719177, -0.6165040731430054, -1.7553269863128662, -0.07624661922454834, -1.211320161819458, 0.42119044065475464, -0.7715246677398682, -0.6454972624778748, -1.108605980873108, 0.14111219346523285, -0.6097124218940735, -0.7490296363830566, 1.3680529594421387, -0.9535717964172363], [-0.1668969839811325, 1.3912088871002197, -0.3568890690803528, -0.08342543244361877, -1.0657830238342285, -0.3188422918319702, -0.39990147948265076, 0.43310534954071045, -1.4360703229904175, -1.0578091144561768, 2.0369913578033447, -1.0215167999267578, 1.9644780158996582, -0.28276732563972473, 0.5356741547584534, 0.09185943752527237, 1.1766167879104614, -0.1944812685251236, 1.4985899925231934, 0.6016175150871277, -2.0849103927612305, -0.5512995719909668, -0.27983149886131287, 0.17707638442516327, 0.758579671382904, 1.2346611022949219, 0.3401992917060852, -0.7351610660552979, -0.9442557692527771, -0.6128268837928772, 0.5139465928077698, 0.6140503883361816, 0.5713584423065186, -0.928203821182251, 0.3462722599506378, 0.23474179208278656, 1.3669153451919556, -0.7262607216835022, 0.1120203509926796, 1.1512762308120728, -0.32899364829063416, -1.27717125415802, 0.8344513177871704, -0.608836829662323, 1.13506019115448, 0.5524778962135315, -0.5753854513168335, -0.5147417783737183, -1.038490891456604, 0.12191596627235413, -0.6822623610496521, 0.7056628465652466, -1.422494888305664, 0.07693855464458466, 0.3336663544178009, 1.6092519760131836, -0.4974699020385742, 0.6719110012054443, 0.6842817068099976, -0.09632843732833862, 1.4011805057525635, -1.7406421899795532, 0.1848224401473999, -2.5675110816955566, 0.8066277503967285, 2.260863780975342, -0.4266416132450104, -1.1574524641036987, -0.40309855341911316, -0.20238618552684784, 1.2050033807754517, 1.5564494132995605, -0.3337741196155548, 1.0761418342590332, 2.4082489013671875, 0.6465437412261963, -0.25846704840660095, 1.5667304992675781, 0.2694283425807953, 0.7597559690475464, -0.39520934224128723, -0.17822612822055817, -0.04632720723748207, 0.9840703010559082, -1.4870761632919312, -1.3671998977661133, -0.39729881286621094, 0.6454337239265442, 1.6148849725723267, -1.2506482601165771, -0.8307231664657593, 0.46996092796325684, 1.8221406936645508, -0.41615718603134155, -1.6458015441894531, -1.1939032077789307, 0.5647501349449158, 0.48738667368888855, -0.18116679787635803, -0.2512892186641693, -0.028180068358778954, 1.0180760622024536, -2.0120184421539307, 0.18895994126796722, -1.0072407722473145, -1.187495231628418, -0.7473475933074951, -1.0936717987060547, -0.07439852505922318, -0.8046709299087524, -0.45870184898376465, 1.5616397857666016, -1.4934149980545044, 1.0851472616195679, 2.3357791900634766, 1.2962563037872314, -1.524040699005127, 0.6983761191368103, 0.38720840215682983, -0.3664090633392334, -0.061588749289512634, 0.2542855143547058, -0.22124382853507996, 1.517749309539795, 0.39730390906333923, 0.19383279979228973, -1.3837617635726929, 2.268242120742798, 0.6802579164505005], [-2.379852771759033, 0.25007858872413635, 0.24704964458942413, 0.9864001870155334, 0.1178336814045906, 1.1343690156936646, -0.4432697594165802, 0.4528869390487671, -1.601118564605713, 0.6304627060890198, -1.1061519384384155, -2.8348183631896973, 0.11881304532289505, 0.7314795255661011, -1.8444877862930298, 0.9925876259803772, -1.5876508951187134, 0.11702635139226913, -0.18096663057804108, 2.298915386199951, 1.7131956815719604, -1.0935777425765991, -0.45341140031814575, -0.057150375097990036, 0.14331857860088348, -1.0933159589767456, 2.064213752746582, -0.284533828496933, 0.6283194422721863, 0.11370459944009781, -0.5913829803466797, 0.45586222410202026, -0.09005743265151978, 0.5687654614448547, 0.17227865755558014, -0.9278759360313416, 0.40624499320983887, -0.42380091547966003, -1.142240285873413, -0.48379454016685486, 0.875920295715332, -1.3701927661895752, 0.04479898512363434, -0.9929649829864502, -0.8590519428253174, -0.5349332690238953, -1.609528660774231, 0.9310961365699768, -0.6038610339164734, 0.5961014032363892, -1.16097092628479, -0.2699902355670929, 0.4839118421077728, -1.0547523498535156, -0.8075066804885864, 0.017384227365255356, -0.1673143059015274, -0.06473632156848907, 0.38579297065734863, 0.08275580406188965, -0.7422923445701599, 0.538936197757721, -0.7898216843605042, -0.4417691230773926, 1.4877370595932007, 1.2614243030548096, -0.7556228041648865, -0.7847273945808411, -0.16664613783359528, 0.3858749568462372, 1.0537605285644531, 1.4695909023284912, 0.13646583259105682, -0.7174903750419617, 0.1320929378271103, 0.6029210090637207, 0.8166622519493103, 1.5668644905090332, -0.7085946798324585, 0.33708420395851135, -0.4326056241989136, -1.1200000047683716, 2.378188133239746, -0.1761072874069214, -0.8239482641220093, -1.2518949508666992, -0.22568486630916595, 0.9139595031738281, -0.10625304281711578, -0.015952954068779945, 0.2477046102285385, 0.6133231520652771, -1.0684322118759155, 0.01076308824121952, -1.057261347770691, -0.543559730052948, -0.8323883414268494, -1.1114444732666016, 0.03496994078159332, -2.61006760597229, -0.06603702902793884, -0.5893309116363525, 0.1495516300201416, 0.7704911231994629, -1.9476579427719116, 0.5569999814033508, -0.14120352268218994, 2.284083366394043, 1.8030482530593872, -0.7838191986083984, -0.31318363547325134, -1.0441811084747314, 1.2237154245376587, -0.08303854614496231, -1.9249118566513062, 0.1742505431175232, 2.112051010131836, 0.9598606824874878, 0.7485634684562683, -0.26118186116218567, 1.3362741470336914, 0.42748698592185974, 0.4010276198387146, -0.9581322073936462, -1.0497087240219116, -3.0423786640167236, -1.2020949125289917, -0.8735920190811157, -0.9459497332572937], [-0.934755802154541, 0.7883578538894653, 0.03379594534635544, 0.7462831139564514, 0.6880485415458679, -0.8923438787460327, 1.4361281394958496, -0.2768307328224182, 0.34475743770599365, -1.1237024068832397, 0.06519834697246552, -2.0644733905792236, 1.105741024017334, -1.6019634008407593, -0.23532740771770477, -1.2020431756973267, 1.323006272315979, 0.45875877141952515, -0.5074804425239563, -0.1934422254562378, 0.15231648087501526, -0.722172737121582, -1.5953530073165894, 1.1795189380645752, -0.49674686789512634, 1.6302120685577393, 0.7625582814216614, 2.0309839248657227, 1.9375513792037964, -1.3487194776535034, -0.10209953784942627, 1.1774406433105469, -0.5359631776809692, 1.1224277019500732, -1.4120701551437378, 0.605278491973877, 0.006589402444660664, 0.2542918920516968, -1.0189927816390991, 0.5357025861740112, -0.5589437484741211, 1.3642830848693848, -1.5920791625976562, -1.2117431163787842, 0.5592306852340698, 0.21677926182746887, 1.0080933570861816, -1.0233380794525146, -1.093428134918213, -0.5739267468452454, 0.986305296421051, 0.027541177347302437, -1.793463945388794, -0.02105814218521118, -0.757714569568634, -1.6939457654953003, -0.05120202153921127, -1.892642855644226, -2.6614348888397217, -0.618524968624115, 2.6297638416290283, -0.14846794307231903, -0.4235180914402008, 0.15324419736862183, -0.05463465675711632, -0.614548921585083, -0.9666985273361206, 1.7627439498901367, 0.4229128956794739, -0.23874981701374054, -0.9417451620101929, 0.5362366437911987, -0.11269709467887878, 1.4020599126815796, -0.3303682804107666, -1.8374927043914795, 0.728489339351654, -0.966923177242279, -2.631322145462036, -0.6303848624229431, 0.007151959929615259, -0.5354771614074707, 1.3570541143417358, -0.9585814476013184, 1.6464641094207764, -2.3404572010040283, 1.0327671766281128, -1.4365310668945312, 1.0483051538467407, 0.9629379510879517, -0.7200594544410706, 0.14839911460876465, -1.0598770380020142, -0.4219094514846802, 0.805824875831604, 0.41950297355651855, 0.16237857937812805, 0.4752330482006073, -2.0373833179473877, -0.25885820388793945, -0.8211561441421509, -1.0195436477661133, -0.6580609679222107, 0.770155668258667, 0.051731813699007034, 0.1843366175889969, -0.5340683460235596, -0.828725278377533, -0.4621196985244751, 0.7654950022697449, -0.9580084085464478, 0.08137936890125275, 0.43979355692863464, 1.2416085004806519, -0.00332133611664176, 0.24917636811733246, -0.5313743352890015, 1.4599127769470215, 1.2404961585998535, -0.7984284162521362, 0.6736571192741394, 0.9163482785224915, 0.7166630625724792, 0.4648233950138092, 0.15017005801200867, 1.5037579536437988, -1.832297444343567, -2.057271957397461, 0.167949378490448]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(transpose, json.loads(_TESTS), 'triton', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")